# COMP8851 — BWGNN Benchmark

## Author Reproduction and Unified Benchmark

**Model:** BWGNN  
**Paper:** Rethinking Graph Neural Networks for Anomaly Detection  
**Primary first dataset:** T-Finance

### Workflow

1. Verify the official BWGNN repository and upstream commit.
2. Inspect the author-requested software environment.
3. Build an isolated BWGNN-compatible environment.
4. Verify T-Finance and T-Social DGL graph structure.
5. Run a 1–2 epoch smoke test.
6. Perform author-mode reproduction.
7. Classify the reproduction evidence.
8. Apply Team Protocol v1 only after author-mode feasibility passes.
9. Run controlled TR40 / TR30 / TR20 / TR10 experiments.
10. Save configuration, environment, metrics and per-epoch timing.

Author-mode and unified results are never mixed.

## Step 6A — Kaggle Runtime Check

The BWGNN notebook uses the same controlled Kaggle hardware as the shared
dataset notebook.

Kaggle may provide two Tesla T4 GPUs, but benchmark code exposes GPU 0 only.

In [1]:
# ============================================================
# STEP 6A — BWGNN RUNTIME CHECK
# ============================================================

import os
import sys
import time
import subprocess

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:6.2f}s] {msg}",
        flush=True
    )

log("START")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Python:", sys.version.replace("\n", " "), flush=True)
print(
    "CUDA_VISIBLE_DEVICES:",
    os.environ["CUDA_VISIBLE_DEVICES"],
    flush=True
)

log("Checking GPU")

result = subprocess.run(
    ["nvidia-smi", "-L"],
    capture_output=True,
    text=True,
    timeout=20
)

print(result.stdout, flush=True)

log("Importing current Kaggle PyTorch")

import torch

print("Current PyTorch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
print("Visible GPUs:", torch.cuda.device_count(), flush=True)

if torch.cuda.is_available():
    print(
        "Visible GPU 0:",
        torch.cuda.get_device_name(0),
        flush=True
    )

log("DONE")

[  0.00s] START
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CUDA_VISIBLE_DEVICES: 0
[  0.00s] Checking GPU
GPU 0: Tesla T4 (UUID: GPU-bb6e2525-baef-c3a1-1723-3fb895da2976)
GPU 1: Tesla T4 (UUID: GPU-8cfe1d11-0900-912b-c822-ef8c4acfe4f7)

[  0.05s] Importing current Kaggle PyTorch
Current PyTorch: 2.10.0+cu128
CUDA available: True
Visible GPUs: 1
Visible GPU 0: Tesla T4
[  4.31s] DONE


## Step 6B — Official BWGNN Repository Identity

This step freezes the identity of the official BWGNN author repository before
any source code is modified.

It records:

- official repository URL
- exact upstream commit hash
- active upstream branch
- latest commit information
- clean Git status
- top-level repository contents
- presence of the expected BWGNN source files

The author repository is cloned into `/kaggle/working`, outside the COMP8851
team repository.

No datasets are required or accessed in this step.

No packages are installed and no author source files are modified.

In [1]:
# ============================================================
# STEP 6B — FREEZE OFFICIAL BWGNN REPOSITORY IDENTITY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import json
import time

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OFFICIAL_URL = (
    "https://github.com/squareRoot3/"
    "Rethinking-Anomaly-Detection.git"
)

REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

BWGNN_WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

EVIDENCE_DIR = (
    BWGNN_WORK /
    "evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — official BWGNN repository identity audit")

print(
    "\nNo dataset input is required for Step 6B.",
    flush=True
)


# ------------------------------------------------------------
# 1. Remove only a previous Kaggle temporary clone
# ------------------------------------------------------------

if REPO_DIR.exists():

    log(
        "Previous /kaggle/working BWGNN clone found."
    )

    log(
        "Removing temporary clone so this audit starts clean..."
    )

    shutil.rmtree(REPO_DIR)

    log("Previous temporary clone removed.")


# ------------------------------------------------------------
# 2. Clone official repository
# ------------------------------------------------------------

log("Cloning official BWGNN repository...")

clone = subprocess.run(
    [
        "git",
        "clone",
        OFFICIAL_URL,
        str(REPO_DIR)
    ],
    capture_output=True,
    text=True,
    timeout=120
)

if clone.stdout:
    print(
        "\n===== GIT CLONE STDOUT =====",
        flush=True
    )
    print(clone.stdout, flush=True)

if clone.stderr:
    print(
        "\n===== GIT CLONE STDERR =====",
        flush=True
    )
    print(clone.stderr, flush=True)

if clone.returncode != 0:

    raise RuntimeError(
        "Official BWGNN repository clone failed. "
        f"Return code: {clone.returncode}"
    )

log("Clone completed successfully.")


# ------------------------------------------------------------
# 3. Git helper
# ------------------------------------------------------------

def git(*args):

    result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            *args
        ],
        capture_output=True,
        text=True,
        timeout=30
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}\n"
            f"{result.stderr}"
        )

    return result.stdout.strip()


# ------------------------------------------------------------
# 4. Read exact upstream identity
# ------------------------------------------------------------

log("Reading exact upstream Git identity...")

remote_url = git(
    "remote",
    "get-url",
    "origin"
)

commit_hash = git(
    "rev-parse",
    "HEAD"
)

branch_name = git(
    "branch",
    "--show-current"
)

status_porcelain = git(
    "status",
    "--porcelain"
)

latest_commit = git(
    "log",
    "-1",
    "--pretty=format:%H%n%an%n%ae%n%ad%n%s",
    "--date=iso-strict"
)


print(
    "\n===== BWGNN UPSTREAM IDENTITY =====",
    flush=True
)

print(
    "Repository URL :",
    remote_url,
    flush=True
)

print(
    "Commit SHA     :",
    commit_hash,
    flush=True
)

print(
    "Branch         :",
    branch_name,
    flush=True
)

print(
    "Git status     :",
    "CLEAN"
    if not status_porcelain
    else "NOT CLEAN",
    flush=True
)


print(
    "\n===== LATEST UPSTREAM COMMIT =====",
    flush=True
)

print(
    latest_commit,
    flush=True
)


# ------------------------------------------------------------
# 5. Inspect repository files
# ------------------------------------------------------------

log("Inspecting top-level repository files...")

top_level = sorted(
    REPO_DIR.iterdir(),
    key=lambda p: p.name.lower()
)

print(
    "\n===== TOP-LEVEL REPOSITORY CONTENTS =====",
    flush=True
)

for path in top_level:

    kind = (
        "DIR "
        if path.is_dir()
        else "FILE"
    )

    print(
        f"{kind} | {path.name}",
        flush=True
    )


# ------------------------------------------------------------
# 6. Verify expected author source files
# ------------------------------------------------------------

expected_files = [
    "BWGNN.py",
    "dataset.py",
    "main.py",
    "readme.md"
]

print(
    "\n===== EXPECTED BWGNN SOURCE FILE CHECK =====",
    flush=True
)

expected_results = {}

for filename in expected_files:

    path = REPO_DIR / filename

    exists = path.is_file()

    expected_results[filename] = exists

    print(
        f"{'PASS' if exists else 'FAIL'} | {filename}",
        flush=True
    )


if not all(expected_results.values()):

    raise RuntimeError(
        "One or more expected BWGNN author files "
        "are missing. Stop before environment setup."
    )


# ------------------------------------------------------------
# 7. Save frozen upstream identity
# ------------------------------------------------------------

log("Saving upstream identity evidence...")

identity_record = {

    "model": "BWGNN",

    "official_repository": remote_url,

    "upstream_commit": commit_hash,

    "upstream_branch": branch_name,

    "git_status": (
        "CLEAN"
        if not status_porcelain
        else "NOT_CLEAN"
    ),

    "expected_source_files": expected_results,

    "cloned_path": str(REPO_DIR),

    "recorded_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    )
}


identity_path = (
    EVIDENCE_DIR /
    "bwgnn_upstream_identity.json"
)

with identity_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        identity_record,
        f,
        indent=2
    )


files_path = (
    EVIDENCE_DIR /
    "bwgnn_upstream_files.txt"
)

with files_path.open(
    "w",
    encoding="utf-8"
) as f:

    for path in top_level:

        kind = (
            "DIR"
            if path.is_dir()
            else "FILE"
        )

        f.write(
            f"{kind}\t{path.name}\n"
        )


print(
    "\n===== SAVED REPOSITORY EVIDENCE =====",
    flush=True
)

print(
    identity_path,
    flush=True
)

print(
    files_path,
    flush=True
)


# ------------------------------------------------------------
# 8. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6B GATE =====",
    flush=True
)

if (
    not status_porcelain
    and all(expected_results.values())
):

    print(
        "PASS — official BWGNN repository cloned, "
        "identity frozen, and source tree is clean.",
        flush=True
    )

else:

    print(
        "FAIL — repository identity gate not satisfied.",
        flush=True
    )


log("DONE — Step 6B complete")

[   0.00s] START — official BWGNN repository identity audit

No dataset input is required for Step 6B.
[   0.00s] Cloning official BWGNN repository...

===== GIT CLONE STDERR =====
Cloning into '/kaggle/working/Rethinking-Anomaly-Detection'...

[   0.89s] Clone completed successfully.
[   0.89s] Reading exact upstream Git identity...

===== BWGNN UPSTREAM IDENTITY =====
Repository URL : https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git
Commit SHA     : de0631f039bbd19c1890b483cc01f1007f596af7
Branch         : master
Git status     : CLEAN

===== LATEST UPSTREAM COMMIT =====
de0631f039bbd19c1890b483cc01f1007f596af7
DSAIL
dsailathkust@163.com
2024-06-25T17:30:37+08:00
Update readme.md
[   0.91s] Inspecting top-level repository files...

===== TOP-LEVEL REPOSITORY CONTENTS =====
DIR  | .git
FILE | BWGNN.py
FILE | dataset.py
FILE | main.py
FILE | readme.md

===== EXPECTED BWGNN SOURCE FILE CHECK =====
PASS | BWGNN.py
PASS | dataset.py
PASS | main.py
PASS | readme.md
[   0.92s

## Step 6C — Author Environment and Code-Path Audit

This step inspects the exact frozen BWGNN repository commit recorded in
Step 6B.

No packages are installed and no source code is modified.

The audit records:

- dependency versions stated by the author repository
- whether a Python version is explicitly specified
- Python package imports used by the source code
- supported dataset names
- dataset file/path expectations
- command-line arguments and training entry point
- author-provided example commands

The output from this step will be used to design the isolated BWGNN
environment.

No benchmark datasets are required yet.

In [2]:
# ============================================================
# STEP 6C — AUTHOR ENVIRONMENT AND CODE-PATH AUDIT
# ============================================================

from pathlib import Path
import hashlib
import json
import re
import time

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — BWGNN author environment/code audit")

print(
    "\nNo dataset input is required for Step 6C.",
    flush=True
)


# ------------------------------------------------------------
# 1. Required files
# ------------------------------------------------------------

FILES = {
    "readme": REPO_DIR / "readme.md",
    "main": REPO_DIR / "main.py",
    "dataset": REPO_DIR / "dataset.py",
    "model": REPO_DIR / "BWGNN.py",
}

log("Checking required repository files...")

for name, path in FILES.items():

    if not path.exists():
        raise FileNotFoundError(path)

    print(
        f"PASS | {name:<8} | "
        f"{path.name:<12} | "
        f"{path.stat().st_size:,} bytes",
        flush=True
    )


# ------------------------------------------------------------
# 2. Read source files
# ------------------------------------------------------------

log("Reading repository text files...")

texts = {}

for name, path in FILES.items():

    texts[name] = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

log("Repository files read successfully.")


# ------------------------------------------------------------
# 3. SHA-256 of source files
# ------------------------------------------------------------

print(
    "\n===== SOURCE FILE SHA-256 =====",
    flush=True
)

source_hashes = {}

for name, path in FILES.items():

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    source_hashes[path.name] = digest

    print(
        f"{path.name:<12} : {digest}",
        flush=True
    )


# ------------------------------------------------------------
# 4. README — environment/dependency evidence
# ------------------------------------------------------------

log("Extracting environment-related README lines...")

README_KEYWORDS = [
    "pytorch",
    "torch",
    "dgl",
    "python",
    "sympy",
    "argparse",
    "sklearn",
    "scikit",
    "cuda",
    "requirement",
    "dependency",
]

print(
    "\n===== README — ENVIRONMENT / DEPENDENCY LINES =====",
    flush=True
)

readme_env_lines = []

for lineno, line in enumerate(
    texts["readme"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in README_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        readme_env_lines.append(record)

        print(record, flush=True)


if not readme_env_lines:
    print(
        "No explicit environment/dependency lines found.",
        flush=True
    )


# ------------------------------------------------------------
# 5. Explicit Python-version check
# ------------------------------------------------------------

log("Checking whether author explicitly specifies Python version...")

python_version_pattern = re.compile(
    r"python\s*(?:==|=|>=|<=|>|<)?\s*"
    r"([23](?:\.\d+){1,2})",
    re.IGNORECASE
)

python_version_matches = (
    python_version_pattern.findall(
        texts["readme"]
    )
)

print(
    "\n===== AUTHOR PYTHON VERSION CHECK =====",
    flush=True
)

if python_version_matches:

    print(
        "Explicit Python version reference(s):",
        sorted(set(python_version_matches)),
        flush=True
    )

else:

    print(
        "NOT SPECIFIED — no explicit Python version "
        "was detected in readme.md.",
        flush=True
    )


# ------------------------------------------------------------
# 6. Imports actually used by source
# ------------------------------------------------------------

log("Extracting imports from Python source files...")

import_pattern = re.compile(
    r"^\s*(?:from\s+([A-Za-z0-9_\.]+)\s+import|"
    r"import\s+([A-Za-z0-9_\.]+))",
    re.MULTILINE
)

print(
    "\n===== SOURCE IMPORTS =====",
    flush=True
)

imports_by_file = {}

for key in ["main", "dataset", "model"]:

    imports = set()

    for match in import_pattern.finditer(
        texts[key]
    ):

        module = (
            match.group(1)
            or match.group(2)
        )

        if module:
            imports.add(module.split(".")[0])

    imports_by_file[
        FILES[key].name
    ] = sorted(imports)

    print(
        f"\n[{FILES[key].name}]",
        flush=True
    )

    for module in sorted(imports):
        print(
            f"  {module}",
            flush=True
        )


# ------------------------------------------------------------
# 7. Dataset-related README evidence
# ------------------------------------------------------------

log("Extracting dataset-related README lines...")

DATASET_KEYWORDS = [
    "yelp",
    "amazon",
    "tfinance",
    "t-finance",
    "tsocial",
    "t-social",
    "dataset",
    "google drive",
]

print(
    "\n===== README — DATASET / RUN INSTRUCTIONS =====",
    flush=True
)

readme_dataset_lines = []

for lineno, line in enumerate(
    texts["readme"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in DATASET_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        readme_dataset_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 8. Dataset loader code paths
# ------------------------------------------------------------

log("Inspecting dataset.py dataset/path logic...")

CODE_DATASET_KEYWORDS = [
    "yelp",
    "amazon",
    "tfinance",
    "tsocial",
    "load_graph",
    "fraud",
    "dataset/",
    "./dataset",
]

print(
    "\n===== dataset.py — DATASET/PATH LINES =====",
    flush=True
)

dataset_code_lines = []

for lineno, line in enumerate(
    texts["dataset"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in CODE_DATASET_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        dataset_code_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 9. main.py CLI arguments
# ------------------------------------------------------------

log("Inspecting main.py command-line arguments...")

print(
    "\n===== main.py — CLI ARGUMENTS =====",
    flush=True
)

cli_lines = []

for lineno, line in enumerate(
    texts["main"].splitlines(),
    start=1
):

    if (
        "add_argument" in line
        or "ArgumentParser" in line
    ):

        record = f"L{lineno:03d}: {line}"

        cli_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 10. Training / optimizer / split evidence
# ------------------------------------------------------------

log("Inspecting training-control code...")

TRAIN_KEYWORDS = [
    "adam",
    "optimizer",
    "train_test_split",
    "random_state",
    "train_ratio",
    "threshold",
    "epoch",
    "auc",
    "f1",
    "recall",
    "precision",
]

print(
    "\n===== main.py — TRAINING / SPLIT / METRIC LINES =====",
    flush=True
)

training_lines = []

for lineno, line in enumerate(
    texts["main"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in TRAIN_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        training_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 11. Save audit evidence
# ------------------------------------------------------------

log("Saving Step 6C evidence...")

audit = {

    "model": "BWGNN",

    "upstream_commit":
        "de0631f039bbd19c1890b483cc01f1007f596af7",

    "source_hashes":
        source_hashes,

    "explicit_python_versions_detected":
        sorted(
            set(python_version_matches)
        ),

    "python_version_status":
        (
            "EXPLICITLY_SPECIFIED"
            if python_version_matches
            else "NOT_SPECIFIED"
        ),

    "imports":
        imports_by_file,

    "readme_environment_lines":
        readme_env_lines,

    "readme_dataset_lines":
        readme_dataset_lines,

    "dataset_code_lines":
        dataset_code_lines,

    "cli_lines":
        cli_lines,

    "training_lines":
        training_lines,
}


audit_path = (
    EVIDENCE_DIR /
    "bwgnn_author_environment_audit.json"
)

with audit_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )


print(
    "\n===== SAVED STEP 6C EVIDENCE =====",
    flush=True
)

print(
    audit_path,
    flush=True
)


# ------------------------------------------------------------
# 12. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6C GATE =====",
    flush=True
)

print(
    "PASS — author environment declarations, "
    "dataset paths and entry-point controls "
    "were inspected without modifying the repository.",
    flush=True
)

log("DONE — Step 6C complete")

[   0.00s] START — BWGNN author environment/code audit

No dataset input is required for Step 6C.
[   0.00s] Checking required repository files...
PASS | readme   | readme.md    | 2,105 bytes
PASS | main     | main.py      | 5,621 bytes
PASS | dataset  | dataset.py   | 2,841 bytes
PASS | model    | BWGNN.py     | 7,109 bytes
[   0.01s] Reading repository text files...
[   0.01s] Repository files read successfully.

===== SOURCE FILE SHA-256 =====
readme.md    : 45b5299c07f3efab1c9b8c3fbc30efd64a1b2836efe890b1b34ceff97f84a886
main.py      : 1642e10e5c799ebd303bb20d66a991a55363276b35855996f5615eba0fd1ea13
dataset.py   : 229b5e5101c63879264802f909e64396a49be694a8bb4815dfea201f9b78d1fb
BWGNN.py     : 0ac45b1104f80fa17c50cd5e3bf9aa6241e008e6b57fdab9366ec69f80cb2f84
[   0.02s] Extracting environment-related README lines...

===== README — ENVIRONMENT / DEPENDENCY LINES =====
L014: - pytorch 1.9.0
L015: - dgl 0.8.1
L016: - sympy
L017: - argparse
L018: - sklearn
L031: python main.py --dataset 

## Step 6D — BWGNN Environment Feasibility Audit

The frozen author repository requires:

- PyTorch 1.9.0
- DGL 0.8.1
- sympy
- argparse
- sklearn

The author repository does not specify a Python version.

The current Kaggle base runtime uses Python 3.12, which is too new for the
official PyTorch 1.9.0 and DGL 0.8.1 wheel combination required for a close
author-compatible reproduction.

A candidate isolated environment is therefore:

- Python 3.9
- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1
- NVIDIA T4, GPU 0 only

Python 3.9 is an operational compatibility choice, not an author-stated
requirement.

This step does not install or modify any package. It checks which environment
creation tools are available in the current Kaggle runtime and records GPU,
driver, disk and Python information before environment construction.

In [3]:
# ============================================================
# STEP 6D — ENVIRONMENT FEASIBILITY AUDIT
# ============================================================

import os
import sys
import shutil
import subprocess
import platform
import json
import time
from pathlib import Path
from datetime import datetime, timezone

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — BWGNN environment feasibility audit")

print(
    "\nNo datasets are required for Step 6D.",
    flush=True
)


# ------------------------------------------------------------
# 1. Current base runtime
# ------------------------------------------------------------

log("Recording current Kaggle base runtime...")

print("\n===== CURRENT KAGGLE BASE =====", flush=True)

print(
    "Python executable :",
    sys.executable,
    flush=True
)

print(
    "Python version    :",
    sys.version.replace("\n", " "),
    flush=True
)

print(
    "Platform          :",
    platform.platform(),
    flush=True
)

print(
    "Architecture      :",
    platform.machine(),
    flush=True
)

print(
    "CUDA_VISIBLE_DEVICES:",
    os.environ.get("CUDA_VISIBLE_DEVICES"),
    flush=True
)


# ------------------------------------------------------------
# 2. Environment manager/tool availability
# ------------------------------------------------------------

log("Checking available environment-management tools...")

tools = [
    "conda",
    "mamba",
    "micromamba",
    "uv",
    "python3.9",
    "python3.10",
    "pip",
    "git"
]

tool_paths = {}

print(
    "\n===== ENVIRONMENT TOOL AVAILABILITY =====",
    flush=True
)

for tool in tools:

    path = shutil.which(tool)

    tool_paths[tool] = path

    print(
        f"{tool:<12} : "
        f"{path if path else 'NOT FOUND'}",
        flush=True
    )


# ------------------------------------------------------------
# 3. GPU + driver
# ------------------------------------------------------------

log("Reading NVIDIA driver/GPU information...")

try:

    gpu_result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu="
            "index,name,driver_version,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(
        "\n===== NVIDIA DRIVER / GPU =====",
        flush=True
    )

    if gpu_result.stdout:
        print(
            gpu_result.stdout.strip(),
            flush=True
        )

    if gpu_result.stderr:
        print(
            gpu_result.stderr.strip(),
            flush=True
        )

except subprocess.TimeoutExpired:

    raise RuntimeError(
        "nvidia-smi timed out after 20 seconds."
    )


# ------------------------------------------------------------
# 4. Current PyTorch details
# ------------------------------------------------------------

log("Reading current PyTorch details...")

import torch

print(
    "\n===== CURRENT PYTORCH =====",
    flush=True
)

print(
    "PyTorch version  :",
    torch.__version__,
    flush=True
)

print(
    "Torch CUDA build :",
    torch.version.cuda,
    flush=True
)

print(
    "CUDA available   :",
    torch.cuda.is_available(),
    flush=True
)

print(
    "Visible GPUs     :",
    torch.cuda.device_count(),
    flush=True
)

if torch.cuda.is_available():

    print(
        "Visible GPU 0    :",
        torch.cuda.get_device_name(0),
        flush=True
    )


# ------------------------------------------------------------
# 5. Disk availability
# ------------------------------------------------------------

log("Checking /kaggle/working disk availability...")

usage = shutil.disk_usage(
    "/kaggle/working"
)

print(
    "\n===== /KAGGLE/WORKING DISK =====",
    flush=True
)

print(
    f"Total : {usage.total / (1024**3):.2f} GiB",
    flush=True
)

print(
    f"Used  : {usage.used / (1024**3):.2f} GiB",
    flush=True
)

print(
    f"Free  : {usage.free / (1024**3):.2f} GiB",
    flush=True
)


# ------------------------------------------------------------
# 6. Candidate environment record
# ------------------------------------------------------------

candidate = {

    "python": "3.9",

    "python_basis":
        "Compatibility choice; author did not specify Python",

    "torch": "1.9.0+cu111",

    "torch_basis":
        "Author requested PyTorch 1.9.0; CUDA 11.1 "
        "selected as compatible official GPU build",

    "dgl": "0.8.1 CUDA 11.1",

    "dgl_basis":
        "Author requested DGL 0.8.1; CUDA 11.1 "
        "selected to match PyTorch build",

    "gpu_policy": "NVIDIA T4, GPU 0 only"
}


print(
    "\n===== CANDIDATE ISOLATED BWGNN ENVIRONMENT =====",
    flush=True
)

for key, value in candidate.items():

    print(
        f"{key:<15}: {value}",
        flush=True
    )


# ------------------------------------------------------------
# 7. Determine available construction path
# ------------------------------------------------------------

print(
    "\n===== ENVIRONMENT CONSTRUCTION OPTIONS =====",
    flush=True
)

if tool_paths["conda"]:

    print(
        "AVAILABLE: conda-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["mamba"]:

    print(
        "AVAILABLE: mamba-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["micromamba"]:

    print(
        "AVAILABLE: micromamba-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["uv"]:

    print(
        "AVAILABLE: uv-managed isolated Python environment",
        flush=True
    )

if tool_paths["python3.9"]:

    print(
        "AVAILABLE: system Python 3.9 executable",
        flush=True
    )

if not any(
    [
        tool_paths["conda"],
        tool_paths["mamba"],
        tool_paths["micromamba"],
        tool_paths["uv"],
        tool_paths["python3.9"],
    ]
):

    print(
        "NO READY ISOLATED-PYTHON TOOL FOUND.",
        flush=True
    )

    print(
        "Do not install old PyTorch into the Python 3.12 base.",
        flush=True
    )


# ------------------------------------------------------------
# 8. Save audit
# ------------------------------------------------------------

log("Saving Step 6D feasibility evidence...")

record = {

    "recorded_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "base_python":
        sys.version.replace("\n", " "),

    "base_python_executable":
        sys.executable,

    "base_torch":
        torch.__version__,

    "base_torch_cuda":
        torch.version.cuda,

    "cuda_visible_devices":
        os.environ.get("CUDA_VISIBLE_DEVICES"),

    "tool_paths":
        tool_paths,

    "disk_free_bytes":
        usage.free,

    "candidate_environment":
        candidate
}


output_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_feasibility.json"
)

with output_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        record,
        f,
        indent=2
    )


print(
    "\n===== SAVED STEP 6D EVIDENCE =====",
    flush=True
)

print(
    output_path,
    flush=True
)


# ------------------------------------------------------------
# 9. Gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6D GATE =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no packages were installed or modified.",
    flush=True
)

log("DONE — Step 6D complete")

[   0.00s] START — BWGNN environment feasibility audit

No datasets are required for Step 6D.
[   0.00s] Recording current Kaggle base runtime...

===== CURRENT KAGGLE BASE =====
Python executable : /usr/bin/python3
Python version    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Architecture      : x86_64
CUDA_VISIBLE_DEVICES: None
[   0.01s] Checking available environment-management tools...

===== ENVIRONMENT TOOL AVAILABILITY =====
conda        : NOT FOUND
mamba        : /usr/local/bin/mamba
micromamba   : NOT FOUND
uv           : /usr/local/bin/uv
python3.9    : NOT FOUND
python3.10   : /usr/bin/python3.10
pip          : /usr/local/bin/pip
git          : /usr/bin/git
[   0.02s] Reading NVIDIA driver/GPU information...

===== NVIDIA DRIVER / GPU =====
0, Tesla T4, 580.159.04, 15360 MiB
1, Tesla T4, 580.159.04, 15360 MiB
[   0.06s] Reading current PyTorch details...

===== CURRENT PYTORCH =====
PyTorch version  : 2.10.0+

## Step 6E — Build the Isolated BWGNN Environment

The author repository explicitly requests PyTorch 1.9.0 and DGL 0.8.1,
but does not specify a Python version.

Python 3.9 is therefore used as a compatibility choice, not as an
author-stated requirement.

The isolated environment will target:

- Python 3.9
- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1
- scikit-learn
- SciPy
- SymPy
- NumPy 1.x
- NVIDIA T4
- GPU 0 only

The existing Kaggle Python 3.12 / PyTorch 2.10 environment will not be
modified.

All BWGNN subprocesses explicitly receive `CUDA_VISIBLE_DEVICES=0`.

In [4]:
# ============================================================
# STEP 6E.1 — GPU POLICY + MAMBA CHECK
# ============================================================

import os
import shutil
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


log("START — preparing isolated-environment policy")

# ------------------------------------------------------------
# Explicit policy for every BWGNN subprocess
# ------------------------------------------------------------

BWGNN_SUBPROCESS_ENV = os.environ.copy()

BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
BWGNN_SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"

print("\n===== BWGNN SUBPROCESS POLICY =====", flush=True)

print(
    "CUDA_VISIBLE_DEVICES =",
    BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"],
    flush=True
)

print(
    "DGLBACKEND           =",
    BWGNN_SUBPROCESS_ENV["DGLBACKEND"],
    flush=True
)


# ------------------------------------------------------------
# Isolated environment location
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

BWGNN_ENV.parent.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Environment path     =",
    BWGNN_ENV,
    flush=True
)


# ------------------------------------------------------------
# Mamba check
# ------------------------------------------------------------

MAMBA = shutil.which("mamba")

if not MAMBA:
    raise RuntimeError(
        "mamba disappeared from the Kaggle runtime."
    )

log(f"mamba located at: {MAMBA}")

log("Reading mamba version...")

result = subprocess.run(
    [MAMBA, "--version"],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== MAMBA VERSION =====",
    flush=True
)

print(
    result.stdout.strip()
    or result.stderr.strip(),
    flush=True
)

if result.returncode != 0:
    raise RuntimeError("mamba version check failed.")


print(
    "\n===== STEP 6E.1 GATE =====",
    flush=True
)

print(
    "PASS — GPU0 policy defined and mamba is available.",
    flush=True
)

log("DONE — Step 6E.1")

[   0.00s] START — preparing isolated-environment policy

===== BWGNN SUBPROCESS POLICY =====
CUDA_VISIBLE_DEVICES = 0
DGLBACKEND           = pytorch
Environment path     = /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
[   0.01s] mamba located at: /usr/local/bin/mamba
[   0.01s] Reading mamba version...

===== MAMBA VERSION =====
0.11.3

===== STEP 6E.1 GATE =====
PASS — GPU0 policy defined and mamba is available.
[   0.42s] DONE — Step 6E.1


### Step 6E.2 — Create Python 3.9 Environment

A fresh Python 3.9 environment is created under `/kaggle/working`.

Only Python and pip are installed at this stage.

No PyTorch, DGL or model dependencies are installed yet.

Progress from mamba is streamed to the notebook while the environment is
created.

In [5]:
# ============================================================
# STEP 6E.2 — CREATE ISOLATED PYTHON 3.9 ENVIRONMENT
# ============================================================

import subprocess
import shutil
import time

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


def run_live(command, env=None, timeout=900):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    start = time.perf_counter()

    try:

        for line in process.stdout:
            print(line.rstrip(), flush=True)

            if time.perf_counter() - start > timeout:
                process.kill()
                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:
        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


log("START — creating BWGNN Python 3.9 environment")

# Remove only an incomplete BWGNN environment from a failed attempt.
if BWGNN_ENV.exists():

    log(
        "Existing BWGNN environment path found. "
        "Removing it for a clean creation."
    )

    shutil.rmtree(BWGNN_ENV)

    log("Previous environment removed.")


log("Starting mamba environment creation...")

run_live(
    [
        MAMBA,
        "create",
        "-y",
        "-p",
        BWGNN_ENV,
        "-c",
        "conda-forge",
        "python=3.9",
        "pip"
    ],
    env=BWGNN_SUBPROCESS_ENV,
    timeout=900
)


PYTHON39 = (
    BWGNN_ENV /
    "bin/python"
)

PIP39 = (
    BWGNN_ENV /
    "bin/pip"
)


if not PYTHON39.exists():
    raise RuntimeError(
        "Python executable was not created."
    )


log("Checking isolated Python...")

result = subprocess.run(
    [
        str(PYTHON39),
        "--version"
    ],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== ISOLATED PYTHON =====",
    flush=True
)

print(
    result.stdout.strip()
    or result.stderr.strip(),
    flush=True
)

print(
    "Executable:",
    PYTHON39,
    flush=True
)


print(
    "\n===== STEP 6E.2 GATE =====",
    flush=True
)

if result.returncode == 0 and "Python 3.9" in (
    result.stdout + result.stderr
):

    print(
        "PASS — isolated Python 3.9 environment created.",
        flush=True
    )

else:

    raise RuntimeError(
        "Python 3.9 environment verification failed."
    )


log("DONE — Step 6E.2")

[   0.00s] START — creating BWGNN Python 3.9 environment
[   0.00s] Starting mamba environment creation...

COMMAND: /usr/local/bin/mamba create -y -p /kaggle/working/comp8851_bwgnn/envs/bwgnn-author -c conda-forge python=3.9 pip
usage: mamba [-h] [--version] [--slow SLOW] [--enable-coverage]
             [--coverage-file COVERAGE_FILE] [--format FORMAT] [--no-color]
             [--tags TAGS]
             [specs ...]
mamba: error: unrecognized arguments: -y -p /kaggle/working/comp8851_bwgnn/envs/bwgnn-author -c conda-forge python=3.9 pip


RuntimeError: Command failed with return code 2

### Step 6E.2 — Environment Creation Correction

The initial environment-creation attempt detected an executable named
`mamba`, but runtime inspection showed that it is not the Conda-compatible
Mamba package manager.

The command failed before creating or modifying the BWGNN environment.

The environment-construction method is therefore changed to `uv`, which is
available in the Kaggle runtime and can download a requested Python version
when it is not already installed.

Python 3.9 remains an operational compatibility choice because the BWGNN
author repository does not specify a Python version.

No author dependency version is changed by this correction.

In [6]:
# ============================================================
# STEP 6E.2 — CORRECTED
# CREATE ISOLATED PYTHON 3.9 ENVIRONMENT WITH UV
# ============================================================

import os
import shutil
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# 1. Re-declare paths and execution policy
# ------------------------------------------------------------

UV = shutil.which("uv")

if not UV:
    raise RuntimeError(
        "uv is no longer available in this Kaggle runtime."
    )

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

BWGNN_ENV.parent.mkdir(
    parents=True,
    exist_ok=True
)

BWGNN_SUBPROCESS_ENV = os.environ.copy()

# Controlled benchmark policy
BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
BWGNN_SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"


# ------------------------------------------------------------
# 2. Live subprocess helper
# ------------------------------------------------------------

def run_live(command, env=None, timeout=900):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    command_start = time.perf_counter()

    try:

        for line in process.stdout:

            print(
                line.rstrip(),
                flush=True
            )

            elapsed = (
                time.perf_counter()
                - command_start
            )

            if elapsed > timeout:

                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:

        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


# ------------------------------------------------------------
# 3. Start audit
# ------------------------------------------------------------

log("START — corrected Python 3.9 environment creation")

print(
    "\n===== ENVIRONMENT CREATION METHOD =====",
    flush=True
)

print(
    "Tool        : uv",
    flush=True
)

print(
    "uv path     :",
    UV,
    flush=True
)

print(
    "Target path :",
    BWGNN_ENV,
    flush=True
)

print(
    "GPU policy  : CUDA_VISIBLE_DEVICES=0",
    flush=True
)


# ------------------------------------------------------------
# 4. Record uv version
# ------------------------------------------------------------

log("Checking uv version...")

uv_version = subprocess.run(
    [UV, "--version"],
    capture_output=True,
    text=True,
    timeout=20
)

print(
    "\n===== UV VERSION =====",
    flush=True
)

print(
    uv_version.stdout.strip()
    or uv_version.stderr.strip(),
    flush=True
)

if uv_version.returncode != 0:
    raise RuntimeError("uv version check failed.")


# ------------------------------------------------------------
# 5. Remove incomplete environment if one exists
# ------------------------------------------------------------

if BWGNN_ENV.exists():

    log(
        "Existing environment directory detected."
    )

    log(
        "Removing incomplete environment before clean creation..."
    )

    shutil.rmtree(BWGNN_ENV)

    log("Previous environment directory removed.")


# ------------------------------------------------------------
# 6. Create Python 3.9 venv
#
# uv will download Python 3.9 automatically if needed.
# --seed installs pip/setuptools/wheel into the venv.
# ------------------------------------------------------------

log(
    "Creating isolated Python 3.9 environment with uv..."
)

run_live(
    [
        UV,
        "venv",
        "--python",
        "3.9",
        "--seed",
        str(BWGNN_ENV)
    ],
    env=BWGNN_SUBPROCESS_ENV,
    timeout=900
)


# ------------------------------------------------------------
# 7. Verify executables
# ------------------------------------------------------------

PYTHON39 = (
    BWGNN_ENV /
    "bin/python"
)

PIP39 = (
    BWGNN_ENV /
    "bin/pip"
)

print(
    "\n===== CREATED EXECUTABLES =====",
    flush=True
)

print(
    "Python:",
    PYTHON39,
    "| exists:",
    PYTHON39.exists(),
    flush=True
)

print(
    "pip   :",
    PIP39,
    "| exists:",
    PIP39.exists(),
    flush=True
)

if not PYTHON39.exists():

    raise RuntimeError(
        "Python executable was not created."
    )


# ------------------------------------------------------------
# 8. Exact Python version
# ------------------------------------------------------------

log("Reading exact isolated Python version...")

python_check = subprocess.run(
    [
        str(PYTHON39),
        "--version"
    ],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

python_version_text = (
    python_check.stdout.strip()
    or python_check.stderr.strip()
)

print(
    "\n===== ISOLATED PYTHON =====",
    flush=True
)

print(
    "Version   :",
    python_version_text,
    flush=True
)

print(
    "Executable:",
    PYTHON39,
    flush=True
)


# ------------------------------------------------------------
# 9. Verify interpreter internals
# ------------------------------------------------------------

log("Checking isolated interpreter details...")

detail_check = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        (
            "import sys, platform; "
            "print('sys.version =', sys.version); "
            "print('executable =', sys.executable); "
            "print('platform =', platform.platform())"
        )
    ],
    capture_output=True,
    text=True,
    timeout=30,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== PYTHON 3.9 DETAILS =====",
    flush=True
)

print(
    detail_check.stdout,
    flush=True
)

if detail_check.returncode != 0:

    print(
        detail_check.stderr,
        flush=True
    )

    raise RuntimeError(
        "Isolated Python detail check failed."
    )


# ------------------------------------------------------------
# 10. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6E.2 GATE =====",
    flush=True
)

if (
    python_check.returncode == 0
    and python_version_text.startswith("Python 3.9")
):

    print(
        "PASS — isolated Python 3.9 environment "
        "created successfully with uv.",
        flush=True
    )

else:

    raise RuntimeError(
        "Python 3.9 environment verification failed."
    )


log("DONE — corrected Step 6E.2 complete")

[   0.00s] START — corrected Python 3.9 environment creation

===== ENVIRONMENT CREATION METHOD =====
Tool        : uv
uv path     : /usr/local/bin/uv
Target path : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
GPU policy  : CUDA_VISIBLE_DEVICES=0
[   0.01s] Checking uv version...

===== UV VERSION =====
uv 0.11.13 (x86_64-unknown-linux-gnu)
[   0.09s] Creating isolated Python 3.9 environment with uv...

COMMAND: /usr/local/bin/uv venv --python 3.9 --seed /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
Using CPython 3.9.25
Creating virtual environment with seed packages at: comp8851_bwgnn/envs/bwgnn-author
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.3
 + pip==26.0.1
 + setuptools==82.0.1
 + wheel==0.48.0
Activate with: source comp8851_bwgnn/envs/bwgnn-author/bin/activate

===== CREATED E

### Step 6E.3A — Exact Dependency Wheel Availability Audit

The BWGNN repository specifies:

- PyTorch 1.9.0
- DGL 0.8.1

It does not specify a CUDA build.

Source inspection also shows that the untouched author entry point does not
explicitly move the model, graph or feature tensor to CUDA.

Before selecting the actual installation build, this step verifies the exact
official Python 3.9 Linux wheels available for:

1. PyTorch 1.9.0 + CUDA 11.1
2. DGL 0.8.1 + CUDA 11.1
3. DGL 0.8.1 CPU

The files are downloaded only for provenance and availability verification.
Nothing is installed in this step.

Each downloaded wheel is SHA-256 hashed and retained for the subsequent
environment installation.

In [7]:
# ============================================================
# STEP 6E.3A — EXACT WHEEL AVAILABILITY AUDIT
# ============================================================

import os
import subprocess
import shutil
import hashlib
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# Paths / environment
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

WHEEL_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/wheels"
)

WHEEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"
SUBPROCESS_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


if not PYTHON39.exists():
    raise RuntimeError(
        "Python 3.9 environment is missing. "
        "Step 6E.2 must pass first."
    )


# ------------------------------------------------------------
# Live command helper
# ------------------------------------------------------------

def run_live(command, timeout=1800):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=SUBPROCESS_ENV
    )

    command_start = time.perf_counter()

    try:

        for line in process.stdout:

            print(line.rstrip(), flush=True)

            elapsed = time.perf_counter() - command_start

            if elapsed > timeout:

                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:

        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


# ------------------------------------------------------------
# SHA-256 helper with progress
# ------------------------------------------------------------

def hash_with_progress(path):

    path = Path(path)

    total = path.stat().st_size
    read_bytes = 0

    hasher = hashlib.sha256()

    chunk_size = 8 * 1024 * 1024
    report_every = 256 * 1024 * 1024
    next_report = report_every

    hash_start = time.perf_counter()

    with path.open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            hasher.update(chunk)
            read_bytes += len(chunk)

            if (
                read_bytes >= next_report
                or read_bytes == total
            ):

                pct = (
                    read_bytes / total * 100
                    if total
                    else 100
                )

                print(
                    f"  HASH "
                    f"{read_bytes / (1024**2):9.1f} / "
                    f"{total / (1024**2):9.1f} MiB | "
                    f"{pct:6.2f}%",
                    flush=True
                )

                next_report += report_every

    return (
        hasher.hexdigest(),
        time.perf_counter() - hash_start
    )


# ------------------------------------------------------------
# Download helper
# ------------------------------------------------------------

def audit_wheel(
    label,
    requirement,
    find_links
):

    print(
        "\n" + "=" * 100,
        flush=True
    )

    print(
        f"DEPENDENCY: {label}",
        flush=True
    )

    print(
        f"Requirement: {requirement}",
        flush=True
    )

    print(
        "=" * 100,
        flush=True
    )

    target = (
        WHEEL_ROOT /
        label.lower()
        .replace(" ", "_")
        .replace("+", "_")
    )

    if target.exists():
        shutil.rmtree(target)

    target.mkdir(
        parents=True,
        exist_ok=True
    )

    log(f"Checking/downloading {label}...")

    run_live(
        [
            str(PYTHON39),
            "-m",
            "pip",
            "download",
            "--no-deps",
            "--only-binary=:all:",
            "--progress-bar",
            "on",
            "--dest",
            str(target),
            requirement,
            "-f",
            find_links
        ]
    )

    wheels = sorted(
        target.glob("*.whl")
    )

    if len(wheels) != 1:

        raise RuntimeError(
            f"{label}: expected exactly one wheel, "
            f"found {len(wheels)}."
        )

    wheel = wheels[0]

    print(
        "\nDOWNLOADED WHEEL:",
        wheel.name,
        flush=True
    )

    print(
        "SIZE:",
        f"{wheel.stat().st_size:,} bytes "
        f"({wheel.stat().st_size / (1024**2):.2f} MiB)",
        flush=True
    )

    log(f"Hashing {wheel.name}...")

    sha256, hash_seconds = hash_with_progress(
        wheel
    )

    print(
        "SHA-256:",
        sha256,
        flush=True
    )

    print(
        "Hash time:",
        f"{hash_seconds:.2f} s",
        flush=True
    )

    return {
        "label": label,
        "requirement": requirement,
        "wheel": str(wheel),
        "wheel_name": wheel.name,
        "size_bytes": wheel.stat().st_size,
        "sha256": sha256
    }


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — exact wheel availability audit")


# ------------------------------------------------------------
# 1. PyTorch 1.9.0 CUDA 11.1
# ------------------------------------------------------------

torch_record = audit_wheel(
    label="PyTorch_1.9.0_cu111",
    requirement="torch==1.9.0+cu111",
    find_links=(
        "https://download.pytorch.org/"
        "whl/torch_stable.html"
    )
)


# ------------------------------------------------------------
# 2. DGL 0.8.1 CUDA 11.1
# ------------------------------------------------------------

dgl_gpu_record = audit_wheel(
    label="DGL_0.8.1_cu111",
    requirement="dgl-cu111==0.8.1",
    find_links=(
        "https://data.dgl.ai/wheels/repo.html"
    )
)


# ------------------------------------------------------------
# 3. DGL 0.8.1 CPU
# ------------------------------------------------------------

dgl_cpu_record = audit_wheel(
    label="DGL_0.8.1_CPU",
    requirement="dgl==0.8.1",
    find_links=(
        "https://data.dgl.ai/wheels/repo.html"
    )
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

wheel_records = [
    torch_record,
    dgl_gpu_record,
    dgl_cpu_record
]


print(
    "\n" + "=" * 100,
    flush=True
)

print(
    "EXACT WHEEL AVAILABILITY SUMMARY",
    flush=True
)

print(
    "=" * 100,
    flush=True
)


for record in wheel_records:

    print(
        f"\n{record['label']}",
        flush=True
    )

    print(
        f"  wheel  : {record['wheel_name']}",
        flush=True
    )

    print(
        f"  size   : {record['size_bytes']:,} bytes",
        flush=True
    )

    print(
        f"  sha256 : {record['sha256']}",
        flush=True
    )


print(
    "\n===== STEP 6E.3A GATE =====",
    flush=True
)

print(
    "PASS — exact PyTorch and both DGL 0.8.1 "
    "candidate wheels are available for Python 3.9 Linux.",
    flush=True
)

print(
    "\nNO PACKAGE HAS BEEN INSTALLED.",
    flush=True
)

log("DONE — Step 6E.3A complete")

[   0.00s] START — exact wheel availability audit

DEPENDENCY: PyTorch_1.9.0_cu111
Requirement: torch==1.9.0+cu111
[   0.01s] Checking/downloading PyTorch_1.9.0_cu111...

COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip download --no-deps --only-binary=:all: --progress-bar on --dest /kaggle/working/comp8851_bwgnn/wheels/pytorch_1.9.0_cu111 torch==1.9.0+cu111 -f https://download.pytorch.org/whl/torch_stable.html
Looking in links: https://download.pytorch.org/whl/torch_stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 14.1 MB/s  0:00:23
Saved ./comp8851_bwgnn/wheels/pytorch_1.9.0_cu111/torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl
Successfully downloaded torch

DOWNLOADED WHEEL: torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl
SIZE: 2,041,351,091 bytes (1946.78 MiB)
[  35.12s] Hashing torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl...
  HASH     256.0 /    1946.8 MiB |  13.15%
  HASH     512.0 /    1946.8 MiB |  26.30%
  HASH     768.0 /    1946.8 Mi

### Step 6E.3B — Install the Frozen BWGNN Stack

Exact author framework versions selected:

- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1

The CUDA-enabled DGL build is selected because the controlled benchmark
requires NVIDIA T4 GPU 0. The CPU DGL 0.8.1 wheel downloaded in Step 6E.3A
is retained only as provenance evidence.

The BWGNN repository does not specify versions for NumPy, SciPy,
scikit-learn, NetworkX, SymPy or other supporting packages.

Therefore, compatible Python 3.9-era versions are pinned operationally and
recorded as project compatibility choices, not author requirements.

The exact previously downloaded PyTorch and DGL wheels are installed from
local files so their SHA-256 identities remain fixed.

In [8]:
# ============================================================
# STEP 6E.3B — INSTALL FROZEN BWGNN STACK
# ============================================================

import os
import subprocess
import json
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

WHEEL_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/wheels"
)

EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TORCH_WHEEL = (
    WHEEL_ROOT /
    "pytorch_1.9.0_cu111" /
    "torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl"
)

DGL_WHEEL = (
    WHEEL_ROOT /
    "dgl_0.8.1_cu111" /
    "dgl_cu111-0.8.1-cp39-cp39-manylinux1_x86_64.whl"
)


for path in [
    PYTHON39,
    TORCH_WHEEL,
    DGL_WHEEL
]:
    if not path.exists():
        raise FileNotFoundError(path)


# ------------------------------------------------------------
# Controlled subprocess environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


# ------------------------------------------------------------
# Live subprocess helper
# ------------------------------------------------------------

def run_live(command, timeout=1200):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=ENV
    )

    started = time.perf_counter()

    try:

        for line in process.stdout:

            print(
                line.rstrip(),
                flush=True
            )

            if (
                time.perf_counter() - started
                > timeout
            ):
                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        rc = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if rc != 0:
        raise RuntimeError(
            f"Command failed with return code {rc}"
        )


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — frozen BWGNN stack installation")


# ------------------------------------------------------------
# 1. Compatibility support packages
#
# These are PROJECT compatibility pins.
# They are NOT claimed as author-specified versions.
# ------------------------------------------------------------

SUPPORT_PACKAGES = [

    "numpy==1.23.5",
    "scipy==1.9.3",

    "scikit-learn==1.1.3",

    "networkx==2.8.8",

    "sympy==1.10.1",

    "requests==2.28.1",
    "tqdm==4.64.1",
    "psutil==5.9.4",

    "typing_extensions==4.4.0",
]


print(
    "\n===== OPERATIONAL COMPATIBILITY PINS =====",
    flush=True
)

for package in SUPPORT_PACKAGES:
    print(
        package,
        flush=True
    )


log("Installing compatibility support packages...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "on",
        *SUPPORT_PACKAGES
    ],
    timeout=900
)


# ------------------------------------------------------------
# 2. Install exact PyTorch wheel
# ------------------------------------------------------------

log("Installing frozen PyTorch 1.9.0+cu111 wheel...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--no-deps",
        str(TORCH_WHEEL)
    ],
    timeout=1200
)


# ------------------------------------------------------------
# 3. Install exact DGL wheel
# ------------------------------------------------------------

log("Installing frozen DGL 0.8.1 cu111 wheel...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--no-deps",
        str(DGL_WHEEL)
    ],
    timeout=900
)


# ------------------------------------------------------------
# 4. pip dependency consistency check
# ------------------------------------------------------------

log("Running pip dependency check...")

pip_check = subprocess.run(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "check"
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=ENV
)


print(
    "\n===== PIP CHECK =====",
    flush=True
)

print(
    pip_check.stdout.strip()
    or pip_check.stderr.strip(),
    flush=True
)


if pip_check.returncode != 0:

    raise RuntimeError(
        "pip dependency consistency check failed."
    )


# ------------------------------------------------------------
# 5. Save selection record
# ------------------------------------------------------------

selection = {

    "author_requested": {
        "torch": "1.9.0",
        "dgl": "0.8.1",
        "python": "NOT SPECIFIED"
    },

    "actual_selected": {
        "python": "3.9.25",
        "torch": "1.9.0+cu111",
        "dgl": "0.8.1 cu111"
    },

    "framework_wheels": {

        "torch": {
            "filename": TORCH_WHEEL.name,
            "sha256":
                "5422d19042e217c2aa94030b16b3fe4da5be9ba8eea46e7e59d40a110955962d"
        },

        "dgl": {
            "filename": DGL_WHEEL.name,
            "sha256":
                "48152794ea2744196e0f7d70d257d3a8f565382855f25efa4edef08e9f9f0b87"
        }
    },

    "operational_compatibility_pins":
        SUPPORT_PACKAGES,

    "gpu_policy":
        "CUDA_VISIBLE_DEVICES=0; NVIDIA Tesla T4"
}


selection_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_selection.json"
)

selection_path.write_text(
    json.dumps(
        selection,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "\n===== SAVED ENVIRONMENT SELECTION =====",
    flush=True
)

print(
    selection_path,
    flush=True
)


print(
    "\n===== STEP 6E.3B GATE =====",
    flush=True
)

print(
    "PASS — exact PyTorch/DGL wheels installed "
    "and pip dependency check passed.",
    flush=True
)

log("DONE — Step 6E.3B complete")

[   0.00s] START — frozen BWGNN stack installation

===== OPERATIONAL COMPATIBILITY PINS =====
numpy==1.23.5
scipy==1.9.3
scikit-learn==1.1.3
networkx==2.8.8
sympy==1.10.1
requests==2.28.1
tqdm==4.64.1
psutil==5.9.4
typing_extensions==4.4.0
[   0.01s] Installing compatibility support packages...

COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip install --progress-bar on numpy==1.23.5 scipy==1.9.3 scikit-learn==1.1.3 networkx==2.8.8 sympy==1.10.1 requests==2.28.1 tqdm==4.64.1 psutil==5.9.4 typing_extensions==4.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 13.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 21.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 28.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 11.5 MB/s  0:00:00


### Step 6E.4 — Verify the Installed BWGNN Environment

The isolated environment is tested in a fresh Python 3.9 subprocess with
GPU visibility explicitly restricted to GPU 0.

The gate verifies:

- Python 3.9
- PyTorch 1.9.0
- PyTorch CUDA 11.1 build
- DGL 0.8.1
- supporting dependency versions
- CUDA availability
- exactly one visible GPU
- Tesla T4 identity
- CUDA tensor execution
- basic DGL graph creation

The Kaggle base Python 3.12 environment is not used for BWGNN execution.

In [9]:
# ============================================================
# STEP 6E.4 — VERIFY INSTALLED BWGNN ENVIRONMENT
# ============================================================

import os
import subprocess
import json
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"


log("START — isolated BWGNN environment verification")


VERIFY_SCRIPT = r'''
import json
import sys

result = {}

result["python"] = sys.version

import numpy
import scipy
import sklearn
import networkx
import sympy
import requests
import tqdm
import psutil

import torch

# Import torch BEFORE DGL so CUDA runtime libraries are loaded.
import dgl


result["numpy"] = numpy.__version__
result["scipy"] = scipy.__version__
result["sklearn"] = sklearn.__version__
result["networkx"] = networkx.__version__
result["sympy"] = sympy.__version__
result["requests"] = requests.__version__
result["tqdm"] = tqdm.__version__
result["psutil"] = psutil.__version__

result["torch"] = torch.__version__
result["torch_cuda_build"] = torch.version.cuda

result["dgl"] = dgl.__version__

result["cuda_available"] = torch.cuda.is_available()
result["visible_gpu_count"] = torch.cuda.device_count()


if torch.cuda.is_available():

    result["gpu0"] = torch.cuda.get_device_name(0)

    x = torch.tensor(
        [1.0, 2.0, 3.0],
        device="cuda:0"
    )

    result["cuda_tensor_sum"] = float(
        x.sum().item()
    )

else:

    result["gpu0"] = None
    result["cuda_tensor_sum"] = None


# Basic DGL CPU graph creation sanity test.
g = dgl.graph(
    ([0, 1], [1, 2])
)

result["dgl_test_nodes"] = g.num_nodes()
result["dgl_test_edges"] = g.num_edges()


print(
    json.dumps(result)
)
'''


log("Launching fresh Python 3.9 subprocess...")

verify = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        VERIFY_SCRIPT
    ],
    capture_output=True,
    text=True,
    timeout=180,
    env=ENV
)


print(
    "\n===== VERIFICATION STDERR =====",
    flush=True
)

print(
    verify.stderr.strip()
    if verify.stderr.strip()
    else "(none)",
    flush=True
)


if verify.returncode != 0:

    print(
        "\n===== VERIFICATION STDOUT =====",
        flush=True
    )

    print(
        verify.stdout,
        flush=True
    )

    raise RuntimeError(
        "Isolated BWGNN environment verification failed."
    )


raw_lines = [
    line
    for line in verify.stdout.splitlines()
    if line.strip()
]

data = json.loads(
    raw_lines[-1]
)


print(
    "\n===== BWGNN ISOLATED ENVIRONMENT =====",
    flush=True
)

for key, value in data.items():

    print(
        f"{key:<22}: {value}",
        flush=True
    )


# ------------------------------------------------------------
# Required gates
# ------------------------------------------------------------

checks = {

    "Python 3.9":
        data["python"].startswith("3.9"),

    "PyTorch 1.9.0":
        data["torch"].startswith("1.9.0"),

    "PyTorch CUDA build 11.1":
        str(data["torch_cuda_build"]).startswith("11.1"),

    "DGL 0.8.1":
        data["dgl"].startswith("0.8.1"),

    "CUDA available":
        data["cuda_available"] is True,

    "Exactly one visible GPU":
        data["visible_gpu_count"] == 1,

    "GPU 0 is Tesla T4":
        "T4" in (data["gpu0"] or "").upper(),

    "CUDA tensor executes":
        data["cuda_tensor_sum"] == 6.0,

    "DGL graph creation":
        (
            data["dgl_test_nodes"] == 3
            and
            data["dgl_test_edges"] == 2
        )
}


print(
    "\n===== ENVIRONMENT GATE CHECKS =====",
    flush=True
)

for name, passed in checks.items():

    print(
        f"{'PASS' if passed else 'FAIL'} | {name}",
        flush=True
    )


if not all(checks.values()):

    raise RuntimeError(
        "BWGNN environment gate failed."
    )


# ------------------------------------------------------------
# Freeze complete environment
# ------------------------------------------------------------

log("Capturing complete package freeze...")

freeze = subprocess.run(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "freeze"
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=ENV,
    check=True
)


freeze_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_freeze.txt"
)

freeze_path.write_text(
    freeze.stdout,
    encoding="utf-8"
)


print(
    "\n===== SAVED ENVIRONMENT FREEZE =====",
    flush=True
)

print(
    freeze_path,
    flush=True
)


print(
    "\n===== STEP 6E GATE =====",
    flush=True
)

print(
    "PASS — isolated BWGNN environment is operational "
    "with PyTorch 1.9.0, DGL 0.8.1 and Tesla T4 GPU 0.",
    flush=True
)

log("DONE — Step 6E complete")

[   0.00s] START — isolated BWGNN environment verification
[   0.00s] Launching fresh Python 3.9 subprocess...

===== VERIFICATION STDERR =====
Traceback (most recent call last):
  File "<string>", line 21, in <module>
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/__init__.py", line 16, in <module>
    from .backend import load_backend, backend_name
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/backend/__init__.py", line 109, in <module>
    load_backend(get_preferred_backend())
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/backend/__init__.py", line 43, in load_backend
    from .._ffi.base import load_tensor_adapter # imports DGL C library
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/_ffi/base.py", line 45, in <module>
    _LIB, _LIB_NAME, _DIR_NAME = _load_lib()
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-au

RuntimeError: Isolated BWGNN environment verification failed.

### Step 6E.4A — CUDA Runtime Library Linkage Audit

The first environment verification successfully reached the installed DGL
package but DGL's native library could not locate `libcublas.so.11`.

This indicates a CUDA runtime library search-path issue rather than an
immediate package-version mismatch.

Before changing any package, this step checks:

- whether `libcublas.so.11` already exists inside the isolated environment
- whether it is bundled with the PyTorch 1.9.0 CUDA wheel
- the location of DGL's native shared library
- the unresolved native dependencies reported by `ldd`
- the current library search path

No package is installed, removed or modified in this step.

In [10]:
# ============================================================
# STEP 6E.4A — CUDA LIBRARY LINKAGE AUDIT
# ============================================================

import os
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE_PACKAGES = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

TORCH_DIR = SITE_PACKAGES / "torch"
DGL_DIR = SITE_PACKAGES / "dgl"


log("START — CUDA runtime library linkage audit")


# ------------------------------------------------------------
# 1. Current relevant environment variables
# ------------------------------------------------------------

print(
    "\n===== CURRENT LIBRARY ENVIRONMENT =====",
    flush=True
)

print(
    "LD_LIBRARY_PATH       :",
    os.environ.get("LD_LIBRARY_PATH"),
    flush=True
)

print(
    "CUDA_VISIBLE_DEVICES  :",
    os.environ.get("CUDA_VISIBLE_DEVICES"),
    flush=True
)

print(
    "Torch directory       :",
    TORCH_DIR,
    flush=True
)

print(
    "DGL directory         :",
    DGL_DIR,
    flush=True
)


# ------------------------------------------------------------
# 2. Search isolated environment for CUDA libraries
# ------------------------------------------------------------

log("Searching isolated environment for libcublas...")

cublas_files = sorted(
    BWGNN_ENV.rglob("libcublas.so*")
)

print(
    "\n===== LIBCUBLAS FILES IN ISOLATED ENVIRONMENT =====",
    flush=True
)

if cublas_files:

    for path in cublas_files:
        print(path, flush=True)

else:

    print(
        "NONE FOUND",
        flush=True
    )


# ------------------------------------------------------------
# 3. Search specifically inside torch/lib
# ------------------------------------------------------------

TORCH_LIB = TORCH_DIR / "lib"

print(
    "\n===== TORCH LIB DIRECTORY =====",
    flush=True
)

print(
    "Path   :",
    TORCH_LIB,
    flush=True
)

print(
    "Exists :",
    TORCH_LIB.exists(),
    flush=True
)


if TORCH_LIB.exists():

    torch_cuda_libs = sorted(
        [
            p
            for p in TORCH_LIB.iterdir()
            if (
                "cublas" in p.name.lower()
                or "cudart" in p.name.lower()
                or "cudnn" in p.name.lower()
                or "cusparse" in p.name.lower()
                or "curand" in p.name.lower()
            )
        ],
        key=lambda p: p.name
    )

    print(
        "\nRelevant CUDA libraries in torch/lib:",
        flush=True
    )

    if torch_cuda_libs:

        for path in torch_cuda_libs:
            print(
                f"  {path.name}",
                flush=True
            )

    else:

        print(
            "  NONE FOUND",
            flush=True
        )


# ------------------------------------------------------------
# 4. Locate DGL native libraries
# ------------------------------------------------------------

log("Locating DGL native shared libraries...")

dgl_shared = sorted(
    DGL_DIR.rglob("*.so")
)

print(
    "\n===== DGL SHARED LIBRARIES =====",
    flush=True
)

for path in dgl_shared:

    print(
        path,
        flush=True
    )


# Prefer libdgl.so for dependency inspection
libdgl_candidates = [
    p
    for p in dgl_shared
    if p.name == "libdgl.so"
]

if not libdgl_candidates:

    raise RuntimeError(
        "Could not locate DGL libdgl.so."
    )

LIBDGL = libdgl_candidates[0]


# ------------------------------------------------------------
# 5. Run ldd on DGL native library
# ------------------------------------------------------------

log("Running ldd on DGL native library...")

ldd = subprocess.run(
    [
        "ldd",
        str(LIBDGL)
    ],
    capture_output=True,
    text=True,
    timeout=60
)


print(
    "\n===== LDD — DGL NATIVE LIBRARY =====",
    flush=True
)

print(
    "Library:",
    LIBDGL,
    flush=True
)

print(
    ldd.stdout,
    flush=True
)

if ldd.stderr:
    print(
        ldd.stderr,
        flush=True
    )


# ------------------------------------------------------------
# 6. Extract missing libraries
# ------------------------------------------------------------

missing_lines = [
    line.strip()
    for line in ldd.stdout.splitlines()
    if "not found" in line.lower()
]


print(
    "\n===== UNRESOLVED DGL DEPENDENCIES =====",
    flush=True
)

if missing_lines:

    for line in missing_lines:
        print(
            line,
            flush=True
        )

else:

    print(
        "NONE",
        flush=True
    )


# ------------------------------------------------------------
# 7. Check PyTorch itself in a clean subprocess
# ------------------------------------------------------------

log("Checking PyTorch CUDA metadata without importing DGL...")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

torch_check = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        (
            "import torch; "
            "print('torch=', torch.__version__); "
            "print('torch_cuda=', torch.version.cuda); "
            "print('cuda_available=', torch.cuda.is_available()); "
            "print('gpu_count=', torch.cuda.device_count()); "
            "print('gpu0=', torch.cuda.get_device_name(0) "
            "if torch.cuda.is_available() else None)"
        )
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=env
)


print(
    "\n===== PYTORCH-ONLY CUDA CHECK =====",
    flush=True
)

print(
    torch_check.stdout,
    flush=True
)

if torch_check.stderr:

    print(
        "STDERR:",
        torch_check.stderr,
        flush=True
    )


print(
    "\n===== STEP 6E.4A STATUS =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no packages or libraries were changed.",
    flush=True
)

log("DONE — Step 6E.4A complete")

[   0.00s] START — CUDA runtime library linkage audit

===== CURRENT LIBRARY ENVIRONMENT =====
LD_LIBRARY_PATH       : /usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64
CUDA_VISIBLE_DEVICES  : None
Torch directory       : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch
DGL directory         : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl
[   0.01s] Searching isolated environment for libcublas...

===== LIBCUBLAS FILES IN ISOLATED ENVIRONMENT =====
NONE FOUND

===== TORCH LIB DIRECTORY =====
Path   : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch/lib
Exists : True

Relevant CUDA libraries in torch/lib:
  libcudart-6d56b25a.so.11.0
[   0.11s] Locating DGL native shared libraries...

===== DGL SHARED LIBRARIES =====
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/_ffi/_cy3/core.cpython-39-x86_64-linux-gnu.so
/kaggle/working/comp8851_b

### Step 6E.4B — NVIDIA CUDA 11 Runtime Availability Audit

The DGL 0.8.1 CUDA 11.1 wheel imports successfully only if its native
dependencies can be resolved.

The previous linkage audit identified three unresolved libraries:

- `libcudart.so.11.0`
- `libcublas.so.11`
- `libcusparse.so.11`

PyTorch 1.9.0+cu111 itself is operational on the Tesla T4, so the framework
installation is retained unchanged.

This step queries NVIDIA's package index for available CUDA 11 runtime,
cuBLAS and cuSPARSE packages before any additional runtime library is
installed.

No package is modified in this step.

In [11]:
# ============================================================
# STEP 6E.4B — NVIDIA CUDA-11 RUNTIME AVAILABILITY AUDIT
# ============================================================

import os
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

if not PYTHON39.exists():
    raise RuntimeError(
        "Isolated Python 3.9 environment is missing."
    )


ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


NVIDIA_INDEX = "https://pypi.nvidia.com"


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def query_versions(package):

    print(
        "\n" + "=" * 90,
        flush=True
    )

    print(
        f"PACKAGE: {package}",
        flush=True
    )

    print(
        "=" * 90,
        flush=True
    )

    command = [
        str(PYTHON39),
        "-m",
        "pip",
        "index",
        "versions",
        package,
        "--index-url",
        NVIDIA_INDEX
    ]

    print(
        "COMMAND:",
        " ".join(command),
        flush=True
    )

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=120,
        env=ENV
    )

    print(
        "\nSTDOUT:",
        flush=True
    )

    print(
        result.stdout.strip()
        if result.stdout.strip()
        else "(none)",
        flush=True
    )

    print(
        "\nSTDERR:",
        flush=True
    )

    print(
        result.stderr.strip()
        if result.stderr.strip()
        else "(none)",
        flush=True
    )

    return {
        "package": package,
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr
    }


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — NVIDIA CUDA-11 runtime availability audit")


packages = [
    "nvidia-cuda-runtime",
    "nvidia-cublas",
    "nvidia-cusparse",
    "nvidia-cuda-runtime-cu11",
    "nvidia-cublas-cu11",
    "nvidia-cusparse-cu11",
]


records = []

for package in packages:

    log(
        f"Querying {package}..."
    )

    records.append(
        query_versions(package)
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "\n" + "=" * 90,
    flush=True
)

print(
    "QUERY SUMMARY",
    flush=True
)

print(
    "=" * 90,
    flush=True
)


for record in records:

    status = (
        "QUERY OK"
        if record["returncode"] == 0
        else "NO RESULT / QUERY FAILED"
    )

    print(
        f"{record['package']:<28} : {status}",
        flush=True
    )


print(
    "\n===== STEP 6E.4B STATUS =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no CUDA runtime package was installed.",
    flush=True
)

log("DONE — Step 6E.4B complete")

[   0.00s] START — NVIDIA CUDA-11 runtime availability audit
[   0.00s] Querying nvidia-cuda-runtime...

PACKAGE: nvidia-cuda-runtime
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip index versions nvidia-cuda-runtime --index-url https://pypi.nvidia.com

STDOUT:
nvidia-cuda-runtime (13.3.29)
Available versions: 13.3.29, 13.2.86, 13.2.75, 13.2.51, 13.1.80, 13.0.96, 13.0.88, 13.0.48, 11.3.58, 11.2.146, 11.2.72, 11.1.74

STDERR:
(none)
[   0.66s] Querying nvidia-cublas...

PACKAGE: nvidia-cublas
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip index versions nvidia-cublas --index-url https://pypi.nvidia.com

STDOUT:
nvidia-cublas (13.6.1.10)
Available versions: 13.6.1.10, 13.6.0.2, 13.5.1.27, 13.4.1.3, 13.4.1.1, 13.4.0.1, 13.3.0.5, 13.2.2.2, 13.2.1.1, 13.2.0.9, 13.1.1.3, 13.1.0.3, 13.0.2.14, 13.0.0.19

STDERR:
(none)
[   1.34s] Querying nvidia-cusparse...

PACKAGE: nvidia-cusparse
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-auth

### Step 6E.4C — Fix DGL CUDA Runtime Linkage

DGL 0.8.1 requires three CUDA 11 shared libraries that are missing from the
current library search path:

- libcudart.so.11.0
- libcublas.so.11
- libcusparse.so.11

CUDA 11 runtime packages are added without changing PyTorch 1.9.0 or
DGL 0.8.1. The environment is then retested.

In [12]:
# ============================================================
# STEP 6E.4C — INSTALL MISSING CUDA 11 LIBRARIES + RETEST DGL
# ============================================================

import os
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

print("START — installing missing CUDA 11 runtime libraries", flush=True)

# ------------------------------------------------------------
# 1. Install only the required CUDA 11 runtime libraries
# ------------------------------------------------------------

packages = [
    "nvidia-cuda-runtime-cu11==11.8.89",
    "nvidia-cublas-cu11==11.11.3.6",
    "nvidia-cusparse-cu11==11.7.5.86",
]

cmd = [
    str(PYTHON39),
    "-m",
    "pip",
    "install",
    "--no-deps",
    *packages
]

print("\nInstalling:", flush=True)

for p in packages:
    print(" ", p, flush=True)

result = subprocess.run(
    cmd,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        "CUDA 11 runtime library installation failed."
    )


# ------------------------------------------------------------
# 2. Find installed NVIDIA library directories
# ------------------------------------------------------------

lib_dirs = []

for pattern in [
    "nvidia/cuda_runtime/lib",
    "nvidia/cublas/lib",
    "nvidia/cusparse/lib",
]:

    path = SITE / pattern

    if path.exists():
        lib_dirs.append(str(path))


print("\n===== CUDA LIBRARY DIRECTORIES =====", flush=True)

for path in lib_dirs:
    print(path, flush=True)


if len(lib_dirs) != 3:
    raise RuntimeError(
        "Could not locate all three NVIDIA CUDA library directories."
    )


# ------------------------------------------------------------
# 3. Build clean BWGNN subprocess environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

existing_ld = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(lib_dirs)
    + (":" + existing_ld if existing_ld else "")
)


# ------------------------------------------------------------
# 4. Re-run ldd
# ------------------------------------------------------------

LIBDGL = SITE / "dgl/libdgl.so"

ldd = subprocess.run(
    ["ldd", str(LIBDGL)],
    capture_output=True,
    text=True,
    env=ENV
)

missing = [
    line.strip()
    for line in ldd.stdout.splitlines()
    if "not found" in line.lower()
]


print("\n===== UNRESOLVED DGL DEPENDENCIES =====", flush=True)

if missing:
    for line in missing:
        print(line, flush=True)
else:
    print("NONE", flush=True)


if missing:
    raise RuntimeError(
        "DGL still has unresolved shared-library dependencies."
    )


# ------------------------------------------------------------
# 5. Fresh Torch + DGL verification
# ------------------------------------------------------------

verify_code = r'''
import torch
import dgl

print("torch =", torch.__version__)
print("torch CUDA =", torch.version.cuda)
print("dgl =", dgl.__version__)
print("CUDA available =", torch.cuda.is_available())
print("visible GPUs =", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0 =", torch.cuda.get_device_name(0))

x = torch.tensor([1., 2., 3.], device="cuda:0")
print("CUDA tensor sum =", x.sum().item())

g = dgl.graph(([0, 1], [1, 2]))
print("DGL nodes =", g.num_nodes())
print("DGL edges =", g.num_edges())
'''

verify = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        verify_code
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=180
)


print("\n===== TORCH + DGL VERIFICATION =====", flush=True)

print(verify.stdout, flush=True)

if verify.stderr:
    print("STDERR:", flush=True)
    print(verify.stderr, flush=True)


if verify.returncode != 0:
    raise RuntimeError(
        "Torch/DGL verification failed."
    )


print("===== STEP 6E.4C GATE =====", flush=True)

print(
    "PASS — DGL 0.8.1 loads successfully with CUDA 11 runtime libraries.",
    flush=True
)

START — installing missing CUDA 11 runtime libraries

Installing:
  nvidia-cuda-runtime-cu11==11.8.89
  nvidia-cublas-cu11==11.11.3.6
  nvidia-cusparse-cu11==11.7.5.86
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 61.0 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 76.6 MB/s  0:00:02


===== CUDA LIBRARY DIRECTORIES =====
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cuda_runtime/lib
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cublas/lib
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cusparse/lib

===== UNRESOLVED DGL DEPENDENCIES =====
NONE

===== TORCH + DGL VERIFICATION =====
torch = 1.9.0+cu111
torch CUDA = 11.1
dgl = 0.8.1
CUDA available = True
visible GPUs = 1
GPU 0 = Tesla T4
CUDA tensor sum = 6.0
DGL nodes = 3
DGL edges = 2

===== STEP 6E.4C GATE ===

## Step 7 — Canonical Dataset Inputs

The benchmark datasets are attached from the same Kaggle dataset sources
used by the dataset registry notebook.

This step verifies that the expected canonical input files are available
before BWGNN loads any graph.

In [13]:
# ============================================================
# STEP 7A — VERIFY CANONICAL DATASET INPUT PATHS
# ============================================================

from pathlib import Path

DATASETS = {
    "YelpChi":
        Path("/kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat"),

    "Amazon":
        Path("/kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat"),

    "T-Finance":
        Path("/kaggle/input/datasets/pathikahmed0007/tfinance/tfinance"),

    "T-Social":
        Path("/kaggle/input/datasets/pathikahmed0007/tsocial/tsocial"),

    "Elliptic Features":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv"),

    "Elliptic Classes":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv"),

    "Elliptic Edges":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv"),

    "FDCompCN":
        Path("/kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl"),
}


print("===== CANONICAL DATASET INPUT CHECK =====")

all_ok = True

for name, path in DATASETS.items():

    exists = path.exists()

    print(
        f"{'PASS' if exists else 'FAIL'} | "
        f"{name:<18} | {path}"
    )

    if not exists:
        all_ok = False


print("\n===== STEP 7A GATE =====")

if all_ok:
    print("PASS — all canonical dataset inputs are attached.")
else:
    raise RuntimeError(
        "One or more canonical dataset inputs are missing."
    )

===== CANONICAL DATASET INPUT CHECK =====
PASS | YelpChi            | /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
PASS | Amazon             | /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat
PASS | T-Finance          | /kaggle/input/datasets/pathikahmed0007/tfinance/tfinance
PASS | T-Social           | /kaggle/input/datasets/pathikahmed0007/tsocial/tsocial
PASS | Elliptic Features  | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv
PASS | Elliptic Classes   | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv
PASS | Elliptic Edges     | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv
PASS | FDCompCN           | /kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl

===== STEP 7A GATE =====
PASS — all canonical dataset inputs are attached.


### Step 7B — T-Finance and T-Social Structural Verification

T-Finance and T-Social are loaded with the verified BWGNN-compatible
DGL 0.8.1 environment.

For each dataset this step records:

- number of nodes and edges
- node-data fields
- feature shape
- label shape
- class counts

T-Finance is additionally compared against the previously established
structural target because its current source-file checksum differs from the
earlier audited mirror.

In [14]:
# ============================================================
# STEP 7B — T-FINANCE / T-SOCIAL DGL STRUCTURAL VERIFICATION
# ============================================================

import os
import json
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

DATASETS = {
    "T-Finance":
        "/kaggle/input/datasets/pathikahmed0007/tfinance/tfinance",

    "T-Social":
        "/kaggle/input/datasets/pathikahmed0007/tsocial/tsocial",
}


# ------------------------------------------------------------
# DGL runtime environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)


# ------------------------------------------------------------
# Separate subprocess = memory released after each dataset
# ------------------------------------------------------------

CHECK_SCRIPT = r'''
import sys
import json
import torch
import dgl
from dgl.data.utils import load_graphs

path = sys.argv[1]
name = sys.argv[2]

print(f"START | loading {name}", flush=True)

graphs, label_dict = load_graphs(path)

if len(graphs) == 0:
    raise RuntimeError("No graph found in DGL file.")

g = graphs[0]

ndata_keys = list(g.ndata.keys())
edata_keys = list(g.edata.keys())

# Feature field
feature = None
feature_key = None

for candidate in ["feature", "feat", "features"]:
    if candidate in g.ndata:
        feature_key = candidate
        feature = g.ndata[candidate]
        break

# Label field
label = None
label_source = None

if "label" in label_dict:
    label = label_dict["label"]
    label_source = "label_dict['label']"

elif "label" in g.ndata:
    label = g.ndata["label"]
    label_source = "graph.ndata['label']"


summary = {
    "dataset": name,
    "dgl_version": dgl.__version__,
    "torch_version": torch.__version__,
    "num_graphs": len(graphs),
    "num_nodes": int(g.num_nodes()),
    "num_edges": int(g.num_edges()),
    "ndata_keys": ndata_keys,
    "edata_keys": edata_keys,
    "feature_key": feature_key,
    "feature_shape":
        list(feature.shape) if feature is not None else None,
    "label_source": label_source,
    "label_shape":
        list(label.shape) if label is not None else None,
}


if label is not None:

    flat = label.reshape(-1).long()

    values, counts = torch.unique(
        flat,
        return_counts=True
    )

    summary["label_counts"] = {
        str(int(v)): int(c)
        for v, c in zip(values, counts)
    }

else:

    summary["label_counts"] = None


print("RESULT_JSON=" + json.dumps(summary), flush=True)
'''


summaries = {}

for name, path in DATASETS.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    result = subprocess.run(
        [
            str(PYTHON39),
            "-c",
            CHECK_SCRIPT,
            path,
            name
        ],
        capture_output=True,
        text=True,
        env=ENV,
        timeout=900
    )

    print(result.stdout, flush=True)

    if result.stderr:
        print("STDERR:", flush=True)
        print(result.stderr, flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed to load."
        )

    json_line = [
        line
        for line in result.stdout.splitlines()
        if line.startswith("RESULT_JSON=")
    ][0]

    summaries[name] = json.loads(
        json_line.replace(
            "RESULT_JSON=",
            "",
            1
        )
    )


# ------------------------------------------------------------
# Human-readable summary
# ------------------------------------------------------------

print("\n===== STRUCTURAL SUMMARY =====")

for name, s in summaries.items():

    print(f"\n{name}")
    print("  DGL         :", s["dgl_version"])
    print("  Nodes       :", f"{s['num_nodes']:,}")
    print("  Edges       :", f"{s['num_edges']:,}")
    print("  Node fields :", s["ndata_keys"])
    print("  Feature key :", s["feature_key"])
    print("  Feature     :", s["feature_shape"])
    print("  Label source:", s["label_source"])
    print("  Label shape :", s["label_shape"])
    print("  Label counts:", s["label_counts"])


# ------------------------------------------------------------
# T-Finance known structural target
# ------------------------------------------------------------

tf = summaries["T-Finance"]

tf_checks = {
    "39,357 nodes":
        tf["num_nodes"] == 39357,

    "42,445,086 stored edges":
        tf["num_edges"] == 42445086,

    "10 feature dimensions":
        (
            tf["feature_shape"] is not None
            and len(tf["feature_shape"]) == 2
            and tf["feature_shape"][1] == 10
        ),

    "37,554 normal labels":
        tf["label_counts"] is not None
        and tf["label_counts"].get("0") == 37554,

    "1,803 fraud labels":
        tf["label_counts"] is not None
        and tf["label_counts"].get("1") == 1803,
}


print("\n===== T-FINANCE STRUCTURAL TARGET =====")

for check, passed in tf_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} | {check}"
    )


print("\n===== STEP 7B GATE =====")

if (
    all(tf_checks.values())
    and summaries["T-Social"]["feature_shape"] is not None
    and summaries["T-Social"]["label_counts"] is not None
):

    print(
        "PASS — T-Finance matches the established structural target "
        "and T-Social loads successfully with features and labels."
    )

else:

    raise RuntimeError(
        "Dataset structural verification requires review."
    )


T-Finance
START | loading T-Finance
RESULT_JSON={"dataset": "T-Finance", "dgl_version": "0.8.1", "torch_version": "1.9.0+cu111", "num_graphs": 1, "num_nodes": 39357, "num_edges": 42445086, "ndata_keys": ["label", "feature"], "edata_keys": [], "feature_key": "feature", "feature_shape": [39357, 10], "label_source": "graph.ndata['label']", "label_shape": [39357, 2], "label_counts": {"0": 39357, "1": 39357}}


T-Social
START | loading T-Social
RESULT_JSON={"dataset": "T-Social", "dgl_version": "0.8.1", "torch_version": "1.9.0+cu111", "num_graphs": 1, "num_nodes": 5781065, "num_edges": 146211016, "ndata_keys": ["feature", "label", "_ID"], "edata_keys": ["_ID"], "feature_key": "feature", "feature_shape": [5781065, 10], "label_source": "graph.ndata['label']", "label_shape": [5781065], "label_counts": {"0": 5606785, "1": 174280}}


===== STRUCTURAL SUMMARY =====

T-Finance
  DGL         : 0.8.1
  Nodes       : 39,357
  Edges       : 42,445,086
  Node fields : ['label', 'feature']
  Feature ke

RuntimeError: Dataset structural verification requires review.

### Step 7B Correction — T-Finance Label Encoding

T-Finance stores labels as a two-column one-hot tensor rather than a
one-dimensional class vector.

The previous verification flattened this tensor, which incorrectly counted
both entries of every row.

This correction converts the one-hot labels to class IDs using `argmax`
before calculating the class distribution.

In [15]:
# ============================================================
# STEP 7B CORRECTION — T-FINANCE LABEL COUNTS
# ============================================================

import subprocess
import json

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

CHECK = r'''
import sys
import json
import torch
from dgl.data.utils import load_graphs

graphs, _ = load_graphs(sys.argv[1])
g = graphs[0]

labels = g.ndata["label"]

print("Original label shape:", list(labels.shape))

# T-Finance uses one-hot labels [N, 2]
if labels.ndim == 2 and labels.shape[1] > 1:
    class_ids = torch.argmax(labels, dim=1)
    encoding = "one-hot -> argmax"
else:
    class_ids = labels.reshape(-1).long()
    encoding = "1D class labels"

values, counts = torch.unique(
    class_ids,
    return_counts=True
)

label_counts = {
    str(int(v)): int(c)
    for v, c in zip(values, counts)
}

result = {
    "nodes": int(g.num_nodes()),
    "edges": int(g.num_edges()),
    "features": list(g.ndata["feature"].shape),
    "label_shape": list(labels.shape),
    "label_encoding": encoding,
    "label_counts": label_counts
}

print("RESULT_JSON=" + json.dumps(result))
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        CHECK,
        TF_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "T-Finance corrected verification failed."
    )

line = [
    x for x in result.stdout.splitlines()
    if x.startswith("RESULT_JSON=")
][0]

tf = json.loads(
    line.replace("RESULT_JSON=", "", 1)
)

print("===== CORRECTED T-FINANCE SUMMARY =====")
print("Nodes        :", f"{tf['nodes']:,}")
print("Edges        :", f"{tf['edges']:,}")
print("Features     :", tf["features"])
print("Label shape  :", tf["label_shape"])
print("Encoding     :", tf["label_encoding"])
print("Label counts :", tf["label_counts"])


checks = {
    "39,357 nodes":
        tf["nodes"] == 39357,

    "42,445,086 stored edges":
        tf["edges"] == 42445086,

    "10 feature dimensions":
        tf["features"] == [39357, 10],

    "37,554 normal":
        tf["label_counts"].get("0") == 37554,

    "1,803 fraud":
        tf["label_counts"].get("1") == 1803,
}


print("\n===== STEP 7B CORRECTED GATE =====")

for name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
    )

if not all(checks.values()):
    raise RuntimeError(
        "T-Finance structure still does not match target."
    )

print(
    "\nPASS — T-Finance matches the established "
    "structural target."
)

print(
    "PASS — T-Social previously loaded successfully "
    "with valid 1D labels and features."
)

Original label shape: [39357, 2]
RESULT_JSON={"nodes": 39357, "edges": 42445086, "features": [39357, 10], "label_shape": [39357, 2], "label_encoding": "one-hot -> argmax", "label_counts": {"0": 37553, "1": 1804}}

===== CORRECTED T-FINANCE SUMMARY =====
Nodes        : 39,357
Edges        : 42,445,086
Features     : [39357, 10]
Label shape  : [39357, 2]
Encoding     : one-hot -> argmax
Label counts : {'0': 37553, '1': 1804}

===== STEP 7B CORRECTED GATE =====
PASS | 39,357 nodes
PASS | 42,445,086 stored edges
PASS | 10 feature dimensions
FAIL | 37,554 normal
FAIL | 1,803 fraud


RuntimeError: T-Finance structure still does not match target.

### Step 7B.1 — T-Finance Label Discrepancy Audit

The current T-Finance file matches the expected graph size and feature
dimension but contains 1,804 fraud labels instead of the established
benchmark count of 1,803.

This step checks the raw two-column label patterns to confirm that the
difference is genuine and not caused by malformed one-hot labels.

In [16]:
# ============================================================
# STEP 7B.1 — T-FINANCE RAW LABEL PATTERN AUDIT
# ============================================================

import subprocess
import json

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

SCRIPT = r'''
import sys
import json
import torch
from dgl.data.utils import load_graphs

graphs, _ = load_graphs(sys.argv[1])
labels = graphs[0].ndata["label"]

unique_rows, counts = torch.unique(
    labels,
    dim=0,
    return_counts=True
)

patterns = {
    str(row.tolist()): int(count)
    for row, count in zip(unique_rows, counts)
}

row_sums = labels.sum(dim=1)

result = {
    "shape": list(labels.shape),
    "patterns": patterns,
    "all_binary":
        bool(torch.all((labels == 0) | (labels == 1))),
    "all_row_sums_one":
        bool(torch.all(row_sums == 1)),
    "class0_argmax":
        int((labels.argmax(1) == 0).sum()),
    "class1_argmax":
        int((labels.argmax(1) == 1).sum()),
}

print(json.dumps(result))
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        TF_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=300
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("T-Finance label audit failed.")

data = json.loads(result.stdout.strip())

print("===== T-FINANCE LABEL AUDIT =====")
print("Shape              :", data["shape"])
print("Unique row patterns:", data["patterns"])
print("All values binary  :", data["all_binary"])
print("Every row sum = 1  :", data["all_row_sums_one"])
print("Normal after argmax:", data["class0_argmax"])
print("Fraud after argmax :", data["class1_argmax"])

===== T-FINANCE LABEL AUDIT =====
Shape              : [39357, 2]
Unique row patterns: {'[0, 1]': 1804, '[1, 0]': 37553}
All values binary  : True
Every row sum = 1  : True
Normal after argmax: 37553
Fraud after argmax : 1804


> **T-Finance dataset note:** The attached T-Finance graph has the expected
> 39,357 nodes, 42,445,086 stored edges and 10 features, but its valid one-hot
> labels produce 37,553 normal and 1,804 fraud nodes. The established reference
> count is 37,554 normal and 1,803 fraud. Therefore this T-Finance copy is
> flagged for source verification against the official BWGNN author dataset
> before final benchmark runs. No labels are modified.

### Unified Split Plan

For the final controlled benchmark, every dataset will be evaluated at multiple
training-set sizes:

- TR40: 40% train / 20% validation / 40% test
- TR30: 30% train / 20% validation / 40% test / 10% unused
- TR20: 20% train / 20% validation / 40% test / 20% unused
- TR10: 10% train / 20% validation / 40% test / 30% unused

For static datasets, training subsets will be nested while validation and test
sets remain fixed.

Elliptic will use the same training-size conditions while preserving temporal
chronology rather than random stratified splitting.

Author-reproduction runs remain separate and retain the author's original split
logic.

## Step 8 — BWGNN Author Smoke Test

A one-epoch smoke test is run using the official BWGNN Yelp heterogeneous
configuration.

This test checks only that the frozen author code can:

- load a supported dataset
- construct BWGNN
- complete forward and backward propagation
- complete one training epoch
- produce validation/test output

The author split logic is retained for this smoke test.

This is not a final benchmark result and is not used for controlled timing.

In [17]:
# ============================================================
# STEP 8 — 1-EPOCH BWGNN AUTHOR SMOKE TEST
# ============================================================

import os
import subprocess
import time
from pathlib import Path

REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

# CUDA runtime library paths required by DGL 0.8.1
cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)

# Avoid interactive prompts
ENV["PYTHONUNBUFFERED"] = "1"


command = [
    str(PYTHON39),
    "main.py",

    "--dataset", "yelp",
    "--train_ratio", "0.01",
    "--hid_dim", "64",
    "--order", "2",
    "--homo", "0",

    # Smoke test only
    "--epoch", "1",
    "--run", "1",
]


print("===== STEP 8 — AUTHOR SMOKE TEST =====", flush=True)

print(
    "Repository:",
    REPO_DIR,
    flush=True
)

print(
    "Command:",
    " ".join(command),
    flush=True
)

print(
    "\nSTART — Yelp 1-epoch smoke test",
    flush=True
)

start = time.perf_counter()

process = subprocess.Popen(
    command,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

return_code = process.wait()

elapsed = time.perf_counter() - start


print(
    f"\nElapsed smoke-test time: {elapsed:.2f} s",
    flush=True
)

print(
    "\n===== STEP 8 GATE =====",
    flush=True
)

if return_code == 0:

    print(
        "PASS — official BWGNN Yelp code completed "
        "one training epoch.",
        flush=True
    )

else:

    raise RuntimeError(
        f"BWGNN smoke test failed "
        f"(return code {return_code})."
    )

===== STEP 8 — AUTHOR SMOKE TEST =====
Repository: /kaggle/working/Rethinking-Anomaly-Detection
Command: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python main.py --dataset yelp --train_ratio 0.01 --hid_dim 64 --order 2 --homo 0 --epoch 1 --run 1

START — Yelp 1-epoch smoke test
Namespace(dataset='yelp', train_ratio=0.01, hid_dim=64, order=2, homo=0, epoch=1, run=1)
Extracting file to /root/.dgl/yelp
Done saving data into cached files.
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 32])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.paramet

## Step 9 — Unified Nested Split Protocol

For the controlled benchmark, static graph datasets use four training-size
conditions: TR40, TR30, TR20 and TR10.

The validation and test sets remain fixed across all conditions. Training
subsets are nested:

`TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40`

Split seed: 2.

These unified splits are separate from the original BWGNN author split logic.

Elliptic will be handled separately because its temporal chronology must be
preserved.

In [18]:
# ============================================================
# STEP 9 — GENERATE NESTED UNIFIED SPLITS
# YelpChi + Amazon
# ============================================================

from pathlib import Path
import numpy as np
import scipy.io as sio
import json

from sklearn.model_selection import train_test_split


SEED = 2

SPLIT_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/shared/splits"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


DATASETS = {
    "yelp": {
        "path":
            "/kaggle/input/datasets/pathikahmed0007/"
            "yelp-chi/YelpChi.mat",
        "label_key": "label",
    },

    "amazon": {
        "path":
            "/kaggle/input/datasets/pathikahmed0007/"
            "amazon/Amazon.mat",
        "label_key": "label",
    },
}


def load_mat_labels(path, label_key):

    mat = sio.loadmat(path)

    labels = np.asarray(
        mat[label_key]
    ).reshape(-1).astype(int)

    return labels


def make_nested_splits(labels, seed=2):

    all_ids = np.arange(
        len(labels)
    )

    # --------------------------------------------------------
    # First lock 40% test
    # --------------------------------------------------------

    remaining_ids, test_ids = train_test_split(
        all_ids,
        test_size=0.40,
        stratify=labels,
        random_state=seed,
        shuffle=True
    )

    # --------------------------------------------------------
    # From remaining 60%, lock 20% of TOTAL as validation.
    #
    # 20 / 60 = 1/3 of remaining pool
    # --------------------------------------------------------

    train40_ids, val_ids = train_test_split(
        remaining_ids,
        test_size=(1 / 3),
        stratify=labels[remaining_ids],
        random_state=seed,
        shuffle=True
    )

    # train40_ids is now 40% of total.

    # --------------------------------------------------------
    # Create nested smaller training sets
    # --------------------------------------------------------

    train30_ids, _ = train_test_split(
        train40_ids,
        train_size=0.75,       # 30 / 40
        stratify=labels[train40_ids],
        random_state=seed,
        shuffle=True
    )

    train20_ids, _ = train_test_split(
        train30_ids,
        train_size=(2 / 3),    # 20 / 30
        stratify=labels[train30_ids],
        random_state=seed,
        shuffle=True
    )

    train10_ids, _ = train_test_split(
        train20_ids,
        train_size=0.50,       # 10 / 20
        stratify=labels[train20_ids],
        random_state=seed,
        shuffle=True
    )

    return {
        "TR40": np.sort(train40_ids),
        "TR30": np.sort(train30_ids),
        "TR20": np.sort(train20_ids),
        "TR10": np.sort(train10_ids),
        "val": np.sort(val_ids),
        "test": np.sort(test_ids),
    }


def verify_nested(splits):

    tr40 = set(splits["TR40"])
    tr30 = set(splits["TR30"])
    tr20 = set(splits["TR20"])
    tr10 = set(splits["TR10"])

    return (
        tr10.issubset(tr20)
        and tr20.issubset(tr30)
        and tr30.issubset(tr40)
    )


print("===== UNIFIED SPLIT GENERATION =====")

for dataset_name, cfg in DATASETS.items():

    print(f"\n--- {dataset_name.upper()} ---")

    labels = load_mat_labels(
        cfg["path"],
        cfg["label_key"]
    )

    splits = make_nested_splits(
        labels,
        SEED
    )

    nested_ok = verify_nested(
        splits
    )

    print(
        "Total:",
        f"{len(labels):,}"
    )

    for key in [
        "TR40",
        "TR30",
        "TR20",
        "TR10",
        "val",
        "test"
    ]:

        ids = splits[key]

        fraud = int(
            labels[ids].sum()
        )

        normal = len(ids) - fraud

        print(
            f"{key:<5} | "
            f"N={len(ids):>6,} | "
            f"normal={normal:>6,} | "
            f"fraud={fraud:>5,}"
        )

    print(
        "Nested training sets:",
        "PASS" if nested_ok else "FAIL"
    )

    if not nested_ok:
        raise RuntimeError(
            f"{dataset_name}: nested split check failed."
        )

    # --------------------------------------------------------
    # Save one compressed split file
    # --------------------------------------------------------

    output = (
        SPLIT_ROOT /
        f"{dataset_name}_seed2_nested_splits.npz"
    )

    np.savez_compressed(
        output,
        **splits
    )

    print(
        "Saved:",
        output
    )


print("\n===== STEP 9 GATE =====")
print(
    "PASS — YelpChi and Amazon unified nested splits created."
)

===== UNIFIED SPLIT GENERATION =====

--- YELP ---
Total: 45,954
TR40  | N=18,381 | normal=15,710 | fraud=2,671
TR30  | N=13,785 | normal=11,782 | fraud=2,003
TR20  | N= 9,190 | normal= 7,855 | fraud=1,335
TR10  | N= 4,595 | normal= 3,927 | fraud=  668
val   | N= 9,191 | normal= 7,856 | fraud=1,335
test  | N=18,382 | normal=15,711 | fraud=2,671
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/yelp_seed2_nested_splits.npz

--- AMAZON ---
Total: 11,944
TR40  | N= 4,777 | normal= 4,448 | fraud=  329
TR30  | N= 3,582 | normal= 3,335 | fraud=  247
TR20  | N= 2,388 | normal= 2,223 | fraud=  165
TR10  | N= 1,194 | normal= 1,112 | fraud=   82
val   | N= 2,389 | normal= 2,225 | fraud=  164
test  | N= 4,778 | normal= 4,450 | fraud=  328
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/amazon_seed2_nested_splits.npz

===== STEP 9 GATE =====
PASS — YelpChi and Amazon unified nested splits created.


### Step 9B — T-Social Splits and FDCompCN Inspection

T-Social uses the same nested static split protocol as YelpChi and Amazon.

FDCompCN is inspected first to determine its graph structure, feature fields
and label encoding before generating its unified splits.

In [19]:
# ============================================================
# STEP 9B — T-SOCIAL SPLITS + FDCOMPCN INSPECTION
# ============================================================

import os
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

SPLIT_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/shared/splits"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


ENV = os.environ.copy()
ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)


# ============================================================
# PART A — T-SOCIAL NESTED SPLITS
# ============================================================

TSOCIAL_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tsocial/tsocial"
)

TSOCIAL_SPLIT_PATH = (
    SPLIT_ROOT /
    "tsocial_seed2_nested_splits.npz"
)

TSOCIAL_SCRIPT = r'''
import sys
import numpy as np
import torch
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]
seed = 2

print("START — loading T-Social", flush=True)

graphs, _ = load_graphs(path)
g = graphs[0]

labels = g.ndata["label"].reshape(-1).cpu().numpy().astype(int)

all_ids = np.arange(len(labels))

# Fixed 40% test
remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=seed,
    shuffle=True
)

# Fixed 20% validation
train40, val_ids = train_test_split(
    remaining,
    test_size=(1/3),
    stratify=labels[remaining],
    random_state=seed,
    shuffle=True
)

# Nested training subsets
train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=seed,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2/3),
    stratify=labels[train30],
    random_state=seed,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=seed,
    shuffle=True
)

splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}

np.savez_compressed(
    output,
    **splits
)

print("\n===== T-SOCIAL UNIFIED SPLITS =====")

for key in ["TR40", "TR30", "TR20", "TR10", "val", "test"]:

    ids = splits[key]

    fraud = int(labels[ids].sum())
    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>9,} | "
        f"normal={normal:>9,} | "
        f"fraud={fraud:>7,}"
    )

nested = (
    set(train10).issubset(set(train20))
    and set(train20).issubset(set(train30))
    and set(train30).issubset(set(train40))
)

print(
    "Nested training sets:",
    "PASS" if nested else "FAIL"
)

print("Saved:", output)

if not nested:
    raise RuntimeError("T-Social nesting failed.")
'''

print("===== T-SOCIAL SPLIT GENERATION =====", flush=True)

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        TSOCIAL_SCRIPT,
        TSOCIAL_PATH,
        str(TSOCIAL_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=1800
)

print(result.stdout, flush=True)

if result.stderr:
    print("STDERR:", flush=True)
    print(result.stderr, flush=True)

if result.returncode != 0:
    raise RuntimeError(
        "T-Social split generation failed."
    )


# ============================================================
# PART B — FDCOMPCN STRUCTURE INSPECTION
# ============================================================

FDCOMP_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

FDCOMP_SCRIPT = r'''
import sys
import torch
import dgl
from dgl.data.utils import load_graphs

graphs, label_dict = load_graphs(sys.argv[1])

print("===== FDCOMPCN STRUCTURE =====")
print("DGL version :", dgl.__version__)
print("Graphs      :", len(graphs))
print("label_dict  :", list(label_dict.keys()))

for i, g in enumerate(graphs):

    print(f"\nGraph {i}")
    print("Graph object:")
    print(g)

    print("Nodes :", g.num_nodes())
    print("Edges :", g.num_edges())

    print(
        "Node data keys:",
        list(g.ndata.keys())
    )

    print(
        "Edge data keys:",
        list(g.edata.keys())
    )

    for key in g.ndata.keys():

        value = g.ndata[key]

        print(
            f"ndata['{key}'] "
            f"shape={list(value.shape)} "
            f"dtype={value.dtype}"
        )

        if "label" in key.lower():

            labels = value

            if labels.ndim == 2 and labels.shape[1] > 1:
                labels = labels.argmax(1)
            else:
                labels = labels.reshape(-1)

            values, counts = torch.unique(
                labels,
                return_counts=True
            )

            print(
                "Label counts:",
                {
                    str(int(v)): int(c)
                    for v, c in zip(values, counts)
                }
            )

for key, value in label_dict.items():

    print(
        f"\nlabel_dict['{key}'] "
        f"shape={list(value.shape)} "
        f"dtype={value.dtype}"
    )

    if "label" in key.lower():

        labels = value

        if labels.ndim == 2 and labels.shape[1] > 1:
            labels = labels.argmax(1)
        else:
            labels = labels.reshape(-1)

        values, counts = torch.unique(
            labels,
            return_counts=True
        )

        print(
            "Label counts:",
            {
                str(int(v)): int(c)
                for v, c in zip(values, counts)
            }
        )
'''

print("\n===== FDCOMPCN INSPECTION =====", flush=True)

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        FDCOMP_SCRIPT,
        FDCOMP_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout, flush=True)

if result.stderr:
    print("STDERR:", flush=True)
    print(result.stderr, flush=True)

if result.returncode != 0:
    raise RuntimeError(
        "FDCompCN inspection failed."
    )


print("\n===== STEP 9B GATE =====")
print(
    "PASS — T-Social splits created; "
    "FDCompCN structure inspected."
)

===== T-SOCIAL SPLIT GENERATION =====
START — loading T-Social

===== T-SOCIAL UNIFIED SPLITS =====
TR40  | N=2,312,426 | normal=2,242,714 | fraud= 69,712
TR30  | N=1,734,319 | normal=1,682,035 | fraud= 52,284
TR20  | N=1,156,212 | normal=1,121,356 | fraud= 34,856
TR10  | N=  578,106 | normal=  560,678 | fraud= 17,428
val   | N=1,156,213 | normal=1,121,357 | fraud= 34,856
test  | N=2,312,426 | normal=2,242,714 | fraud= 69,712
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/tsocial_seed2_nested_splits.npz


===== FDCOMPCN INSPECTION =====
===== FDCOMPCN STRUCTURE =====
DGL version : 0.8.1
Graphs      : 1
label_dict  : []

Graph 0
Graph object:
Graph(num_nodes={'company': 5317},
      num_edges={('company', 'homo', 'company'): 10059, ('company', 'invest_bc2bc', 'company'): 8505, ('company', 'provide_bc2bc', 'company'): 5944, ('company', 'sale_bc2bc', 'company'): 6244},
      metagraph=[('company', 'company', 'homo'), ('company', 'company', 'invest_bc2bc'), 

### Step 9C — FDCompCN Unified Nested Splits

FDCompCN uses the same static nested split protocol:

- TR40, TR30, TR20, TR10
- fixed validation set
- fixed test set
- split seed 2
- TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40

The dataset's original train/validation/test masks are not used for the
unified benchmark splits.

In [20]:
# ============================================================
# STEP 9C — FDCOMPCN UNIFIED NESTED SPLITS
# ============================================================

import os
import subprocess
from pathlib import Path

FDCOMP_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

FDCOMP_SPLIT_PATH = (
    SPLIT_ROOT /
    "fdcompcn_seed2_nested_splits.npz"
)

SCRIPT = r'''
import sys
import numpy as np
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]
seed = 2

graphs, _ = load_graphs(path)
g = graphs[0]

labels = (
    g.ndata["label"]
    .reshape(-1)
    .cpu()
    .numpy()
    .astype(int)
)

all_ids = np.arange(len(labels))

# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=seed,
    shuffle=True
)

# ------------------------------------------------------------
# Fixed 20% validation
# ------------------------------------------------------------

train40, val_ids = train_test_split(
    remaining,
    test_size=(1 / 3),
    stratify=labels[remaining],
    random_state=seed,
    shuffle=True
)

# ------------------------------------------------------------
# Nested training subsets
# ------------------------------------------------------------

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=seed,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2 / 3),
    stratify=labels[train30],
    random_state=seed,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=seed,
    shuffle=True
)

splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}

# ------------------------------------------------------------
# Verify nesting
# ------------------------------------------------------------

nested = (
    set(train10).issubset(set(train20))
    and set(train20).issubset(set(train30))
    and set(train30).issubset(set(train40))
)

if not nested:
    raise RuntimeError("FDCompCN nesting failed.")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.savez_compressed(
    output,
    **splits
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("===== FDCOMPCN UNIFIED SPLITS =====")

for key in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[key]

    fraud = int(labels[ids].sum())
    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>5,} | "
        f"normal={normal:>5,} | "
        f"fraud={fraud:>4,}"
    )

print(
    "Nested training sets:",
    "PASS"
)

print(
    "Saved:",
    output
)
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        FDCOMP_PATH,
        str(FDCOMP_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=600
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "FDCompCN split generation failed."
    )

print("\n===== STEP 9C GATE =====")
print(
    "PASS — FDCompCN unified nested splits created."
)

===== FDCOMPCN UNIFIED SPLITS =====
TR40  | N=2,126 | normal=1,903 | fraud= 223
TR30  | N=1,594 | normal=1,427 | fraud= 167
TR20  | N=1,062 | normal=  951 | fraud= 111
TR10  | N=  531 | normal=  475 | fraud=  56
val   | N=1,064 | normal=  952 | fraud= 112
test  | N=2,127 | normal=1,903 | fraud= 224
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/fdcompcn_seed2_nested_splits.npz


===== STEP 9C GATE =====
PASS — FDCompCN unified nested splits created.


### Step 9D — Elliptic Chronological Unified Splits

Elliptic uses chronological rather than random splits.

The feature CSV has no header, so it is loaded with `header=None`.
Only known-label transactions are included in train/validation/test masks.
Unknown transactions remain available as graph nodes.

Whole time steps are preserved, with nested TR10/TR20/TR30/TR40 training
windows and fixed later validation/test periods.

In [22]:
# ============================================================
# STEP 9D — ELLIPTIC CHRONOLOGICAL SPLITS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

FEATURES_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_features.csv"
)

CLASSES_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_classes.csv"
)

OUTPUT_PATH = (
    SPLIT_ROOT /
    "elliptic_chronological_nested_splits.npz"
)

# Features file has NO header
features = pd.read_csv(
    FEATURES_PATH,
    header=None
)

# Classes file has header: txId,class
classes = pd.read_csv(
    CLASSES_PATH
)

print("Features rows :", f"{len(features):,}")
print("Classes rows  :", f"{len(classes):,}")
print("Feature cols  :", features.shape[1])

if len(features) != 203769:
    raise RuntimeError(
        "Unexpected Elliptic feature-row count."
    )

if features.shape[1] != 167:
    raise RuntimeError(
        "Unexpected Elliptic feature-column count."
    )

# Column 0 = transaction ID
# Column 1 = time step
features = features.copy()

features["_node_id"] = np.arange(
    len(features),
    dtype=np.int64
)

base = pd.DataFrame({
    "txId": features.iloc[:, 0],
    "time_step": features.iloc[:, 1].astype(int),
    "_node_id": features["_node_id"],
})

classes["txId"] = classes["txId"].astype(
    base["txId"].dtype
)

df = base.merge(
    classes,
    on="txId",
    how="left",
    validate="one_to_one"
)

# Known labels only
known = df[
    df["class"].astype(str).isin(["1", "2"])
].copy()

# 1 = illicit/fraud
# 2 = licit/normal
known["label"] = (
    known["class"]
    .astype(str)
    .map({"1": 1, "2": 0})
    .astype(int)
)

known = known.sort_values(
    ["time_step", "txId"]
).reset_index(drop=True)

print(
    "Known-label nodes  :",
    f"{len(known):,}"
)

print(
    "Unknown-label nodes:",
    f"{len(df) - len(known):,}"
)

print(
    "Fraud:",
    f"{int(known['label'].sum()):,}"
)

print(
    "Normal:",
    f"{len(known) - int(known['label'].sum()):,}"
)

# ------------------------------------------------------------
# Counts per complete time step
# ------------------------------------------------------------

step_counts = (
    known.groupby("time_step")
    .size()
    .sort_index()
)

steps = step_counts.index.to_numpy()
total = len(known)

# ------------------------------------------------------------
# Closest complete-time-step 40/20/40 split
# ------------------------------------------------------------

best = None

for train_end in steps[:-2]:

    later = steps[steps > train_end]

    for val_end in later[:-1]:

        n_train = int(
            step_counts[
                step_counts.index <= train_end
            ].sum()
        )

        n_val = int(
            step_counts[
                (step_counts.index > train_end)
                &
                (step_counts.index <= val_end)
            ].sum()
        )

        n_test = total - n_train - n_val

        ratios = np.array([
            n_train / total,
            n_val / total,
            n_test / total
        ])

        error = np.abs(
            ratios - np.array([0.40, 0.20, 0.40])
        )

        score = (
            error.max(),
            error.sum()
        )

        if best is None or score < best["score"]:

            best = {
                "train_end": int(train_end),
                "val_end": int(val_end),
                "score": score
            }

TR40_END = best["train_end"]
VAL_END = best["val_end"]

cumulative = step_counts.cumsum()

def closest_training_cutoff(ratio):

    candidates = cumulative[
        cumulative.index <= TR40_END
    ]

    target = ratio * total

    return int(
        (candidates - target)
        .abs()
        .idxmin()
    )

TR10_END = closest_training_cutoff(0.10)
TR20_END = closest_training_cutoff(0.20)
TR30_END = closest_training_cutoff(0.30)

def train_ids(end_step):

    return np.sort(
        known.loc[
            known["time_step"] <= end_step,
            "_node_id"
        ].to_numpy(dtype=np.int64)
    )

splits = {
    "TR10": train_ids(TR10_END),
    "TR20": train_ids(TR20_END),
    "TR30": train_ids(TR30_END),
    "TR40": train_ids(TR40_END),

    "val": np.sort(
        known.loc[
            (known["time_step"] > TR40_END)
            &
            (known["time_step"] <= VAL_END),
            "_node_id"
        ].to_numpy(dtype=np.int64)
    ),

    "test": np.sort(
        known.loc[
            known["time_step"] > VAL_END,
            "_node_id"
        ].to_numpy(dtype=np.int64)
    ),
}

# Verify nesting
assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

# Verify no overlap
assert not (set(splits["TR40"]) & set(splits["val"]))
assert not (set(splits["TR40"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))

np.savez_compressed(
    OUTPUT_PATH,
    **splits,
    TR10_end_step=np.array([TR10_END]),
    TR20_end_step=np.array([TR20_END]),
    TR30_end_step=np.array([TR30_END]),
    TR40_end_step=np.array([TR40_END]),
    val_end_step=np.array([VAL_END]),
)

label_map = dict(
    zip(
        known["_node_id"],
        known["label"]
    )
)

print("\n===== ELLIPTIC CHRONOLOGICAL BOUNDARIES =====")
print(f"TR10 : <= time step {TR10_END}")
print(f"TR20 : <= time step {TR20_END}")
print(f"TR30 : <= time step {TR30_END}")
print(f"TR40 : <= time step {TR40_END}")
print(f"VAL  : {TR40_END + 1}–{VAL_END}")
print(f"TEST : > {VAL_END}")

print("\n===== ELLIPTIC UNIFIED SPLITS =====")

for key in [
    "TR40", "TR30", "TR20",
    "TR10", "val", "test"
]:

    ids = splits[key]

    fraud = sum(
        label_map[int(i)]
        for i in ids
    )

    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>6,} | "
        f"{len(ids)/total*100:>6.2f}% | "
        f"normal={normal:>6,} | "
        f"fraud={fraud:>5,}"
    )

print("\nNested training sets: PASS")
print("Saved:", OUTPUT_PATH)

print("\n===== STEP 9D GATE =====")
print(
    "PASS — corrected Elliptic chronological splits created."
)

Features rows : 203,769
Classes rows  : 203,769
Feature cols  : 167
Known-label nodes  : 46,564
Unknown-label nodes: 157,205
Fraud: 4,545
Normal: 42,019

===== ELLIPTIC CHRONOLOGICAL BOUNDARIES =====
TR10 : <= time step 3
TR20 : <= time step 7
TR30 : <= time step 12
TR40 : <= time step 20
VAL  : 21–31
TEST : > 31

===== ELLIPTIC UNIFIED SPLITS =====
TR40  | N=18,889 |  40.57% | normal=17,118 | fraud=1,771
TR30  | N=13,670 |  29.36% | normal=12,999 | fraud=  671
TR20  | N= 9,553 |  20.52% | normal= 9,362 | fraud=  191
TR10  | N= 4,543 |   9.76% | normal= 4,497 | fraud=   46
val   | N= 8,726 |  18.74% | normal= 7,437 | fraud=1,289
test  | N=18,949 |  40.69% | normal=17,464 | fraud=1,485

Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/elliptic_chronological_nested_splits.npz

===== STEP 9D GATE =====
PASS — corrected Elliptic chronological splits created.


### Step 9E — T-Finance Unified Nested Splits

The attached T-Finance graph contains valid one-hot labels with 1,804 fraud
nodes and 37,553 normal nodes.

BWGNN itself converts these labels with `argmax(1)` and does not enforce a
fixed anomaly count.

Therefore, the current canonical file is split without modifying any label.

The unified protocol uses:

- split seed 2
- fixed validation and test sets
- TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40

The observed 1,804-fraud source count is retained in the split metadata for
provenance.

In [23]:
# ============================================================
# STEP 9E — T-FINANCE UNIFIED NESTED SPLITS
# ============================================================

import subprocess
from pathlib import Path

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

TF_SPLIT_PATH = (
    SPLIT_ROOT /
    "tfinance_seed2_nested_splits.npz"
)

SCRIPT = r'''
import sys
import numpy as np
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]

SEED = 2

graphs, _ = load_graphs(path)
g = graphs[0]

raw_labels = g.ndata["label"]

# Same conversion used by BWGNN author code
labels = (
    raw_labels.argmax(1)
    .cpu()
    .numpy()
    .astype(int)
)

print("===== T-FINANCE SOURCE =====")
print("Nodes   :", f"{g.num_nodes():,}")
print("Edges   :", f"{g.num_edges():,}")
print("Features:", list(g.ndata["feature"].shape))
print("Normal  :", f"{int((labels == 0).sum()):,}")
print("Fraud   :", f"{int((labels == 1).sum()):,}")


all_ids = np.arange(len(labels))

# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=SEED,
    shuffle=True
)

# ------------------------------------------------------------
# Fixed 20% validation
# ------------------------------------------------------------

train40, val_ids = train_test_split(
    remaining,
    test_size=(1 / 3),
    stratify=labels[remaining],
    random_state=SEED,
    shuffle=True
)

# ------------------------------------------------------------
# Nested training subsets
# ------------------------------------------------------------

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=SEED,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2 / 3),
    stratify=labels[train30],
    random_state=SEED,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=SEED,
    shuffle=True
)


splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

assert not (
    set(splits["TR40"])
    & set(splits["val"])
)

assert not (
    set(splits["TR40"])
    & set(splits["test"])
)

assert not (
    set(splits["val"])
    & set(splits["test"])
)


# ------------------------------------------------------------
# Save splits + provenance metadata
# ------------------------------------------------------------

np.savez_compressed(
    output,
    **splits,
    seed=np.array([SEED]),
    source_nodes=np.array([len(labels)]),
    source_normal=np.array([(labels == 0).sum()]),
    source_fraud=np.array([(labels == 1).sum()]),
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n===== T-FINANCE UNIFIED SPLITS =====")

for key in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[key]

    fraud = int(
        labels[ids].sum()
    )

    normal = (
        len(ids) - fraud
    )

    print(
        f"{key:<5} | "
        f"N={len(ids):>6,} | "
        f"normal={normal:>6,} | "
        f"fraud={fraud:>4,}"
    )


print(
    "\nNested training sets: PASS"
)

print(
    "Saved:",
    output
)

print(
    "\n===== STEP 9E GATE ====="
)

print(
    "PASS — current canonical T-Finance file "
    "was split without modifying labels."
)
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        TF_PATH,
        str(TF_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "T-Finance split generation failed."
    )

===== T-FINANCE SOURCE =====
Nodes   : 39,357
Edges   : 42,445,086
Features: [39357, 10]
Normal  : 37,553
Fraud   : 1,804

===== T-FINANCE UNIFIED SPLITS =====
TR40  | N=15,742 | normal=15,021 | fraud= 721
TR30  | N=11,806 | normal=11,265 | fraud= 541
TR20  | N= 7,870 | normal= 7,509 | fraud= 361
TR10  | N= 3,935 | normal= 3,754 | fraud= 181
val   | N= 7,872 | normal= 7,511 | fraud= 361
test  | N=15,743 | normal=15,021 | fraud= 722

Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/tfinance_seed2_nested_splits.npz

===== STEP 9E GATE =====
PASS — current canonical T-Finance file was split without modifying labels.



## Step 10 — Unified BWGNN YelpChi TR40 Smoke Test

This is a one-epoch unified-protocol smoke test using the canonical YelpChi
dataset.

Configuration:

- Training split: TR40
- Validation: fixed 20% validation set
- Test: fixed 40% test set
- Split seed: 2
- Training seed: 2
- Model: BWGNN-Hetero
- Hidden dimension: 64
- Wavelet order: 2
- Optimizer: Adam
- GPU: Tesla T4, GPU 0 only
- Epochs: 1

The frozen author repository is not modified. A separate compatibility copy of
`BWGNN.py` changes only temporary zero-tensor allocation so those tensors are
created on the same device as the input features.

In [24]:
# ============================================================
# STEP 10 — UNIFIED YELPCHI TR40 1-EPOCH GPU SMOKE TEST
# ============================================================

import os
import re
import subprocess
from pathlib import Path


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

ADAPTER = WORK / "adapters"
ADAPTER.mkdir(parents=True, exist_ok=True)

PYTHON39 = (
    WORK / "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

AUTHOR_MODEL = REPO / "BWGNN.py"

PATCHED_MODEL = ADAPTER / "BWGNN_gpu.py"

RUNNER = ADAPTER / "yelp_tr40_smoke.py"


# ------------------------------------------------------------
# 1. Verify frozen upstream commit
# ------------------------------------------------------------

commit = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

EXPECTED_COMMIT = (
    "de0631f039bbd19c1890b483cc01f1007f596af7"
)

if commit != EXPECTED_COMMIT:
    raise RuntimeError(
        f"Unexpected BWGNN commit: {commit}"
    )

print(
    "Upstream commit:",
    commit,
    flush=True
)


# ------------------------------------------------------------
# 2. Create minimal GPU-compatible model copy
# ------------------------------------------------------------

author_code = AUTHOR_MODEL.read_text(
    encoding="utf-8"
)

patched_code, n_changes = re.subn(
    r"torch\.zeros\(\[len\(in_feat\),\s*0\]\)",
    "torch.zeros([len(in_feat), 0], device=in_feat.device)",
    author_code
)

if n_changes != 4:
    raise RuntimeError(
        f"Expected 4 device patches, found {n_changes}."
    )

PATCHED_MODEL.write_text(
    patched_code,
    encoding="utf-8"
)

print(
    "GPU compatibility patches:",
    n_changes,
    flush=True
)

print(
    "Frozen author repository modified: NO",
    flush=True
)


# ------------------------------------------------------------
# 3. Write unified smoke runner
# ------------------------------------------------------------

RUNNER.write_text(
r'''
import sys
import random
import numpy as np
import scipy.io as sio
import scipy.sparse as sp

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN_Hetero


# ============================================================
# SETTINGS
# ============================================================

SEED = 2

DEVICE = torch.device("cuda:0")

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "yelp-chi/YelpChi.mat"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "yelp_seed2_nested_splits.npz"
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# LOAD CANONICAL YELPCHI
# ============================================================

print(
    "START — loading canonical YelpChi",
    flush=True
)

mat = sio.loadmat(DATA_PATH)


# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

features = mat["features"]

if sp.issparse(features):
    features = features.toarray()

features = torch.tensor(
    np.asarray(features),
    dtype=torch.float32
)


# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

labels = torch.tensor(
    np.asarray(
        mat["label"]
    ).reshape(-1),
    dtype=torch.long
)


# ------------------------------------------------------------
# Heterogeneous relations
# ------------------------------------------------------------

relations = {}

for relation in [
    "net_rsr",
    "net_rtr",
    "net_rur"
]:

    matrix = mat[relation].tocoo()

    src = torch.tensor(
        matrix.row,
        dtype=torch.int64
    )

    dst = torch.tensor(
        matrix.col,
        dtype=torch.int64
    )

    relations[
        (
            "review",
            relation,
            "review"
        )
    ] = (
        src,
        dst
    )


graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)


print(
    "Nodes      :",
    graph.num_nodes(),
    flush=True
)

print(
    "Edges      :",
    graph.num_edges(),
    flush=True
)

print(
    "Relations  :",
    graph.canonical_etypes,
    flush=True
)

print(
    "Features   :",
    tuple(features.shape),
    flush=True
)

print(
    "Labels     :",
    tuple(labels.shape),
    flush=True
)


# ============================================================
# LOAD UNIFIED SPLITS
# ============================================================

split = np.load(
    SPLIT_PATH
)

train_ids = torch.tensor(
    split["TR40"],
    dtype=torch.long
)

val_ids = torch.tensor(
    split["val"],
    dtype=torch.long
)

test_ids = torch.tensor(
    split["test"],
    dtype=torch.long
)


print(
    "TR40 / VAL / TEST:",
    len(train_ids),
    len(val_ids),
    len(test_ids),
    flush=True
)


# ============================================================
# GPU 0
# ============================================================

graph = graph.to(DEVICE)

features = features.to(DEVICE)
labels = labels.to(DEVICE)

train_ids = train_ids.to(DEVICE)
val_ids = val_ids.to(DEVICE)
test_ids = test_ids.to(DEVICE)


print(
    "GPU        :",
    torch.cuda.get_device_name(0),
    flush=True
)

print(
    "Visible GPU:",
    torch.cuda.device_count(),
    flush=True
)


# ============================================================
# MODEL
# ============================================================

model = BWGNN_Hetero(
    in_feats=features.shape[1],
    h_feats=64,
    num_classes=2,
    graph=graph,
    d=2
).to(DEVICE)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    betas=(0.9, 0.999),
    eps=1e-8
)


# ============================================================
# TRAIN-ONLY CLASS WEIGHT
# ============================================================

train_labels = labels[train_ids]

normal_count = int(
    (train_labels == 0).sum().item()
)

fraud_count = int(
    (train_labels == 1).sum().item()
)

class_weight = torch.tensor(
    [
        1.0,
        normal_count / fraud_count
    ],
    dtype=torch.float32,
    device=DEVICE
)


print(
    "Train normal/fraud:",
    normal_count,
    fraud_count,
    flush=True
)

print(
    "Fraud class weight:",
    f"{normal_count / fraud_count:.6f}",
    flush=True
)


# ============================================================
# ONE TRAINING EPOCH
# ============================================================

print(
    "\nSTART — TR40 epoch 1",
    flush=True
)

model.train()

logits = model(features)

loss = F.cross_entropy(
    logits[train_ids],
    labels[train_ids],
    weight=class_weight
)

optimizer.zero_grad()
loss.backward()
optimizer.step()


# ============================================================
# VALIDATION
# ============================================================

model.eval()

with torch.no_grad():

    logits = model(features)

    probabilities = torch.softmax(
        logits,
        dim=1
    )[:, 1]


val_y = (
    labels[val_ids]
    .detach()
    .cpu()
    .numpy()
)

val_prob = (
    probabilities[val_ids]
    .detach()
    .cpu()
    .numpy()
)


val_auprc = average_precision_score(
    val_y,
    val_prob
)


# ============================================================
# VALIDATION THRESHOLD
#
# 0.01 -> 0.99
# Max Macro-F1
# Tie 1: higher fraud recall
# Tie 2: threshold closer to 0.50
# ============================================================

candidates = []

for threshold in np.arange(
    0.01,
    1.00,
    0.01
):

    prediction = (
        val_prob >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        val_y,
        prediction,
        average="macro"
    )

    fraud_recall = recall_score(
        val_y,
        prediction,
        zero_division=0
    )

    candidates.append(
        (
            macro_f1,
            fraud_recall,
            -abs(float(threshold) - 0.50),
            float(threshold)
        )
    )


best = max(candidates)

best_macro_f1 = best[0]
best_recall = best[1]
best_threshold = best[3]


print(
    "\n===== UNIFIED VALIDATION =====",
    flush=True
)

print(
    f"Loss          : {loss.item():.6f}",
    flush=True
)

print(
    f"Val AUPRC     : {val_auprc:.6f}",
    flush=True
)

print(
    f"Val threshold : {best_threshold:.2f}",
    flush=True
)

print(
    f"Val Macro-F1  : {best_macro_f1:.6f}",
    flush=True
)

print(
    f"Val fraud REC : {best_recall:.6f}",
    flush=True
)


# ============================================================
# TEST
#
# Threshold already selected from validation.
# Test is not used to choose it.
# ============================================================

test_y = (
    labels[test_ids]
    .detach()
    .cpu()
    .numpy()
)

test_prob = (
    probabilities[test_ids]
    .detach()
    .cpu()
    .numpy()
)

test_prediction = (
    test_prob >= best_threshold
).astype(int)


test_auprc = average_precision_score(
    test_y,
    test_prob
)

test_auroc = roc_auc_score(
    test_y,
    test_prob
)

test_macro_f1 = f1_score(
    test_y,
    test_prediction,
    average="macro"
)

test_recall = recall_score(
    test_y,
    test_prediction,
    zero_division=0
)


print(
    "\n===== UNIFIED TEST =====",
    flush=True
)

print(
    f"AUPRC     : {test_auprc:.6f}",
    flush=True
)

print(
    f"AUROC     : {test_auroc:.6f}",
    flush=True
)

print(
    f"Macro-F1  : {test_macro_f1:.6f}",
    flush=True
)

print(
    f"Fraud REC : {test_recall:.6f}",
    flush=True
)


print(
    "\n===== STEP 10 GATE =====",
    flush=True
)

print(
    "PASS — canonical YelpChi TR40 unified "
    "1-epoch GPU smoke test completed.",
    flush=True
)
''',
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Controlled subprocess environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(
        str(path)
        for path in cuda_dirs
    )
    +
    (
        ":" + existing_ld
        if existing_ld
        else ""
    )
)


# ------------------------------------------------------------
# 5. Run smoke test
# ------------------------------------------------------------

print(
    "\n===== STEP 10 RUN =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:

    print(
        line.rstrip(),
        flush=True
    )

return_code = process.wait()


if return_code != 0:

    raise RuntimeError(
        f"Step 10 failed with return code "
        f"{return_code}."
    )

Upstream commit: de0631f039bbd19c1890b483cc01f1007f596af7
GPU compatibility patches: 4
Frozen author repository modified: NO

===== STEP 10 RUN =====
START — loading canonical YelpChi
Nodes      : 45954
Edges      : 8051348
Relations  : [('review', 'net_rsr', 'review'), ('review', 'net_rtr', 'review'), ('review', 'net_rur', 'review')]
Features   : (45954, 32)
Labels     : (45954,)
TR40 / VAL / TEST: 18381 9191 18382
GPU        : Tesla T4
Visible GPU: 1
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 32])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 192])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([2, 64])
<class 'torch.nn.parameter.Parameter'> torch.Size([2])
Train normal/fraud: 15710 2671
Fraud class weight: 5.881692

START — TR40 epoch 

## Step 11 — YelpChi TR40 Hyperparameter Tuning

BWGNN is tuned only on the YelpChi TR40 condition using training seed 2.

Search space:

- Hidden dimension: 32, 64
- Wavelet order: 2, 3
- Learning rate: 0.001, 0.005, 0.01

Protocol:

- Maximum 12 trials
- Maximum 100 epochs per trial
- Early stopping patience: 20 epochs
- Adam beta1 = 0.9
- Adam beta2 = 0.999
- Adam epsilon = 1e-8
- Model selection and checkpointing: validation AUPRC

The test set is not evaluated during tuning.

After tuning, the winning configuration is frozen for the remaining YelpChi
training-size conditions and training seeds.

In [25]:
# ============================================================
# STEP 11 — YELPCHI TR40 TUNING (12 TRIALS)
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PYTHON39 = (
    WORK / "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK / "envs/bwgnn-author/lib/python3.9/site-packages"
)

ADAPTER = WORK / "adapters"

CHECKPOINT_DIR = (
    WORK / "checkpoints/yelp"
)

RESULT_DIR = (
    WORK / "results/tuning"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNNER = (
    ADAPTER / "yelp_tr40_tune.py"
)


RUNNER.write_text(r'''
import sys
import gc
import csv
import json
import random
import contextlib
import io

from pathlib import Path

import numpy as np
import scipy.io as sio
import scipy.sparse as sp

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN_Hetero


# ============================================================
# SETTINGS
# ============================================================

SEED = 2
MAX_EPOCHS = 100
PATIENCE = 20

DEVICE = torch.device("cuda:0")

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "yelp-chi/YelpChi.mat"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "yelp_seed2_nested_splits.npz"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/checkpoints/yelp"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/results/tuning"
)


# ============================================================
# LOAD DATA ONCE
# ============================================================

print(
    "START — loading YelpChi for TR40 tuning",
    flush=True
)

mat = sio.loadmat(DATA_PATH)

features = mat["features"]

if sp.issparse(features):
    features = features.toarray()

features = torch.tensor(
    np.asarray(features),
    dtype=torch.float32
)

labels = torch.tensor(
    np.asarray(mat["label"]).reshape(-1),
    dtype=torch.long
)


relations = {}

for relation in [
    "net_rsr",
    "net_rtr",
    "net_rur"
]:

    m = mat[relation].tocoo()

    relations[
        ("review", relation, "review")
    ] = (
        torch.tensor(
            m.row,
            dtype=torch.int64
        ),
        torch.tensor(
            m.col,
            dtype=torch.int64
        )
    )


graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)


split = np.load(SPLIT_PATH)

train_ids = torch.tensor(
    split["TR40"],
    dtype=torch.long
)

val_ids = torch.tensor(
    split["val"],
    dtype=torch.long
)


graph = graph.to(DEVICE)

features = features.to(DEVICE)
labels = labels.to(DEVICE)

train_ids = train_ids.to(DEVICE)
val_ids = val_ids.to(DEVICE)


# ============================================================
# TRAIN-ONLY CLASS WEIGHT
# ============================================================

train_labels = labels[train_ids]

normal = int(
    (train_labels == 0).sum().item()
)

fraud = int(
    (train_labels == 1).sum().item()
)

class_weight = torch.tensor(
    [1.0, normal / fraud],
    dtype=torch.float32,
    device=DEVICE
)


print(
    "TR40:",
    len(train_ids),
    "| VAL:",
    len(val_ids),
    flush=True
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
    flush=True
)


# ============================================================
# EXACT 12-TRIAL SEARCH SPACE
# ============================================================

trials = []

trial_id = 0

for hidden in [32, 64]:

    for order in [2, 3]:

        for lr in [
            0.001,
            0.005,
            0.01
        ]:

            trial_id += 1

            trials.append({
                "trial": trial_id,
                "hidden": hidden,
                "order": order,
                "lr": lr
            })


results = []


# ============================================================
# TUNING
# ============================================================

for cfg in trials:

    trial = cfg["trial"]

    print(
        "\n========================================",
        flush=True
    )

    print(
        f"TRIAL {trial:02d}/12 | "
        f"hidden={cfg['hidden']} | "
        f"order={cfg['order']} | "
        f"lr={cfg['lr']}",
        flush=True
    )

    # Same tuning seed for every trial
    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # Suppress constructor parameter printouts
    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        model = BWGNN_Hetero(
            in_feats=features.shape[1],
            h_feats=cfg["hidden"],
            num_classes=2,
            graph=graph,
            d=cfg["order"]
        ).to(DEVICE)


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        betas=(0.9, 0.999),
        eps=1e-8
    )


    best_auprc = -1.0
    best_epoch = -1

    epochs_without_improvement = 0

    checkpoint = (
        CHECKPOINT_DIR /
        f"trial_{trial:02d}.pt"
    )


    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        # -----------------------------
        # TRAIN
        # -----------------------------

        model.train()

        logits = model(features)

        loss = F.cross_entropy(
            logits[train_ids],
            labels[train_ids],
            weight=class_weight
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        # -----------------------------
        # VALIDATION
        # -----------------------------

        model.eval()

        with torch.no_grad():

            logits = model(features)

            val_prob = torch.softmax(
                logits,
                dim=1
            )[val_ids, 1]

        val_y = (
            labels[val_ids]
            .detach()
            .cpu()
            .numpy()
        )

        val_prob_np = (
            val_prob
            .detach()
            .cpu()
            .numpy()
        )

        val_auprc = (
            average_precision_score(
                val_y,
                val_prob_np
            )
        )


        # -----------------------------
        # CHECKPOINT BY VAL AUPRC
        # -----------------------------

        if val_auprc > best_auprc:

            best_auprc = val_auprc
            best_epoch = epoch

            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint
            )

        else:

            epochs_without_improvement += 1


        if (
            epoch == 1
            or epoch % 10 == 0
            or epochs_without_improvement >= PATIENCE
        ):

            print(
                f"epoch={epoch:03d} | "
                f"loss={loss.item():.5f} | "
                f"val_AUPRC={val_auprc:.5f} | "
                f"best={best_auprc:.5f}"
                f"@{best_epoch}",
                flush=True
            )


        if epochs_without_improvement >= PATIENCE:

            print(
                f"Early stop at epoch {epoch}",
                flush=True
            )

            break


    results.append({
        "trial": trial,
        "hidden": cfg["hidden"],
        "order": cfg["order"],
        "lr": cfg["lr"],
        "best_epoch": best_epoch,
        "best_val_auprc": best_auprc,
        "checkpoint": str(checkpoint)
    })


    print(
        f"TRIAL {trial:02d} DONE | "
        f"best val AUPRC="
        f"{best_auprc:.6f} "
        f"at epoch {best_epoch}",
        flush=True
    )


    del model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# SELECT WINNER USING VALIDATION AUPRC ONLY
# ============================================================

winner = max(
    results,
    key=lambda x: x[
        "best_val_auprc"
    ]
)


# ============================================================
# SAVE RESULTS
# ============================================================

csv_path = (
    RESULT_DIR /
    "yelp_tr40_seed2_tuning.csv"
)

with csv_path.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys()
    )

    writer.writeheader()
    writer.writerows(results)


winner_path = (
    RESULT_DIR /
    "yelp_best_config.json"
)

winner_path.write_text(
    json.dumps(
        winner,
        indent=2
    )
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n===== YELP TR40 TUNING RESULTS =====",
    flush=True
)

for r in sorted(
    results,
    key=lambda x:
        x["best_val_auprc"],
    reverse=True
):

    print(
        f"trial={r['trial']:02d} | "
        f"hidden={r['hidden']:>2} | "
        f"order={r['order']} | "
        f"lr={r['lr']:<5} | "
        f"epoch={r['best_epoch']:>3} | "
        f"val AUPRC={r['best_val_auprc']:.6f}",
        flush=True
    )


print(
    "\n===== WINNING CONFIGURATION =====",
    flush=True
)

print(
    "Hidden dimension:",
    winner["hidden"],
    flush=True
)

print(
    "Wavelet order   :",
    winner["order"],
    flush=True
)

print(
    "Learning rate   :",
    winner["lr"],
    flush=True
)

print(
    "Best epoch      :",
    winner["best_epoch"],
    flush=True
)

print(
    "Best Val AUPRC  :",
    f"{winner['best_val_auprc']:.6f}",
    flush=True
)

print(
    "\nTEST SET ACCESSED DURING TUNING: NO",
    flush=True
)

print(
    "\n===== STEP 11 GATE =====",
    flush=True
)

print(
    "PASS — 12-trial YelpChi TR40 tuning completed "
    "using validation AUPRC only.",
    flush=True
)
''')


# ============================================================
# ISOLATED RUNTIME
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(
        str(x)
        for x in cuda_dirs
    )
    +
    (
        ":" + existing
        if existing
        else ""
    )
)


# ============================================================
# RUN
# ============================================================

print(
    "===== STEP 11 — STARTING 12 TRIALS =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = process.wait()

if rc != 0:

    raise RuntimeError(
        f"Step 11 failed with return code {rc}."
    )

===== STEP 11 — STARTING 12 TRIALS =====
START — loading YelpChi for TR40 tuning
TR40: 18381 | VAL: 9191
GPU: Tesla T4

TRIAL 01/12 | hidden=32 | order=2 | lr=0.001
epoch=001 | loss=0.69139 | val_AUPRC=0.26851 | best=0.26851@1
epoch=010 | loss=0.63800 | val_AUPRC=0.29383 | best=0.29383@10
epoch=020 | loss=0.60324 | val_AUPRC=0.29241 | best=0.30302@13
epoch=030 | loss=0.58924 | val_AUPRC=0.35875 | best=0.35875@30
epoch=040 | loss=0.55592 | val_AUPRC=0.38986 | best=0.38986@40
epoch=050 | loss=0.51963 | val_AUPRC=0.43248 | best=0.43248@50
epoch=060 | loss=0.48765 | val_AUPRC=0.48798 | best=0.48798@60
epoch=070 | loss=0.47207 | val_AUPRC=0.51665 | best=0.51665@70
epoch=080 | loss=0.46456 | val_AUPRC=0.52845 | best=0.52845@80
epoch=090 | loss=0.45703 | val_AUPRC=0.53632 | best=0.53632@90
epoch=100 | loss=0.45001 | val_AUPRC=0.54732 | best=0.54732@100
TRIAL 01 DONE | best val AUPRC=0.547318 at epoch 100

TRIAL 02/12 | hidden=32 | order=2 | lr=0.005
epoch=001 | loss=0.69139 | val_AUPRC=0.2663

## Step 12 — Final YelpChi Unified Benchmark Runs

The winning TR40 configuration is frozen:

- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.01

Final runs use:

- Splits: TR40, TR30, TR20, TR10
- Training seeds: 2, 42, 72
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint criterion: validation AUPRC
- Validation threshold grid: 0.01–0.99
- Threshold objective: maximum validation Macro-F1
- Threshold tie-breaks: fraud recall, then closeness to 0.50
- GPU: Tesla T4, GPU 0 only

The test set is evaluated only after checkpoint and threshold selection are
complete.

In [26]:
# ============================================================
# STEP 12 — FINAL YELPCHI UNIFIED RUNS
# 4 SPLITS × 3 SEEDS = 12 RUNS
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PYTHON39 = (
    WORK / "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK / "envs/bwgnn-author/lib/python3.9/site-packages"
)

ADAPTER = WORK / "adapters"

RUNNER = ADAPTER / "yelp_final_runs.py"

RESULT_DIR = WORK / "results/unified/yelp"
CHECKPOINT_DIR = WORK / "checkpoints/yelp_final"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


RUNNER.write_text(r'''
import sys
import gc
import csv
import json
import time
import random
import contextlib
import io

from pathlib import Path

import numpy as np
import scipy.io as sio
import scipy.sparse as sp

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN_Hetero


# ============================================================
# LOCKED CONFIG
# ============================================================

HIDDEN = 64
ORDER = 3
LR = 0.01

MAX_EPOCHS = 100
PATIENCE = 20

SEEDS = [2, 42, 72]

SPLIT_NAMES = [
    "TR40",
    "TR30",
    "TR20",
    "TR10"
]

DEVICE = torch.device("cuda:0")


DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "yelp-chi/YelpChi.mat"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "yelp_seed2_nested_splits.npz"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/results/unified/yelp"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/checkpoints/yelp_final"
)


# ============================================================
# LOAD DATA ONCE
# ============================================================

print("START — loading canonical YelpChi", flush=True)

mat = sio.loadmat(DATA_PATH)

features = mat["features"]

if sp.issparse(features):
    features = features.toarray()

features = torch.tensor(
    np.asarray(features),
    dtype=torch.float32
)

labels = torch.tensor(
    np.asarray(mat["label"]).reshape(-1),
    dtype=torch.long
)


relations = {}

for relation in [
    "net_rsr",
    "net_rtr",
    "net_rur"
]:

    m = mat[relation].tocoo()

    relations[
        ("review", relation, "review")
    ] = (
        torch.tensor(
            m.row,
            dtype=torch.int64
        ),
        torch.tensor(
            m.col,
            dtype=torch.int64
        )
    )


graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)

split_data = np.load(SPLIT_PATH)

val_ids_cpu = torch.tensor(
    split_data["val"],
    dtype=torch.long
)

test_ids_cpu = torch.tensor(
    split_data["test"],
    dtype=torch.long
)


graph = graph.to(DEVICE)
features = features.to(DEVICE)
labels = labels.to(DEVICE)

val_ids = val_ids_cpu.to(DEVICE)
test_ids = test_ids_cpu.to(DEVICE)


print(
    "GPU:",
    torch.cuda.get_device_name(0),
    flush=True
)

print(
    "Visible GPUs:",
    torch.cuda.device_count(),
    flush=True
)


# ============================================================
# THRESHOLD SELECTION
# ============================================================

def select_threshold(y_true, probabilities):

    candidates = []

    for threshold in np.arange(
        0.01,
        1.00,
        0.01
    ):

        prediction = (
            probabilities >= threshold
        ).astype(int)

        macro_f1 = f1_score(
            y_true,
            prediction,
            average="macro"
        )

        fraud_recall = recall_score(
            y_true,
            prediction,
            zero_division=0
        )

        candidates.append(
            (
                macro_f1,
                fraud_recall,
                -abs(float(threshold) - 0.5),
                float(threshold)
            )
        )

    return max(candidates)


# ============================================================
# FINAL RUNS
# ============================================================

results = []

total_runs = len(SPLIT_NAMES) * len(SEEDS)
run_number = 0


for split_name in SPLIT_NAMES:

    train_ids_cpu = torch.tensor(
        split_data[split_name],
        dtype=torch.long
    )

    train_ids = train_ids_cpu.to(DEVICE)

    for seed in SEEDS:

        run_number += 1

        print(
            "\n============================================",
            flush=True
        )

        print(
            f"RUN {run_number:02d}/{total_runs} | "
            f"{split_name} | seed={seed}",
            flush=True
        )

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------

        random.seed(seed)
        np.random.seed(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            model = BWGNN_Hetero(
                in_feats=features.shape[1],
                h_feats=HIDDEN,
                num_classes=2,
                graph=graph,
                d=ORDER
            ).to(DEVICE)


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            betas=(0.9, 0.999),
            eps=1e-8
        )


        # ----------------------------------------------------
        # TRAIN-ONLY class weighting
        # ----------------------------------------------------

        train_labels = labels[train_ids]

        normal_count = int(
            (train_labels == 0).sum().item()
        )

        fraud_count = int(
            (train_labels == 1).sum().item()
        )

        class_weight = torch.tensor(
            [
                1.0,
                normal_count / fraud_count
            ],
            dtype=torch.float32,
            device=DEVICE
        )


        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        best_val_auprc = -1.0
        best_epoch = -1
        no_improvement = 0

        checkpoint = (
            CHECKPOINT_DIR /
            f"{split_name}_seed{seed}.pt"
        )

        torch.cuda.synchronize()
        start_time = time.perf_counter()


        for epoch in range(
            1,
            MAX_EPOCHS + 1
        ):

            model.train()

            logits = model(features)

            loss = F.cross_entropy(
                logits[train_ids],
                labels[train_ids],
                weight=class_weight
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()


            # -----------------------------------------------
            # Validation checkpoint criterion
            # -----------------------------------------------

            model.eval()

            with torch.no_grad():

                logits = model(features)

                val_prob = torch.softmax(
                    logits,
                    dim=1
                )[val_ids, 1]


            val_y = (
                labels[val_ids]
                .detach()
                .cpu()
                .numpy()
            )

            val_prob_np = (
                val_prob
                .detach()
                .cpu()
                .numpy()
            )

            val_auprc = average_precision_score(
                val_y,
                val_prob_np
            )


            if val_auprc > best_val_auprc:

                best_val_auprc = val_auprc
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    model.state_dict(),
                    checkpoint
                )

            else:

                no_improvement += 1


            if (
                epoch == 1
                or epoch % 20 == 0
                or no_improvement >= PATIENCE
            ):

                print(
                    f"epoch={epoch:03d} | "
                    f"loss={loss.item():.5f} | "
                    f"val_AUPRC={val_auprc:.5f} | "
                    f"best={best_val_auprc:.5f}"
                    f"@{best_epoch}",
                    flush=True
                )


            if no_improvement >= PATIENCE:
                break


        torch.cuda.synchronize()

        train_seconds = (
            time.perf_counter()
            - start_time
        )


        # ----------------------------------------------------
        # Restore BEST VALIDATION checkpoint
        # ----------------------------------------------------

        model.load_state_dict(
            torch.load(
                checkpoint,
                map_location=DEVICE
            )
        )

        model.eval()


        with torch.no_grad():

            logits = model(features)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]


        # ----------------------------------------------------
        # Validation threshold selection
        # ----------------------------------------------------

        val_y = (
            labels[val_ids]
            .cpu()
            .numpy()
        )

        val_prob = (
            probabilities[val_ids]
            .cpu()
            .numpy()
        )


        threshold_result = select_threshold(
            val_y,
            val_prob
        )

        val_macro_f1 = threshold_result[0]
        val_fraud_recall = threshold_result[1]
        threshold = threshold_result[3]


        # ----------------------------------------------------
        # TEST ONCE
        # ----------------------------------------------------

        test_y = (
            labels[test_ids]
            .cpu()
            .numpy()
        )

        test_prob = (
            probabilities[test_ids]
            .cpu()
            .numpy()
        )

        test_pred = (
            test_prob >= threshold
        ).astype(int)


        test_auprc = average_precision_score(
            test_y,
            test_prob
        )

        test_auroc = roc_auc_score(
            test_y,
            test_prob
        )

        test_macro_f1 = f1_score(
            test_y,
            test_pred,
            average="macro"
        )

        test_recall = recall_score(
            test_y,
            test_pred,
            zero_division=0
        )

        test_precision = precision_score(
            test_y,
            test_pred,
            zero_division=0
        )


        result = {
            "dataset": "yelp",
            "split": split_name,
            "seed": seed,

            "hidden": HIDDEN,
            "order": ORDER,
            "lr": LR,

            "best_epoch": best_epoch,
            "val_auprc": best_val_auprc,

            "val_threshold": threshold,
            "val_macro_f1": val_macro_f1,
            "val_fraud_recall": val_fraud_recall,

            "test_auprc": test_auprc,
            "test_auroc": test_auroc,
            "test_macro_f1": test_macro_f1,
            "test_fraud_recall": test_recall,
            "test_fraud_precision": test_precision,

            "training_seconds": train_seconds
        }


        results.append(result)


        print(
            f"DONE | {split_name} seed={seed} | "
            f"epoch={best_epoch} | "
            f"val AUPRC={best_val_auprc:.6f} | "
            f"test AUPRC={test_auprc:.6f} | "
            f"test AUROC={test_auroc:.6f} | "
            f"Macro-F1={test_macro_f1:.6f}",
            flush=True
        )


        del model
        del optimizer

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# SAVE RAW RESULTS
# ============================================================

csv_path = (
    RESULT_DIR /
    "yelp_unified_all_runs.csv"
)

with csv_path.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys()
    )

    writer.writeheader()

    writer.writerows(results)


json_path = (
    RESULT_DIR /
    "yelp_unified_all_runs.json"
)

json_path.write_text(
    json.dumps(
        results,
        indent=2
    )
)


# ============================================================
# SPLIT SUMMARY
# ============================================================

print(
    "\n===== YELPCHI FINAL SUMMARY =====",
    flush=True
)


for split_name in SPLIT_NAMES:

    rows = [
        r
        for r in results
        if r["split"] == split_name
    ]

    print(
        f"\n{split_name}",
        flush=True
    )

    for metric in [
        "test_auprc",
        "test_auroc",
        "test_macro_f1",
        "test_fraud_recall"
    ]:

        values = np.array(
            [
                r[metric]
                for r in rows
            ]
        )

        print(
            f"  {metric:<18} "
            f"{values.mean():.6f} "
            f"± {values.std(ddof=1):.6f}",
            flush=True
        )


print(
    "\nResults CSV:",
    csv_path,
    flush=True
)

print(
    "Results JSON:",
    json_path,
    flush=True
)

print(
    "\n===== STEP 12 GATE =====",
    flush=True
)

print(
    "PASS — YelpChi TR40/TR30/TR20/TR10 "
    "completed for seeds 2, 42 and 72.",
    flush=True
)
''')


# ============================================================
# RUNTIME
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(
        str(x)
        for x in cuda_dirs
    )
    +
    (
        ":" + existing
        if existing
        else ""
    )
)


print(
    "===== STEP 12 — STARTING 12 FINAL YELPCHI RUNS =====",
    flush=True
)


process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)


for line in process.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = process.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 12 failed with return code {rc}."
    )

===== STEP 12 — STARTING 12 FINAL YELPCHI RUNS =====
START — loading canonical YelpChi
GPU: Tesla T4
Visible GPUs: 1

RUN 01/12 | TR40 | seed=2
epoch=001 | loss=0.70117 | val_AUPRC=0.26748 | best=0.26748@1
epoch=020 | loss=0.48440 | val_AUPRC=0.47088 | best=0.47517@18
epoch=040 | loss=0.43628 | val_AUPRC=0.56553 | best=0.57417@36
epoch=060 | loss=0.49983 | val_AUPRC=0.46078 | best=0.62550@54
epoch=074 | loss=0.41382 | val_AUPRC=0.60364 | best=0.62550@54
DONE | TR40 seed=2 | epoch=54 | val AUPRC=0.625501 | test AUPRC=0.622404 | test AUROC=0.893031 | Macro-F1=0.750934

RUN 02/12 | TR40 | seed=42
epoch=001 | loss=0.68648 | val_AUPRC=0.25660 | best=0.25660@1
epoch=020 | loss=0.49306 | val_AUPRC=0.45404 | best=0.45404@20
epoch=040 | loss=0.44389 | val_AUPRC=0.56172 | best=0.56172@40
epoch=060 | loss=0.39775 | val_AUPRC=0.61938 | best=0.61938@60
epoch=080 | loss=0.35275 | val_AUPRC=0.64448 | best=0.64448@80
epoch=100 | loss=0.32423 | val_AUPRC=0.66108 | best=0.66305@98
DONE | TR40 seed=42 | 

## Step 13 — BWGNN-Homo YelpChi Verification

The previous YelpChi unified experiment used BWGNN-Hetero.

For consistency with the six-dataset unified benchmark, this step verifies
BWGNN-Homo on the same canonical YelpChi graph and TR40 split.

The heterogeneous relation graph is converted using DGL's homogeneous
conversion and self-loops are added, matching the original BWGNN author's
`homo=True` preprocessing.

The previous heterogeneous results are retained separately.

In [27]:
# ============================================================
# STEP 13 — YELPCHI BWGNN-HOMO 1-EPOCH SMOKE
# ============================================================

from pathlib import Path
import os
import subprocess

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = (
    WORK / "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK / "envs/bwgnn-author/lib/python3.9/site-packages"
)

SOURCE_RUNNER = (
    ADAPTER / "yelp_tr40_smoke.py"
)

HOMO_RUNNER = (
    ADAPTER / "yelp_tr40_homo_smoke.py"
)

if not SOURCE_RUNNER.exists():
    raise FileNotFoundError(SOURCE_RUNNER)


# ------------------------------------------------------------
# Convert existing unified smoke runner to Homo mode
# ------------------------------------------------------------

code = SOURCE_RUNNER.read_text(
    encoding="utf-8"
)

code = code.replace(
    "from BWGNN_gpu import BWGNN_Hetero",
    "from BWGNN_gpu import BWGNN"
)

code = code.replace(
    "model = BWGNN_Hetero(",
    "model = BWGNN("
)


# ------------------------------------------------------------
# Match author's homo preprocessing
# ------------------------------------------------------------

graph_block = '''graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)
'''

replacement = graph_block + '''
# Match BWGNN author homo=True preprocessing
graph = dgl.to_homogeneous(graph)
graph = dgl.add_self_loop(graph)
'''

if graph_block not in code:
    raise RuntimeError(
        "Could not locate graph construction block."
    )

code = code.replace(
    graph_block,
    replacement,
    1
)


# ------------------------------------------------------------
# Tighten reproducibility
# ------------------------------------------------------------

seed_line = (
    "torch.cuda.manual_seed_all(SEED)"
)

code = code.replace(
    seed_line,
    seed_line
    + "\n"
    + "torch.backends.cudnn.deterministic = True\n"
    + "torch.backends.cudnn.benchmark = False",
    1
)


code = code.replace(
    "PASS — canonical YelpChi TR40 unified "
    "1-epoch GPU smoke test completed.",
    "PASS — canonical YelpChi BWGNN-Homo TR40 "
    "1-epoch GPU smoke test completed."
)


HOMO_RUNNER.write_text(
    code,
    encoding="utf-8"
)


print(
    "Created:",
    HOMO_RUNNER,
    flush=True
)

print(
    "Frozen author repository modified: NO",
    flush=True
)


# ------------------------------------------------------------
# Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# Execute
# ------------------------------------------------------------

print(
    "\n===== STEP 13 — BWGNN-HOMO YELP SMOKE =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(HOMO_RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 13 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/yelp_tr40_homo_smoke.py
Frozen author repository modified: NO

===== STEP 13 — BWGNN-HOMO YELP SMOKE =====
START — loading canonical YelpChi
Nodes      : 45954
Edges      : 8097302
Relations  : [('_N', '_E', '_N')]
Features   : (45954, 32)
Labels     : (45954,)
TR40 / VAL / TEST: 18381 9191 18382
GPU        : Tesla T4
Visible GPU: 1
Train normal/fraud: 15710 2671
Fraud class weight: 5.881692

START — TR40 epoch 1

===== UNIFIED VALIDATION =====
Loss          : 0.693316
Val AUPRC     : 0.238522
Val threshold : 0.64
Val Macro-F1  : 0.557665
Val fraud REC : 0.173783

===== UNIFIED TEST =====
AUPRC     : 0.244852
AUROC     : 0.649130
Macro-F1  : 0.567020
Fraud REC : 0.194684

===== STEP 10 GATE =====
PASS — canonical YelpChi TR40 unified 1-epoch GPU smoke test completed.


## Step 14 — BWGNN-Homo YelpChi TR40 Tuning

BWGNN-Homo is tuned on YelpChi TR40 using the same controlled tuning protocol:

- Seed: 2
- Maximum trials: 12
- Maximum epochs: 100
- Patience: 20
- Checkpoint criterion: validation AUPRC
- Test set not accessed during tuning

Search space:

- Hidden dimension: 32, 64
- Wavelet order: 2, 3
- Learning rate: 0.001, 0.005, 0.01

In [28]:
# ============================================================
# STEP 14 — CREATE + RUN YELPCHI HOMO TUNER
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

SOURCE = ADAPTER / "yelp_tr40_tune.py"
HOMO = ADAPTER / "yelp_tr40_homo_tune.py"

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

code = SOURCE.read_text(encoding="utf-8")


# ------------------------------------------------------------
# 1. Hetero model -> Homo model
# ------------------------------------------------------------

code = code.replace(
    "from BWGNN_gpu import BWGNN_Hetero",
    "from BWGNN_gpu import BWGNN"
)

code = code.replace(
    "model = BWGNN_Hetero(",
    "model = BWGNN("
)


# ------------------------------------------------------------
# 2. Convert Yelp relation graph to homogeneous graph
#    + author-style self loops
# ------------------------------------------------------------

old_graph = '''graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)
'''

new_graph = old_graph + '''
graph = dgl.to_homogeneous(graph)
graph = dgl.add_self_loop(graph)
'''

if old_graph not in code:
    raise RuntimeError(
        "Could not locate graph-construction block."
    )

code = code.replace(
    old_graph,
    new_graph,
    1
)


# ------------------------------------------------------------
# 3. Separate Homo output directories/files
# ------------------------------------------------------------

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp"',
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp_homo"'
)

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/results/tuning"',
    '"/kaggle/working/comp8851_bwgnn/results/tuning_homo"'
)

code = code.replace(
    '"yelp_tr40_seed2_tuning.csv"',
    '"yelp_homo_tr40_seed2_tuning.csv"'
)

code = code.replace(
    '"yelp_best_config.json"',
    '"yelp_homo_best_config.json"'
)


# ------------------------------------------------------------
# 4. Deterministic GPU settings
# ------------------------------------------------------------

seed_line = "torch.cuda.manual_seed_all(SEED)"

replacement = (
    seed_line
    + "\n"
    + "    torch.backends.cudnn.deterministic = True\n"
    + "    torch.backends.cudnn.benchmark = False"
)

code = code.replace(
    seed_line,
    replacement
)


# ------------------------------------------------------------
# 5. Change report labels
# ------------------------------------------------------------

code = code.replace(
    "YELP TR40 TUNING RESULTS",
    "YELP HOMO TR40 TUNING RESULTS"
)

code = code.replace(
    "12-trial YelpChi TR40 tuning completed",
    "12-trial YelpChi BWGNN-Homo TR40 tuning completed"
)


# ------------------------------------------------------------
# 6. Save runner
# ------------------------------------------------------------

HOMO.write_text(
    code,
    encoding="utf-8"
)

print("Created:", HOMO, flush=True)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# 8. Execute
# ------------------------------------------------------------

print(
    "\n===== STEP 14 — STARTING HOMO TUNING =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(HOMO)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 14 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/yelp_tr40_homo_tune.py

===== STEP 14 — STARTING HOMO TUNING =====
START — loading YelpChi for TR40 tuning
TR40: 18381 | VAL: 9191
GPU: Tesla T4

TRIAL 01/12 | hidden=32 | order=2 | lr=0.001
Traceback (most recent call last):
  File "/kaggle/working/comp8851_bwgnn/adapters/yelp_tr40_homo_tune.py", line 349, in <module>
    torch.save(
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch/serialization.py", line 376, in save
    with _open_file_like(f, 'wb') as opened_file:
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch/serialization.py", line 230, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch/serialization.py", line 211, in __init__
    super(_open_file, self).__init__(open(name, mode))
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/w

RuntimeError: Step 14 failed with return code 1.

In [29]:
# ============================================================
# STEP 14 FIX — CREATE HOMO OUTPUT DIRECTORIES + RERUN
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

HOMO = WORK / "adapters/yelp_tr40_homo_tune.py"

# Create missing directories
(WORK / "checkpoints/yelp_homo").mkdir(
    parents=True,
    exist_ok=True
)

(WORK / "results/tuning_homo").mkdir(
    parents=True,
    exist_ok=True
)

print("PASS — Homo output directories created", flush=True)


# Runtime environment
ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


print(
    "\n===== STEP 14 — RESTARTING HOMO TUNING =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(HOMO)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 14 failed with return code {rc}."
    )

PASS — Homo output directories created

===== STEP 14 — RESTARTING HOMO TUNING =====
START — loading YelpChi for TR40 tuning
TR40: 18381 | VAL: 9191
GPU: Tesla T4

TRIAL 01/12 | hidden=32 | order=2 | lr=0.001
epoch=001 | loss=0.69404 | val_AUPRC=0.17107 | best=0.17107@1
epoch=010 | loss=0.69133 | val_AUPRC=0.22907 | best=0.22907@10
epoch=020 | loss=0.68586 | val_AUPRC=0.25096 | best=0.25096@20
epoch=030 | loss=0.66565 | val_AUPRC=0.26750 | best=0.26750@30
epoch=040 | loss=0.62599 | val_AUPRC=0.28922 | best=0.28922@40
epoch=050 | loss=0.59415 | val_AUPRC=0.33228 | best=0.33228@50
epoch=060 | loss=0.57303 | val_AUPRC=0.37281 | best=0.37281@60
epoch=070 | loss=0.56730 | val_AUPRC=0.38196 | best=0.38196@70
epoch=080 | loss=0.56277 | val_AUPRC=0.38987 | best=0.38987@80
epoch=090 | loss=0.55743 | val_AUPRC=0.40144 | best=0.40144@90
epoch=100 | loss=0.55148 | val_AUPRC=0.41336 | best=0.41336@100
TRIAL 01 DONE | best val AUPRC=0.413357 at epoch 100

TRIAL 02/12 | hidden=32 | order=2 | lr=0.005

## Step 15 — Final YelpChi BWGNN-Homo Unified Runs

The winning BWGNN-Homo configuration is frozen:

- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005

Final evaluation uses:

- TR40, TR30, TR20, TR10
- Seeds 2, 42, 72
- Maximum 100 epochs
- Patience 20
- Validation AUPRC checkpointing
- Validation-only threshold selection
- Test evaluation only after model and threshold selection

In [30]:
# ============================================================
# STEP 15 — FINAL YELPCHI BWGNN-HOMO RUNS
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

SOURCE = ADAPTER / "yelp_final_runs.py"
HOMO = ADAPTER / "yelp_homo_final_runs.py"

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

code = SOURCE.read_text(encoding="utf-8")


# ------------------------------------------------------------
# 1. Hetero -> Homo model
# ------------------------------------------------------------

if "from BWGNN_gpu import BWGNN_Hetero" not in code:
    raise RuntimeError("Hetero import not found.")

code = code.replace(
    "from BWGNN_gpu import BWGNN_Hetero",
    "from BWGNN_gpu import BWGNN",
    1
)

if "model = BWGNN_Hetero(" not in code:
    raise RuntimeError("Hetero model constructor not found.")

code = code.replace(
    "model = BWGNN_Hetero(",
    "model = BWGNN(",
    1
)


# ------------------------------------------------------------
# 2. Convert Yelp graph to Homo + self-loops
# ------------------------------------------------------------

graph_block = '''graph = dgl.heterograph(
    relations,
    num_nodes_dict={
        "review": len(labels)
    }
)
'''

if graph_block not in code:
    raise RuntimeError("Graph construction block not found.")

code = code.replace(
    graph_block,
    graph_block + '''
graph = dgl.to_homogeneous(graph)
graph = dgl.add_self_loop(graph)
''',
    1
)


# ------------------------------------------------------------
# 3. Freeze Homo winning learning rate
# ------------------------------------------------------------

if "LR = 0.01" not in code:
    raise RuntimeError("Original LR setting not found.")

code = code.replace(
    "LR = 0.01",
    "LR = 0.005",
    1
)


# ------------------------------------------------------------
# 4. Separate Homo result/checkpoint locations
# ------------------------------------------------------------

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/results/unified/yelp"',
    '"/kaggle/working/comp8851_bwgnn/results/unified/yelp_homo"'
)

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp_final"',
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp_homo_final"'
)

code = code.replace(
    '"yelp_unified_all_runs.csv"',
    '"yelp_homo_unified_all_runs.csv"'
)

code = code.replace(
    '"yelp_unified_all_runs.json"',
    '"yelp_homo_unified_all_runs.json"'
)

code = code.replace(
    '"dataset": "yelp"',
    '"dataset": "yelp_homo"'
)


# ------------------------------------------------------------
# 5. Deterministic seed settings
# ------------------------------------------------------------

seed_line = "torch.cuda.manual_seed_all(seed)"

if seed_line not in code:
    raise RuntimeError("Seed block not found.")

code = code.replace(
    seed_line,
    seed_line
    + "\n"
    + "        torch.backends.cudnn.deterministic = True\n"
    + "        torch.backends.cudnn.benchmark = False",
    1
)


# ------------------------------------------------------------
# 6. Update summary labels
# ------------------------------------------------------------

code = code.replace(
    "YELPCHI FINAL SUMMARY",
    "YELPCHI BWGNN-HOMO FINAL SUMMARY"
)

code = code.replace(
    "PASS — YelpChi TR40/TR30/TR20/TR10",
    "PASS — YelpChi BWGNN-Homo TR40/TR30/TR20/TR10"
)


# ------------------------------------------------------------
# 7. Save Homo runner
# ------------------------------------------------------------

HOMO.write_text(
    code,
    encoding="utf-8"
)

print("Created:", HOMO, flush=True)

# Make sure new directories exist
(WORK / "results/unified/yelp_homo").mkdir(
    parents=True,
    exist_ok=True
)

(WORK / "checkpoints/yelp_homo_final").mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 8. Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# 9. Execute 12 final Homo runs
# ------------------------------------------------------------

print(
    "\n===== STEP 15 — STARTING 12 FINAL YELPCHI HOMO RUNS =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(HOMO)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 15 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/yelp_homo_final_runs.py

===== STEP 15 — STARTING 12 FINAL YELPCHI HOMO RUNS =====
START — loading canonical YelpChi
GPU: Tesla T4
Visible GPUs: 1

RUN 01/12 | TR40 | seed=2
epoch=001 | loss=0.69135 | val_AUPRC=0.26341 | best=0.26341@1
epoch=020 | loss=0.56734 | val_AUPRC=0.39746 | best=0.39746@20
epoch=040 | loss=0.52970 | val_AUPRC=0.45698 | best=0.45698@40
epoch=060 | loss=0.48202 | val_AUPRC=0.48564 | best=0.48564@60
epoch=080 | loss=0.45211 | val_AUPRC=0.51076 | best=0.51076@80
epoch=100 | loss=0.43656 | val_AUPRC=0.52466 | best=0.52599@98
DONE | TR40 seed=2 | epoch=98 | val AUPRC=0.525988 | test AUPRC=0.527437 | test AUROC=0.836291 | Macro-F1=0.706258

RUN 02/12 | TR40 | seed=42
epoch=001 | loss=0.69439 | val_AUPRC=0.23579 | best=0.23579@1
epoch=020 | loss=0.57672 | val_AUPRC=0.37444 | best=0.37444@20
epoch=040 | loss=0.52619 | val_AUPRC=0.45095 | best=0.45222@39
epoch=060 | loss=0.49146 | val_AUPRC=0.48073 | best=0.48170@59
epoch=

## Step 16 — Correct Amazon Unified Nested Splits

Amazon contains 11,944 graph nodes, but the first 3,305 nodes are unlabeled.

For supervised benchmarking, only node IDs 3305–11943 are eligible:

- Labeled nodes: 8,639
- Normal: 7,818
- Fraud: 821
- Unlabeled/excluded: 3,305

The unified split uses:

- Split seed: 2
- Fixed validation and test sets
- TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40

The previous Amazon split created from all 11,944 nodes is overwritten.

In [31]:
# ============================================================
# STEP 16 — CORRECT AMAZON UNIFIED SPLITS
# EXCLUDE UNLABELED NODES 0–3304
# ============================================================

import numpy as np
import scipy.io as sio
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 2

AMAZON_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "amazon/Amazon.mat"
)

SPLIT_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/shared/splits"
)

OUTPUT = (
    SPLIT_ROOT /
    "amazon_seed2_nested_splits.npz"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Load labels
# ------------------------------------------------------------

mat = sio.loadmat(AMAZON_PATH)

labels = (
    np.asarray(mat["label"])
    .reshape(-1)
    .astype(int)
)

n_nodes = len(labels)

if n_nodes != 11944:
    raise RuntimeError(
        f"Unexpected Amazon node count: {n_nodes}"
    )


# ------------------------------------------------------------
# Amazon supervised eligible nodes
#
# IDs 0–3304 are unlabeled in the benchmark.
# ------------------------------------------------------------

eligible_ids = np.arange(
    3305,
    n_nodes
)

eligible_labels = labels[
    eligible_ids
]


normal_count = int(
    (eligible_labels == 0).sum()
)

fraud_count = int(
    (eligible_labels == 1).sum()
)


print("===== AMAZON LABELED SOURCE =====")

print(
    "Total graph nodes :",
    f"{n_nodes:,}"
)

print(
    "Excluded unlabeled:",
    f"{3305:,}"
)

print(
    "Eligible labeled  :",
    f"{len(eligible_ids):,}"
)

print(
    "Normal            :",
    f"{normal_count:,}"
)

print(
    "Fraud             :",
    f"{fraud_count:,}"
)


# Hard validation against canonical benchmark counts

if len(eligible_ids) != 8639:
    raise RuntimeError(
        "Expected 8,639 labeled Amazon nodes."
    )

if normal_count != 7818:
    raise RuntimeError(
        f"Expected 7,818 normal nodes, got {normal_count}."
    )

if fraud_count != 821:
    raise RuntimeError(
        f"Expected 821 fraud nodes, got {fraud_count}."
    )


# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining, test_ids = train_test_split(
    eligible_ids,
    test_size=0.40,
    stratify=labels[eligible_ids],
    random_state=SEED,
    shuffle=True
)


# ------------------------------------------------------------
# Fixed 20% validation
#
# remaining = 60%
# 1/3 of remaining = 20% overall
# ------------------------------------------------------------

train40, val_ids = train_test_split(
    remaining,
    test_size=(1 / 3),
    stratify=labels[remaining],
    random_state=SEED,
    shuffle=True
)


# ------------------------------------------------------------
# Nested TR30 / TR20 / TR10
# ------------------------------------------------------------

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=SEED,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2 / 3),
    stratify=labels[train30],
    random_state=SEED,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=SEED,
    shuffle=True
)


splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

assert not (
    set(splits["TR40"])
    & set(splits["val"])
)

assert not (
    set(splits["TR40"])
    & set(splits["test"])
)

assert not (
    set(splits["val"])
    & set(splits["test"])
)

# No unlabeled node may appear anywhere
for name, ids in splits.items():

    if np.any(ids < 3305):

        raise RuntimeError(
            f"Unlabeled Amazon node found in {name}."
        )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.savez_compressed(
    OUTPUT,
    **splits,
    seed=np.array([SEED]),
    total_graph_nodes=np.array([n_nodes]),
    excluded_unlabeled=np.array([3305]),
    eligible_labeled=np.array([len(eligible_ids)]),
    eligible_normal=np.array([normal_count]),
    eligible_fraud=np.array([fraud_count]),
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "\n===== CORRECT AMAZON UNIFIED SPLITS ====="
)

for key in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[key]

    y = labels[ids]

    normal = int(
        (y == 0).sum()
    )

    fraud = int(
        (y == 1).sum()
    )

    print(
        f"{key:<5} | "
        f"N={len(ids):>5,} | "
        f"normal={normal:>5,} | "
        f"fraud={fraud:>3,}"
    )


print(
    "\nNested training sets: PASS"
)

print(
    "Unlabeled nodes excluded: PASS"
)

print(
    "Saved:",
    OUTPUT
)

print(
    "\n===== STEP 16 GATE ====="
)

print(
    "PASS — Amazon unified splits rebuilt "
    "using labeled nodes only."
)

===== AMAZON LABELED SOURCE =====
Total graph nodes : 11,944
Excluded unlabeled: 3,305
Eligible labeled  : 8,639
Normal            : 7,818
Fraud             : 821

===== CORRECT AMAZON UNIFIED SPLITS =====
TR40  | N=3,455 | normal=3,126 | fraud=329
TR30  | N=2,591 | normal=2,344 | fraud=247
TR20  | N=1,727 | normal=1,562 | fraud=165
TR10  | N=  863 | normal=  781 | fraud= 82
val   | N=1,728 | normal=1,564 | fraud=164
test  | N=3,456 | normal=3,128 | fraud=328

Nested training sets: PASS
Unlabeled nodes excluded: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/amazon_seed2_nested_splits.npz

===== STEP 16 GATE =====
PASS — Amazon unified splits rebuilt using labeled nodes only.


## Step 17 — Amazon BWGNN-Homo TR40 Smoke Test

This is a one-epoch smoke test using the corrected Amazon unified split.

Configuration:

- Eligible supervised nodes: IDs 3305–11943
- Training split: TR40
- Fixed validation and test sets
- Split seed: 2
- Training seed: 2
- Model: BWGNN-Homo
- GPU: Tesla T4, GPU 0 only
- Epochs: 1

The full graph is retained for message passing, while only labeled eligible
nodes are used for train/validation/test supervision.

In [32]:
# ============================================================
# STEP 17 — AMAZON BWGNN-HOMO 1-EPOCH SMOKE TEST
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

SOURCE = ADAPTER / "yelp_tr40_homo_smoke.py"
AMAZON = ADAPTER / "amazon_tr40_homo_smoke.py"

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

code = SOURCE.read_text(encoding="utf-8")


# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

code = code.replace(
    "yelp-chi/YelpChi.mat",
    "amazon/Amazon.mat"
)

code = code.replace(
    "yelp_seed2_nested_splits.npz",
    "amazon_seed2_nested_splits.npz"
)


# ------------------------------------------------------------
# Amazon relations
# ------------------------------------------------------------

code = code.replace(
    '"net_rsr"',
    '"net_upu"'
)

code = code.replace(
    '"net_rtr"',
    '"net_usu"'
)

code = code.replace(
    '"net_rur"',
    '"net_uvu"'
)


# ------------------------------------------------------------
# Labels / display names
# ------------------------------------------------------------

code = code.replace(
    "YelpChi",
    "Amazon"
)

code = code.replace(
    '"review"',
    '"user"'
)


# ------------------------------------------------------------
# Add Amazon structural gates
# ------------------------------------------------------------

marker = '''print(
    "Labels     :",
    tuple(labels.shape),
    flush=True
)
'''

check_block = marker + '''

# Amazon canonical structural verification
assert graph.num_nodes() == 11944
assert graph.num_edges() == 9569592
assert tuple(features.shape) == (11944, 25)
assert len(labels) == 11944

print(
    "Amazon structural gate: PASS",
    flush=True
)
'''

if marker not in code:
    raise RuntimeError(
        "Could not locate structural-output block."
    )

code = code.replace(
    marker,
    check_block,
    1
)


# ------------------------------------------------------------
# Correct gate label
# ------------------------------------------------------------

code = code.replace(
    "===== STEP 10 GATE =====",
    "===== STEP 17 GATE ====="
)

code = code.replace(
    "PASS — canonical Amazon TR40 unified "
    "1-epoch GPU smoke test completed.",
    "PASS — Amazon BWGNN-Homo TR40 "
    "1-epoch GPU smoke test completed."
)


# ------------------------------------------------------------
# Save Amazon runner
# ------------------------------------------------------------

AMAZON.write_text(
    code,
    encoding="utf-8"
)

print(
    "Created:",
    AMAZON,
    flush=True
)


# ------------------------------------------------------------
# Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

print(
    "\n===== STEP 17 — AMAZON HOMO SMOKE =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(AMAZON)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 17 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/amazon_tr40_homo_smoke.py

===== STEP 17 — AMAZON HOMO SMOKE =====
START — loading canonical Amazon
Nodes      : 11944
Edges      : 9569592
Relations  : [('_N', '_E', '_N')]
Features   : (11944, 25)
Labels     : (11944,)
Amazon structural gate: PASS
TR40 / VAL / TEST: 3455 1728 3456
GPU        : Tesla T4
Visible GPU: 1
Train normal/fraud: 3126 329
Fraud class weight: 9.501520

START — TR40 epoch 1

===== UNIFIED VALIDATION =====
Loss          : 1.170653
Val AUPRC     : 0.291033
Val threshold : 0.03
Val Macro-F1  : 0.491841
Val fraud REC : 0.024390

===== UNIFIED TEST =====
AUPRC     : 0.323026
AUROC     : 0.831003
Macro-F1  : 0.504657
Fraud REC : 0.036585

===== STEP 17 GATE =====
PASS — canonical Amazon TR40 unified 1-epoch GPU smoke test completed.


## Step 18 — Amazon BWGNN-Homo TR40 Tuning

BWGNN-Homo is tuned on the corrected Amazon TR40 split.

Protocol:

- Training seed: 2
- Maximum trials: 12
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint criterion: validation AUPRC
- Test set is not accessed during tuning

Search space:

- Hidden dimension: 32, 64
- Wavelet order: 2, 3
- Learning rate: 0.001, 0.005, 0.01

In [33]:
# ============================================================
# STEP 18 — AMAZON BWGNN-HOMO TR40 TUNING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

SOURCE = ADAPTER / "yelp_tr40_homo_tune.py"
AMAZON = ADAPTER / "amazon_tr40_homo_tune.py"

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

code = SOURCE.read_text(encoding="utf-8")


# ------------------------------------------------------------
# 1. Dataset paths
# ------------------------------------------------------------

code = code.replace(
    "yelp-chi/YelpChi.mat",
    "amazon/Amazon.mat"
)

code = code.replace(
    "yelp_seed2_nested_splits.npz",
    "amazon_seed2_nested_splits.npz"
)


# ------------------------------------------------------------
# 2. Amazon relation names
# ------------------------------------------------------------

code = code.replace(
    '"net_rsr"',
    '"net_upu"'
)

code = code.replace(
    '"net_rtr"',
    '"net_usu"'
)

code = code.replace(
    '"net_rur"',
    '"net_uvu"'
)


# ------------------------------------------------------------
# 3. Node type
# ------------------------------------------------------------

code = code.replace(
    '"review"',
    '"user"'
)


# ------------------------------------------------------------
# 4. Separate Amazon output locations
# ------------------------------------------------------------

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp_homo"',
    '"/kaggle/working/comp8851_bwgnn/checkpoints/amazon_homo"'
)

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/results/tuning_homo"',
    '"/kaggle/working/comp8851_bwgnn/results/tuning_amazon_homo"'
)

code = code.replace(
    '"yelp_homo_tr40_seed2_tuning.csv"',
    '"amazon_homo_tr40_seed2_tuning.csv"'
)

code = code.replace(
    '"yelp_homo_best_config.json"',
    '"amazon_homo_best_config.json"'
)


# ------------------------------------------------------------
# 5. Output labels
# ------------------------------------------------------------

code = code.replace(
    "YelpChi",
    "Amazon"
)

code = code.replace(
    "YELP HOMO TR40 TUNING RESULTS",
    "AMAZON HOMO TR40 TUNING RESULTS"
)

code = code.replace(
    "12-trial Amazon BWGNN-Homo TR40 tuning completed",
    "12-trial Amazon BWGNN-Homo TR40 tuning completed"
)


# ------------------------------------------------------------
# 6. Save runner
# ------------------------------------------------------------

AMAZON.write_text(
    code,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Ensure output directories exist
# ------------------------------------------------------------

(WORK / "checkpoints/amazon_homo").mkdir(
    parents=True,
    exist_ok=True
)

(WORK / "results/tuning_amazon_homo").mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Created:",
    AMAZON,
    flush=True
)


# ------------------------------------------------------------
# 8. Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# 9. Execute
# ------------------------------------------------------------

print(
    "\n===== STEP 18 — STARTING AMAZON HOMO TUNING =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(AMAZON)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 18 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/amazon_tr40_homo_tune.py

===== STEP 18 — STARTING AMAZON HOMO TUNING =====
START — loading Amazon for TR40 tuning
TR40: 3455 | VAL: 1728
GPU: Tesla T4

TRIAL 01/12 | hidden=32 | order=2 | lr=0.001
epoch=001 | loss=1.52654 | val_AUPRC=0.20576 | best=0.20576@1
epoch=010 | loss=0.61912 | val_AUPRC=0.27300 | best=0.57750@6
epoch=020 | loss=0.47914 | val_AUPRC=0.79414 | best=0.79414@20
epoch=030 | loss=0.38750 | val_AUPRC=0.83629 | best=0.83629@30
epoch=040 | loss=0.31897 | val_AUPRC=0.86611 | best=0.86611@40
epoch=050 | loss=0.27928 | val_AUPRC=0.88062 | best=0.88062@50
epoch=060 | loss=0.25731 | val_AUPRC=0.88458 | best=0.88508@59
epoch=070 | loss=0.24087 | val_AUPRC=0.88491 | best=0.88557@67
epoch=080 | loss=0.22838 | val_AUPRC=0.88551 | best=0.88625@76
epoch=090 | loss=0.21782 | val_AUPRC=0.88636 | best=0.88636@90
epoch=100 | loss=0.20948 | val_AUPRC=0.88487 | best=0.88639@91
TRIAL 01 DONE | best val AUPRC=0.886389 at epoch 91

TRIAL 02/

## Step 19 — Final Amazon BWGNN-Homo Unified Runs

The winning Amazon configuration is frozen:

- Hidden dimension: 32
- Wavelet order: 2
- Learning rate: 0.01

Final evaluation:

- TR40, TR30, TR20, TR10
- Seeds: 2, 42, 72
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint selection: validation AUPRC
- Threshold selection: validation set only
- Test evaluation only after checkpoint and threshold selection

In [34]:
# ============================================================
# STEP 19 — FINAL AMAZON BWGNN-HOMO RUNS
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

SOURCE = ADAPTER / "yelp_homo_final_runs.py"
AMAZON = ADAPTER / "amazon_homo_final_runs.py"

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

code = SOURCE.read_text(encoding="utf-8")


# ------------------------------------------------------------
# 1. Amazon dataset + split
# ------------------------------------------------------------

code = code.replace(
    "yelp-chi/YelpChi.mat",
    "amazon/Amazon.mat"
)

code = code.replace(
    "yelp_seed2_nested_splits.npz",
    "amazon_seed2_nested_splits.npz"
)


# ------------------------------------------------------------
# 2. Amazon relations
# ------------------------------------------------------------

code = code.replace(
    '"net_rsr"',
    '"net_upu"'
)

code = code.replace(
    '"net_rtr"',
    '"net_usu"'
)

code = code.replace(
    '"net_rur"',
    '"net_uvu"'
)

code = code.replace(
    '"review"',
    '"user"'
)


# ------------------------------------------------------------
# 3. Freeze Amazon winning configuration
# ------------------------------------------------------------

code = code.replace(
    "HIDDEN = 64",
    "HIDDEN = 32",
    1
)

code = code.replace(
    "ORDER = 3",
    "ORDER = 2",
    1
)

code = code.replace(
    "LR = 0.005",
    "LR = 0.01",
    1
)


# ------------------------------------------------------------
# 4. Amazon output locations
# ------------------------------------------------------------

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/results/unified/yelp_homo"',
    '"/kaggle/working/comp8851_bwgnn/results/unified/amazon_homo"'
)

code = code.replace(
    '"/kaggle/working/comp8851_bwgnn/checkpoints/yelp_homo_final"',
    '"/kaggle/working/comp8851_bwgnn/checkpoints/amazon_homo_final"'
)

code = code.replace(
    '"yelp_homo_unified_all_runs.csv"',
    '"amazon_homo_unified_all_runs.csv"'
)

code = code.replace(
    '"yelp_homo_unified_all_runs.json"',
    '"amazon_homo_unified_all_runs.json"'
)

code = code.replace(
    '"dataset": "yelp_homo"',
    '"dataset": "amazon_homo"'
)


# ------------------------------------------------------------
# 5. Output labels
# ------------------------------------------------------------

code = code.replace(
    "YELPCHI BWGNN-HOMO FINAL SUMMARY",
    "AMAZON BWGNN-HOMO FINAL SUMMARY"
)

code = code.replace(
    "YelpChi BWGNN-Homo",
    "Amazon BWGNN-Homo"
)


# ------------------------------------------------------------
# 6. Save runner
# ------------------------------------------------------------

AMAZON.write_text(
    code,
    encoding="utf-8"
)

(WORK / "results/unified/amazon_homo").mkdir(
    parents=True,
    exist_ok=True
)

(WORK / "checkpoints/amazon_homo_final").mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Created:",
    AMAZON,
    flush=True
)


# ------------------------------------------------------------
# 7. Runtime
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ------------------------------------------------------------
# 8. Execute
# ------------------------------------------------------------

print(
    "\n===== STEP 19 — STARTING 12 FINAL AMAZON RUNS =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(AMAZON)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 19 failed with return code {rc}."
    )

Created: /kaggle/working/comp8851_bwgnn/adapters/amazon_homo_final_runs.py

===== STEP 19 — STARTING 12 FINAL AMAZON RUNS =====
START — loading canonical YelpChi
GPU: Tesla T4
Visible GPUs: 1

RUN 01/12 | TR40 | seed=2
epoch=001 | loss=1.52654 | val_AUPRC=0.07612 | best=0.07612@1
epoch=020 | loss=0.28472 | val_AUPRC=0.86993 | best=0.86993@20
epoch=040 | loss=0.21923 | val_AUPRC=0.89246 | best=0.89277@39
epoch=060 | loss=0.18000 | val_AUPRC=0.89728 | best=0.89728@60
epoch=080 | loss=0.14758 | val_AUPRC=0.89747 | best=0.90879@78
epoch=098 | loss=0.13011 | val_AUPRC=0.90365 | best=0.90879@78
DONE | TR40 seed=2 | epoch=78 | val AUPRC=0.908785 | test AUPRC=0.901418 | test AUROC=0.983758 | Macro-F1=0.919264

RUN 02/12 | TR40 | seed=42
epoch=001 | loss=0.94524 | val_AUPRC=0.55615 | best=0.55615@1
epoch=020 | loss=0.39045 | val_AUPRC=0.85160 | best=0.86453@15
epoch=040 | loss=0.25190 | val_AUPRC=0.88843 | best=0.89048@39
epoch=060 | loss=0.20868 | val_AUPRC=0.89513 | best=0.89563@51
epoch=071 

## Step 20 — T-Finance BWGNN TR40 Smoke Test

This one-epoch smoke test uses the current canonical T-Finance DGL graph and
the fixed unified split.

Configuration:

- Nodes: 39,357
- Stored edges: 42,445,086
- Features: 10
- Current canonical labels: 37,553 normal / 1,804 fraud
- Training split: TR40
- Split seed: 2
- Training seed: 2
- Model: BWGNN
- GPU: Tesla T4, GPU 0 only
- Epochs: 1

T-Finance is loaded directly from its DGL graph file. No labels are modified
and no self-loops are added.

In [35]:
# ============================================================
# STEP 20 — T-FINANCE BWGNN TR40 1-EPOCH SMOKE TEST
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

RUNNER = ADAPTER / "tfinance_tr40_smoke.py"


RUNNER.write_text(r'''
import sys
import random
import numpy as np

import torch
import torch.nn.functional as F
from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


# ============================================================
# SETTINGS
# ============================================================

SEED = 2
DEVICE = torch.device("cuda:0")

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "tfinance_seed2_nested_splits.npz"
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# LOAD T-FINANCE EXACTLY AS STORED
# ============================================================

print(
    "START — loading canonical T-Finance",
    flush=True
)

graphs, _ = load_graphs(DATA_PATH)

graph = graphs[0]

features = graph.ndata["feature"].float()

raw_labels = graph.ndata["label"]

# Same conversion used by frozen BWGNN dataset.py
if raw_labels.ndim == 2:
    labels = raw_labels.argmax(1).long()
else:
    labels = raw_labels.long().squeeze(-1)


# ============================================================
# STRUCTURAL GATE
# ============================================================

print(
    "Nodes      :",
    graph.num_nodes(),
    flush=True
)

print(
    "Edges      :",
    graph.num_edges(),
    flush=True
)

print(
    "Features   :",
    tuple(features.shape),
    flush=True
)

print(
    "Labels     :",
    tuple(labels.shape),
    flush=True
)

normal_total = int(
    (labels == 0).sum().item()
)

fraud_total = int(
    (labels == 1).sum().item()
)

print(
    "Normal/Fraud:",
    normal_total,
    fraud_total,
    flush=True
)


assert graph.num_nodes() == 39357
assert graph.num_edges() == 42445086
assert tuple(features.shape) == (39357, 10)
assert normal_total == 37553
assert fraud_total == 1804

print(
    "T-Finance structural gate: PASS",
    flush=True
)


# ============================================================
# UNIFIED SPLITS
# ============================================================

split = np.load(SPLIT_PATH)

train_ids = torch.tensor(
    split["TR40"],
    dtype=torch.long
)

val_ids = torch.tensor(
    split["val"],
    dtype=torch.long
)

test_ids = torch.tensor(
    split["test"],
    dtype=torch.long
)


print(
    "TR40 / VAL / TEST:",
    len(train_ids),
    len(val_ids),
    len(test_ids),
    flush=True
)


# ============================================================
# GPU 0
# ============================================================

graph = graph.to(DEVICE)

features = features.to(DEVICE)
labels = labels.to(DEVICE)

train_ids = train_ids.to(DEVICE)
val_ids = val_ids.to(DEVICE)
test_ids = test_ids.to(DEVICE)


print(
    "GPU        :",
    torch.cuda.get_device_name(0),
    flush=True
)

print(
    "Visible GPU:",
    torch.cuda.device_count(),
    flush=True
)


# ============================================================
# MODEL
# ============================================================

model = BWGNN(
    in_feats=features.shape[1],
    h_feats=64,
    num_classes=2,
    graph=graph,
    d=2
).to(DEVICE)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    betas=(0.9, 0.999),
    eps=1e-8
)


# ============================================================
# TRAIN-ONLY CLASS WEIGHT
# ============================================================

train_labels = labels[train_ids]

normal = int(
    (train_labels == 0).sum().item()
)

fraud = int(
    (train_labels == 1).sum().item()
)

class_weight = torch.tensor(
    [
        1.0,
        normal / fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)


print(
    "Train normal/fraud:",
    normal,
    fraud,
    flush=True
)

print(
    "Fraud class weight:",
    f"{normal / fraud:.6f}",
    flush=True
)


# ============================================================
# ONE EPOCH
# ============================================================

print(
    "\nSTART — TR40 epoch 1",
    flush=True
)

model.train()

logits = model(features)

loss = F.cross_entropy(
    logits[train_ids],
    labels[train_ids],
    weight=class_weight
)

optimizer.zero_grad()
loss.backward()
optimizer.step()


# ============================================================
# VALIDATION
# ============================================================

model.eval()

with torch.no_grad():

    logits = model(features)

    probabilities = torch.softmax(
        logits,
        dim=1
    )[:, 1]


val_y = (
    labels[val_ids]
    .cpu()
    .numpy()
)

val_prob = (
    probabilities[val_ids]
    .cpu()
    .numpy()
)


val_auprc = average_precision_score(
    val_y,
    val_prob
)


# ------------------------------------------------------------
# Validation threshold
# ------------------------------------------------------------

candidates = []

for threshold in np.arange(
    0.01,
    1.00,
    0.01
):

    pred = (
        val_prob >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        val_y,
        pred,
        average="macro"
    )

    fraud_recall = recall_score(
        val_y,
        pred,
        zero_division=0
    )

    candidates.append(
        (
            macro_f1,
            fraud_recall,
            -abs(float(threshold) - 0.50),
            float(threshold)
        )
    )


best = max(candidates)

threshold = best[3]


print(
    "\n===== UNIFIED VALIDATION =====",
    flush=True
)

print(
    f"Loss          : {loss.item():.6f}",
    flush=True
)

print(
    f"Val AUPRC     : {val_auprc:.6f}",
    flush=True
)

print(
    f"Val threshold : {threshold:.2f}",
    flush=True
)

print(
    f"Val Macro-F1  : {best[0]:.6f}",
    flush=True
)

print(
    f"Val fraud REC : {best[1]:.6f}",
    flush=True
)


# ============================================================
# TEST — SMOKE DIAGNOSTIC ONLY
# ============================================================

test_y = (
    labels[test_ids]
    .cpu()
    .numpy()
)

test_prob = (
    probabilities[test_ids]
    .cpu()
    .numpy()
)

test_pred = (
    test_prob >= threshold
).astype(int)


print(
    "\n===== UNIFIED TEST =====",
    flush=True
)

print(
    f"AUPRC     : "
    f"{average_precision_score(test_y, test_prob):.6f}",
    flush=True
)

print(
    f"AUROC     : "
    f"{roc_auc_score(test_y, test_prob):.6f}",
    flush=True
)

print(
    f"Macro-F1  : "
    f"{f1_score(test_y, test_pred, average='macro'):.6f}",
    flush=True
)

print(
    f"Fraud REC : "
    f"{recall_score(test_y, test_pred, zero_division=0):.6f}",
    flush=True
)


print(
    "\n===== STEP 20 GATE =====",
    flush=True
)

print(
    "PASS — T-Finance BWGNN TR40 "
    "1-epoch GPU smoke test completed.",
    flush=True
)
''',
encoding="utf-8"
)


# ============================================================
# RUNTIME
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ============================================================
# RUN
# ============================================================

print(
    "===== STEP 20 — T-FINANCE SMOKE =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 20 failed with return code {rc}."
    )

===== STEP 20 — T-FINANCE SMOKE =====
START — loading canonical T-Finance
Nodes      : 39357
Edges      : 42445086
Features   : (39357, 10)
Labels     : (39357,)
Normal/Fraud: 37553 1804
T-Finance structural gate: PASS
TR40 / VAL / TEST: 15742 7872 15743
GPU        : Tesla T4
Visible GPU: 1
Train normal/fraud: 15021 721
Fraud class weight: 20.833564

START — TR40 epoch 1

===== UNIFIED VALIDATION =====
Loss          : 32.363659
Val AUPRC     : 0.107551
Val threshold : 0.50
Val Macro-F1  : 0.488266
Val fraud REC : 0.000000

===== UNIFIED TEST =====
AUPRC     : 0.104994
AUROC     : 0.549562
Macro-F1  : 0.488266
Fraud REC : 0.000000

===== STEP 20 GATE =====
PASS — T-Finance BWGNN TR40 1-epoch GPU smoke test completed.


## Step 21 — T-Finance BWGNN TR40 Tuning

BWGNN is tuned on the fixed T-Finance TR40 split.

Protocol:

- Training seed: 2
- Maximum trials: 12
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint criterion: validation AUPRC
- Test set is not accessed during tuning

Search space:

- Hidden dimension: 32, 64
- Wavelet order: 2, 3
- Learning rate: 0.001, 0.005, 0.01

The current canonical T-Finance labels are retained unchanged:
37,553 normal and 1,804 fraud.

In [36]:
# ============================================================
# STEP 21 — T-FINANCE BWGNN TR40 TUNING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
ADAPTER = WORK / "adapters"

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

RUNNER = ADAPTER / "tfinance_tr40_tune.py"

CHECKPOINT_DIR = WORK / "checkpoints/tfinance"
RESULT_DIR = WORK / "results/tuning_tfinance"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


RUNNER.write_text(r'''
import sys
import gc
import csv
import json
import random
import contextlib
import io

from pathlib import Path

import numpy as np

import torch
import torch.nn.functional as F

from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


# ============================================================
# SETTINGS
# ============================================================

SEED = 2

MAX_EPOCHS = 100
PATIENCE = 20

DEVICE = torch.device("cuda:0")

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "tfinance_seed2_nested_splits.npz"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/checkpoints/tfinance"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/results/tuning_tfinance"
)


# ============================================================
# LOAD DATA ONCE
# ============================================================

print(
    "START — loading T-Finance for TR40 tuning",
    flush=True
)

graphs, _ = load_graphs(DATA_PATH)

graph = graphs[0]

features = graph.ndata["feature"].float()

raw_labels = graph.ndata["label"]

if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .long()
    )

else:

    labels = (
        raw_labels
        .long()
        .squeeze(-1)
    )


# ============================================================
# STRUCTURAL CHECK
# ============================================================

assert graph.num_nodes() == 39357
assert graph.num_edges() == 42445086
assert tuple(features.shape) == (39357, 10)

assert int((labels == 0).sum()) == 37553
assert int((labels == 1).sum()) == 1804


# ============================================================
# LOAD TR40 + VALIDATION
# ============================================================

split = np.load(SPLIT_PATH)

train_ids = torch.tensor(
    split["TR40"],
    dtype=torch.long
)

val_ids = torch.tensor(
    split["val"],
    dtype=torch.long
)


# ============================================================
# GPU
# ============================================================

graph = graph.to(DEVICE)

features = features.to(DEVICE)
labels = labels.to(DEVICE)

train_ids = train_ids.to(DEVICE)
val_ids = val_ids.to(DEVICE)


print(
    "TR40:",
    len(train_ids),
    "| VAL:",
    len(val_ids),
    flush=True
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
    flush=True
)


# ============================================================
# TRAIN-ONLY CLASS WEIGHT
# ============================================================

train_labels = labels[train_ids]

normal = int(
    (train_labels == 0)
    .sum()
    .item()
)

fraud = int(
    (train_labels == 1)
    .sum()
    .item()
)

class_weight = torch.tensor(
    [
        1.0,
        normal / fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)


print(
    "Train normal/fraud:",
    normal,
    fraud,
    flush=True
)

print(
    "Fraud class weight:",
    f"{normal / fraud:.6f}",
    flush=True
)


# ============================================================
# EXACT 12-TRIAL SEARCH
# ============================================================

trials = []

trial_number = 0

for hidden in [32, 64]:

    for order in [2, 3]:

        for lr in [
            0.001,
            0.005,
            0.01
        ]:

            trial_number += 1

            trials.append({
                "trial": trial_number,
                "hidden": hidden,
                "order": order,
                "lr": lr
            })


results = []


# ============================================================
# TUNING
# ============================================================

for cfg in trials:

    trial = cfg["trial"]

    print(
        "\n========================================",
        flush=True
    )

    print(
        f"TRIAL {trial:02d}/12 | "
        f"hidden={cfg['hidden']} | "
        f"order={cfg['order']} | "
        f"lr={cfg['lr']}",
        flush=True
    )


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        model = BWGNN(
            in_feats=features.shape[1],
            h_feats=cfg["hidden"],
            num_classes=2,
            graph=graph,
            d=cfg["order"]
        ).to(DEVICE)


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        betas=(0.9, 0.999),
        eps=1e-8
    )


    best_val_auprc = -1.0
    best_epoch = -1
    no_improvement = 0

    checkpoint = (
        CHECKPOINT_DIR /
        f"trial_{trial:02d}.pt"
    )


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        # -----------------------------
        # TRAIN
        # -----------------------------

        model.train()

        logits = model(features)

        loss = F.cross_entropy(
            logits[train_ids],
            labels[train_ids],
            weight=class_weight
        )

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite loss in trial "
                f"{trial} epoch {epoch}"
            )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        # -----------------------------
        # VALIDATION
        # -----------------------------

        model.eval()

        with torch.no_grad():

            logits = model(features)

            val_prob = torch.softmax(
                logits,
                dim=1
            )[val_ids, 1]


        val_y = (
            labels[val_ids]
            .detach()
            .cpu()
            .numpy()
        )

        val_prob_np = (
            val_prob
            .detach()
            .cpu()
            .numpy()
        )


        if not np.isfinite(
            val_prob_np
        ).all():

            raise RuntimeError(
                f"Non-finite validation probabilities "
                f"in trial {trial} epoch {epoch}"
            )


        val_auprc = average_precision_score(
            val_y,
            val_prob_np
        )


        # -----------------------------
        # CHECKPOINT
        # -----------------------------

        if val_auprc > best_val_auprc:

            best_val_auprc = val_auprc
            best_epoch = epoch

            no_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint
            )

        else:

            no_improvement += 1


        # -----------------------------
        # Progress
        # -----------------------------

        if (
            epoch == 1
            or epoch % 10 == 0
            or no_improvement >= PATIENCE
        ):

            print(
                f"epoch={epoch:03d} | "
                f"loss={loss.item():.5f} | "
                f"val_AUPRC={val_auprc:.5f} | "
                f"best={best_val_auprc:.5f}"
                f"@{best_epoch}",
                flush=True
            )


        # -----------------------------
        # Early stop
        # -----------------------------

        if no_improvement >= PATIENCE:

            print(
                f"Early stop at epoch {epoch}",
                flush=True
            )

            break


    # ========================================================
    # RECORD TRIAL
    # ========================================================

    results.append({
        "trial": trial,
        "hidden": cfg["hidden"],
        "order": cfg["order"],
        "lr": cfg["lr"],
        "best_epoch": best_epoch,
        "best_val_auprc": best_val_auprc,
        "checkpoint": str(checkpoint)
    })


    print(
        f"TRIAL {trial:02d} DONE | "
        f"best val AUPRC="
        f"{best_val_auprc:.6f} "
        f"at epoch {best_epoch}",
        flush=True
    )


    del model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# SELECT WINNER — VALIDATION AUPRC ONLY
# ============================================================

winner = max(
    results,
    key=lambda x:
        x["best_val_auprc"]
)


# ============================================================
# SAVE RESULTS
# ============================================================

csv_path = (
    RESULT_DIR /
    "tfinance_tr40_seed2_tuning.csv"
)

with csv_path.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys()
    )

    writer.writeheader()
    writer.writerows(results)


winner_path = (
    RESULT_DIR /
    "tfinance_best_config.json"
)

winner_path.write_text(
    json.dumps(
        winner,
        indent=2
    )
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n===== T-FINANCE TR40 TUNING RESULTS =====",
    flush=True
)

for r in sorted(
    results,
    key=lambda x:
        x["best_val_auprc"],
    reverse=True
):

    print(
        f"trial={r['trial']:02d} | "
        f"hidden={r['hidden']:>2} | "
        f"order={r['order']} | "
        f"lr={r['lr']:<5} | "
        f"epoch={r['best_epoch']:>3} | "
        f"val AUPRC="
        f"{r['best_val_auprc']:.6f}",
        flush=True
    )


print(
    "\n===== WINNING CONFIGURATION =====",
    flush=True
)

print(
    "Hidden dimension:",
    winner["hidden"],
    flush=True
)

print(
    "Wavelet order   :",
    winner["order"],
    flush=True
)

print(
    "Learning rate   :",
    winner["lr"],
    flush=True
)

print(
    "Best epoch      :",
    winner["best_epoch"],
    flush=True
)

print(
    "Best Val AUPRC  :",
    f"{winner['best_val_auprc']:.6f}",
    flush=True
)

print(
    "\nTEST SET ACCESSED DURING TUNING: NO",
    flush=True
)

print(
    "\n===== STEP 21 GATE =====",
    flush=True
)

print(
    "PASS — 12-trial T-Finance TR40 tuning "
    "completed using validation AUPRC only.",
    flush=True
)
''',
encoding="utf-8"
)


# ============================================================
# RUNTIME
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing if existing else "")
)


# ============================================================
# EXECUTE
# ============================================================

print(
    "===== STEP 21 — STARTING T-FINANCE TUNING =====",
    flush=True
)

process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

rc = process.wait()

if rc != 0:
    raise RuntimeError(
        f"Step 21 failed with return code {rc}."
    )

===== STEP 21 — STARTING T-FINANCE TUNING =====
START — loading T-Finance for TR40 tuning
TR40: 15742 | VAL: 7872
GPU: Tesla T4
Train normal/fraud: 15021 721
Fraud class weight: 20.833564

TRIAL 01/12 | hidden=32 | order=2 | lr=0.001
epoch=001 | loss=28.77529 | val_AUPRC=0.04201 | best=0.04201@1
epoch=010 | loss=4.42148 | val_AUPRC=0.08886 | best=0.27088@7
epoch=020 | loss=3.12364 | val_AUPRC=0.14873 | best=0.38032@15
epoch=030 | loss=1.87686 | val_AUPRC=0.30915 | best=0.41874@27
epoch=040 | loss=1.26564 | val_AUPRC=0.20739 | best=0.46310@35
epoch=050 | loss=0.86401 | val_AUPRC=0.31558 | best=0.50491@47
epoch=060 | loss=0.73988 | val_AUPRC=0.48817 | best=0.50491@47
epoch=070 | loss=0.61171 | val_AUPRC=0.51528 | best=0.53618@62
epoch=080 | loss=0.51995 | val_AUPRC=0.56528 | best=0.56528@80
epoch=090 | loss=0.47281 | val_AUPRC=0.58630 | best=0.61150@89
epoch=100 | loss=0.43101 | val_AUPRC=0.61166 | best=0.62339@97
TRIAL 01 DONE | best val AUPRC=0.623388 at epoch 97

TRIAL 02/12 | hidden=

## Step 22 — Final T-Finance BWGNN Unified Runs with Timing

The winning T-Finance configuration is frozen:

- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005

Final runs:

- TR40, TR30, TR20, TR10
- Training seeds: 2, 42, 72
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint criterion: validation AUPRC
- Threshold selection: validation Macro-F1
- Test evaluated only after model and threshold selection

Timing evidence is recorded for every run:

- Per-epoch training time
- Per-epoch validation time
- Total training time
- Mean and standard deviation of epoch training time
- Full-graph inference latency
- Peak GPU memory
- Dataset load/preprocessing time

Inference latency is measured as model forward-pass latency on the complete graph,
using 3 warm-up passes followed by 10 timed passes on Tesla T4 GPU 0.

In [1]:
# ============================================================
# RECOVERY CHECK BEFORE RECREATING STEP 22
# ============================================================

from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/comp8851_bwgnn")

checks = {
    "WORK directory":
        WORK.exists(),

    "BWGNN environment":
        (WORK / "envs/bwgnn-author/bin/python").exists(),

    "GPU-patched BWGNN":
        (WORK / "adapters/BWGNN_gpu.py").exists(),

    "T-Finance split":
        (WORK / "shared/splits/tfinance_seed2_nested_splits.npz").exists(),

    "Step 22 runner":
        (WORK / "adapters/tfinance_final_timed.py").exists(),

    "Step 21 best config":
        (WORK / "results/tuning_tfinance/tfinance_best_config.json").exists(),
}

print("===== RECOVERY CHECK =====")

for name, status in checks.items():
    print(f"{name:<25}: {status}")


print("\n===== OLD STEP 22 PROCESS =====")

proc = subprocess.run(
    [
        "bash",
        "-lc",
        "pgrep -af tfinance_final_timed.py || true"
    ],
    capture_output=True,
    text=True
)

print(
    proc.stdout.strip()
    if proc.stdout.strip()
    else "No old Step 22 process running."
)


print("\n===== EXISTING T-FINANCE FINAL FILES =====")

result_dir = WORK / "results/unified/tfinance"

if result_dir.exists():
    for p in sorted(result_dir.rglob("*")):
        if p.is_file():
            print(p)
else:
    print("No Step 22 result directory yet.")


print("\n===== GPU =====")

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.used,memory.total",
        "--format=csv,noheader"
    ]
)

===== RECOVERY CHECK =====
WORK directory           : False
BWGNN environment        : False
GPU-patched BWGNN        : False
T-Finance split          : False
Step 22 runner           : False
Step 21 best config      : False

===== OLD STEP 22 PROCESS =====
84 bash -lc pgrep -af tfinance_final_timed.py || true

===== EXISTING T-FINANCE FINAL FILES =====
No Step 22 result directory yet.

===== GPU =====
Tesla T4, 0 MiB, 15360 MiB
Tesla T4, 0 MiB, 15360 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv,noheader'], returncode=0)

In [2]:
# ============================================================
# STEP 22A — FRESH-SESSION MINIMAL RECOVERY FOR T-FINANCE
# ============================================================

import os
import re
import sys
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
REPO = Path("/kaggle/working/Rethinking-Anomaly-Detection")
ENV_DIR = WORK / "envs/bwgnn-author"
ADAPTER = WORK / "adapters"
SPLIT_DIR = WORK / "shared/splits"

TF_PATH = Path(
    "/kaggle/input/datasets/pathikahmed0007/tfinance/tfinance"
)

EXPECTED_COMMIT = (
    "de0631f039bbd19c1890b483cc01f1007f596af7"
)

for d in [
    WORK,
    ADAPTER,
    SPLIT_DIR,
    WORK / "results",
    WORK / "checkpoints",
]:
    d.mkdir(parents=True, exist_ok=True)


def run(cmd, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(
        list(map(str, cmd)),
        check=True,
        env=env
    )


# ============================================================
# 1. CHECK ATTACHED T-FINANCE DATASET
# ============================================================

print("===== 1. DATASET CHECK =====", flush=True)

if not TF_PATH.exists():
    raise FileNotFoundError(
        f"T-Finance is not attached at:\n{TF_PATH}"
    )

print("PASS — T-Finance attached:", TF_PATH, flush=True)


# ============================================================
# 2. CLONE EXACT BWGNN SOURCE
# ============================================================

print("\n===== 2. BWGNN SOURCE =====", flush=True)

if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git",
    "clone",
    "https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git",
    REPO
])

run([
    "git",
    "-C",
    REPO,
    "checkout",
    EXPECTED_COMMIT
])

commit = subprocess.check_output(
    [
        "git",
        "-C",
        REPO,
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()

assert commit == EXPECTED_COMMIT

print(
    "PASS — exact upstream commit:",
    commit,
    flush=True
)


# ============================================================
# 3. INSTALL UV
# ============================================================

print("\n===== 3. UV =====", flush=True)

if shutil.which("uv") is None:
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "uv==0.11.13"
    ])

UV = shutil.which("uv")

if UV is None:
    raise RuntimeError("uv installation failed.")

print("uv:", UV, flush=True)


# ============================================================
# 4. CREATE PYTHON 3.9 ENVIRONMENT
# ============================================================

print("\n===== 4. PYTHON 3.9 ENV =====", flush=True)

run([
    UV,
    "python",
    "install",
    "3.9.25"
])

if ENV_DIR.exists():
    shutil.rmtree(ENV_DIR)

run([
    UV,
    "venv",
    "--python",
    "3.9.25",
    ENV_DIR
])

PYTHON39 = ENV_DIR / "bin/python"


# ============================================================
# 5. INSTALL AUTHOR-COMPATIBLE PACKAGES
# ============================================================

print("\n===== 5. PACKAGES =====", flush=True)

run([
    UV,
    "pip",
    "install",
    "--python",
    PYTHON39,
    "torch==1.9.0+cu111",
    "--find-links",
    "https://download.pytorch.org/whl/torch_stable.html"
])

run([
    UV,
    "pip",
    "install",
    "--python",
    PYTHON39,
    "dgl-cu111==0.8.1",
    "--find-links",
    "https://data.dgl.ai/wheels/repo.html"
])

run([
    UV,
    "pip",
    "install",
    "--python",
    PYTHON39,

    "numpy==1.23.5",
    "scipy==1.9.3",
    "scikit-learn==1.1.3",
    "networkx==2.8.8",
    "sympy==1.10.1",
    "requests==2.28.1",
    "tqdm==4.64.1",
    "psutil==5.9.4",
    "typing_extensions==4.4.0",

    "nvidia-cuda-runtime-cu11==11.8.89",
    "nvidia-cublas-cu11==11.11.3.6",
    "nvidia-cusparse-cu11==11.7.5.86",
])


# ============================================================
# 6. CUDA LIBRARY PATH
# ============================================================

SITE = (
    ENV_DIR /
    "lib/python3.9/site-packages"
)

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

RUNTIME_ENV = os.environ.copy()

RUNTIME_ENV["CUDA_VISIBLE_DEVICES"] = "0"
RUNTIME_ENV["DGLBACKEND"] = "pytorch"
RUNTIME_ENV["PYTHONUNBUFFERED"] = "1"

existing_ld = RUNTIME_ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

RUNTIME_ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + (":" + existing_ld if existing_ld else "")
)


# ============================================================
# 7. VERIFY ENVIRONMENT + T4 GPU 0
# ============================================================

print("\n===== 6. ENVIRONMENT VERIFICATION =====", flush=True)

verification = r'''
import torch
import dgl

print("Python environment : PASS")
print("Torch              :", torch.__version__)
print("Torch CUDA         :", torch.version.cuda)
print("DGL                :", dgl.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("Visible GPUs       :", torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1

print("GPU 0              :", torch.cuda.get_device_name(0))

x = torch.tensor([1., 2., 3.], device="cuda:0")
print("CUDA tensor sum    :", x.sum().item())

g = dgl.graph(([0, 1], [1, 0])).to("cuda:0")
print("DGL CUDA graph     : PASS")
'''

run([
    PYTHON39,
    "-c",
    verification
], env=RUNTIME_ENV)


# ============================================================
# 8. RECREATE GPU-COMPATIBILITY COPY
# ============================================================

print("\n===== 7. BWGNN GPU PATCH =====", flush=True)

author_code = (
    REPO / "BWGNN.py"
).read_text(
    encoding="utf-8"
)

patched_code, patch_count = re.subn(
    r"torch\.zeros\(\[len\(in_feat\),\s*0\]\)",
    "torch.zeros([len(in_feat), 0], device=in_feat.device)",
    author_code
)

if patch_count != 4:
    raise RuntimeError(
        f"Expected 4 GPU allocation patches; got {patch_count}"
    )

PATCHED_MODEL = ADAPTER / "BWGNN_gpu.py"

PATCHED_MODEL.write_text(
    patched_code,
    encoding="utf-8"
)

print(
    "GPU allocation patches:",
    patch_count,
    flush=True
)

print(
    "Frozen author repository modified: NO",
    flush=True
)


# ============================================================
# 9. RECREATE EXACT T-FINANCE NESTED SPLIT
# ============================================================

print("\n===== 8. T-FINANCE SPLIT =====", flush=True)

SPLIT_PATH = (
    SPLIT_DIR /
    "tfinance_seed2_nested_splits.npz"
)

split_script = r'''
import sys
import numpy as np

from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

DATA_PATH = sys.argv[1]
OUTPUT = sys.argv[2]

SEED = 2

graphs, _ = load_graphs(DATA_PATH)
g = graphs[0]

raw = g.ndata["label"]

labels = (
    raw.argmax(1)
    .cpu()
    .numpy()
    .astype(int)
)

assert g.num_nodes() == 39357
assert g.num_edges() == 42445086
assert int((labels == 0).sum()) == 37553
assert int((labels == 1).sum()) == 1804

all_ids = np.arange(len(labels))

remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=SEED,
    shuffle=True
)

train40, val_ids = train_test_split(
    remaining,
    test_size=(1/3),
    stratify=labels[remaining],
    random_state=SEED,
    shuffle=True
)

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=SEED,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2/3),
    stratify=labels[train30],
    random_state=SEED,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=SEED,
    shuffle=True
)

splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}

assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

assert not (set(splits["TR40"]) & set(splits["val"]))
assert not (set(splits["TR40"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))

np.savez_compressed(
    OUTPUT,
    **splits,
    seed=np.array([SEED]),
    source_nodes=np.array([39357]),
    source_normal=np.array([37553]),
    source_fraud=np.array([1804]),
)

print("TR40:", len(train40))
print("TR30:", len(train30))
print("TR20:", len(train20))
print("TR10:", len(train10))
print("VAL :", len(val_ids))
print("TEST:", len(test_ids))

assert len(train40) == 15742
assert len(train30) == 11806
assert len(train20) == 7870
assert len(train10) == 3935
assert len(val_ids) == 7872
assert len(test_ids) == 15743

print("T-Finance nested split: PASS")
'''

run([
    PYTHON39,
    "-c",
    split_script,
    TF_PATH,
    SPLIT_PATH
], env=RUNTIME_ENV)


# ============================================================
# FINAL RECOVERY GATE
# ============================================================

print(
    "\n============================================",
    flush=True
)

print(
    "STEP 22A RECOVERY GATE: PASS",
    flush=True
)

print(
    "BWGNN source      : RESTORED",
    flush=True
)

print(
    "Python 3.9 env    : RESTORED",
    flush=True
)

print(
    "Torch/DGL CUDA    : RESTORED",
    flush=True
)

print(
    "BWGNN GPU adapter : RESTORED",
    flush=True
)

print(
    "T-Finance split   : RESTORED",
    flush=True
)

print(
    "\nFrozen T-Finance config from Step 21:",
    flush=True
)

print(
    "hidden=64 | order=3 | lr=0.005",
    flush=True
)

print(
    "\nReady to recreate Step 22 timed final runner.",
    flush=True
)

===== 1. DATASET CHECK =====
PASS — T-Finance attached: /kaggle/input/datasets/pathikahmed0007/tfinance/tfinance

===== 2. BWGNN SOURCE =====

$ git clone https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git /kaggle/working/Rethinking-Anomaly-Detection


Cloning into '/kaggle/working/Rethinking-Anomaly-Detection'...



$ git -C /kaggle/working/Rethinking-Anomaly-Detection checkout de0631f039bbd19c1890b483cc01f1007f596af7
PASS — exact upstream commit: de0631f039bbd19c1890b483cc01f1007f596af7

===== 3. UV =====
uv: /usr/local/bin/uv

===== 4. PYTHON 3.9 ENV =====

$ /usr/local/bin/uv python install 3.9.25


Note: switching to 'de0631f039bbd19c1890b483cc01f1007f596af7'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at de0631f Update readme.md



$ /usr/local/bin/uv venv --python 3.9.25 /kaggle/working/comp8851_bwgnn/envs/bwgnn-author


...


$ /usr/local/bin/uv pip install --python /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python numpy==1.23.5 scipy==1.9.3 scikit-learn==1.1.3 networkx==2.8.8 sympy==1.10.1 requests==2.28.1 tqdm==4.64.1 psutil==5.9.4 typing_extensions==4.4.0 nvidia-cuda-runtime-cu11==11.8.89 nvidia-cublas-cu11==11.11.3.6 nvidia-cusparse-cu11==11.7.5.86


Installed 10 packages in 508ms
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + dgl-cu111==0.8.1
 + idna==3.19
 + networkx==3.2.1
 + numpy==2.0.2
 + requests==2.32.5
 + scipy==1.13.1
 + tqdm==4.70.0
 + urllib3==2.6.3
Using Python 3.9.25 environment at: comp8851_bwgnn/envs/bwgnn-author
Resolved 19 packages in 673ms
Prepared 17 packages in 7.76s
Uninstalled 8 packages in 114ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.



===== 6. ENVIRONMENT VERIFICATION =====

$ /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -c 
import torch
import dgl

print("Python environment : PASS")
print("Torch              :", torch.__version__)
print("Torch CUDA         :", torch.version.cuda)
print("DGL                :", dgl.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("Visible GPUs       :", torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1

print("GPU 0              :", torch.cuda.get_device_name(0))

x = torch.tensor([1., 2., 3.], device="cuda:0")
print("CUDA tensor sum    :", x.sum().item())

g = dgl.graph(([0, 1], [1, 0])).to("cuda:0")
print("DGL CUDA graph     : PASS")



Installed 17 packages in 841ms
 - charset-normalizer==3.5.1
 + charset-normalizer==2.1.1
 + joblib==1.5.3
 + mpmath==1.4.1
 - networkx==3.2.1
 + networkx==2.8.8
 - numpy==2.0.2
 + numpy==1.23.5
 + nvidia-cublas-cu11==11.11.3.6
 + nvidia-cuda-runtime-cu11==11.8.89
 + nvidia-cusparse-cu11==11.7.5.86
 + psutil==5.9.4
 - requests==2.32.5
 + requests==2.28.1
 + scikit-learn==1.1.3
 - scipy==1.13.1
 + scipy==1.9.3
 + sympy==1.10.1
 + threadpoolctl==3.6.0
 - tqdm==4.70.0
 + tqdm==4.64.1
 - typing-extensions==4.16.0
 + typing-extensions==4.4.0
 - urllib3==2.6.3
 + urllib3==1.26.20


Python environment : PASS
Torch              : 1.9.0+cu111
Torch CUDA         : 11.1
DGL                : 0.8.1
CUDA available     : True
Visible GPUs       : 1
GPU 0              : Tesla T4
CUDA tensor sum    : 6.0
DGL CUDA graph     : PASS

===== 7. BWGNN GPU PATCH =====
GPU allocation patches: 4
Frozen author repository modified: NO

===== 8. T-FINANCE SPLIT =====

$ /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -c 
import sys
import numpy as np

from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

DATA_PATH = sys.argv[1]
OUTPUT = sys.argv[2]

SEED = 2

graphs, _ = load_graphs(DATA_PATH)
g = graphs[0]

raw = g.ndata["label"]

labels = (
    raw.argmax(1)
    .cpu()
    .numpy()
    .astype(int)
)

assert g.num_nodes() == 39357
assert g.num_edges() == 42445086
assert int((labels == 0).sum()) == 37553
assert int((labels == 1).sum()) == 1804

all_ids = np.arange(len(labels))

remaining, test_ids = train_test_split(
    all_ids,
    

## Step 22B — Final T-Finance BWGNN Unified Benchmark with Timing

Frozen configuration selected on TR40 validation data:

- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005

Final benchmark:

- TR40, TR30, TR20, TR10
- Training seeds: 2, 42, 72
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint selection: validation AUPRC
- Threshold selection: validation Macro-F1
- Test evaluation only after checkpoint and threshold selection

Timing recorded:

- Raw training time for every epoch
- Validation time for every epoch
- Total and mean training time
- Training-time standard deviation
- Full-graph inference latency
- Peak GPU memory
- Load/preprocessing time

Inference latency uses 3 warm-up passes followed by 10 measured full-graph
forward passes on Tesla T4 GPU 0.

The runner is restart-safe: completed split/seed runs are saved individually
and skipped if execution must later be resumed.

In [3]:
# ============================================================
# STEP 22B — RESTART-SAFE FINAL T-FINANCE TIMED BENCHMARK
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

ADAPTER = WORK / "adapters"
ADAPTER.mkdir(parents=True, exist_ok=True)

PYTHON39 = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

RUNNER = ADAPTER / "tfinance_final_timed_resumable.py"

RESULT_DIR = WORK / "results/unified/tfinance"
CHECKPOINT_DIR = WORK / "checkpoints/tfinance_final"
TIMING_DIR = RESULT_DIR / "epoch_times"
RUN_DIR = RESULT_DIR / "runs"

LOG_FILE = WORK / "tfinance_step22b.log"
PID_FILE = WORK / "tfinance_step22b.pid"

for directory in [
    RESULT_DIR,
    CHECKPOINT_DIR,
    TIMING_DIR,
    RUN_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# WRITE RESTART-SAFE RUNNER
# ============================================================

RUNNER.write_text(r'''
import sys
import gc
import csv
import json
import time
import math
import random
import platform
import contextlib
import io

from datetime import datetime, timezone
from pathlib import Path

import numpy as np

import torch
import torch.nn.functional as F

import dgl
from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix
)


sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


# ============================================================
# LOCKED PROTOCOL
# ============================================================

MODEL_NAME = "BWGNN"
DATASET_NAME = "T-Finance"
MODE = "unified"

UPSTREAM_COMMIT = (
    "de0631f039bbd19c1890b483cc01f1007f596af7"
)

DATASET_SHA256 = (
    "b7d853ec4079e9f7137c03f78a044b1c33d0ff5ed6caa297b895542a7c8e3700"
)

HIDDEN = 64
ORDER = 3
LR = 0.005

MAX_EPOCHS = 100
PATIENCE = 20

SPLIT_SEED = 2

TRAIN_SEEDS = [
    2,
    42,
    72
]

SPLIT_NAMES = [
    "TR40",
    "TR30",
    "TR20",
    "TR10"
]

INFERENCE_WARMUPS = 3
INFERENCE_REPEATS = 10

DEVICE = torch.device("cuda:0")


DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/splits/"
    "tfinance_seed2_nested_splits.npz"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/results/unified/tfinance"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/checkpoints/tfinance_final"
)

TIMING_DIR = RESULT_DIR / "epoch_times"
RUN_DIR = RESULT_DIR / "runs"


# ============================================================
# HELPERS
# ============================================================

def gmean_score(y_true, y_pred):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return float(
        math.sqrt(
            sensitivity * specificity
        )
    )


def select_threshold(
    y_true,
    probabilities
):

    candidates = []

    for threshold in np.arange(
        0.01,
        1.00,
        0.01
    ):

        prediction = (
            probabilities >= threshold
        ).astype(int)

        macro_f1 = f1_score(
            y_true,
            prediction,
            average="macro"
        )

        fraud_recall = recall_score(
            y_true,
            prediction,
            zero_division=0
        )

        candidates.append(
            (
                macro_f1,
                fraud_recall,
                -abs(
                    float(threshold) - 0.50
                ),
                float(threshold)
            )
        )

    return max(candidates)


# ============================================================
# LOAD + PREPROCESS
# ============================================================

print(
    "===== STEP 22B — T-FINANCE FINAL TIMED RUNS =====",
    flush=True
)

print(
    "START — loading canonical T-Finance",
    flush=True
)

load_start = time.perf_counter()

graphs, _ = load_graphs(
    DATA_PATH
)

graph = graphs[0]

features = (
    graph.ndata["feature"]
    .float()
)

raw_labels = graph.ndata["label"]

if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .long()
    )

else:

    labels = (
        raw_labels
        .long()
        .squeeze(-1)
    )


# Structural gate

assert graph.num_nodes() == 39357
assert graph.num_edges() == 42445086
assert tuple(features.shape) == (39357, 10)

normal_total = int(
    (labels == 0).sum().item()
)

fraud_total = int(
    (labels == 1).sum().item()
)

assert normal_total == 37553
assert fraud_total == 1804


split_data = np.load(
    SPLIT_PATH
)

val_ids = torch.tensor(
    split_data["val"],
    dtype=torch.long
)

test_ids = torch.tensor(
    split_data["test"],
    dtype=torch.long
)


# GPU

graph = graph.to(DEVICE)
features = features.to(DEVICE)
labels = labels.to(DEVICE)

val_ids = val_ids.to(DEVICE)
test_ids = test_ids.to(DEVICE)

torch.cuda.synchronize()

load_preprocess_seconds = (
    time.perf_counter()
    - load_start
)


GPU_NAME = torch.cuda.get_device_name(0)

assert torch.cuda.device_count() == 1


print(
    "Nodes       :",
    graph.num_nodes(),
    flush=True
)

print(
    "Edges       :",
    graph.num_edges(),
    flush=True
)

print(
    "Features    :",
    tuple(features.shape),
    flush=True
)

print(
    "Normal/Fraud:",
    normal_total,
    fraud_total,
    flush=True
)

print(
    "GPU         :",
    GPU_NAME,
    flush=True
)

print(
    "Visible GPU :",
    torch.cuda.device_count(),
    flush=True
)

print(
    "Load/preprocess:",
    f"{load_preprocess_seconds:.4f}s",
    flush=True
)


# ============================================================
# SAVE FROZEN CONFIG
# ============================================================

config_path = (
    RESULT_DIR /
    "tfinance_frozen_config.json"
)

config_path.write_text(
    json.dumps(
        {
            "model": MODEL_NAME,
            "dataset": DATASET_NAME,
            "mode": MODE,
            "hidden": HIDDEN,
            "order": ORDER,
            "learning_rate": LR,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "split_seed": SPLIT_SEED,
            "train_seeds": TRAIN_SEEDS,
            "optimizer": "Adam",
            "beta1": 0.9,
            "beta2": 0.999,
            "epsilon": 1e-8,
            "checkpoint_metric": "validation AUPRC",
            "threshold_rule":
                "validation Macro-F1; tie fraud recall; then closest to 0.5",
            "inference_warmups":
                INFERENCE_WARMUPS,
            "inference_repeats":
                INFERENCE_REPEATS,
            "upstream_commit":
                UPSTREAM_COMMIT,
            "dataset_sha256":
                DATASET_SHA256
        },
        indent=2
    )
)


# ============================================================
# FINAL 12 RUNS
# ============================================================

all_results = []

total_runs = (
    len(SPLIT_NAMES)
    * len(TRAIN_SEEDS)
)

run_index = 0


for split_name in SPLIT_NAMES:

    train_ids = torch.tensor(
        split_data[split_name],
        dtype=torch.long,
        device=DEVICE
    )


    for seed in TRAIN_SEEDS:

        run_index += 1

        run_key = (
            f"{split_name}_seed{seed}"
        )

        run_json = (
            RUN_DIR /
            f"{run_key}.json"
        )


        # ====================================================
        # RESUME SUPPORT
        # ====================================================

        if run_json.exists():

            saved = json.loads(
                run_json.read_text()
            )

            all_results.append(
                saved
            )

            print(
                "\n============================================",
                flush=True
            )

            print(
                f"RUN {run_index:02d}/{total_runs} | "
                f"{split_name} | seed={seed}",
                flush=True
            )

            print(
                "SKIP — completed result already exists.",
                flush=True
            )

            continue


        print(
            "\n============================================",
            flush=True
        )

        print(
            f"RUN {run_index:02d}/{total_runs} | "
            f"{split_name} | seed={seed}",
            flush=True
        )


        checkpoint = (
            CHECKPOINT_DIR /
            f"{run_key}.pt"
        )

        epoch_csv = (
            TIMING_DIR /
            f"{run_key}_epoch_times.csv"
        )


        # If this run previously crashed halfway,
        # restart only this run cleanly.

        if checkpoint.exists():
            checkpoint.unlink()

        if epoch_csv.exists():
            epoch_csv.unlink()


        # ====================================================
        # SEED
        # ====================================================

        random.seed(seed)
        np.random.seed(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


        # ====================================================
        # MODEL
        # ====================================================

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            model = BWGNN(
                in_feats=features.shape[1],
                h_feats=HIDDEN,
                num_classes=2,
                graph=graph,
                d=ORDER
            ).to(DEVICE)


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            betas=(0.9, 0.999),
            eps=1e-8
        )


        parameter_count = int(
            sum(
                p.numel()
                for p in model.parameters()
                if p.requires_grad
            )
        )


        # ====================================================
        # TRAIN-ONLY CLASS WEIGHT
        # ====================================================

        train_labels = labels[
            train_ids
        ]

        train_normal = int(
            (train_labels == 0)
            .sum()
            .item()
        )

        train_fraud = int(
            (train_labels == 1)
            .sum()
            .item()
        )

        class_weight = torch.tensor(
            [
                1.0,
                train_normal / train_fraud
            ],
            dtype=torch.float32,
            device=DEVICE
        )


        print(
            "Train normal/fraud:",
            train_normal,
            train_fraud,
            flush=True
        )


        # ====================================================
        # MEMORY RESET
        # ====================================================

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


        # ====================================================
        # EPOCH CSV — WRITE INCREMENTALLY
        # ====================================================

        epoch_fields = [
            "epoch",
            "loss",
            "train_seconds",
            "validation_seconds",
            "cumulative_train_seconds",
            "val_auprc",
            "best_val_auprc",
            "best_epoch"
        ]

        epoch_file = epoch_csv.open(
            "w",
            newline=""
        )

        epoch_writer = csv.DictWriter(
            epoch_file,
            fieldnames=epoch_fields
        )

        epoch_writer.writeheader()
        epoch_file.flush()


        # ====================================================
        # TRAIN
        # ====================================================

        best_val_auprc = -1.0
        best_epoch = -1
        no_improvement = 0

        train_times = []
        validation_times = []

        cumulative_train_seconds = 0.0

        fit_wall_start = (
            time.perf_counter()
        )


        for epoch in range(
            1,
            MAX_EPOCHS + 1
        ):

            # -----------------------------------------------
            # TRAINING ONLY
            # -----------------------------------------------

            model.train()

            torch.cuda.synchronize()

            train_start = (
                time.perf_counter()
            )


            logits = model(
                features
            )

            loss = F.cross_entropy(
                logits[train_ids],
                labels[train_ids],
                weight=class_weight
            )


            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite loss | "
                    f"{run_key} | epoch {epoch}"
                )


            optimizer.zero_grad()

            loss.backward()

            optimizer.step()


            torch.cuda.synchronize()

            train_seconds = (
                time.perf_counter()
                - train_start
            )


            train_times.append(
                train_seconds
            )

            cumulative_train_seconds += (
                train_seconds
            )


            # -----------------------------------------------
            # VALIDATION — SEPARATE TIMER
            # Includes forward + score transfer + AUPRC.
            # -----------------------------------------------

            model.eval()

            torch.cuda.synchronize()

            validation_start = (
                time.perf_counter()
            )


            with torch.no_grad():

                val_logits = model(
                    features
                )

                val_probability = (
                    torch.softmax(
                        val_logits,
                        dim=1
                    )[val_ids, 1]
                )


            torch.cuda.synchronize()

            val_y = (
                labels[val_ids]
                .detach()
                .cpu()
                .numpy()
            )

            val_probability_np = (
                val_probability
                .detach()
                .cpu()
                .numpy()
            )


            val_auprc = float(
                average_precision_score(
                    val_y,
                    val_probability_np
                )
            )


            validation_seconds = (
                time.perf_counter()
                - validation_start
            )


            validation_times.append(
                validation_seconds
            )


            # -----------------------------------------------
            # VALIDATION CHECKPOINT
            # -----------------------------------------------

            if val_auprc > best_val_auprc:

                best_val_auprc = (
                    val_auprc
                )

                best_epoch = epoch

                no_improvement = 0

                torch.save(
                    model.state_dict(),
                    checkpoint
                )

            else:

                no_improvement += 1


            # -----------------------------------------------
            # IMMEDIATE RAW TIMING RECORD
            # -----------------------------------------------

            epoch_writer.writerow(
                {
                    "epoch":
                        epoch,

                    "loss":
                        float(loss.item()),

                    "train_seconds":
                        train_seconds,

                    "validation_seconds":
                        validation_seconds,

                    "cumulative_train_seconds":
                        cumulative_train_seconds,

                    "val_auprc":
                        val_auprc,

                    "best_val_auprc":
                        best_val_auprc,

                    "best_epoch":
                        best_epoch
                }
            )

            epoch_file.flush()


            if (
                epoch == 1
                or epoch % 10 == 0
                or no_improvement >= PATIENCE
            ):

                print(
                    f"epoch={epoch:03d} | "
                    f"loss={loss.item():.5f} | "
                    f"val_AUPRC={val_auprc:.5f} | "
                    f"best={best_val_auprc:.5f}"
                    f"@{best_epoch} | "
                    f"train={train_seconds:.4f}s | "
                    f"val={validation_seconds:.4f}s",
                    flush=True
                )


            if no_improvement >= PATIENCE:

                print(
                    f"Early stop at epoch {epoch}",
                    flush=True
                )

                break


        epoch_file.close()


        fit_wall_seconds = (
            time.perf_counter()
            - fit_wall_start
        )


        train_times_np = np.asarray(
            train_times,
            dtype=float
        )

        validation_times_np = np.asarray(
            validation_times,
            dtype=float
        )


        total_train_seconds = float(
            train_times_np.sum()
        )

        mean_epoch_train_seconds = float(
            train_times_np.mean()
        )

        std_epoch_train_seconds = float(
            train_times_np.std(ddof=1)
            if len(train_times_np) > 1
            else 0.0
        )

        total_validation_seconds = float(
            validation_times_np.sum()
        )

        mean_validation_seconds = float(
            validation_times_np.mean()
        )


        training_peak_gpu_memory_mb = float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )


        # ====================================================
        # RESTORE BEST CHECKPOINT
        # ====================================================

        model.load_state_dict(
            torch.load(
                checkpoint,
                map_location=DEVICE
            )
        )

        model.eval()


        # ====================================================
        # FULL-GRAPH INFERENCE LATENCY
        #
        # 3 warmups + 10 measured passes
        # ====================================================

        with torch.no_grad():

            for _ in range(
                INFERENCE_WARMUPS
            ):

                _ = model(
                    features
                )

                torch.cuda.synchronize()


        inference_samples_ms = []


        with torch.no_grad():

            for _ in range(
                INFERENCE_REPEATS
            ):

                torch.cuda.synchronize()

                inference_start = (
                    time.perf_counter()
                )

                _ = model(
                    features
                )

                torch.cuda.synchronize()

                elapsed_ms = (
                    time.perf_counter()
                    - inference_start
                ) * 1000.0

                inference_samples_ms.append(
                    elapsed_ms
                )


        inference_latency_mean_ms = float(
            np.mean(
                inference_samples_ms
            )
        )

        inference_latency_std_ms = float(
            np.std(
                inference_samples_ms,
                ddof=1
            )
        )


        # ====================================================
        # FINAL SCORES
        #
        # Outside timed latency region
        # ====================================================

        with torch.no_grad():

            final_logits = model(
                features
            )

            probabilities = (
                torch.softmax(
                    final_logits,
                    dim=1
                )[:, 1]
            )


        # ====================================================
        # VALIDATION THRESHOLD
        # ====================================================

        val_y = (
            labels[val_ids]
            .cpu()
            .numpy()
        )

        val_prob = (
            probabilities[val_ids]
            .cpu()
            .numpy()
        )


        threshold_result = (
            select_threshold(
                val_y,
                val_prob
            )
        )


        val_macro_f1 = float(
            threshold_result[0]
        )

        val_fraud_recall = float(
            threshold_result[1]
        )

        threshold = float(
            threshold_result[3]
        )


        # ====================================================
        # TEST ONCE
        # ====================================================

        test_y = (
            labels[test_ids]
            .cpu()
            .numpy()
        )

        test_prob = (
            probabilities[test_ids]
            .cpu()
            .numpy()
        )

        test_pred = (
            test_prob >= threshold
        ).astype(int)


        test_auprc = float(
            average_precision_score(
                test_y,
                test_prob
            )
        )

        test_auroc = float(
            roc_auc_score(
                test_y,
                test_prob
            )
        )

        test_macro_f1 = float(
            f1_score(
                test_y,
                test_pred,
                average="macro"
            )
        )

        test_fraud_precision = float(
            precision_score(
                test_y,
                test_pred,
                zero_division=0
            )
        )

        test_fraud_recall = float(
            recall_score(
                test_y,
                test_pred,
                zero_division=0
            )
        )

        test_fraud_f1 = float(
            f1_score(
                test_y,
                test_pred,
                pos_label=1,
                zero_division=0
            )
        )

        test_gmean = (
            gmean_score(
                test_y,
                test_pred
            )
        )


        final_peak_gpu_memory_mb = float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )


        # ====================================================
        # RUN ID / RESULT
        # ====================================================

        timestamp = (
            datetime.now(timezone.utc)
            .strftime("%Y%m%dT%H%M%SZ")
        )

        run_id = (
            f"{timestamp}_BWGNN_TFinance_"
            f"unified_{split_name}_"
            f"splitS{SPLIT_SEED}_trainS{seed}"
        )


        result = {

            "run_id":
                run_id,

            "model":
                MODEL_NAME,

            "dataset":
                DATASET_NAME,

            "mode":
                MODE,

            "split":
                split_name,

            "split_seed":
                SPLIT_SEED,

            "train_seed":
                seed,

            "upstream_commit":
                UPSTREAM_COMMIT,

            "dataset_sha256":
                DATASET_SHA256,

            "gpu":
                GPU_NAME,

            "visible_gpu_count":
                torch.cuda.device_count(),

            "python":
                platform.python_version(),

            "torch":
                torch.__version__,

            "dgl":
                dgl.__version__,

            "hidden":
                HIDDEN,

            "order":
                ORDER,

            "learning_rate":
                LR,

            "parameter_count":
                parameter_count,

            "completed_epochs":
                len(train_times),

            "best_epoch":
                best_epoch,

            "best_val_auprc":
                best_val_auprc,

            "val_threshold":
                threshold,

            "val_macro_f1":
                val_macro_f1,

            "val_fraud_recall":
                val_fraud_recall,

            "test_auprc":
                test_auprc,

            "test_auroc":
                test_auroc,

            "test_macro_f1":
                test_macro_f1,

            "test_fraud_precision":
                test_fraud_precision,

            "test_fraud_recall":
                test_fraud_recall,

            "test_fraud_f1":
                test_fraud_f1,

            "test_gmean":
                test_gmean,

            "load_preprocess_seconds":
                load_preprocess_seconds,

            "total_train_seconds":
                total_train_seconds,

            "mean_epoch_train_seconds":
                mean_epoch_train_seconds,

            "std_epoch_train_seconds":
                std_epoch_train_seconds,

            "total_validation_seconds":
                total_validation_seconds,

            "mean_validation_seconds":
                mean_validation_seconds,

            "fit_wall_seconds":
                fit_wall_seconds,

            "inference_latency_mean_ms":
                inference_latency_mean_ms,

            "inference_latency_std_ms":
                inference_latency_std_ms,

            "inference_warmups":
                INFERENCE_WARMUPS,

            "inference_repeats":
                INFERENCE_REPEATS,

            "inference_samples_ms":
                inference_samples_ms,

            "training_peak_gpu_memory_mb":
                training_peak_gpu_memory_mb,

            "final_peak_gpu_memory_mb":
                final_peak_gpu_memory_mb,

            "epoch_timing_csv":
                str(epoch_csv),

            "checkpoint":
                str(checkpoint)
        }


        # ====================================================
        # SAVE THIS RUN IMMEDIATELY
        # ====================================================

        run_json.write_text(
            json.dumps(
                result,
                indent=2
            )
        )


        all_results.append(
            result
        )


        print(
            f"DONE | "
            f"{split_name} seed={seed} | "
            f"best_epoch={best_epoch} | "
            f"AUPRC={test_auprc:.6f} | "
            f"AUROC={test_auroc:.6f} | "
            f"Macro-F1={test_macro_f1:.6f}",
            flush=True
        )

        print(
            f"TIMING | "
            f"train_total={total_train_seconds:.3f}s | "
            f"epoch={mean_epoch_train_seconds:.4f}"
            f"±{std_epoch_train_seconds:.4f}s | "
            f"inference={inference_latency_mean_ms:.3f}"
            f"±{inference_latency_std_ms:.3f}ms | "
            f"peak={final_peak_gpu_memory_mb:.1f}MB",
            flush=True
        )


        del model
        del optimizer

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# FINAL AGGREGATION
# ============================================================

# Reload canonical set of completed run files.

all_results = []

for split_name in SPLIT_NAMES:

    for seed in TRAIN_SEEDS:

        path = (
            RUN_DIR /
            f"{split_name}_seed{seed}.json"
        )

        if not path.exists():

            raise RuntimeError(
                f"Missing completed run: {path}"
            )

        all_results.append(
            json.loads(
                path.read_text()
            )
        )


# ============================================================
# MASTER CSV
# ============================================================

csv_path = (
    RESULT_DIR /
    "tfinance_unified_all_runs_timed.csv"
)


csv_fields = [
    key
    for key in all_results[0].keys()
    if key != "inference_samples_ms"
]


with csv_path.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()

    for row in all_results:

        writer.writerow(
            {
                key: value
                for key, value in row.items()
                if key != "inference_samples_ms"
            }
        )


# ============================================================
# MASTER JSON
# ============================================================

json_path = (
    RESULT_DIR /
    "tfinance_unified_all_runs_timed.json"
)

json_path.write_text(
    json.dumps(
        all_results,
        indent=2
    )
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n===== T-FINANCE FINAL SUMMARY =====",
    flush=True
)


for split_name in SPLIT_NAMES:

    rows = [
        r
        for r in all_results
        if r["split"] == split_name
    ]

    print(
        f"\n{split_name}",
        flush=True
    )


    summary_metrics = [
        "test_auprc",
        "test_auroc",
        "test_macro_f1",
        "test_fraud_precision",
        "test_fraud_recall",
        "test_fraud_f1",
        "test_gmean",
        "mean_epoch_train_seconds",
        "total_train_seconds",
        "inference_latency_mean_ms",
        "final_peak_gpu_memory_mb"
    ]


    for metric in summary_metrics:

        values = np.array(
            [
                float(r[metric])
                for r in rows
            ]
        )

        print(
            f"  {metric:<30} "
            f"{values.mean():.6f} "
            f"± {values.std(ddof=1):.6f}",
            flush=True
        )


print(
    "\nResults CSV:",
    csv_path,
    flush=True
)

print(
    "Results JSON:",
    json_path,
    flush=True
)

print(
    "Raw epoch timing:",
    TIMING_DIR,
    flush=True
)

print(
    "Individual run records:",
    RUN_DIR,
    flush=True
)


print(
    "\n===== STEP 22B GATE =====",
    flush=True
)

print(
    "PASS — T-Finance final unified benchmark "
    "completed for all 12 runs with per-epoch timing, "
    "inference latency and GPU memory evidence.",
    flush=True
)
''',
encoding="utf-8"
)


# ============================================================
# CONTROLLED CHILD ENVIRONMENT
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"
ENV["PYTHONHASHSEED"] = "2"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(
        str(path)
        for path in cuda_dirs
    )
    +
    (
        ":" + existing_ld
        if existing_ld
        else ""
    )
)


# ============================================================
# PREVENT DOUBLE LAUNCH
# ============================================================

if PID_FILE.exists():

    try:
        previous_pid = int(
            PID_FILE.read_text().strip()
        )

        proc_path = Path(
            f"/proc/{previous_pid}/cmdline"
        )

        if proc_path.exists():

            cmdline = (
                proc_path
                .read_bytes()
                .replace(b"\x00", b" ")
                .decode(
                    errors="ignore"
                )
            )

            if str(RUNNER) in cmdline:

                raise RuntimeError(
                    f"Step 22B is already running "
                    f"with PID {previous_pid}.\n"
                    f"Do NOT launch another copy."
                )

    except ValueError:
        pass


# ============================================================
# DETACHED BACKGROUND LAUNCH
# ============================================================

log_handle = LOG_FILE.open(
    "a",
    buffering=1
)

log_handle.write(
    "\n\n============================================\n"
    "NEW STEP 22B LAUNCH\n"
    "============================================\n"
)

log_handle.flush()


process = subprocess.Popen(
    [
        str(PYTHON39),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=ENV,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log_handle.close()


print(
    "===== STEP 22B STARTED ====="
)

print(
    "PID:",
    process.pid
)

print(
    "Log:",
    LOG_FILE
)

print(
    "Runner:",
    RUNNER
)

print(
    "\nThe benchmark is now running in the background."
)

print(
    "Do NOT run this launch cell again while it is running."
)

===== STEP 22B STARTED =====
PID: 199
Log: /kaggle/working/comp8851_bwgnn/tfinance_step22b.log
Runner: /kaggle/working/comp8851_bwgnn/adapters/tfinance_final_timed_resumable.py

The benchmark is now running in the background.
Do NOT run this launch cell again while it is running.


In [4]:
# ============================================================
# STEP 22C — CHECK T-FINANCE BACKGROUND RUN
# ============================================================

from pathlib import Path
import os

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "tfinance_step22b.pid"
LOG_FILE = WORK / "tfinance_step22b.log"
RUN_DIR = WORK / "results/unified/tfinance/runs"

print("===== STEP 22B STATUS =====")

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    proc_path = Path(f"/proc/{pid}/cmdline")
    running = proc_path.exists()

    print("PID:", pid)
    print("Process running:", running)
else:
    print("No PID file found.")
    running = False

completed = []

if RUN_DIR.exists():
    completed = sorted(
        p.stem
        for p in RUN_DIR.glob("*.json")
    )

print(f"\nCompleted final runs: {len(completed)}/12")

for name in completed:
    print(" ✓", name)

print("\n===== LATEST LOG =====")

if LOG_FILE.exists():
    lines = LOG_FILE.read_text(
        errors="replace"
    ).splitlines()

    for line in lines[-60:]:
        print(line)
else:
    print("No log file found.")

===== STEP 22B STATUS =====
PID: 199
Process running: True

Completed final runs: 12/12
 ✓ TR10_seed2
 ✓ TR10_seed42
 ✓ TR10_seed72
 ✓ TR20_seed2
 ✓ TR20_seed42
 ✓ TR20_seed72
 ✓ TR30_seed2
 ✓ TR30_seed42
 ✓ TR30_seed72
 ✓ TR40_seed2
 ✓ TR40_seed42
 ✓ TR40_seed72

===== LATEST LOG =====

TR40
  test_auprc                     0.702817 ± 0.130686
  test_auroc                     0.925711 ± 0.035176
  test_macro_f1                  0.825136 ± 0.057709
  test_fraud_precision           0.737330 ± 0.155912
  test_fraud_recall              0.615882 ± 0.109710
  test_fraud_f1                  0.665242 ± 0.109913
  test_gmean                     0.778273 ± 0.069312
  mean_epoch_train_seconds       0.665238 ± 0.065551
  total_train_seconds            29.195167 ± 15.367312
  inference_latency_mean_ms      338.641478 ± 26.931364
  final_peak_gpu_memory_mb       2385.909993 ± 546.734918

TR30
  test_auprc                     0.804245 ± 0.097219
  test_auroc                     0.950566 ± 0.024869
 

In [5]:
# ============================================================
# STEP 23 — PACKAGE T-FINANCE REPRODUCIBILITY EVIDENCE
# ============================================================

from pathlib import Path
import shutil

WORK = Path("/kaggle/working/comp8851_bwgnn")
EXPORT = Path("/kaggle/working/bwgnn_tfinance_evidence")

if EXPORT.exists():
    shutil.rmtree(EXPORT)

EXPORT.mkdir(parents=True)


def copy_file(src, dst):
    src = Path(src)
    dst = Path(dst)

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("✓", src)
    else:
        print("MISSING:", src)


# ------------------------------------------------------------
# Final T-Finance results
# ------------------------------------------------------------

RESULT = WORK / "results/unified/tfinance"

for name in [
    "tfinance_unified_all_runs_timed.csv",
    "tfinance_unified_all_runs_timed.json",
    "tfinance_frozen_config.json",
]:
    copy_file(
        RESULT / name,
        EXPORT / "results" / name
    )


# ------------------------------------------------------------
# Individual run records
# ------------------------------------------------------------

run_dir = RESULT / "runs"

if run_dir.exists():
    shutil.copytree(
        run_dir,
        EXPORT / "results/runs"
    )
    print("✓ individual run JSON files")


# ------------------------------------------------------------
# Raw per-epoch timing
# ------------------------------------------------------------

timing_dir = RESULT / "epoch_times"

if timing_dir.exists():
    shutil.copytree(
        timing_dir,
        EXPORT / "results/epoch_times"
    )
    print("✓ raw epoch timing CSV files")


# ------------------------------------------------------------
# Exact split used
# ------------------------------------------------------------

copy_file(
    WORK / "shared/splits/tfinance_seed2_nested_splits.npz",
    EXPORT / "splits/tfinance_seed2_nested_splits.npz"
)


# ------------------------------------------------------------
# Model adapter + final runner
# ------------------------------------------------------------

copy_file(
    WORK / "adapters/BWGNN_gpu.py",
    EXPORT / "adapters/BWGNN_gpu.py"
)

copy_file(
    WORK / "adapters/tfinance_final_timed_resumable.py",
    EXPORT / "adapters/tfinance_final_timed_resumable.py"
)


# ------------------------------------------------------------
# Execution log
# ------------------------------------------------------------

copy_file(
    Path("/kaggle/working/tfinance_step22b.log"),
    EXPORT / "logs/tfinance_step22b.log"
)


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

ZIP_BASE = Path(
    "/kaggle/working/bwgnn_tfinance_evidence_20260909"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=EXPORT
)

print("\n============================================")
print("STEP 23 GATE: PASS")
print("Evidence ZIP:", zip_path)

print("\nContains:")

for p in sorted(EXPORT.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(EXPORT))

✓ /kaggle/working/comp8851_bwgnn/results/unified/tfinance/tfinance_unified_all_runs_timed.csv
✓ /kaggle/working/comp8851_bwgnn/results/unified/tfinance/tfinance_unified_all_runs_timed.json
✓ /kaggle/working/comp8851_bwgnn/results/unified/tfinance/tfinance_frozen_config.json
✓ individual run JSON files
✓ raw epoch timing CSV files
✓ /kaggle/working/comp8851_bwgnn/shared/splits/tfinance_seed2_nested_splits.npz
✓ /kaggle/working/comp8851_bwgnn/adapters/BWGNN_gpu.py
✓ /kaggle/working/comp8851_bwgnn/adapters/tfinance_final_timed_resumable.py
MISSING: /kaggle/working/tfinance_step22b.log

STEP 23 GATE: PASS
Evidence ZIP: /kaggle/working/bwgnn_tfinance_evidence_20260909.zip

Contains:
 - adapters/BWGNN_gpu.py
 - adapters/tfinance_final_timed_resumable.py
 - results/epoch_times/TR10_seed2_epoch_times.csv
 - results/epoch_times/TR10_seed42_epoch_times.csv
 - results/epoch_times/TR10_seed72_epoch_times.csv
 - results/epoch_times/TR20_seed2_epoch_times.csv
 - results/epoch_times/TR20_seed42_epoch

In [6]:
# ============================================================
# STEP 23A — ADD MISSING STEP 22B LOG TO EVIDENCE ZIP
# ============================================================

from pathlib import Path
import shutil

WORK = Path("/kaggle/working/comp8851_bwgnn")
EXPORT = Path("/kaggle/working/bwgnn_tfinance_evidence")

LOG = WORK / "tfinance_step22b.log"

if not LOG.exists():
    raise FileNotFoundError(
        f"Expected log not found: {LOG}"
    )

(EXPORT / "logs").mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    LOG,
    EXPORT / "logs/tfinance_step22b.log"
)

ZIP_BASE = Path(
    "/kaggle/working/bwgnn_tfinance_evidence_20260909"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=EXPORT
)

print("===== STEP 23A GATE =====")
print("PASS — Step 22B execution log added.")
print("Updated ZIP:", zip_path)
print(
    "Log size:",
    (EXPORT / "logs/tfinance_step22b.log").stat().st_size,
    "bytes"
)

===== STEP 23A GATE =====
PASS — Step 22B execution log added.
Updated ZIP: /kaggle/working/bwgnn_tfinance_evidence_20260909.zip
Log size: 16169 bytes


## Step 24A — Additional Computational-Efficiency Backfill Setup

**Purpose of this step**

This is additional reproducibility and computational-efficiency work performed
after the primary BWGNN benchmark runs were completed.

It does **not** replace or modify the previously completed benchmark results:

- YelpChi primary final benchmark: **Step 15**
- Amazon primary final benchmark: **Step 19**

The earlier final performance results remain the primary benchmark results.

This additional section is required because the earlier YelpChi and Amazon
final runners did not separately preserve:

- raw per-epoch training-only time,
- validation time,
- total training-only time,
- mean and standard deviation of epoch training time,
- full-graph inference latency,
- peak GPU memory.

Therefore, YelpChi and Amazon will be rerun using their already frozen
hyperparameters and the same controlled split protocol solely to obtain
comparable computational-efficiency evidence.

No hyperparameter tuning is repeated, and test data is not used for model,
checkpoint, threshold, or configuration selection.

### Frozen configurations

**YelpChi BWGNN-Homo**
- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005

**Amazon BWGNN-Homo**
- Hidden dimension: 32
- Wavelet order: 2
- Learning rate: 0.01

### Controlled protocol retained

- Split seed: 2
- Training seeds: 2, 42, 72
- Ratios: TR40, TR30, TR20, TR10
- Validation: 20%
- Test: fixed 40%
- Nested training sets
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint criterion: validation AUPRC
- Threshold selection: validation Macro-F1

All new timing outputs are stored in separate `timing_backfill` directories so
the original Step 15 and Step 19 result directories remain untouched.

In [7]:
# ============================================================
# STEP 24A — ADDITIONAL TIMING BACKFILL SETUP
#
# Supplemental to:
#   Step 15 — YelpChi primary final benchmark
#   Step 19 — Amazon primary final benchmark
#
# DOES NOT modify previous benchmark result directories.
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

WORK = Path("/kaggle/working/comp8851_bwgnn")

YELP_PATH = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "yelp-chi/YelpChi.mat"
)

AMAZON_PATH = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "amazon/Amazon.mat"
)

BACKFILL_SPLIT_DIR = (
    WORK / "shared/timing_backfill_splits"
)

BACKFILL_RESULT_DIR = (
    WORK / "results/timing_backfill"
)

BACKFILL_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

(BACKFILL_RESULT_DIR / "yelp_homo").mkdir(
    parents=True,
    exist_ok=True
)

(BACKFILL_RESULT_DIR / "amazon_homo").mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# EXPECTED CANONICAL DATASET HASHES
# ============================================================

EXPECTED_YELP_SHA256 = (
    "fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42"
)

EXPECTED_AMAZON_SHA256 = (
    "4b7e3f9cccc62b736792707393ccd74332a1a0592dba128ac6b2989bf1ee9d63"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ============================================================
# 1. DATASET PRESENCE + HASH VERIFICATION
# ============================================================

print(
    "===== STEP 24A — DATASET VERIFICATION ====="
)

assert YELP_PATH.exists(), (
    f"Missing YelpChi dataset: {YELP_PATH}"
)

assert AMAZON_PATH.exists(), (
    f"Missing Amazon dataset: {AMAZON_PATH}"
)


yelp_hash = sha256_file(
    YELP_PATH
)

amazon_hash = sha256_file(
    AMAZON_PATH
)


print(
    "YelpChi SHA256:",
    yelp_hash
)

print(
    "Amazon SHA256 :",
    amazon_hash
)


assert yelp_hash == EXPECTED_YELP_SHA256
assert amazon_hash == EXPECTED_AMAZON_SHA256


print(
    "PASS — canonical YelpChi and Amazon files verified."
)


# ============================================================
# 2. LOAD LABELS ONLY
# ============================================================

print(
    "\n===== LABEL VERIFICATION ====="
)

yelp_mat = loadmat(
    YELP_PATH
)

amazon_mat = loadmat(
    AMAZON_PATH
)


if "label" not in yelp_mat:
    raise KeyError(
        "YelpChi.mat does not contain expected 'label' key."
    )

if "label" not in amazon_mat:
    raise KeyError(
        "Amazon.mat does not contain expected 'label' key."
    )


yelp_labels = (
    np.asarray(
        yelp_mat["label"]
    )
    .reshape(-1)
    .astype(int)
)

amazon_labels_full = (
    np.asarray(
        amazon_mat["label"]
    )
    .reshape(-1)
    .astype(int)
)


# ============================================================
# 3. YELPCHI STRUCTURAL CHECK
# ============================================================

assert len(yelp_labels) == 45954

print(
    "YelpChi graph nodes:",
    len(yelp_labels)
)

print(
    "YelpChi normal/fraud:",
    int((yelp_labels == 0).sum()),
    int((yelp_labels == 1).sum())
)


# ============================================================
# 4. AMAZON SUPERVISED-BENCHMARK ELIGIBILITY
#
# Official BWGNN logic excludes node IDs 0–3304.
# Eligible supervised IDs = 3305–11943.
# ============================================================

assert len(amazon_labels_full) == 11944

AMAZON_FIRST_LABELED_ID = 3305

amazon_eligible_ids = np.arange(
    AMAZON_FIRST_LABELED_ID,
    len(amazon_labels_full)
)

amazon_eligible_labels = (
    amazon_labels_full[
        amazon_eligible_ids
    ]
)

assert len(amazon_eligible_ids) == 8639

assert int(
    (amazon_eligible_labels == 0).sum()
) == 7818

assert int(
    (amazon_eligible_labels == 1).sum()
) == 821


print(
    "\nAmazon graph nodes:",
    len(amazon_labels_full)
)

print(
    "Amazon excluded IDs:",
    "0–3304"
)

print(
    "Amazon eligible nodes:",
    len(amazon_eligible_ids)
)

print(
    "Amazon eligible normal/fraud:",
    int((amazon_eligible_labels == 0).sum()),
    int((amazon_eligible_labels == 1).sum())
)


# ============================================================
# 5. NESTED SPLIT HELPER
# ============================================================

def create_nested_split(
    eligible_ids,
    labels,
    seed=2
):

    eligible_ids = np.asarray(
        eligible_ids
    )

    eligible_labels = labels[
        eligible_ids
    ]


    # --------------------------------------------------------
    # Fixed TEST = 40%
    # Remaining = 60%
    # --------------------------------------------------------

    remaining_ids, test_ids = (
        train_test_split(
            eligible_ids,
            test_size=0.40,
            stratify=eligible_labels,
            random_state=seed,
            shuffle=True
        )
    )


    remaining_labels = labels[
        remaining_ids
    ]


    # --------------------------------------------------------
    # From the remaining 60%:
    # 1/3 -> validation = 20% overall
    # 2/3 -> TR40 = 40% overall
    # --------------------------------------------------------

    train40, val_ids = (
        train_test_split(
            remaining_ids,
            test_size=(1 / 3),
            stratify=remaining_labels,
            random_state=seed,
            shuffle=True
        )
    )


    # --------------------------------------------------------
    # Nested TR30 ⊂ TR40
    # --------------------------------------------------------

    train30, _ = (
        train_test_split(
            train40,
            train_size=0.75,
            stratify=labels[train40],
            random_state=seed,
            shuffle=True
        )
    )


    # --------------------------------------------------------
    # Nested TR20 ⊂ TR30
    # --------------------------------------------------------

    train20, _ = (
        train_test_split(
            train30,
            train_size=(2 / 3),
            stratify=labels[train30],
            random_state=seed,
            shuffle=True
        )
    )


    # --------------------------------------------------------
    # Nested TR10 ⊂ TR20
    # --------------------------------------------------------

    train10, _ = (
        train_test_split(
            train20,
            train_size=0.50,
            stratify=labels[train20],
            random_state=seed,
            shuffle=True
        )
    )


    splits = {
        "TR40": np.sort(train40),
        "TR30": np.sort(train30),
        "TR20": np.sort(train20),
        "TR10": np.sort(train10),
        "val": np.sort(val_ids),
        "test": np.sort(test_ids)
    }


    # Nested-set checks

    assert set(
        splits["TR10"]
    ) <= set(
        splits["TR20"]
    )

    assert set(
        splits["TR20"]
    ) <= set(
        splits["TR30"]
    )

    assert set(
        splits["TR30"]
    ) <= set(
        splits["TR40"]
    )


    # Leakage checks

    assert not (
        set(splits["TR40"])
        & set(splits["val"])
    )

    assert not (
        set(splits["TR40"])
        & set(splits["test"])
    )

    assert not (
        set(splits["val"])
        & set(splits["test"])
    )


    return splits


# ============================================================
# 6. RECREATE YELPCHI SPLIT
# ============================================================

print(
    "\n===== YELPCHI TIMING-BACKFILL SPLIT ====="
)

yelp_ids = np.arange(
    len(yelp_labels)
)

yelp_splits = create_nested_split(
    yelp_ids,
    yelp_labels,
    seed=2
)


expected_yelp_sizes = {
    "TR40": 18381,
    "TR30": 13785,
    "TR20": 9190,
    "TR10": 4595,
    "val": 9191,
    "test": 18382
}


for name, expected in (
    expected_yelp_sizes.items()
):

    actual = len(
        yelp_splits[name]
    )

    print(
        f"{name:<5}: {actual}"
    )

    assert actual == expected


YELP_SPLIT_PATH = (
    BACKFILL_SPLIT_DIR /
    "yelp_seed2_nested_splits.npz"
)

np.savez_compressed(
    YELP_SPLIT_PATH,
    **yelp_splits,
    seed=np.array([2]),
    source_nodes=np.array([45954])
)


print(
    "YelpChi nested split: PASS"
)


# ============================================================
# 7. RECREATE CORRECTED AMAZON SPLIT
# ============================================================

print(
    "\n===== AMAZON TIMING-BACKFILL SPLIT ====="
)

amazon_splits = create_nested_split(
    amazon_eligible_ids,
    amazon_labels_full,
    seed=2
)


expected_amazon_sizes = {
    "TR40": 3455,
    "TR30": 2591,
    "TR20": 1727,
    "TR10": 863,
    "val": 1728,
    "test": 3456
}


for name, expected in (
    expected_amazon_sizes.items()
):

    actual = len(
        amazon_splits[name]
    )

    print(
        f"{name:<5}: {actual}"
    )

    assert actual == expected


# Verify no excluded Amazon node appears

for name in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    assert np.all(
        amazon_splits[name]
        >= AMAZON_FIRST_LABELED_ID
    )


AMAZON_SPLIT_PATH = (
    BACKFILL_SPLIT_DIR /
    "amazon_seed2_nested_splits.npz"
)

np.savez_compressed(
    AMAZON_SPLIT_PATH,
    **amazon_splits,
    seed=np.array([2]),
    source_nodes=np.array([11944]),
    excluded_first_id=np.array([0]),
    excluded_last_id=np.array([3304]),
    eligible_nodes=np.array([8639])
)


print(
    "Amazon nested split: PASS"
)


# ============================================================
# 8. FROZEN CONFIG RECORD
# ============================================================

import json

CONFIG_PATH = (
    BACKFILL_RESULT_DIR /
    "timing_backfill_frozen_configs.json"
)

CONFIG_PATH.write_text(
    json.dumps(
        {
            "purpose":
                "Additional computational-efficiency "
                "backfill only; does not replace "
                "primary benchmark metrics.",

            "yelp_homo": {
                "primary_result_step": 15,
                "hidden": 64,
                "order": 3,
                "learning_rate": 0.005
            },

            "amazon_homo": {
                "primary_result_step": 19,
                "hidden": 32,
                "order": 2,
                "learning_rate": 0.01
            },

            "split_seed": 2,

            "train_seeds": [
                2,
                42,
                72
            ],

            "ratios": [
                "TR40",
                "TR30",
                "TR20",
                "TR10"
            ]
        },
        indent=2
    )
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n============================================"
)

print(
    "STEP 24A GATE: PASS"
)

print(
    "Purpose: ADDITIONAL timing backfill only."
)

print(
    "Step 15 YelpChi primary results: UNCHANGED"
)

print(
    "Step 19 Amazon primary results: UNCHANGED"
)

print(
    "YelpChi timing split:",
    YELP_SPLIT_PATH
)

print(
    "Amazon timing split:",
    AMAZON_SPLIT_PATH
)

print(
    "Backfill outputs will use:",
    BACKFILL_RESULT_DIR
)

print(
    "\nReady for Step 24B — "
    "YelpChi computational-efficiency backfill."
)

===== STEP 24A — DATASET VERIFICATION =====
YelpChi SHA256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
Amazon SHA256 : 4b7e3f9cccc62b736792707393ccd74332a1a0592dba128ac6b2989bf1ee9d63
PASS — canonical YelpChi and Amazon files verified.

===== LABEL VERIFICATION =====
YelpChi graph nodes: 45954
YelpChi normal/fraud: 39277 6677

Amazon graph nodes: 11944
Amazon excluded IDs: 0–3304
Amazon eligible nodes: 8639
Amazon eligible normal/fraud: 7818 821

===== YELPCHI TIMING-BACKFILL SPLIT =====
TR40 : 18381
TR30 : 13785
TR20 : 9190
TR10 : 4595
val  : 9191
test : 18382
YelpChi nested split: PASS

===== AMAZON TIMING-BACKFILL SPLIT =====
TR40 : 3455
TR30 : 2591
TR20 : 1727
TR10 : 863
val  : 1728
test : 3456
Amazon nested split: PASS

STEP 24A GATE: PASS
Purpose: ADDITIONAL timing backfill only.
Step 15 YelpChi primary results: UNCHANGED
Step 19 Amazon primary results: UNCHANGED
YelpChi timing split: /kaggle/working/comp8851_bwgnn/shared/timing_backfill_splits/yelp_seed2_ne

## Step 24B — YelpChi Computational-Efficiency Backfill

**Relation to earlier work:** This is an additional computational-efficiency
rerun for the primary YelpChi BWGNN-Homo benchmark completed in **Step 15**.

**Step 15 remains the primary YelpChi performance result.**

This step does not retune BWGNN, replace Step 15 metrics, or modify the
original `results/unified/yelp_homo/` outputs.

The purpose is only to collect timing and efficiency evidence that was not
separately preserved during Step 15.

### Frozen Step 15 configuration

- Model: BWGNN-Homo
- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005
- Split seed: 2
- Training seeds: 2, 42, 72
- Ratios: TR40, TR30, TR20, TR10
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint selection: validation AUPRC
- Threshold selection: validation Macro-F1

### Graph construction

The canonical YelpChi `.mat` file is reconstructed using the three original
relations (`net_rsr`, `net_rtr`, and `net_rur`). The graph is then converted
to the homogeneous BWGNN representation and one self-loop is added per node,
matching the BWGNN-Homo processing used for the primary benchmark.

Expected graph:

- Nodes: 45,954
- Relation-wise edges before self-loops: 8,051,348
- Homogeneous edges after self-loops: 8,097,302
- Features: 32

### Additional evidence recorded

For every ratio × training-seed run:

- Raw training-only time for every epoch
- Validation time separately
- Total training-only time
- Mean and standard deviation of epoch training time
- Full-graph inference latency
- Peak GPU memory
- Parameter count

Performance metrics are also recorded as a verification check, but these
supplemental rerun metrics do **not replace Step 15**.

All outputs are written separately under:

`results/timing_backfill/yelp_homo/`

In [ ]:
# ============================================================
# STEP 24B RECOVERY CHECK
# Do NOT launch anything here.
# ============================================================

from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "yelp_step24b.pid"
LOG_FILE = WORK / "yelp_step24b.log"
RUN_DIR = WORK / "results/timing_backfill/yelp_homo/runs"

print("===== STEP 24B RECOVERY CHECK =====")

# PID
if PID_FILE.exists():
    print("PID file:", PID_FILE.read_text().strip())
else:
    print("PID file: NOT FOUND")

# Actual process
p = subprocess.run(
    [
        "bash",
        "-lc",
        "pgrep -af yelp_homo_timing_backfill.py || true"
    ],
    capture_output=True,
    text=True
)

real_lines = [
    line for line in p.stdout.splitlines()
    if "pgrep -af" not in line
]

print("\nYelp timing process:")

if real_lines:
    for line in real_lines:
        print(line)
else:
    print("NOT RUNNING")

# Completed runs
completed = []

if RUN_DIR.exists():
    completed = sorted(
        x.stem for x in RUN_DIR.glob("*.json")
    )

print(f"\nCompleted runs: {len(completed)}/12")

for x in completed:
    print(" ✓", x)

# Log
print("\n===== LATEST LOG =====")

if LOG_FILE.exists():
    lines = LOG_FILE.read_text(
        errors="replace"
    ).splitlines()

    for line in lines[-40:]:
        print(line)
else:
    print("No Yelp Step 24B log file yet.")

In [8]:
print("KERNEL OK")

KERNEL OK


In [9]:
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

print("WORK:", WORK.exists())
print("24A split:", (WORK / "shared/timing_backfill_splits/yelp_seed2_nested_splits.npz").exists())
print("24B runner:", (WORK / "adapters/yelp_homo_timing_backfill.py").exists())

WORK: True
24A split: True
24B runner: False


## Step 24B — YelpChi Computational-Efficiency Backfill

**Additional work related to Step 15.**

The primary YelpChi BWGNN-Homo benchmark was completed in **Step 15**.
Its original performance results remain unchanged.

This step is a supplemental rerun performed only to obtain the
computational-efficiency evidence that was not separately preserved during
Step 15.

The frozen Step 15 configuration is reused without any retuning:

- Hidden dimension: 64
- Wavelet order: 3
- Learning rate: 0.005
- Split seed: 2
- Training seeds: 2, 42, 72
- Ratios: TR40, TR30, TR20, TR10
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint selection: validation AUPRC

The same YelpChi BWGNN-Homo graph construction is retained:

- 45,954 nodes
- 8,051,348 relation edges before self-loops
- 8,097,302 homogeneous edges after self-loops
- 32 input features

Additional evidence collected:

- Raw per-epoch training-only time
- Validation time separately
- Total training-only time
- Mean and standard deviation of epoch training time
- Full-graph inference latency
- Peak GPU memory
- Parameter count

Performance metrics are recorded only as a verification check.
They do **not replace the Step 15 primary metrics**.

All outputs are written separately under:

`results/timing_backfill/yelp_homo/`

In [10]:
from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/comp8851_bwgnn")

print("Environment:",
      (WORK / "envs/bwgnn-author/bin/python").exists())

print("BWGNN adapter:",
      (WORK / "adapters/BWGNN_gpu.py").exists())

print("Yelp split:",
      (WORK / "shared/timing_backfill_splits/yelp_seed2_nested_splits.npz").exists())

p = subprocess.run(
    ["bash", "-lc",
     "ps -ef | grep '[y]elp_homo_timing_backfill.py' || true"],
    capture_output=True,
    text=True
)

print("Old Yelp process:",
      p.stdout.strip() if p.stdout.strip() else "NONE")

Environment: True
BWGNN adapter: True
Yelp split: True
Old Yelp process: NONE


In [11]:
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
RUNNER = WORK / "adapters/yelp_homo_timing_backfill.py"

code = r'''
import sys, gc, csv, json, time, math, random, io, contextlib
from pathlib import Path

import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

sys.path.insert(0, "/kaggle/working/comp8851_bwgnn/adapters")
from BWGNN_gpu import BWGNN


# ============================================================
# FIXED STEP 15 CONFIG
# ============================================================

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "yelp-chi/YelpChi.mat"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/"
    "timing_backfill_splits/yelp_seed2_nested_splits.npz"
)

ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "results/timing_backfill/yelp_homo"
)

RUN_DIR = ROOT / "runs"
TIME_DIR = ROOT / "epoch_times"

CKPT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "checkpoints/timing_backfill/yelp_homo"
)

for d in [ROOT, RUN_DIR, TIME_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


HIDDEN = 64
ORDER = 3
LR = 0.005

MAX_EPOCHS = 100
PATIENCE = 20

SEEDS = [2, 42, 72]
RATIOS = ["TR40", "TR30", "TR20", "TR10"]

DEVICE = torch.device("cuda:0")

WARMUPS = 3
REPEATS = 10


# ============================================================
# HELPERS
# ============================================================

def choose_threshold(y, p):

    best = None

    for t in np.arange(0.01, 1.00, 0.01):

        pred = (p >= t).astype(int)

        candidate = (
            f1_score(y, pred, average="macro"),
            recall_score(y, pred, zero_division=0),
            -abs(float(t) - 0.5),
            float(t)
        )

        if best is None or candidate > best:
            best = candidate

    return best[3]


def gmean(y, pred):

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0

    return math.sqrt(sensitivity * specificity)


# ============================================================
# LOAD YELPCHI
# ============================================================

print("===== STEP 24B — YELPCHI TIMING BACKFILL =====", flush=True)
print("Supplemental to Step 15 only.", flush=True)

load_start = time.perf_counter()

m = loadmat(DATA_PATH)

features_np = m["features"]

if sp.issparse(features_np):
    features_np = features_np.toarray()

features_np = np.asarray(
    features_np,
    dtype=np.float32
)

labels_np = np.asarray(
    m["label"]
).reshape(-1).astype(np.int64)

assert features_np.shape == (45954, 32)
assert labels_np.shape == (45954,)
assert int((labels_np == 0).sum()) == 39277
assert int((labels_np == 1).sum()) == 6677


# ============================================================
# THREE AUTHOR RELATIONS
# ============================================================

relations = {}
edge_total = 0

for rel, key in [
    ("rsr", "net_rsr"),
    ("rtr", "net_rtr"),
    ("rur", "net_rur")
]:

    a = sp.coo_matrix(m[key])

    src = torch.from_numpy(
        a.row.astype(np.int64)
    )

    dst = torch.from_numpy(
        a.col.astype(np.int64)
    )

    edge_total += len(src)

    relations[
        ("review", rel, "review")
    ] = (src, dst)


assert edge_total == 8051348

hg = dgl.heterograph(
    relations,
    num_nodes_dict={"review": 45954}
)

hg.nodes["review"].data["feature"] = (
    torch.from_numpy(features_np)
)

hg.nodes["review"].data["label"] = (
    torch.from_numpy(labels_np)
)

g = dgl.to_homogeneous(
    hg,
    ndata=["feature", "label"]
)

g = dgl.add_self_loop(g)

assert g.num_nodes() == 45954
assert g.num_edges() == 8097302


features = g.ndata["feature"].float()
labels = g.ndata["label"].long()

splits = np.load(SPLIT_PATH)

val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long
)

test_ids = torch.tensor(
    splits["test"],
    dtype=torch.long
)


g = g.to(DEVICE)
features = features.to(DEVICE)
labels = labels.to(DEVICE)

val_ids = val_ids.to(DEVICE)
test_ids = test_ids.to(DEVICE)

torch.cuda.synchronize()

load_seconds = time.perf_counter() - load_start


print("Nodes:", g.num_nodes(), flush=True)
print("Edges:", g.num_edges(), flush=True)
print("Features:", tuple(features.shape), flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("Load/preprocess:", f"{load_seconds:.3f}s", flush=True)


del m, hg, relations, features_np, labels_np
gc.collect()


# ============================================================
# FINAL TIMING RUNS
# ============================================================

for ratio in RATIOS:

    train_ids = torch.tensor(
        splits[ratio],
        dtype=torch.long,
        device=DEVICE
    )

    for seed in SEEDS:

        key = f"{ratio}_seed{seed}"

        result_file = RUN_DIR / f"{key}.json"
        epoch_file = TIME_DIR / f"{key}_epoch_times.csv"
        checkpoint = CKPT_DIR / f"{key}.pt"


        print(
            "\n============================================",
            flush=True
        )

        print(
            f"START | {ratio} | seed={seed}",
            flush=True
        )


        # completed run survives relaunch
        if result_file.exists():

            print("SKIP — already complete.", flush=True)
            continue


        if epoch_file.exists():
            epoch_file.unlink()

        if checkpoint.exists():
            checkpoint.unlink()


        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


        with contextlib.redirect_stdout(io.StringIO()):

            model = BWGNN(
                in_feats=32,
                h_feats=HIDDEN,
                num_classes=2,
                graph=g,
                d=ORDER
            ).to(DEVICE)


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            betas=(0.9, 0.999),
            eps=1e-8
        )


        params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )


        train_y = labels[train_ids]

        n0 = int(
            (train_y == 0).sum().item()
        )

        n1 = int(
            (train_y == 1).sum().item()
        )

        weight = torch.tensor(
            [1.0, n0 / n1],
            dtype=torch.float32,
            device=DEVICE
        )


        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


        train_times = []
        val_times = []

        best_ap = -1.0
        best_epoch = -1
        stale = 0
        cumulative = 0.0


        with epoch_file.open("w", newline="") as f:

            writer = csv.writer(f)

            writer.writerow([
                "epoch",
                "loss",
                "train_seconds",
                "validation_seconds",
                "cumulative_train_seconds",
                "val_auprc",
                "best_val_auprc",
                "best_epoch"
            ])

            for epoch in range(1, MAX_EPOCHS + 1):

                # ============================================
                # TRAINING ONLY
                # ============================================

                model.train()

                torch.cuda.synchronize()
                t0 = time.perf_counter()

                logits = model(features)

                loss = F.cross_entropy(
                    logits[train_ids],
                    labels[train_ids],
                    weight=weight
                )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                torch.cuda.synchronize()

                train_sec = time.perf_counter() - t0

                train_times.append(train_sec)
                cumulative += train_sec


                # ============================================
                # VALIDATION SEPARATELY
                # ============================================

                model.eval()

                torch.cuda.synchronize()
                v0 = time.perf_counter()

                with torch.no_grad():

                    vp = torch.softmax(
                        model(features),
                        dim=1
                    )[val_ids, 1]

                torch.cuda.synchronize()

                vy = labels[val_ids].cpu().numpy()
                vp = vp.cpu().numpy()

                val_ap = average_precision_score(
                    vy,
                    vp
                )

                val_sec = time.perf_counter() - v0

                val_times.append(val_sec)


                if val_ap > best_ap:

                    best_ap = float(val_ap)
                    best_epoch = epoch
                    stale = 0

                    torch.save(
                        model.state_dict(),
                        checkpoint
                    )

                else:

                    stale += 1


                writer.writerow([
                    epoch,
                    float(loss.item()),
                    train_sec,
                    val_sec,
                    cumulative,
                    float(val_ap),
                    best_ap,
                    best_epoch
                ])

                f.flush()


                if (
                    epoch == 1
                    or epoch % 10 == 0
                    or stale >= PATIENCE
                ):

                    print(
                        f"{key} | epoch={epoch:03d} | "
                        f"val_AP={val_ap:.5f} | "
                        f"best={best_ap:.5f}@{best_epoch} | "
                        f"train={train_sec:.4f}s",
                        flush=True
                    )


                if stale >= PATIENCE:

                    print(
                        f"Early stop: {epoch}",
                        flush=True
                    )

                    break


        train_times = np.asarray(train_times)
        val_times = np.asarray(val_times)


        # ====================================================
        # BEST CHECKPOINT
        # ============================================================

        model.load_state_dict(
            torch.load(
                checkpoint,
                map_location=DEVICE
            )
        )

        model.eval()


        # ====================================================
        # LATENCY
        # ============================================================

        with torch.no_grad():

            for _ in range(WARMUPS):

                _ = model(features)
                torch.cuda.synchronize()


        latency = []

        with torch.no_grad():

            for _ in range(REPEATS):

                torch.cuda.synchronize()
                t0 = time.perf_counter()

                _ = model(features)

                torch.cuda.synchronize()

                latency.append(
                    (time.perf_counter() - t0) * 1000
                )


        # ====================================================
        # PERFORMANCE VERIFICATION ONLY
        # ============================================================

        with torch.no_grad():

            probability = torch.softmax(
                model(features),
                dim=1
            )[:, 1]


        vy = labels[val_ids].cpu().numpy()
        vp = probability[val_ids].cpu().numpy()

        threshold = choose_threshold(vy, vp)


        ty = labels[test_ids].cpu().numpy()
        tp = probability[test_ids].cpu().numpy()

        pred = (tp >= threshold).astype(int)


        result = {

            "purpose":
                "Supplemental computational-efficiency "
                "backfill for Step 15.",

            "primary_result_step":
                15,

            "primary_results_replaced":
                False,

            "dataset":
                "YelpChi",

            "model":
                "BWGNN-Homo",

            "split":
                ratio,

            "split_seed":
                2,

            "train_seed":
                seed,

            "hidden":
                HIDDEN,

            "order":
                ORDER,

            "learning_rate":
                LR,

            "completed_epochs":
                len(train_times),

            "best_epoch":
                best_epoch,

            "best_val_auprc":
                best_ap,

            "parameter_count":
                int(params),

            "load_preprocess_seconds":
                float(load_seconds),

            "total_train_seconds":
                float(train_times.sum()),

            "mean_epoch_train_seconds":
                float(train_times.mean()),

            "std_epoch_train_seconds":
                float(
                    train_times.std(ddof=1)
                    if len(train_times) > 1
                    else 0
                ),

            "total_validation_seconds":
                float(val_times.sum()),

            "mean_validation_seconds":
                float(val_times.mean()),

            "inference_latency_mean_ms":
                float(np.mean(latency)),

            "inference_latency_std_ms":
                float(
                    np.std(latency, ddof=1)
                ),

            "inference_samples_ms":
                latency,

            "peak_gpu_memory_mb":
                float(
                    torch.cuda.max_memory_allocated()
                    / 1024**2
                ),

            "val_threshold":
                float(threshold),

            "verification_test_auprc":
                float(
                    average_precision_score(ty, tp)
                ),

            "verification_test_auroc":
                float(
                    roc_auc_score(ty, tp)
                ),

            "verification_test_macro_f1":
                float(
                    f1_score(
                        ty,
                        pred,
                        average="macro"
                    )
                ),

            "verification_test_fraud_precision":
                float(
                    precision_score(
                        ty,
                        pred,
                        zero_division=0
                    )
                ),

            "verification_test_fraud_recall":
                float(
                    recall_score(
                        ty,
                        pred,
                        zero_division=0
                    )
                ),

            "verification_test_fraud_f1":
                float(
                    f1_score(
                        ty,
                        pred,
                        pos_label=1,
                        zero_division=0
                    )
                ),

            "verification_test_gmean":
                float(
                    gmean(ty, pred)
                )
        }


        result_file.write_text(
            json.dumps(
                result,
                indent=2
            )
        )


        print(
            f"DONE | {key} | "
            f"train={result['mean_epoch_train_seconds']:.4f}s/epoch | "
            f"latency={result['inference_latency_mean_ms']:.3f}ms",
            flush=True
        )


        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# AGGREGATE
# ============================================================

results = []

for ratio in RATIOS:

    for seed in SEEDS:

        p = RUN_DIR / f"{ratio}_seed{seed}.json"

        if not p.exists():

            raise RuntimeError(
                f"Missing run: {p}"
            )

        results.append(
            json.loads(
                p.read_text()
            )
        )


(ROOT / "yelp_timing_backfill_all_runs.json").write_text(
    json.dumps(
        results,
        indent=2
    )
)


csv_path = ROOT / "yelp_timing_backfill_all_runs.csv"

fields = [
    k
    for k in results[0]
    if k != "inference_samples_ms"
]

with csv_path.open("w", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fields
    )

    writer.writeheader()

    for r in results:

        writer.writerow({
            k: v
            for k, v in r.items()
            if k != "inference_samples_ms"
        })


print(
    "\n===== YELPCHI TIMING-BACKFILL SUMMARY =====",
    flush=True
)

for ratio in RATIOS:

    rows = [
        r for r in results
        if r["split"] == ratio
    ]

    print(f"\n{ratio}", flush=True)

    for metric in [
        "mean_epoch_train_seconds",
        "total_train_seconds",
        "inference_latency_mean_ms",
        "peak_gpu_memory_mb"
    ]:

        x = np.asarray([
            r[metric]
            for r in rows
        ])

        print(
            f"  {metric:<30} "
            f"{x.mean():.6f} ± {x.std(ddof=1):.6f}",
            flush=True
        )


print(
    "\n===== STEP 24B GATE =====",
    flush=True
)

print(
    "PASS — YelpChi computational-efficiency "
    "backfill complete for all 12 runs.",
    flush=True
)

print(
    "Step 15 primary results remain unchanged.",
    flush=True
)
'''

RUNNER.write_text(
    code,
    encoding="utf-8"
)

print("===== STEP 24B-1 =====")
print("PASS — compact runner created.")
print("Runner:", RUNNER)
print("Size:", RUNNER.stat().st_size, "bytes")

===== STEP 24B-1 =====
PASS — compact runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/yelp_homo_timing_backfill.py
Size: 17590 bytes


In [12]:
# ============================================================
# STEP 24B-2 — LAUNCH YELP TIMING BACKFILL
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = WORK / "adapters/yelp_homo_timing_backfill.py"
PYTHON = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

LOG = WORK / "yelp_step24b.log"
PID_FILE = WORK / "yelp_step24b.pid"

assert RUNNER.exists()
assert PYTHON.exists()


# Check actual runner, not pgrep itself
check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[y]elp_homo_timing_backfill.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "Yelp Step 24B is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(map(str, cuda))
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


log = open(LOG, "a", buffering=1)

process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log.close()


print("===== STEP 24B STARTED =====")
print("PID:", process.pid)
print("Log:", LOG)
print("Additional timing work for Step 15 only.")
print("Do NOT run this launcher again.")

===== STEP 24B STARTED =====
PID: 234
Log: /kaggle/working/comp8851_bwgnn/yelp_step24b.log
Additional timing work for Step 15 only.
Do NOT run this launcher again.


### Step 24C — YelpChi Timing-Backfill Progress Check

This cell monitors the supplemental Step 24B timing run only.

It does not start or modify any experiment and may be rerun safely.

In [14]:
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "yelp_step24b.pid"
LOG = WORK / "yelp_step24b.log"

RUN_DIR = (
    WORK /
    "results/timing_backfill/yelp_homo/runs"
)

print("===== STEP 24B STATUS =====")

if PID_FILE.exists():

    pid = int(
        PID_FILE.read_text().strip()
    )

    cmd = Path(
        f"/proc/{pid}/cmdline"
    )

    running = (
        cmd.exists()
        and b"yelp_homo_timing_backfill.py"
        in cmd.read_bytes()
    )

    print("PID:", pid)
    print("Process running:", running)

else:

    print("No PID file.")


runs = (
    sorted(RUN_DIR.glob("*.json"))
    if RUN_DIR.exists()
    else []
)

print(
    f"\nCompleted timing runs: {len(runs)}/12"
)

for p in runs:
    print(" ✓", p.stem)


print("\n===== LATEST LOG =====")

if LOG.exists():

    lines = LOG.read_text(
        errors="replace"
    ).splitlines()

    for line in lines[-50:]:
        print(line)

else:

    print("No log yet.")

===== STEP 24B STATUS =====
PID: 234
Process running: False

Completed timing runs: 12/12
 ✓ TR10_seed2
 ✓ TR10_seed42
 ✓ TR10_seed72
 ✓ TR20_seed2
 ✓ TR20_seed42
 ✓ TR20_seed72
 ✓ TR30_seed2
 ✓ TR30_seed42
 ✓ TR30_seed72
 ✓ TR40_seed2
 ✓ TR40_seed42
 ✓ TR40_seed72

===== LATEST LOG =====
TR10_seed42 | epoch=040 | val_AP=0.39708 | best=0.39975@36 | train=0.1601s
TR10_seed42 | epoch=050 | val_AP=0.39353 | best=0.40494@48 | train=0.1524s
TR10_seed42 | epoch=060 | val_AP=0.40486 | best=0.40539@58 | train=0.1520s
TR10_seed42 | epoch=070 | val_AP=0.40565 | best=0.41220@68 | train=0.1593s
TR10_seed42 | epoch=080 | val_AP=0.40511 | best=0.41288@75 | train=0.1590s
TR10_seed42 | epoch=090 | val_AP=0.41231 | best=0.41619@81 | train=0.1547s
TR10_seed42 | epoch=100 | val_AP=0.40152 | best=0.41619@81 | train=0.1558s
DONE | TR10_seed42 | train=0.1576s/epoch | latency=78.538ms

START | TR10 | seed=72
TR10_seed72 | epoch=001 | val_AP=0.23628 | best=0.23628@1 | train=0.1409s
TR10_seed72 | epoch=010 | v

## Step 25 — Amazon Computational-Efficiency Backfill

**Additional work related to Step 19.**

The primary Amazon BWGNN-Homo benchmark was completed in **Step 19**.
Its original performance results remain unchanged.

This step is a supplemental rerun performed only to obtain the
computational-efficiency evidence that was not separately preserved during
Step 19.

No hyperparameter tuning is repeated.

### Frozen Step 19 configuration

- Model: BWGNN-Homo
- Hidden dimension: 32
- Wavelet order: 2
- Learning rate: 0.01
- Split seed: 2
- Training seeds: 2, 42, 72
- Ratios: TR40, TR30, TR20, TR10
- Maximum epochs: 100
- Early stopping patience: 20
- Checkpoint selection: validation AUPRC
- Threshold selection: validation Macro-F1

### Amazon supervised benchmark eligibility

The graph contains 11,944 nodes, but the official BWGNN Amazon benchmark
excludes node IDs `0–3304` from supervised splitting.

Eligible supervised nodes:

- IDs: 3305–11943
- Total: 8,639
- Normal: 7,818
- Fraud: 821

The complete graph is still used for transductive message passing.

### Graph construction

Amazon is reconstructed from its three original relations and converted to
the homogeneous BWGNN representation with one self-loop per node.

Expected graph:

- Nodes: 11,944
- Relation edges before self-loops: 9,557,648
- Homogeneous edges after self-loops: 9,569,592
- Features: 25

### Additional evidence recorded

For every ratio × training-seed run:

- Raw per-epoch training-only time
- Validation time separately
- Total training-only time
- Mean and standard deviation of epoch training time
- Full-graph inference latency
- Peak GPU memory
- Parameter count

Performance metrics are also recorded as a verification check only.
They do **not replace Step 19 primary results**.

All new outputs are written separately under:

`results/timing_backfill/amazon_homo/`

In [15]:
# ============================================================
# STEP 25A — AMAZON TIMING-BACKFILL SAFETY CHECK
# ============================================================

from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/comp8851_bwgnn")

print(
    "Environment:",
    (WORK / "envs/bwgnn-author/bin/python").exists()
)

print(
    "BWGNN adapter:",
    (WORK / "adapters/BWGNN_gpu.py").exists()
)

print(
    "Amazon split:",
    (
        WORK /
        "shared/timing_backfill_splits/"
        "amazon_seed2_nested_splits.npz"
    ).exists()
)

p = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[a]mazon_homo_timing_backfill.py' || true"
    ],
    capture_output=True,
    text=True
)

print(
    "Old Amazon process:",
    p.stdout.strip()
    if p.stdout.strip()
    else "NONE"
)

Environment: True
BWGNN adapter: True
Amazon split: True
Old Amazon process: NONE


### Step 25B — Create Amazon Timing-Backfill Runner

This cell only creates the supplemental Amazon timing runner.

It does not start training and does not modify the Step 19 primary result
directory.

In [16]:
# ============================================================
# STEP 25B — CREATE AMAZON TIMING RUNNER
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/amazon_homo_timing_backfill.py"
)

RUNNER.parent.mkdir(
    parents=True,
    exist_ok=True
)

code = r'''
import sys
import gc
import csv
import json
import time
import math
import random
import contextlib
import io

from pathlib import Path

import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


# ============================================================
# FROZEN STEP 19 CONFIG
# ============================================================

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "amazon/Amazon.mat"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/shared/"
    "timing_backfill_splits/"
    "amazon_seed2_nested_splits.npz"
)

ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "results/timing_backfill/amazon_homo"
)

RUN_DIR = ROOT / "runs"
TIME_DIR = ROOT / "epoch_times"

CKPT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "checkpoints/timing_backfill/amazon_homo"
)

for directory in [
    ROOT,
    RUN_DIR,
    TIME_DIR,
    CKPT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


HIDDEN = 32
ORDER = 2
LR = 0.01

MAX_EPOCHS = 100
PATIENCE = 20

SEEDS = [
    2,
    42,
    72
]

RATIOS = [
    "TR40",
    "TR30",
    "TR20",
    "TR10"
]

DEVICE = torch.device("cuda:0")

WARMUPS = 3
REPEATS = 10


# ============================================================
# HELPERS
# ============================================================

def choose_threshold(y, probability):

    best = None

    for threshold in np.arange(
        0.01,
        1.00,
        0.01
    ):

        prediction = (
            probability >= threshold
        ).astype(int)

        candidate = (
            f1_score(
                y,
                prediction,
                average="macro"
            ),
            recall_score(
                y,
                prediction,
                zero_division=0
            ),
            -abs(
                float(threshold) - 0.5
            ),
            float(threshold)
        )

        if (
            best is None
            or candidate > best
        ):
            best = candidate

    return best[3]


def gmean(y, prediction):

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            prediction,
            labels=[0, 1]
        ).ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn)
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else 0.0
    )

    return math.sqrt(
        sensitivity * specificity
    )


# ============================================================
# LOAD AMAZON
# ============================================================

print(
    "===== STEP 25 — AMAZON TIMING BACKFILL =====",
    flush=True
)

print(
    "Supplemental to Step 19 only.",
    flush=True
)

load_start = (
    time.perf_counter()
)

mat = loadmat(
    DATA_PATH
)


# ============================================================
# FEATURES / LABELS
# ============================================================

features_np = mat[
    "features"
]

if sp.issparse(
    features_np
):
    features_np = (
        features_np.toarray()
    )

features_np = np.asarray(
    features_np,
    dtype=np.float32
)

labels_np = np.asarray(
    mat["label"]
).reshape(-1).astype(
    np.int64
)


assert features_np.shape == (
    11944,
    25
)

assert labels_np.shape == (
    11944,
)


eligible = labels_np[
    3305:
]

assert len(eligible) == 8639

assert int(
    (eligible == 0).sum()
) == 7818

assert int(
    (eligible == 1).sum()
) == 821


# ============================================================
# THREE AMAZON RELATIONS
# ============================================================

relations = {}

edge_total = 0

for rel, key in [
    ("upu", "net_upu"),
    ("usu", "net_usu"),
    ("uvu", "net_uvu")
]:

    if key not in mat:

        raise KeyError(
            f"Amazon.mat missing relation: {key}"
        )

    adjacency = sp.coo_matrix(
        mat[key]
    )

    src = torch.from_numpy(
        adjacency.row.astype(
            np.int64
        )
    )

    dst = torch.from_numpy(
        adjacency.col.astype(
            np.int64
        )
    )

    edge_total += len(src)

    relations[
        ("user", rel, "user")
    ] = (
        src,
        dst
    )


assert edge_total == 9557648


hetero = dgl.heterograph(
    relations,
    num_nodes_dict={
        "user": 11944
    }
)

hetero.nodes[
    "user"
].data[
    "feature"
] = torch.from_numpy(
    features_np
)

hetero.nodes[
    "user"
].data[
    "label"
] = torch.from_numpy(
    labels_np
)


# ============================================================
# HOMOGENEOUS + SELF-LOOPS
# ============================================================

graph = dgl.to_homogeneous(
    hetero,
    ndata=[
        "feature",
        "label"
    ]
)

graph = dgl.add_self_loop(
    graph
)


assert graph.num_nodes() == 11944

assert graph.num_edges() == 9569592


features = (
    graph.ndata[
        "feature"
    ].float()
)

labels = (
    graph.ndata[
        "label"
    ].long()
)


# ============================================================
# SPLITS
# ============================================================

splits = np.load(
    SPLIT_PATH
)

val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long
)

test_ids = torch.tensor(
    splits["test"],
    dtype=torch.long
)


assert int(
    np.min(
        splits["TR40"]
    )
) >= 3305

assert int(
    np.min(
        splits["val"]
    )
) >= 3305

assert int(
    np.min(
        splits["test"]
    )
) >= 3305


# ============================================================
# GPU
# ============================================================

graph = graph.to(
    DEVICE
)

features = features.to(
    DEVICE
)

labels = labels.to(
    DEVICE
)

val_ids = val_ids.to(
    DEVICE
)

test_ids = test_ids.to(
    DEVICE
)

torch.cuda.synchronize()

load_seconds = (
    time.perf_counter()
    - load_start
)


print(
    "Graph nodes:",
    graph.num_nodes(),
    flush=True
)

print(
    "Relation edges:",
    edge_total,
    flush=True
)

print(
    "Homo + self-loop edges:",
    graph.num_edges(),
    flush=True
)

print(
    "Features:",
    tuple(
        features.shape
    ),
    flush=True
)

print(
    "Eligible supervised nodes:",
    8639,
    flush=True
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
    flush=True
)

print(
    "Load/preprocess:",
    f"{load_seconds:.3f}s",
    flush=True
)


del (
    mat,
    hetero,
    relations,
    features_np,
    labels_np
)

gc.collect()


# ============================================================
# 12 SUPPLEMENTAL TIMING RUNS
# ============================================================

for ratio in RATIOS:

    train_ids = torch.tensor(
        splits[ratio],
        dtype=torch.long,
        device=DEVICE
    )


    for seed in SEEDS:

        key = (
            f"{ratio}_seed{seed}"
        )

        result_file = (
            RUN_DIR /
            f"{key}.json"
        )

        epoch_file = (
            TIME_DIR /
            f"{key}_epoch_times.csv"
        )

        checkpoint = (
            CKPT_DIR /
            f"{key}.pt"
        )


        print(
            "\n============================================",
            flush=True
        )

        print(
            f"START | {ratio} | seed={seed}",
            flush=True
        )


        # ----------------------------------------------------
        # RESUME — skip completed runs
        # ----------------------------------------------------

        if result_file.exists():

            print(
                "SKIP — already complete.",
                flush=True
            )

            continue


        if epoch_file.exists():
            epoch_file.unlink()

        if checkpoint.exists():
            checkpoint.unlink()


        # ====================================================
        # REPRODUCIBILITY
        # ====================================================

        random.seed(seed)
        np.random.seed(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


        # ====================================================
        # MODEL
        # ====================================================

        with contextlib.redirect_stdout(
            io.StringIO()
        ):

            model = BWGNN(
                in_feats=25,
                h_feats=HIDDEN,
                num_classes=2,
                graph=graph,
                d=ORDER
            ).to(
                DEVICE
            )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            betas=(
                0.9,
                0.999
            ),
            eps=1e-8
        )


        parameter_count = int(
            sum(
                p.numel()
                for p in model.parameters()
                if p.requires_grad
            )
        )


        # ====================================================
        # TRAIN-ONLY CLASS WEIGHT
        # ====================================================

        train_labels = labels[
            train_ids
        ]

        normal = int(
            (
                train_labels == 0
            ).sum().item()
        )

        fraud = int(
            (
                train_labels == 1
            ).sum().item()
        )

        weight = torch.tensor(
            [
                1.0,
                normal / fraud
            ],
            dtype=torch.float32,
            device=DEVICE
        )


        print(
            "Train normal/fraud:",
            normal,
            fraud,
            flush=True
        )


        gc.collect()
        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


        # ====================================================
        # EPOCH STATE
        # ====================================================

        train_times = []
        validation_times = []

        best_ap = -1.0
        best_epoch = -1

        stale = 0
        cumulative = 0.0


        # ====================================================
        # RAW TIMING CSV
        # ====================================================

        with epoch_file.open(
            "w",
            newline=""
        ) as f:

            writer = csv.writer(
                f
            )

            writer.writerow([
                "epoch",
                "loss",
                "train_seconds",
                "validation_seconds",
                "cumulative_train_seconds",
                "val_auprc",
                "best_val_auprc",
                "best_epoch"
            ])


            # =================================================
            # EPOCHS
            # =================================================

            for epoch in range(
                1,
                MAX_EPOCHS + 1
            ):

                # =============================================
                # TRAINING TIMER
                # =============================================

                model.train()

                torch.cuda.synchronize()

                train_start = (
                    time.perf_counter()
                )


                logits = model(
                    features
                )

                loss = F.cross_entropy(
                    logits[
                        train_ids
                    ],
                    labels[
                        train_ids
                    ],
                    weight=weight
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()


                torch.cuda.synchronize()

                train_seconds = (
                    time.perf_counter()
                    - train_start
                )


                train_times.append(
                    train_seconds
                )

                cumulative += (
                    train_seconds
                )


                # =============================================
                # VALIDATION SEPARATELY
                # =============================================

                model.eval()

                torch.cuda.synchronize()

                validation_start = (
                    time.perf_counter()
                )


                with torch.no_grad():

                    val_probability = (
                        torch.softmax(
                            model(
                                features
                            ),
                            dim=1
                        )[
                            val_ids,
                            1
                        ]
                    )


                torch.cuda.synchronize()


                val_y = (
                    labels[
                        val_ids
                    ]
                    .cpu()
                    .numpy()
                )

                val_probability = (
                    val_probability
                    .cpu()
                    .numpy()
                )


                val_ap = float(
                    average_precision_score(
                        val_y,
                        val_probability
                    )
                )


                validation_seconds = (
                    time.perf_counter()
                    - validation_start
                )


                validation_times.append(
                    validation_seconds
                )


                # =============================================
                # CHECKPOINT
                # =============================================

                if val_ap > best_ap:

                    best_ap = (
                        val_ap
                    )

                    best_epoch = (
                        epoch
                    )

                    stale = 0

                    torch.save(
                        model.state_dict(),
                        checkpoint
                    )

                else:

                    stale += 1


                # =============================================
                # RAW EPOCH RECORD
                # =============================================

                writer.writerow([
                    epoch,
                    float(
                        loss.item()
                    ),
                    train_seconds,
                    validation_seconds,
                    cumulative,
                    val_ap,
                    best_ap,
                    best_epoch
                ])

                f.flush()


                # =============================================
                # VISIBLE LOG PROGRESS
                #
                # Every 5 epochs rather than 10.
                # =============================================

                if (
                    epoch == 1
                    or epoch % 5 == 0
                    or stale >= PATIENCE
                ):

                    print(
                        f"{key} | "
                        f"epoch={epoch:03d}/{MAX_EPOCHS} | "
                        f"val_AP={val_ap:.5f} | "
                        f"best={best_ap:.5f}"
                        f"@{best_epoch} | "
                        f"train={train_seconds:.4f}s | "
                        f"patience={stale}/{PATIENCE}",
                        flush=True
                    )


                if stale >= PATIENCE:

                    print(
                        f"Early stop: {epoch}",
                        flush=True
                    )

                    break


        # ====================================================
        # TIMING SUMMARY
        # ====================================================

        train_times = np.asarray(
            train_times,
            dtype=float
        )

        validation_times = np.asarray(
            validation_times,
            dtype=float
        )


        # ====================================================
        # RESTORE BEST
        # ====================================================

        model.load_state_dict(
            torch.load(
                checkpoint,
                map_location=DEVICE
            )
        )

        model.eval()


        # ====================================================
        # INFERENCE LATENCY
        # ====================================================

        with torch.no_grad():

            for _ in range(
                WARMUPS
            ):

                _ = model(
                    features
                )

                torch.cuda.synchronize()


        latency = []


        with torch.no_grad():

            for _ in range(
                REPEATS
            ):

                torch.cuda.synchronize()

                start = (
                    time.perf_counter()
                )

                _ = model(
                    features
                )

                torch.cuda.synchronize()

                latency.append(
                    (
                        time.perf_counter()
                        - start
                    ) * 1000.0
                )


        # ====================================================
        # VERIFICATION METRICS ONLY
        # ====================================================

        with torch.no_grad():

            probability = (
                torch.softmax(
                    model(
                        features
                    ),
                    dim=1
                )[:, 1]
            )


        val_y = (
            labels[
                val_ids
            ]
            .cpu()
            .numpy()
        )

        val_probability = (
            probability[
                val_ids
            ]
            .cpu()
            .numpy()
        )


        threshold = (
            choose_threshold(
                val_y,
                val_probability
            )
        )


        test_y = (
            labels[
                test_ids
            ]
            .cpu()
            .numpy()
        )

        test_probability = (
            probability[
                test_ids
            ]
            .cpu()
            .numpy()
        )

        prediction = (
            test_probability
            >= threshold
        ).astype(
            int
        )


        result = {

            "purpose":
                "Supplemental computational-efficiency "
                "backfill for Step 19.",

            "primary_result_step":
                19,

            "primary_results_replaced":
                False,

            "dataset":
                "Amazon",

            "model":
                "BWGNN-Homo",

            "split":
                ratio,

            "split_seed":
                2,

            "train_seed":
                seed,

            "hidden":
                HIDDEN,

            "order":
                ORDER,

            "learning_rate":
                LR,

            "completed_epochs":
                int(
                    len(
                        train_times
                    )
                ),

            "best_epoch":
                best_epoch,

            "best_val_auprc":
                best_ap,

            "parameter_count":
                parameter_count,

            "load_preprocess_seconds":
                float(
                    load_seconds
                ),

            "total_train_seconds":
                float(
                    train_times.sum()
                ),

            "mean_epoch_train_seconds":
                float(
                    train_times.mean()
                ),

            "std_epoch_train_seconds":
                float(
                    train_times.std(
                        ddof=1
                    )
                    if len(
                        train_times
                    ) > 1
                    else 0.0
                ),

            "total_validation_seconds":
                float(
                    validation_times.sum()
                ),

            "mean_validation_seconds":
                float(
                    validation_times.mean()
                ),

            "inference_latency_mean_ms":
                float(
                    np.mean(
                        latency
                    )
                ),

            "inference_latency_std_ms":
                float(
                    np.std(
                        latency,
                        ddof=1
                    )
                ),

            "inference_samples_ms":
                latency,

            "peak_gpu_memory_mb":
                float(
                    torch.cuda.max_memory_allocated()
                    / (1024 ** 2)
                ),

            "val_threshold":
                float(
                    threshold
                ),

            "verification_test_auprc":
                float(
                    average_precision_score(
                        test_y,
                        test_probability
                    )
                ),

            "verification_test_auroc":
                float(
                    roc_auc_score(
                        test_y,
                        test_probability
                    )
                ),

            "verification_test_macro_f1":
                float(
                    f1_score(
                        test_y,
                        prediction,
                        average="macro"
                    )
                ),

            "verification_test_fraud_precision":
                float(
                    precision_score(
                        test_y,
                        prediction,
                        zero_division=0
                    )
                ),

            "verification_test_fraud_recall":
                float(
                    recall_score(
                        test_y,
                        prediction,
                        zero_division=0
                    )
                ),

            "verification_test_fraud_f1":
                float(
                    f1_score(
                        test_y,
                        prediction,
                        pos_label=1,
                        zero_division=0
                    )
                ),

            "verification_test_gmean":
                float(
                    gmean(
                        test_y,
                        prediction
                    )
                )
        }


        result_file.write_text(
            json.dumps(
                result,
                indent=2
            )
        )


        print(
            f"DONE | {key} | "
            f"epochs={len(train_times)} | "
            f"train="
            f"{result['mean_epoch_train_seconds']:.4f}s/epoch | "
            f"latency="
            f"{result['inference_latency_mean_ms']:.3f}ms",
            flush=True
        )


        del (
            model,
            optimizer
        )

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# AGGREGATE
# ============================================================

results = []


for ratio in RATIOS:

    for seed in SEEDS:

        path = (
            RUN_DIR /
            f"{ratio}_seed{seed}.json"
        )

        if not path.exists():

            raise RuntimeError(
                f"Missing completed run: {path}"
            )

        results.append(
            json.loads(
                path.read_text()
            )
        )


master_json = (
    ROOT /
    "amazon_timing_backfill_all_runs.json"
)

master_json.write_text(
    json.dumps(
        results,
        indent=2
    )
)


master_csv = (
    ROOT /
    "amazon_timing_backfill_all_runs.csv"
)

fields = [
    key
    for key in results[0]
    if key
    != "inference_samples_ms"
]

with master_csv.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fields
    )

    writer.writeheader()

    for row in results:

        writer.writerow({
            key: value
            for key, value
            in row.items()
            if key
            != "inference_samples_ms"
        })


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n===== AMAZON TIMING-BACKFILL SUMMARY =====",
    flush=True
)


for ratio in RATIOS:

    rows = [
        row
        for row in results
        if row["split"] == ratio
    ]

    print(
        f"\n{ratio}",
        flush=True
    )


    for metric in [
        "mean_epoch_train_seconds",
        "total_train_seconds",
        "inference_latency_mean_ms",
        "peak_gpu_memory_mb"
    ]:

        values = np.asarray([
            row[metric]
            for row in rows
        ])

        print(
            f"  {metric:<30} "
            f"{values.mean():.6f} "
            f"± {values.std(ddof=1):.6f}",
            flush=True
        )


print(
    "\n===== STEP 25 GATE =====",
    flush=True
)

print(
    "PASS — Amazon computational-efficiency "
    "backfill complete for all 12 runs.",
    flush=True
)

print(
    "Step 19 primary results remain unchanged.",
    flush=True
)
'''

RUNNER.write_text(
    code,
    encoding="utf-8"
)


print(
    "===== STEP 25B ====="
)

print(
    "PASS — Amazon timing runner created."
)

print(
    "Runner:",
    RUNNER
)

print(
    "Size:",
    RUNNER.stat().st_size,
    "bytes"
)

===== STEP 25B =====
PASS — Amazon timing runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/amazon_homo_timing_backfill.py
Size: 26041 bytes


### Step 25C — Launch Amazon Timing Backfill

This cell starts the Step 25 supplemental Amazon timing benchmark in the
background.

Run this launcher **once only**.

The training continues independently of the progress-monitor cell.

In [18]:
# ============================================================
# STEP 25C — LAUNCH AMAZON TIMING BACKFILL
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

RUNNER = (
    WORK /
    "adapters/amazon_homo_timing_backfill.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = (
    WORK /
    "amazon_step25.log"
)

PID_FILE = (
    WORK /
    "amazon_step25.pid"
)


assert RUNNER.exists()
assert PYTHON.exists()


check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[a]mazon_homo_timing_backfill.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "Amazon Step 25 is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env[
    "CUDA_VISIBLE_DEVICES"
] = "0"

env[
    "DGLBACKEND"
] = "pytorch"

env[
    "PYTHONUNBUFFERED"
] = "1"


cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env[
    "LD_LIBRARY_PATH"
] = (
    ":".join(
        map(
            str,
            cuda
        )
    )
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)


log_handle = open(
    LOG,
    "a",
    buffering=1
)


process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)


PID_FILE.write_text(
    str(
        process.pid
    )
)

log_handle.close()


print(
    "===== STEP 25 STARTED ====="
)

print(
    "Purpose: ADDITIONAL Amazon timing evidence for Step 19."
)

print(
    "Step 19 primary results are NOT being modified."
)

print(
    "PID:",
    process.pid
)

print(
    "Log:",
    LOG
)

print(
    "\nDo NOT run this launcher again."
)

print(
    "Run Step 25D for LIVE progress."
)

RuntimeError: Amazon Step 25 is already running:
root         255      58 99 06:36 ?        00:00:04 /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python /kaggle/working/comp8851_bwgnn/adapters/amazon_homo_timing_backfill.py


### Step 25D — Live Amazon Timing-Backfill Progress

The Amazon Step 25 background runner is already active.

The Step 25C duplicate-launch warning is not an experiment failure; it is the
safety guard preventing a second Amazon process from using the same GPU.

This cell only monitors the active supplemental timing run for Step 19.
It refreshes every 3 seconds and does not modify or restart training.

In [19]:
# ============================================================
# STEP 25D — ROBUST LIVE AMAZON PROGRESS MONITOR
# ============================================================

from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER_NAME = (
    "/kaggle/working/comp8851_bwgnn/"
    "adapters/amazon_homo_timing_backfill.py"
)

LOG = WORK / "amazon_step25.log"

RUN_DIR = (
    WORK /
    "results/timing_backfill/amazon_homo/runs"
)


def find_amazon_process():

    result = subprocess.run(
        ["ps", "-eo", "pid,args"],
        capture_output=True,
        text=True
    )

    matches = []

    for line in result.stdout.splitlines():

        if RUNNER_NAME in line:

            parts = line.strip().split(
                maxsplit=1
            )

            if parts:
                matches.append(
                    int(parts[0])
                )

    return matches


try:

    while True:

        clear_output(
            wait=True
        )

        pids = find_amazon_process()

        running = (
            len(pids) > 0
        )


        # ----------------------------------------------------
        # Completed run records
        # ----------------------------------------------------

        runs = (
            sorted(
                RUN_DIR.glob("*.json")
            )
            if RUN_DIR.exists()
            else []
        )


        print(
            "===== STEP 25 — LIVE AMAZON PROGRESS ====="
        )

        print(
            "Process running:",
            running
        )

        print(
            "PID:",
            pids[0]
            if pids
            else "NONE"
        )

        print(
            f"Completed runs: {len(runs)}/12"
        )


        for p in runs:

            print(
                " ✓",
                p.stem
            )


        # ----------------------------------------------------
        # Latest progress from log
        # ----------------------------------------------------

        print(
            "\n----- CURRENT TRAINING -----"
        )


        lines = []

        if LOG.exists():

            lines = LOG.read_text(
                errors="replace"
            ).splitlines()


            useful = [
                line
                for line in lines
                if (
                    line.startswith(
                        "START |"
                    )
                    or "epoch=" in line
                    or line.startswith(
                        "DONE |"
                    )
                    or line.startswith(
                        "Early stop:"
                    )
                )
            ]


            if useful:

                for line in useful[-10:]:

                    print(line)

            else:

                print(
                    "Loading Amazon graph/model..."
                )

        else:

            print(
                "Waiting for Amazon log..."
            )


        # ----------------------------------------------------
        # Successful completion
        # ----------------------------------------------------

        if len(runs) == 12:

            print(
                "\n============================================"
            )

            print(
                "ALL 12 AMAZON TIMING RUNS COMPLETE"
            )

            # allow runner a moment to print summary
            time.sleep(2)

            if LOG.exists():

                final_lines = LOG.read_text(
                    errors="replace"
                ).splitlines()

                print(
                    "\n----- FINAL SUMMARY / GATE -----"
                )

                for line in final_lines[-65:]:

                    print(line)

            break


        # ----------------------------------------------------
        # Process died before completing all runs
        # ----------------------------------------------------

        if not running:

            print(
                "\nWARNING — Amazon process stopped "
                "before 12/12 runs completed."
            )

            if lines:

                print(
                    "\n----- LAST LOG LINES -----"
                )

                for line in lines[-50:]:

                    print(line)

            break


        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(3)


except KeyboardInterrupt:

    print(
        "\nLive monitor stopped."
    )

    print(
        "Amazon training continues in the background."
    )

===== STEP 25 — LIVE AMAZON PROGRESS =====
Process running: False
PID: NONE
Completed runs: 12/12
 ✓ TR10_seed2
 ✓ TR10_seed42
 ✓ TR10_seed72
 ✓ TR20_seed2
 ✓ TR20_seed42
 ✓ TR20_seed72
 ✓ TR30_seed2
 ✓ TR30_seed42
 ✓ TR30_seed72
 ✓ TR40_seed2
 ✓ TR40_seed42
 ✓ TR40_seed72

----- CURRENT TRAINING -----
TR10_seed72 | epoch=020/100 | val_AP=0.87051 | best=0.87051@20 | train=0.0541s | patience=0/20
TR10_seed72 | epoch=025/100 | val_AP=0.87232 | best=0.87232@25 | train=0.0535s | patience=0/20
TR10_seed72 | epoch=030/100 | val_AP=0.87538 | best=0.87538@30 | train=0.0529s | patience=0/20
TR10_seed72 | epoch=035/100 | val_AP=0.87215 | best=0.87569@31 | train=0.0535s | patience=4/20
TR10_seed72 | epoch=040/100 | val_AP=0.87285 | best=0.87569@31 | train=0.0535s | patience=9/20
TR10_seed72 | epoch=045/100 | val_AP=0.87393 | best=0.87569@31 | train=0.0539s | patience=14/20
TR10_seed72 | epoch=050/100 | val_AP=0.86932 | best=0.87569@31 | train=0.0540s | patience=19/20
TR10_seed72 | epoch=051/100 |

## Step 26 — Export Supplemental Computational-Efficiency Evidence

This step packages the additional computational-efficiency evidence produced
for the previously completed YelpChi and Amazon BWGNN-Homo benchmarks.

### Relation to previous benchmark results

- YelpChi primary benchmark: **Step 15**
- YelpChi timing backfill: **Step 24B**

- Amazon primary benchmark: **Step 19**
- Amazon timing backfill: **Step 25**

The timing-backfill results are supplemental evidence only and do not replace
the primary performance results reported in Steps 15 and 19.

### Exported evidence

For both YelpChi and Amazon, the package includes:

- All 12 individual ratio × seed run records
- Raw per-epoch training and validation timing CSV files
- Aggregated timing CSV and JSON files
- Exact nested split files used for the timing reruns
- Timing-backfill runner scripts
- Execution logs

Large model checkpoint `.pt` files are intentionally excluded.

The resulting ZIP is retained as reproducibility evidence before proceeding to
the next BWGNN dataset.

In [20]:
# ============================================================
# STEP 26 — PACKAGE YELPCHI + AMAZON TIMING EVIDENCE
# ============================================================

from pathlib import Path
import shutil
import json

WORK = Path("/kaggle/working/comp8851_bwgnn")

EXPORT = Path(
    "/kaggle/working/bwgnn_yelp_amazon_timing_evidence"
)

ZIP_BASE = Path(
    "/kaggle/working/"
    "bwgnn_yelp_amazon_timing_evidence_20260909"
)

if EXPORT.exists():
    shutil.rmtree(EXPORT)

EXPORT.mkdir(
    parents=True,
    exist_ok=True
)


def copy_file(src, dst):

    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(
            f"Required evidence missing:\n{src}"
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        src,
        dst
    )

    print(
        "✓",
        src
    )


def copy_dir(src, dst):

    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(
            f"Required evidence directory missing:\n{src}"
        )

    if dst.exists():
        shutil.rmtree(dst)

    shutil.copytree(
        src,
        dst
    )

    print(
        "✓",
        src
    )


# ============================================================
# 1. VERIFY COMPLETED RUN COUNTS
# ============================================================

YELP_RESULT = (
    WORK /
    "results/timing_backfill/yelp_homo"
)

AMAZON_RESULT = (
    WORK /
    "results/timing_backfill/amazon_homo"
)


yelp_runs = list(
    (YELP_RESULT / "runs").glob(
        "*.json"
    )
)

amazon_runs = list(
    (AMAZON_RESULT / "runs").glob(
        "*.json"
    )
)


print(
    "===== STEP 26 — EVIDENCE VERIFICATION ====="
)

print(
    "YelpChi completed runs:",
    len(yelp_runs),
    "/12"
)

print(
    "Amazon completed runs:",
    len(amazon_runs),
    "/12"
)


assert len(yelp_runs) == 12
assert len(amazon_runs) == 12


yelp_epoch_files = list(
    (
        YELP_RESULT /
        "epoch_times"
    ).glob(
        "*.csv"
    )
)

amazon_epoch_files = list(
    (
        AMAZON_RESULT /
        "epoch_times"
    ).glob(
        "*.csv"
    )
)


assert len(yelp_epoch_files) == 12
assert len(amazon_epoch_files) == 12


print(
    "YelpChi epoch timing files:",
    len(yelp_epoch_files),
    "/12"
)

print(
    "Amazon epoch timing files:",
    len(amazon_epoch_files),
    "/12"
)


# ============================================================
# 2. YELPCHI EVIDENCE
# ============================================================

print(
    "\n===== YELPCHI ====="
)

copy_file(
    YELP_RESULT /
    "yelp_timing_backfill_all_runs.csv",

    EXPORT /
    "yelp/results/"
    "yelp_timing_backfill_all_runs.csv"
)

copy_file(
    YELP_RESULT /
    "yelp_timing_backfill_all_runs.json",

    EXPORT /
    "yelp/results/"
    "yelp_timing_backfill_all_runs.json"
)

copy_dir(
    YELP_RESULT /
    "runs",

    EXPORT /
    "yelp/results/runs"
)

copy_dir(
    YELP_RESULT /
    "epoch_times",

    EXPORT /
    "yelp/results/epoch_times"
)

copy_file(
    WORK /
    "shared/timing_backfill_splits/"
    "yelp_seed2_nested_splits.npz",

    EXPORT /
    "yelp/splits/"
    "yelp_seed2_nested_splits.npz"
)

copy_file(
    WORK /
    "adapters/"
    "yelp_homo_timing_backfill.py",

    EXPORT /
    "yelp/runner/"
    "yelp_homo_timing_backfill.py"
)

copy_file(
    WORK /
    "yelp_step24b.log",

    EXPORT /
    "yelp/logs/"
    "yelp_step24b.log"
)


# ============================================================
# 3. AMAZON EVIDENCE
# ============================================================

print(
    "\n===== AMAZON ====="
)

copy_file(
    AMAZON_RESULT /
    "amazon_timing_backfill_all_runs.csv",

    EXPORT /
    "amazon/results/"
    "amazon_timing_backfill_all_runs.csv"
)

copy_file(
    AMAZON_RESULT /
    "amazon_timing_backfill_all_runs.json",

    EXPORT /
    "amazon/results/"
    "amazon_timing_backfill_all_runs.json"
)

copy_dir(
    AMAZON_RESULT /
    "runs",

    EXPORT /
    "amazon/results/runs"
)

copy_dir(
    AMAZON_RESULT /
    "epoch_times",

    EXPORT /
    "amazon/results/epoch_times"
)

copy_file(
    WORK /
    "shared/timing_backfill_splits/"
    "amazon_seed2_nested_splits.npz",

    EXPORT /
    "amazon/splits/"
    "amazon_seed2_nested_splits.npz"
)

copy_file(
    WORK /
    "adapters/"
    "amazon_homo_timing_backfill.py",

    EXPORT /
    "amazon/runner/"
    "amazon_homo_timing_backfill.py"
)

copy_file(
    WORK /
    "amazon_step25.log",

    EXPORT /
    "amazon/logs/"
    "amazon_step25.log"
)


# ============================================================
# 4. SHARED BACKFILL CONFIG
# ============================================================

shared_config = (
    WORK /
    "results/timing_backfill/"
    "timing_backfill_frozen_configs.json"
)

if shared_config.exists():

    copy_file(
        shared_config,

        EXPORT /
        "shared/"
        "timing_backfill_frozen_configs.json"
    )


# ============================================================
# 5. README / MANIFEST
# ============================================================

manifest = {
    "purpose":
        "Supplemental computational-efficiency evidence.",

    "primary_results": {
        "YelpChi":
            "Step 15 — unchanged",

        "Amazon":
            "Step 19 — unchanged"
    },

    "supplemental_steps": {
        "YelpChi":
            "Step 24B",

        "Amazon":
            "Step 25"
    },

    "split_seed":
        2,

    "training_seeds":
        [
            2,
            42,
            72
        ],

    "training_ratios":
        [
            "TR40",
            "TR30",
            "TR20",
            "TR10"
        ],

    "yelp_completed_runs":
        len(
            yelp_runs
        ),

    "amazon_completed_runs":
        len(
            amazon_runs
        ),

    "checkpoints_included":
        False
}


(
    EXPORT /
    "manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


# ============================================================
# 6. ZIP
# ============================================================

zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=EXPORT
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n============================================"
)

print(
    "STEP 26 GATE: PASS"
)

print(
    "YelpChi timing evidence: 12/12"
)

print(
    "Amazon timing evidence: 12/12"
)

print(
    "Primary Step 15 and Step 19 results: UNCHANGED"
)

print(
    "Checkpoints included: NO"
)

print(
    "\nEvidence ZIP:"
)

print(
    zip_path
)

print(
    "\nFiles packaged:"
)

for path in sorted(
    EXPORT.rglob("*")
):

    if path.is_file():

        print(
            " -",
            path.relative_to(
                EXPORT
            )
        )

===== STEP 26 — EVIDENCE VERIFICATION =====
YelpChi completed runs: 12 /12
Amazon completed runs: 12 /12
YelpChi epoch timing files: 12 /12
Amazon epoch timing files: 12 /12

===== YELPCHI =====
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/yelp_homo/yelp_timing_backfill_all_runs.csv
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/yelp_homo/yelp_timing_backfill_all_runs.json
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/yelp_homo/runs
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/yelp_homo/epoch_times
✓ /kaggle/working/comp8851_bwgnn/shared/timing_backfill_splits/yelp_seed2_nested_splits.npz
✓ /kaggle/working/comp8851_bwgnn/adapters/yelp_homo_timing_backfill.py
✓ /kaggle/working/comp8851_bwgnn/yelp_step24b.log

===== AMAZON =====
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/amazon_homo/amazon_timing_backfill_all_runs.csv
✓ /kaggle/working/comp8851_bwgnn/results/timing_backfill/amazon_homo/amazon_timing_backfill_all_runs.json
✓ /kagg

## Step 27A — T-Social Canonical Dataset and Unified Split Setup

This step begins the primary BWGNN benchmark for **T-Social**.

Unlike Steps 24–26, this is **not a timing backfill**. T-Social has not yet
been benchmarked with BWGNN in this notebook.

### Purpose

This step:

- verifies the canonical T-Social dataset file,
- verifies graph structure, features and labels,
- recreates the controlled nested split using split seed 2,
- checks class counts and split nesting,
- saves the exact split for later smoke, tuning and final runs.

No model training or test evaluation is performed in this step.

### Canonical T-Social structure

Expected:

- Nodes: 5,781,065
- Edges: 146,211,016
- Features: 10
- Normal nodes: 5,606,785
- Fraud nodes: 174,280

### Unified controlled split

- TR40: 40% training
- TR30: 30% training
- TR20: 20% training
- TR10: 10% training
- Validation: fixed 20%
- Test: fixed 40%
- Split seed: 2

The training subsets are nested:

`TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40`

The validation and test sets remain fixed for all later T-Social runs.

In [21]:
# ============================================================
# STEP 27A — T-SOCIAL DATASET + CONTROLLED SPLIT
# ============================================================

import os
import hashlib
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PYTHON39 = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

DATA_PATH = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "tsocial/tsocial"
)

SPLIT_DIR = (
    WORK /
    "shared/splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_PATH = (
    SPLIT_DIR /
    "tsocial_seed2_nested_splits.npz"
)


EXPECTED_SHA256 = (
    "8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0"
)


# ============================================================
# 1. FILE PRESENCE
# ============================================================

print(
    "===== STEP 27A — T-SOCIAL DATASET VERIFICATION =====",
    flush=True
)

if not DATA_PATH.exists():

    raise FileNotFoundError(
        f"T-Social is not attached:\n{DATA_PATH}"
    )

print(
    "PASS — T-Social file found:",
    DATA_PATH,
    flush=True
)


# ============================================================
# 2. SHA256
# ============================================================

print(
    "\nCalculating SHA256...",
    flush=True
)

sha = hashlib.sha256()

with DATA_PATH.open("rb") as f:

    while True:

        block = f.read(
            1024 * 1024 * 8
        )

        if not block:
            break

        sha.update(
            block
        )

actual_sha256 = (
    sha.hexdigest()
)

print(
    "T-Social SHA256:",
    actual_sha256,
    flush=True
)

assert (
    actual_sha256
    == EXPECTED_SHA256
)

print(
    "PASS — canonical T-Social hash verified.",
    flush=True
)


# ============================================================
# 3. CONTROLLED ENVIRONMENT
# ============================================================

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PYTHONUNBUFFERED"] = "1"


cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

ENV["LD_LIBRARY_PATH"] = (
    ":".join(
        str(path)
        for path in cuda_dirs
    )
    + ":"
    + ENV.get(
        "LD_LIBRARY_PATH",
        ""
    )
)


# ============================================================
# 4. CREATE SPLIT INSIDE AUTHOR ENV
# ============================================================

split_script = r'''
import sys
import numpy as np
import torch

from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split


DATA_PATH = sys.argv[1]
OUTPUT_PATH = sys.argv[2]

SEED = 2


print(
    "Loading T-Social graph on CPU...",
    flush=True
)

graphs, _ = load_graphs(
    DATA_PATH
)

graph = graphs[0]


# ============================================================
# STRUCTURE
# ============================================================

print(
    "Graph loaded.",
    flush=True
)

print(
    "Nodes:",
    graph.num_nodes(),
    flush=True
)

print(
    "Edges:",
    graph.num_edges(),
    flush=True
)


assert graph.num_nodes() == 5781065
assert graph.num_edges() == 146211016


# ============================================================
# FEATURES
# ============================================================

if "feature" not in graph.ndata:

    raise KeyError(
        "T-Social graph is missing ndata['feature']"
    )


features = graph.ndata[
    "feature"
]

print(
    "Feature shape:",
    tuple(features.shape),
    flush=True
)

assert tuple(
    features.shape
) == (
    5781065,
    10
)


# ============================================================
# LABELS
# ============================================================

if "label" not in graph.ndata:

    raise KeyError(
        "T-Social graph is missing ndata['label']"
    )


raw_labels = graph.ndata[
    "label"
]


if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .cpu()
        .numpy()
        .astype(np.int64)
    )

else:

    labels = (
        raw_labels
        .reshape(-1)
        .cpu()
        .numpy()
        .astype(np.int64)
    )


normal = int(
    (labels == 0).sum()
)

fraud = int(
    (labels == 1).sum()
)


print(
    "Normal:",
    normal,
    flush=True
)

print(
    "Fraud:",
    fraud,
    flush=True
)


assert normal == 5606785
assert fraud == 174280


# ============================================================
# CONTROLLED NESTED SPLIT
# ============================================================

print(
    "\nCreating split seed 2...",
    flush=True
)


all_ids = np.arange(
    len(labels),
    dtype=np.int64
)


# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining_ids, test_ids = (
    train_test_split(
        all_ids,
        test_size=0.40,
        stratify=labels,
        random_state=SEED,
        shuffle=True
    )
)


# ------------------------------------------------------------
# Remaining 60%:
# 40% training + 20% validation
# ------------------------------------------------------------

train40, val_ids = (
    train_test_split(
        remaining_ids,
        test_size=(1 / 3),
        stratify=labels[
            remaining_ids
        ],
        random_state=SEED,
        shuffle=True
    )
)


# ------------------------------------------------------------
# Nested TR30
# ------------------------------------------------------------

train30, _ = (
    train_test_split(
        train40,
        train_size=0.75,
        stratify=labels[
            train40
        ],
        random_state=SEED,
        shuffle=True
    )
)


# ------------------------------------------------------------
# Nested TR20
# ------------------------------------------------------------

train20, _ = (
    train_test_split(
        train30,
        train_size=(2 / 3),
        stratify=labels[
            train30
        ],
        random_state=SEED,
        shuffle=True
    )
)


# ------------------------------------------------------------
# Nested TR10
# ------------------------------------------------------------

train10, _ = (
    train_test_split(
        train20,
        train_size=0.50,
        stratify=labels[
            train20
        ],
        random_state=SEED,
        shuffle=True
    )
)


splits = {

    "TR40":
        np.sort(
            train40
        ),

    "TR30":
        np.sort(
            train30
        ),

    "TR20":
        np.sort(
            train20
        ),

    "TR10":
        np.sort(
            train10
        ),

    "val":
        np.sort(
            val_ids
        ),

    "test":
        np.sort(
            test_ids
        )
}


# ============================================================
# SIZE GATE
# ============================================================

expected_sizes = {

    "TR40":
        2312426,

    "TR30":
        1734319,

    "TR20":
        1156212,

    "TR10":
        578106,

    "val":
        1156213,

    "test":
        2312426
}


print(
    "\n===== SPLIT SIZES =====",
    flush=True
)


for name, expected in (
    expected_sizes.items()
):

    actual = len(
        splits[name]
    )

    print(
        f"{name:<5}: "
        f"{actual:,}",
        flush=True
    )

    assert actual == expected


# ============================================================
# CLASS COUNTS
# ============================================================

print(
    "\n===== SPLIT CLASS COUNTS =====",
    flush=True
)


for name in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[
        name
    ]

    n0 = int(
        (
            labels[ids]
            == 0
        ).sum()
    )

    n1 = int(
        (
            labels[ids]
            == 1
        ).sum()
    )

    print(
        f"{name:<5} | "
        f"normal={n0:,} | "
        f"fraud={n1:,}",
        flush=True
    )


# Expected class counts from controlled split

assert int(
    (
        labels[
            splits["TR40"]
        ] == 0
    ).sum()
) == 2242714

assert int(
    (
        labels[
            splits["TR40"]
        ] == 1
    ).sum()
) == 69712


assert int(
    (
        labels[
            splits["TR30"]
        ] == 0
    ).sum()
) == 1682035

assert int(
    (
        labels[
            splits["TR30"]
        ] == 1
    ).sum()
) == 52284


assert int(
    (
        labels[
            splits["TR20"]
        ] == 0
    ).sum()
) == 1121356

assert int(
    (
        labels[
            splits["TR20"]
        ] == 1
    ).sum()
) == 34856


assert int(
    (
        labels[
            splits["TR10"]
        ] == 0
    ).sum()
) == 560678

assert int(
    (
        labels[
            splits["TR10"]
        ] == 1
    ).sum()
) == 17428


assert int(
    (
        labels[
            splits["val"]
        ] == 0
    ).sum()
) == 1121357

assert int(
    (
        labels[
            splits["val"]
        ] == 1
    ).sum()
) == 34856


assert int(
    (
        labels[
            splits["test"]
        ] == 0
    ).sum()
) == 2242714

assert int(
    (
        labels[
            splits["test"]
        ] == 1
    ).sum()
) == 69712


# ============================================================
# NESTING
# ============================================================

print(
    "\nChecking nested training subsets...",
    flush=True
)


assert set(
    splits["TR10"]
) <= set(
    splits["TR20"]
)

assert set(
    splits["TR20"]
) <= set(
    splits["TR30"]
)

assert set(
    splits["TR30"]
) <= set(
    splits["TR40"]
)


print(
    "Nested TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40: PASS",
    flush=True
)


# ============================================================
# LEAKAGE
# ============================================================

assert not (
    set(
        splits["TR40"]
    )
    & set(
        splits["val"]
    )
)

assert not (
    set(
        splits["TR40"]
    )
    & set(
        splits["test"]
    )
)

assert not (
    set(
        splits["val"]
    )
    & set(
        splits["test"]
    )
)


print(
    "Train/validation/test overlap: NONE",
    flush=True
)


# ============================================================
# SAVE
# ============================================================

np.savez_compressed(
    OUTPUT_PATH,

    **splits,

    seed=np.array(
        [SEED],
        dtype=np.int64
    ),

    source_nodes=np.array(
        [5781065],
        dtype=np.int64
    ),

    source_edges=np.array(
        [146211016],
        dtype=np.int64
    ),

    source_normal=np.array(
        [5606785],
        dtype=np.int64
    ),

    source_fraud=np.array(
        [174280],
        dtype=np.int64
    )
)


print(
    "\nSaved:",
    OUTPUT_PATH,
    flush=True
)

print(
    "\n===== STEP 27A GATE =====",
    flush=True
)

print(
    "PASS — canonical T-Social verified and "
    "controlled nested split created.",
    flush=True
)
'''


process = subprocess.Popen(
    [
        str(
            PYTHON39
        ),
        "-c",
        split_script,
        str(
            DATA_PATH
        ),
        str(
            SPLIT_PATH
        )
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)


for line in process.stdout:

    print(
        line.rstrip(),
        flush=True
    )


return_code = (
    process.wait()
)


if return_code != 0:

    raise RuntimeError(
        f"Step 27A failed with "
        f"return code {return_code}"
    )

===== STEP 27A — T-SOCIAL DATASET VERIFICATION =====
PASS — T-Social file found: /kaggle/input/datasets/pathikahmed0007/tsocial/tsocial

Calculating SHA256...
T-Social SHA256: 8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0
PASS — canonical T-Social hash verified.
Loading T-Social graph on CPU...
Graph loaded.
Nodes: 5781065
Edges: 146211016
Feature shape: (5781065, 10)
Normal: 5606785
Fraud: 174280

Creating split seed 2...

===== SPLIT SIZES =====
TR40 : 2,312,426
TR30 : 1,734,319
TR20 : 1,156,212
TR10 : 578,106
val  : 1,156,213
test : 2,312,426

===== SPLIT CLASS COUNTS =====
TR40  | normal=2,242,714 | fraud=69,712
TR30  | normal=1,682,035 | fraud=52,284
TR20  | normal=1,121,356 | fraud=34,856
TR10  | normal=560,678 | fraud=17,428
val   | normal=1,121,357 | fraud=34,856
test  | normal=2,242,714 | fraud=69,712

Checking nested training subsets...
Nested TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40: PASS
Train/validation/test overlap: NONE

Saved: /kaggle/working/comp8851_bwgnn/shared/s

## Step 27B — T-Social BWGNN Compatibility and 1-Epoch Smoke Test

This is the first BWGNN execution test on T-Social.

### Purpose

T-Social is substantially larger than YelpChi, Amazon and T-Finance:

- Nodes: 5,781,065
- Edges: 146,211,016
- Features: 10

Therefore, before tuning, this step verifies that the author-compatible BWGNN
pipeline can execute on the controlled Tesla T4 environment.

### Author-compatible T-Social handling

T-Social is loaded directly from the canonical DGL graph.

Unlike YelpChi and Amazon, no heterograph-to-homogeneous conversion and no
additional self-loops are introduced.

The stored graph is preserved as supplied.

### Smoke configuration

- Training split: TR40
- Split seed: 2
- Training seed: 2
- Hidden dimension: 64
- Wavelet order: 2
- Learning rate: 0.01
- Epochs: 1
- GPU: Tesla T4, GPU 0 only

### Smoke-test scope

The test verifies:

1. canonical graph loading,
2. GPU graph transfer,
3. BWGNN model construction,
4. one training forward pass,
5. loss calculation,
6. backward pass,
7. optimizer update,
8. one validation forward pass.

The fixed **test set is not evaluated in this smoke test**.

Timing and GPU-memory numbers produced here are diagnostic only.
Final computational-efficiency measurements will be collected during the
controlled final runs.

A live monitor is provided in Step 27C so execution progress can be observed
without blind waiting.

In [23]:
# ============================================================
# STEP 27B-1 — CREATE T-SOCIAL 1-EPOCH SMOKE RUNNER
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_smoke.py"
)

RUNNER.parent.mkdir(
    parents=True,
    exist_ok=True
)

code = r'''
import sys
import gc
import json
import time
import random
import traceback
from pathlib import Path

import numpy as np

import torch
import torch.nn.functional as F

from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


# ============================================================
# LOCKED SMOKE CONFIG
# ============================================================

DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tsocial/tsocial"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/"
    "shared/splits/"
    "tsocial_seed2_nested_splits.npz"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "results/smoke/tsocial"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH = (
    RESULT_DIR /
    "tsocial_bwgnn_smoke.json"
)

HIDDEN = 64
ORDER = 2
LR = 0.01

SPLIT_SEED = 2
TRAIN_SEED = 2

DEVICE = torch.device(
    "cuda:0"
)


# ============================================================
# HELPERS
# ============================================================

def gpu_memory():

    return {
        "allocated_mb":
            torch.cuda.memory_allocated()
            / (1024 ** 2),

        "reserved_mb":
            torch.cuda.memory_reserved()
            / (1024 ** 2),

        "peak_allocated_mb":
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
    }


def show_memory(label):

    m = gpu_memory()

    print(
        f"{label} | "
        f"allocated={m['allocated_mb']:.1f} MB | "
        f"reserved={m['reserved_mb']:.1f} MB | "
        f"peak={m['peak_allocated_mb']:.1f} MB",
        flush=True
    )


# ============================================================
# EXECUTION
# ============================================================

try:

    print(
        "===== STEP 27B — T-SOCIAL BWGNN SMOKE =====",
        flush=True
    )

    print(
        "Test set access: NO",
        flush=True
    )


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    random.seed(
        TRAIN_SEED
    )

    np.random.seed(
        TRAIN_SEED
    )

    torch.manual_seed(
        TRAIN_SEED
    )

    torch.cuda.manual_seed_all(
        TRAIN_SEED
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    assert torch.cuda.is_available()
    assert torch.cuda.device_count() == 1


    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True
    )


    # ========================================================
    # STAGE 1 — CPU GRAPH LOAD
    # ========================================================

    print(
        "\n[STAGE 1/8] Loading canonical T-Social graph on CPU...",
        flush=True
    )

    load_start = (
        time.perf_counter()
    )


    graphs, _ = load_graphs(
        DATA_PATH
    )

    graph = graphs[0]


    cpu_load_seconds = (
        time.perf_counter()
        - load_start
    )


    assert graph.num_nodes() == 5781065
    assert graph.num_edges() == 146211016

    assert "feature" in graph.ndata
    assert "label" in graph.ndata

    assert tuple(
        graph.ndata["feature"].shape
    ) == (
        5781065,
        10
    )


    print(
        "[STAGE 1/8] PASS",
        flush=True
    )

    print(
        "Nodes:",
        graph.num_nodes(),
        flush=True
    )

    print(
        "Edges:",
        graph.num_edges(),
        flush=True
    )

    print(
        "CPU load time:",
        f"{cpu_load_seconds:.3f}s",
        flush=True
    )


    # ========================================================
    # STAGE 2 — SPLIT LOAD
    # ========================================================

    print(
        "\n[STAGE 2/8] Loading controlled split...",
        flush=True
    )


    splits = np.load(
        SPLIT_PATH
    )


    assert len(
        splits["TR40"]
    ) == 2312426

    assert len(
        splits["val"]
    ) == 1156213

    assert len(
        splits["test"]
    ) == 2312426


    print(
        "[STAGE 2/8] PASS",
        flush=True
    )

    print(
        "TR40:",
        len(
            splits["TR40"]
        ),
        flush=True
    )

    print(
        "VAL:",
        len(
            splits["val"]
        ),
        flush=True
    )


    # ========================================================
    # STAGE 3 — GPU GRAPH TRANSFER
    # ========================================================

    print(
        "\n[STAGE 3/8] Moving full T-Social graph to GPU...",
        flush=True
    )

    print(
        "This is a large 146M-edge transfer.",
        flush=True
    )


    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


    transfer_start = (
        time.perf_counter()
    )


    graph = graph.to(
        DEVICE
    )


    torch.cuda.synchronize()


    gpu_transfer_seconds = (
        time.perf_counter()
        - transfer_start
    )


    print(
        "[STAGE 3/8] PASS",
        flush=True
    )

    print(
        "GPU transfer time:",
        f"{gpu_transfer_seconds:.3f}s",
        flush=True
    )

    show_memory(
        "After graph transfer"
    )


    # ========================================================
    # FEATURES + LABELS
    # ========================================================

    features = (
        graph.ndata[
            "feature"
        ].float()
    )


    raw_labels = graph.ndata[
        "label"
    ]


    if raw_labels.ndim == 2:

        labels = (
            raw_labels
            .argmax(1)
            .long()
        )

    else:

        labels = (
            raw_labels
            .reshape(-1)
            .long()
        )


    normal = int(
        (
            labels == 0
        ).sum().item()
    )

    fraud = int(
        (
            labels == 1
        ).sum().item()
    )


    assert normal == 5606785
    assert fraud == 174280


    # ========================================================
    # STAGE 4 — SPLIT TENSORS
    # ========================================================

    print(
        "\n[STAGE 4/8] Preparing TR40 and validation indices...",
        flush=True
    )


    train_ids = torch.tensor(
        splits["TR40"],
        dtype=torch.long,
        device=DEVICE
    )

    val_ids = torch.tensor(
        splits["val"],
        dtype=torch.long,
        device=DEVICE
    )


    train_labels = labels[
        train_ids
    ]


    train_normal = int(
        (
            train_labels == 0
        ).sum().item()
    )

    train_fraud = int(
        (
            train_labels == 1
        ).sum().item()
    )


    assert train_normal == 2242714
    assert train_fraud == 69712


    class_weight = torch.tensor(
        [
            1.0,
            train_normal / train_fraud
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    print(
        "[STAGE 4/8] PASS",
        flush=True
    )

    print(
        "TR40 normal/fraud:",
        train_normal,
        train_fraud,
        flush=True
    )

    print(
        "Fraud class weight:",
        f"{train_normal / train_fraud:.6f}",
        flush=True
    )

    show_memory(
        "After split preparation"
    )


    # ========================================================
    # STAGE 5 — MODEL
    # ========================================================

    print(
        "\n[STAGE 5/8] Constructing BWGNN...",
        flush=True
    )


    model = BWGNN(
        in_feats=10,
        h_feats=HIDDEN,
        num_classes=2,
        graph=graph,
        d=ORDER
    ).to(
        DEVICE
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(
            0.9,
            0.999
        ),
        eps=1e-8
    )


    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    )


    print(
        "[STAGE 5/8] PASS",
        flush=True
    )

    print(
        "Parameters:",
        parameter_count,
        flush=True
    )

    show_memory(
        "After model construction"
    )


    # ========================================================
    # STAGE 6 — ONE TRAINING EPOCH
    # ========================================================

    print(
        "\n[STAGE 6/8] Starting 1 training epoch...",
        flush=True
    )

    print(
        "Running full-graph forward + backward + optimizer update.",
        flush=True
    )


    model.train()


    torch.cuda.synchronize()

    train_start = (
        time.perf_counter()
    )


    logits = model(
        features
    )


    print(
        "[STAGE 6/8] Forward pass complete.",
        flush=True
    )

    show_memory(
        "After training forward"
    )


    loss = F.cross_entropy(
        logits[
            train_ids
        ],
        labels[
            train_ids
        ],
        weight=class_weight
    )


    if not torch.isfinite(
        loss
    ):

        raise RuntimeError(
            "Non-finite training loss."
        )


    print(
        "[STAGE 6/8] Loss computed. Starting backward...",
        flush=True
    )


    optimizer.zero_grad()

    loss.backward()


    print(
        "[STAGE 6/8] Backward pass complete. Updating optimizer...",
        flush=True
    )


    optimizer.step()


    torch.cuda.synchronize()


    train_seconds = (
        time.perf_counter()
        - train_start
    )


    print(
        "[STAGE 6/8] PASS",
        flush=True
    )

    print(
        "Loss:",
        float(
            loss.item()
        ),
        flush=True
    )

    print(
        "Training-only seconds:",
        f"{train_seconds:.3f}",
        flush=True
    )

    show_memory(
        "After optimizer step"
    )


    # Free training logits before validation.

    del logits
    del loss

    gc.collect()
    torch.cuda.empty_cache()


    # ========================================================
    # STAGE 7 — VALIDATION
    # ========================================================

    print(
        "\n[STAGE 7/8] Running validation forward pass...",
        flush=True
    )


    model.eval()


    torch.cuda.synchronize()

    val_start = (
        time.perf_counter()
    )


    with torch.no_grad():

        val_logits = model(
            features
        )


    torch.cuda.synchronize()


    print(
        "[STAGE 7/8] Validation forward complete.",
        flush=True
    )


    val_probability = (
        torch.softmax(
            val_logits,
            dim=1
        )[
            val_ids,
            1
        ]
        .detach()
        .cpu()
        .numpy()
    )


    val_y = (
        labels[
            val_ids
        ]
        .detach()
        .cpu()
        .numpy()
    )


    val_auprc = float(
        average_precision_score(
            val_y,
            val_probability
        )
    )


    val_auroc = float(
        roc_auc_score(
            val_y,
            val_probability
        )
    )


    validation_seconds = (
        time.perf_counter()
        - val_start
    )


    print(
        "[STAGE 7/8] PASS",
        flush=True
    )

    print(
        "Validation AUPRC:",
        f"{val_auprc:.6f}",
        flush=True
    )

    print(
        "Validation AUROC:",
        f"{val_auroc:.6f}",
        flush=True
    )

    print(
        "Validation seconds:",
        f"{validation_seconds:.3f}",
        flush=True
    )


    del val_logits


    # ========================================================
    # STAGE 8 — SAVE SMOKE RECORD
    # ========================================================

    print(
        "\n[STAGE 8/8] Saving smoke evidence...",
        flush=True
    )


    result = {

        "dataset":
            "T-Social",

        "model":
            "BWGNN",

        "purpose":
            "1-epoch compatibility smoke",

        "test_accessed":
            False,

        "nodes":
            graph.num_nodes(),

        "edges":
            graph.num_edges(),

        "features":
            10,

        "split":
            "TR40",

        "split_seed":
            SPLIT_SEED,

        "train_seed":
            TRAIN_SEED,

        "hidden":
            HIDDEN,

        "order":
            ORDER,

        "learning_rate":
            LR,

        "loss":
            float(
                F.cross_entropy(
                    model(features)[train_ids],
                    labels[train_ids],
                    weight=class_weight
                ).detach().item()
            ),

        "val_auprc":
            val_auprc,

        "val_auroc":
            val_auroc,

        "cpu_load_seconds":
            cpu_load_seconds,

        "gpu_transfer_seconds":
            gpu_transfer_seconds,

        "train_seconds":
            train_seconds,

        "validation_seconds":
            validation_seconds,

        "peak_gpu_memory_mb":
            float(
                torch.cuda.max_memory_allocated()
                / (1024 ** 2)
            ),

        "parameter_count":
            parameter_count
    }


    RESULT_PATH.write_text(
        json.dumps(
            result,
            indent=2
        )
    )


    print(
        "[STAGE 8/8] PASS",
        flush=True
    )


    print(
        "\n===== STEP 27B GATE =====",
        flush=True
    )

    print(
        "PASS — T-Social BWGNN completed "
        "1 full training epoch and validation on Tesla T4.",
        flush=True
    )

    print(
        "Test set accessed: NO",
        flush=True
    )


except Exception as error:

    print(
        "\n===== STEP 27B GATE =====",
        flush=True
    )

    print(
        "FAIL — T-Social smoke did not complete.",
        flush=True
    )

    print(
        "Error type:",
        type(error).__name__,
        flush=True
    )

    print(
        "Error:",
        str(error),
        flush=True
    )


    if (
        "out of memory"
        in str(error).lower()
    ):

        print(
            "Classification: CUDA OUT OF MEMORY",
            flush=True
        )

        print(
            "This is a model/dataset feasibility result "
            "for full-graph T-Social on this T4 configuration.",
            flush=True
        )


    traceback.print_exc()

    sys.exit(1)
'''

RUNNER.write_text(
    code,
    encoding="utf-8"
)

print(
    "===== STEP 27B-1 ====="
)

print(
    "PASS — T-Social smoke runner created."
)

print(
    "Runner:",
    RUNNER
)

===== STEP 27B-1 =====
PASS — T-Social smoke runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/tsocial_bwgnn_smoke.py


In [24]:
# ============================================================
# STEP 27B-2 — REMOVE UNNECESSARY SECOND FORWARD
# ============================================================

from pathlib import Path

RUNNER = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "adapters/tsocial_bwgnn_smoke.py"
)

text = RUNNER.read_text()

old = '''        "loss":
            float(
                F.cross_entropy(
                    model(features)[train_ids],
                    labels[train_ids],
                    weight=class_weight
                ).detach().item()
            ),'''

new = '''        "loss":
            training_loss_value,'''

assert old in text

text = text.replace(
    old,
    new
)

# Store loss value before deleting loss tensor.
marker = '''    print(
        "[STAGE 6/8] PASS",
        flush=True
    )'''

replacement = '''    training_loss_value = float(
        loss.item()
    )

    print(
        "[STAGE 6/8] PASS",
        flush=True
    )'''

assert marker in text

text = text.replace(
    marker,
    replacement,
    1
)

RUNNER.write_text(
    text
)

print(
    "PASS — unnecessary second full-graph forward removed."
)

PASS — unnecessary second full-graph forward removed.


### Step 27B-3 — Launch T-Social Smoke Test

This cell launches the 1-epoch T-Social compatibility smoke in the background.

Run this launcher only once.

Use Step 27C immediately afterward to observe live execution status.

In [25]:
# ============================================================
# STEP 27B-3 — LAUNCH T-SOCIAL SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_smoke.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = (
    WORK /
    "tsocial_step27b.log"
)

PID_FILE = (
    WORK /
    "tsocial_step27b.pid"
)


assert RUNNER.exists()
assert PYTHON.exists()


check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[t]social_bwgnn_smoke.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "T-Social Step 27B is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(
        map(
            str,
            cuda
        )
    )
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)


log_handle = open(
    LOG,
    "a",
    buffering=1
)


process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)


PID_FILE.write_text(
    str(
        process.pid
    )
)

log_handle.close()


print(
    "===== STEP 27B STARTED ====="
)

print(
    "PID:",
    process.pid
)

print(
    "Log:",
    LOG
)

print(
    "Run Step 27C now for live progress."
)

===== STEP 27B STARTED =====
PID: 286
Log: /kaggle/working/comp8851_bwgnn/tsocial_step27b.log
Run Step 27C now for live progress.


### Step 27C — Live T-Social Smoke Progress

This monitor refreshes every 3 seconds.

It shows:

- whether the smoke process is still running,
- process elapsed time,
- the current execution stage,
- graph/GPU memory milestones,
- training forward/backward progress,
- validation progress,
- the final compatibility gate.

Stopping this monitor does not stop the background smoke process.

In [26]:
# ============================================================
# STEP 27C — LIVE T-SOCIAL SMOKE MONITOR
# ============================================================

from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

PID_FILE = (
    WORK /
    "tsocial_step27b.pid"
)

LOG = (
    WORK /
    "tsocial_step27b.log"
)


try:

    while True:

        clear_output(
            wait=True
        )


        print(
            "===== STEP 27C — LIVE T-SOCIAL SMOKE ====="
        )


        # ====================================================
        # PROCESS
        # ====================================================

        running = False
        pid = None
        elapsed = "N/A"


        if PID_FILE.exists():

            pid = int(
                PID_FILE
                .read_text()
                .strip()
            )


            cmd_path = Path(
                f"/proc/{pid}/cmdline"
            )


            if cmd_path.exists():

                running = (
                    b"tsocial_bwgnn_smoke.py"
                    in cmd_path.read_bytes()
                )


                if running:

                    ps = subprocess.run(
                        [
                            "ps",
                            "-p",
                            str(pid),
                            "-o",
                            "etime="
                        ],
                        capture_output=True,
                        text=True
                    )

                    elapsed = (
                        ps.stdout.strip()
                        or "N/A"
                    )


        print(
            "PID:",
            pid
        )

        print(
            "Process running:",
            running
        )

        print(
            "Elapsed:",
            elapsed
        )


        # ====================================================
        # GPU STATUS
        # ====================================================

        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total",
                "--format=csv,noheader,nounits",
                "-i",
                "0"
            ],
            capture_output=True,
            text=True
        )


        print(
            "GPU memory:",
            gpu.stdout.strip(),
            "MiB (used,total)"
        )


        # ====================================================
        # LOG
        # ====================================================

        print(
            "\n----- CURRENT SMOKE STAGE -----"
        )


        lines = []

        if LOG.exists():

            lines = (
                LOG
                .read_text(
                    errors="replace"
                )
                .splitlines()
            )


            useful = [
                line
                for line in lines
                if (
                    "[STAGE" in line
                    or "After graph" in line
                    or "After split" in line
                    or "After model" in line
                    or "After training" in line
                    or "After optimizer" in line
                    or "Validation AUPRC" in line
                    or "Validation AUROC" in line
                    or "Error" in line
                    or "Classification:" in line
                    or "STEP 27B GATE" in line
                    or line.startswith(
                        "PASS — T-Social"
                    )
                    or line.startswith(
                        "FAIL — T-Social"
                    )
                )
            ]


            if useful:

                for line in useful[-18:]:

                    print(line)

            else:

                print(
                    "Process started; waiting for first stage..."
                )

        else:

            print(
                "Waiting for log file..."
            )


        # ====================================================
        # PROCESS ENDED
        # ====================================================

        if not running:

            print(
                "\n============================================"
            )

            print(
                "T-Social smoke process has ended."
            )


            if lines:

                print(
                    "\n----- FINAL LOG -----"
                )

                for line in lines[-50:]:

                    print(line)

            break


        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(
            3
        )


except KeyboardInterrupt:

    print(
        "\nMonitor stopped."
    )

    print(
        "T-Social smoke continues in the background."
    )

===== STEP 27C — LIVE T-SOCIAL SMOKE =====
PID: 286
Process running: False
Elapsed: N/A
GPU memory: 0, 15360 MiB (used,total)

----- CURRENT SMOKE STAGE -----
[STAGE 1/8] Loading canonical T-Social graph on CPU...
[STAGE 1/8] PASS
[STAGE 2/8] Loading controlled split...
[STAGE 2/8] PASS
[STAGE 3/8] Moving full T-Social graph to GPU...
[STAGE 3/8] PASS
After graph transfer | allocated=2276.1 MB | reserved=2278.0 MB | peak=2276.1 MB
[STAGE 4/8] Preparing TR40 and validation indices...
[STAGE 4/8] PASS
After split preparation | allocated=3026.9 MB | reserved=3056.0 MB | peak=3046.9 MB
[STAGE 5/8] Constructing BWGNN...
[STAGE 5/8] PASS
After model construction | allocated=3026.9 MB | reserved=3056.0 MB | peak=3046.9 MB
[STAGE 6/8] Starting 1 training epoch...
terminate called after throwing an instance of 'c10::CUDAOutOfMemoryError'
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x42 (0x7ad921053a22 in /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-pac

## Step 27D — T-Social Sparse-Format Compatibility Retry

The initial Step 27B smoke test failed with CUDA out-of-memory during the
first BWGNN forward pass.

### Observed Step 27B result

The following stages succeeded:

- canonical T-Social graph loading,
- controlled split loading,
- transfer of the 146,211,016-edge graph to Tesla T4,
- TR40 index preparation,
- BWGNN model construction.

The failure occurred during the first training forward pass while DGL was
lazily converting the graph's sparse representation for inbound message
passing.

### Compatibility retry

This step performs a minimal storage-level compatibility adaptation:

- the canonical graph remains unchanged,
- node and edge sets remain unchanged,
- features and labels remain unchanged,
- the Step 27A split remains unchanged,
- BWGNN configuration remains hidden=64, order=2, lr=0.01,
- no sampling or graph reduction is introduced.

The graph is converted to a single CSC sparse representation on CPU before
being transferred to GPU. This prevents DGL from performing the large
COO-to-inbound-sparse conversion during the first GPU forward pass.

This is a storage-format compatibility test, not a new model configuration.

The test set is not evaluated.

If this retry still fails with CUDA OOM, the full-graph hidden-64 BWGNN
configuration will be classified as infeasible on the controlled Tesla T4
environment, and a lower-memory compatibility configuration will be tested
separately rather than silently changing the benchmark.

In [27]:
# ============================================================
# STEP 27D-1 — CREATE T-SOCIAL CSC COMPATIBILITY SMOKE
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_csc_smoke.py"
)

RUNNER.parent.mkdir(
    parents=True,
    exist_ok=True
)

code = r'''
import sys
import gc
import json
import time
import random
import traceback

from pathlib import Path

import numpy as np

import torch
import torch.nn.functional as F

from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

sys.path.insert(
    0,
    "/kaggle/working/comp8851_bwgnn/adapters"
)

from BWGNN_gpu import BWGNN


DATA_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tsocial/tsocial"
)

SPLIT_PATH = (
    "/kaggle/working/comp8851_bwgnn/"
    "shared/splits/tsocial_seed2_nested_splits.npz"
)

RESULT_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "results/smoke/tsocial"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH = (
    RESULT_DIR /
    "tsocial_bwgnn_csc_smoke.json"
)

HIDDEN = 64
ORDER = 2
LR = 0.01

SEED = 2

DEVICE = torch.device("cuda:0")


def mem(label):

    print(
        f"{label} | "
        f"allocated="
        f"{torch.cuda.memory_allocated()/1024**2:.1f} MB | "
        f"reserved="
        f"{torch.cuda.memory_reserved()/1024**2:.1f} MB | "
        f"peak="
        f"{torch.cuda.max_memory_allocated()/1024**2:.1f} MB",
        flush=True
    )


try:

    print(
        "===== STEP 27D — T-SOCIAL CSC COMPATIBILITY RETRY =====",
        flush=True
    )

    print(
        "Test set accessed: NO",
        flush=True
    )


    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    # ========================================================
    # STAGE 1
    # ========================================================

    print(
        "\n[STAGE 1/9] Loading canonical T-Social...",
        flush=True
    )

    graphs, _ = load_graphs(
        DATA_PATH
    )

    graph = graphs[0]

    assert graph.num_nodes() == 5781065
    assert graph.num_edges() == 146211016

    print(
        "[STAGE 1/9] PASS",
        flush=True
    )

    print(
        "Initial sparse formats:",
        graph.formats(),
        flush=True
    )


    # ========================================================
    # STAGE 2 — CPU CSC
    # ========================================================

    print(
        "\n[STAGE 2/9] Materializing CSC format on CPU...",
        flush=True
    )

    format_start = time.perf_counter()

    graph = graph.formats(
        "csc"
    )

    format_seconds = (
        time.perf_counter()
        - format_start
    )

    print(
        "[STAGE 2/9] PASS",
        flush=True
    )

    print(
        "CSC formats:",
        graph.formats(),
        flush=True
    )

    print(
        "CPU format conversion:",
        f"{format_seconds:.3f}s",
        flush=True
    )


    # ========================================================
    # STAGE 3 — SPLITS
    # ========================================================

    print(
        "\n[STAGE 3/9] Loading TR40/validation split...",
        flush=True
    )

    splits = np.load(
        SPLIT_PATH
    )

    train_ids_cpu = torch.tensor(
        splits["TR40"],
        dtype=torch.long
    )

    val_ids_cpu = torch.tensor(
        splits["val"],
        dtype=torch.long
    )

    print(
        "[STAGE 3/9] PASS",
        flush=True
    )


    # ========================================================
    # STAGE 4 — GPU TRANSFER
    # ========================================================

    print(
        "\n[STAGE 4/9] Moving CSC-only graph to GPU...",
        flush=True
    )

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    transfer_start = time.perf_counter()

    graph = graph.to(
        DEVICE
    )

    torch.cuda.synchronize()

    transfer_seconds = (
        time.perf_counter()
        - transfer_start
    )

    print(
        "[STAGE 4/9] PASS",
        flush=True
    )

    print(
        "GPU sparse formats:",
        graph.formats(),
        flush=True
    )

    print(
        "Transfer:",
        f"{transfer_seconds:.3f}s",
        flush=True
    )

    mem(
        "After CSC graph transfer"
    )


    # ========================================================
    # STAGE 5 — DATA
    # ========================================================

    print(
        "\n[STAGE 5/9] Preparing features, labels and indices...",
        flush=True
    )

    features = (
        graph.ndata["feature"]
        .float()
    )

    raw_labels = graph.ndata[
        "label"
    ]

    if raw_labels.ndim == 2:

        labels = (
            raw_labels.argmax(1)
            .long()
        )

    else:

        labels = (
            raw_labels.reshape(-1)
            .long()
        )

    train_ids = train_ids_cpu.to(
        DEVICE
    )

    val_ids = val_ids_cpu.to(
        DEVICE
    )

    train_labels = labels[
        train_ids
    ]

    normal = int(
        (train_labels == 0)
        .sum()
        .item()
    )

    fraud = int(
        (train_labels == 1)
        .sum()
        .item()
    )

    weight = torch.tensor(
        [
            1.0,
            normal / fraud
        ],
        dtype=torch.float32,
        device=DEVICE
    )

    print(
        "[STAGE 5/9] PASS",
        flush=True
    )

    print(
        "TR40 normal/fraud:",
        normal,
        fraud,
        flush=True
    )

    mem(
        "After indices"
    )


    # ========================================================
    # STAGE 6 — MODEL
    # ========================================================

    print(
        "\n[STAGE 6/9] Constructing BWGNN...",
        flush=True
    )

    model = BWGNN(
        in_feats=10,
        h_feats=HIDDEN,
        num_classes=2,
        graph=graph,
        d=ORDER
    ).to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    parameter_count = int(
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    )

    print(
        "[STAGE 6/9] PASS",
        flush=True
    )

    print(
        "Parameters:",
        parameter_count,
        flush=True
    )

    mem(
        "After model"
    )


    # ========================================================
    # STAGE 7 — TRAIN
    # ========================================================

    print(
        "\n[STAGE 7/9] Starting full-graph training forward...",
        flush=True
    )

    model.train()

    torch.cuda.synchronize()

    start = time.perf_counter()

    logits = model(
        features
    )

    print(
        "[STAGE 7/9] Forward complete.",
        flush=True
    )

    mem(
        "After forward"
    )

    loss = F.cross_entropy(
        logits[train_ids],
        labels[train_ids],
        weight=weight
    )

    training_loss = float(
        loss.item()
    )

    print(
        "[STAGE 7/9] Loss complete. Starting backward...",
        flush=True
    )

    optimizer.zero_grad()

    loss.backward()

    print(
        "[STAGE 7/9] Backward complete.",
        flush=True
    )

    optimizer.step()

    torch.cuda.synchronize()

    train_seconds = (
        time.perf_counter()
        - start
    )

    print(
        "[STAGE 7/9] PASS",
        flush=True
    )

    print(
        "Loss:",
        training_loss,
        flush=True
    )

    print(
        "Training seconds:",
        f"{train_seconds:.3f}",
        flush=True
    )

    mem(
        "After optimizer"
    )

    del logits, loss

    gc.collect()
    torch.cuda.empty_cache()


    # ========================================================
    # STAGE 8 — VALIDATION
    # ========================================================

    print(
        "\n[STAGE 8/9] Validation forward...",
        flush=True
    )

    model.eval()

    torch.cuda.synchronize()

    val_start = time.perf_counter()

    with torch.no_grad():

        logits = model(
            features
        )

        val_probability = (
            torch.softmax(
                logits,
                dim=1
            )[val_ids, 1]
            .cpu()
            .numpy()
        )

    torch.cuda.synchronize()

    val_y = (
        labels[val_ids]
        .cpu()
        .numpy()
    )

    val_auprc = float(
        average_precision_score(
            val_y,
            val_probability
        )
    )

    val_auroc = float(
        roc_auc_score(
            val_y,
            val_probability
        )
    )

    val_seconds = (
        time.perf_counter()
        - val_start
    )

    print(
        "[STAGE 8/9] PASS",
        flush=True
    )

    print(
        "Validation AUPRC:",
        f"{val_auprc:.6f}",
        flush=True
    )

    print(
        "Validation AUROC:",
        f"{val_auroc:.6f}",
        flush=True
    )


    # ========================================================
    # STAGE 9 — SAVE
    # ========================================================

    result = {
        "dataset":
            "T-Social",

        "model":
            "BWGNN",

        "purpose":
            "CSC storage-format compatibility smoke",

        "graph_changed":
            False,

        "sampling_used":
            False,

        "test_accessed":
            False,

        "hidden":
            HIDDEN,

        "order":
            ORDER,

        "learning_rate":
            LR,

        "sparse_format":
            "csc",

        "format_seconds":
            format_seconds,

        "gpu_transfer_seconds":
            transfer_seconds,

        "train_seconds":
            train_seconds,

        "validation_seconds":
            val_seconds,

        "training_loss":
            training_loss,

        "val_auprc":
            val_auprc,

        "val_auroc":
            val_auroc,

        "parameter_count":
            parameter_count,

        "peak_gpu_memory_mb":
            float(
                torch.cuda.max_memory_allocated()
                / (1024 ** 2)
            )
    }

    RESULT_PATH.write_text(
        json.dumps(
            result,
            indent=2
        )
    )

    print(
        "\n[STAGE 9/9] PASS",
        flush=True
    )

    print(
        "\n===== STEP 27D GATE =====",
        flush=True
    )

    print(
        "PASS — T-Social full-graph BWGNN "
        "completed with CSC storage adaptation.",
        flush=True
    )

    print(
        "Test set accessed: NO",
        flush=True
    )


except Exception as error:

    print(
        "\n===== STEP 27D GATE =====",
        flush=True
    )

    print(
        "FAIL — CSC compatibility retry did not complete.",
        flush=True
    )

    print(
        "Error type:",
        type(error).__name__,
        flush=True
    )

    print(
        "Error:",
        str(error),
        flush=True
    )

    if (
        "out of memory"
        in str(error).lower()
    ):

        print(
            "Classification: CUDA OUT OF MEMORY",
            flush=True
        )

    traceback.print_exc()

    sys.exit(1)
'''

RUNNER.write_text(
    code,
    encoding="utf-8"
)

print(
    "===== STEP 27D-1 ====="
)

print(
    "PASS — CSC compatibility runner created."
)

print(
    "Runner:",
    RUNNER
)

===== STEP 27D-1 =====
PASS — CSC compatibility runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/tsocial_bwgnn_csc_smoke.py


In [28]:
# ============================================================
# STEP 27D-2 — LAUNCH CSC COMPATIBILITY SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_csc_smoke.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = (
    WORK /
    "tsocial_step27d.log"
)

PID_FILE = (
    WORK /
    "tsocial_step27d.pid"
)

assert RUNNER.exists()


check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[t]social_bwgnn_csc_smoke.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "Step 27D is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(
        map(str, cuda)
    )
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)


log_handle = open(
    LOG,
    "a",
    buffering=1
)

process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log_handle.close()


print(
    "===== STEP 27D STARTED ====="
)

print(
    "PID:",
    process.pid
)

print(
    "Run Step 27E now for live progress."
)

===== STEP 27D STARTED =====
PID: 302
Run Step 27E now for live progress.


### Step 27E — Live T-Social CSC Compatibility Monitor

This cell monitors the Step 27D compatibility retry every 3 seconds.

It does not start, stop or modify the experiment.

In [31]:
from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

PID_FILE = WORK / "tsocial_step27d.pid"
LOG = WORK / "tsocial_step27d.log"

try:

    while True:

        clear_output(wait=True)

        print(
            "===== STEP 27E — LIVE T-SOCIAL CSC RETRY ====="
        )

        running = False
        pid = None
        elapsed = "N/A"

        if PID_FILE.exists():

            pid = int(
                PID_FILE.read_text().strip()
            )

            cmd = Path(
                f"/proc/{pid}/cmdline"
            )

            running = (
                cmd.exists()
                and b"tsocial_bwgnn_csc_smoke.py"
                in cmd.read_bytes()
            )

            if running:

                ps = subprocess.run(
                    [
                        "ps",
                        "-p",
                        str(pid),
                        "-o",
                        "etime="
                    ],
                    capture_output=True,
                    text=True
                )

                elapsed = (
                    ps.stdout.strip()
                    or "N/A"
                )


        print("PID:", pid)
        print("Process running:", running)
        print("Elapsed:", elapsed)


        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total",
                "--format=csv,noheader,nounits",
                "-i",
                "0"
            ],
            capture_output=True,
            text=True
        )

        print(
            "GPU memory:",
            gpu.stdout.strip(),
            "MiB (used,total)"
        )


        print(
            "\n----- CURRENT STAGE -----"
        )

        lines = []

        if LOG.exists():

            lines = (
                LOG.read_text(
                    errors="replace"
                ).splitlines()
            )

            useful = [
                line
                for line in lines
                if (
                    "[STAGE" in line
                    or "formats:" in line
                    or "After CSC" in line
                    or "After indices" in line
                    or "After model" in line
                    or "After forward" in line
                    or "After optimizer" in line
                    or "Validation AUPRC" in line
                    or "Classification:" in line
                    or "STEP 27D GATE" in line
                    or line.startswith(
                        "PASS — T-Social"
                    )
                    or line.startswith(
                        "FAIL — CSC"
                    )
                )
            ]

            for line in useful[-20:]:
                print(line)

        else:

            print(
                "Waiting for log..."
            )


        if not running:

            print(
                "\n============================================"
            )

            print(
                "Step 27D process ended."
            )

            if lines:

                print(
                    "\n----- FINAL LOG -----"
                )

                for line in lines[-50:]:
                    print(line)

            break


        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(3)


except KeyboardInterrupt:

    print(
        "\nMonitor stopped; background retry continues."
    )

===== STEP 27E — LIVE T-SOCIAL CSC RETRY =====
PID: 302
Process running: False
Elapsed: N/A
GPU memory: 0, 15360 MiB (used,total)

----- CURRENT STAGE -----
Initial sparse formats: {'created': ['csr'], 'not created': ['coo', 'csc']}
[STAGE 2/9] Materializing CSC format on CPU...
[STAGE 2/9] PASS
CSC formats: {'created': ['csc'], 'not created': []}
[STAGE 3/9] Loading TR40/validation split...
[STAGE 3/9] PASS
[STAGE 4/9] Moving CSC-only graph to GPU...
[STAGE 4/9] PASS
GPU sparse formats: {'created': ['csc'], 'not created': []}
After CSC graph transfer | allocated=2276.1 MB | reserved=2278.0 MB | peak=2276.1 MB
[STAGE 5/9] Preparing features, labels and indices...
[STAGE 5/9] PASS
After indices | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 6/9] Constructing BWGNN...
[STAGE 6/9] PASS
After model | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 7/9] Starting full-graph training forward...
===== STEP 27D GATE =====
FAIL — CSC compatibility retry did n

## Step 27F — T-Social Hidden-64 Full-Graph Compatibility Decision

Two controlled compatibility tests were performed before T-Social tuning.

### Step 27B — Original stored graph format

Configuration:

- Hidden dimension: 64
- Wavelet order: 2
- Learning rate: 0.01
- TR40
- Training seed: 2
- Tesla T4 GPU 0
- Full-graph execution

Result:

- Dataset loading: PASS
- GPU graph transfer: PASS
- Split preparation: PASS
- BWGNN construction: PASS
- First training forward: **CUDA OOM**

The initial failure occurred while DGL was lazily constructing the sparse
representation required for inbound message passing.

### Step 27D — CSC storage-format retry

The same graph, labels, splits and BWGNN configuration were retained, but the
graph was materialized as CSC on CPU before GPU transfer.

Result:

- CSC construction: PASS
- GPU graph transfer: PASS
- Split preparation: PASS
- BWGNN construction: PASS
- First training forward: **CUDA OOM**

The second failure occurred inside DGL/CuSPARSE `gspmm` during the actual
BWGNN message-passing operation.

### Compatibility decision

Full-graph BWGNN with `hidden=64, order=2` is therefore classified as
**infeasible on the controlled 15 GB Tesla T4 environment for T-Social**.

This is a hardware/model compatibility finding, not a performance result.

The test set was not accessed.

A lower-memory compatibility configuration is tested next. Any such change is
reported explicitly rather than silently altering the benchmark.

In [32]:
# ============================================================
# STEP 27F — SAVE T-SOCIAL COMPATIBILITY DECISION
# ============================================================

from pathlib import Path
import json

WORK = Path("/kaggle/working/comp8851_bwgnn")

OUT = (
    WORK /
    "results/smoke/tsocial/"
    "tsocial_hidden64_compatibility_decision.json"
)

OUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

record = {
    "dataset": "T-Social",
    "model": "BWGNN",
    "environment": "Tesla T4 15 GB, GPU 0",
    "full_graph": True,

    "configuration": {
        "hidden": 64,
        "order": 2,
        "learning_rate": 0.01
    },

    "step27b_original_format": {
        "graph_load": "PASS",
        "gpu_transfer": "PASS",
        "model_construction": "PASS",
        "training_forward": "FAIL",
        "failure": "CUDA OOM"
    },

    "step27d_csc_retry": {
        "graph_load": "PASS",
        "csc_materialization": "PASS",
        "gpu_transfer": "PASS",
        "model_construction": "PASS",
        "training_forward": "FAIL",
        "failure": "CUDA OOM during DGL/CuSPARSE gspmm"
    },

    "compatibility_decision":
        "hidden=64 full-graph BWGNN is infeasible on the controlled T4.",

    "test_accessed": False
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    )
)

print("===== STEP 27F GATE =====")
print("PASS — hidden=64 infeasibility recorded.")
print("Test set accessed: NO")
print("Saved:", OUT)

===== STEP 27F GATE =====
PASS — hidden=64 infeasibility recorded.
Test set accessed: NO
Saved: /kaggle/working/comp8851_bwgnn/results/smoke/tsocial/tsocial_hidden64_compatibility_decision.json


## Step 27G — T-Social Lower-Memory Full-Graph Compatibility Smoke

Because the frozen hidden-64 smoke configuration failed twice with CUDA
out-of-memory, this step tests a lower-memory full-graph configuration.

### Compatibility configuration

- Hidden dimension: **32**
- Wavelet order: 2
- Learning rate: 0.01
- Split: TR40
- Split seed: 2
- Training seed: 2
- Epochs: 1
- Sparse format: CSC
- GPU: Tesla T4, GPU 0 only

Only the hidden dimension is reduced.

The following remain unchanged:

- canonical T-Social graph,
- all 146,211,016 edges,
- all 5,781,065 nodes,
- features and labels,
- Step 27A split,
- full-graph message passing,
- optimizer,
- wavelet order,
- learning rate.

This is a **compatibility test**, not hyperparameter tuning.

The test set is not evaluated.

If hidden=32 succeeds, the feasible configuration space will be used for the
subsequent validation-only tuning step. Hidden=64 will remain documented as
hardware-infeasible on the controlled T4.

In [37]:
# ============================================================
# STEP 27G-1 — CREATE T-SOCIAL HIDDEN-32 CSC SMOKE
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

SOURCE = (
    WORK /
    "adapters/tsocial_bwgnn_csc_smoke.py"
)

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h32_csc_smoke.py"
)

assert SOURCE.exists()

text = SOURCE.read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# Change ONLY compatibility hidden dimension + labels/outputs
# ------------------------------------------------------------

text = text.replace(
    "HIDDEN = 64",
    "HIDDEN = 32"
)

text = text.replace(
    "tsocial_bwgnn_csc_smoke.json",
    "tsocial_bwgnn_h32_csc_smoke.json"
)

text = text.replace(
    "===== STEP 27D — T-SOCIAL CSC COMPATIBILITY RETRY =====",
    "===== STEP 27G — T-SOCIAL HIDDEN-32 CSC SMOKE ====="
)

text = text.replace(
    "===== STEP 27D GATE =====",
    "===== STEP 27G GATE ====="
)

text = text.replace(
    "CSC storage-format compatibility smoke",
    "hidden-32 CSC full-graph compatibility smoke"
)

text = text.replace(
    "PASS — T-Social full-graph BWGNN completed with CSC storage adaptation.",
    "PASS — T-Social hidden=32 full-graph BWGNN completed on Tesla T4."
)

text = text.replace(
    "FAIL — CSC compatibility retry did not complete.",
    "FAIL — T-Social hidden=32 compatibility smoke did not complete."
)

RUNNER.write_text(
    text,
    encoding="utf-8"
)

print("===== STEP 27G-1 =====")
print("PASS — hidden=32 compatibility runner created.")
print("Runner:", RUNNER)

===== STEP 27G-1 =====
PASS — hidden=32 compatibility runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/tsocial_bwgnn_h32_csc_smoke.py


In [38]:
# ============================================================
# STEP 27G-2 — LAUNCH HIDDEN-32 T-SOCIAL SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h32_csc_smoke.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = WORK / "tsocial_step27g.log"
PID_FILE = WORK / "tsocial_step27g.pid"

assert RUNNER.exists()
assert PYTHON.exists()


check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[t]social_bwgnn_h32_csc_smoke.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():
    raise RuntimeError(
        "Step 27G is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(
        map(str, cuda)
    )
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)


log_handle = open(
    LOG,
    "a",
    buffering=1
)

process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log_handle.close()

print("===== STEP 27G STARTED =====")
print("PID:", process.pid)
print("Compatibility config: hidden=32, order=2, lr=0.01")
print("Run Step 27H now for live progress.")

===== STEP 27G STARTED =====
PID: 344
Compatibility config: hidden=32, order=2, lr=0.01
Run Step 27H now for live progress.


### Step 27H — Live T-Social Hidden-32 Compatibility Monitor

This monitor refreshes every 3 seconds and displays the current compatibility
stage, GPU memory usage and final gate.

It does not modify or restart the Step 27G experiment.

In [41]:
# ============================================================
# STEP 27H — LIVE T-SOCIAL HIDDEN-32 MONITOR
# ============================================================

from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "tsocial_step27g.pid"
LOG = WORK / "tsocial_step27g.log"


try:

    while True:

        clear_output(
            wait=True
        )

        print(
            "===== STEP 27H — LIVE T-SOCIAL HIDDEN-32 SMOKE ====="
        )


        # ----------------------------------------------------
        # Process status
        # ----------------------------------------------------

        pid = None
        running = False
        elapsed = "N/A"

        if PID_FILE.exists():

            pid = int(
                PID_FILE.read_text().strip()
            )

            cmd = Path(
                f"/proc/{pid}/cmdline"
            )

            running = (
                cmd.exists()
                and b"tsocial_bwgnn_h32_csc_smoke.py"
                in cmd.read_bytes()
            )

            if running:

                ps = subprocess.run(
                    [
                        "ps",
                        "-p",
                        str(pid),
                        "-o",
                        "etime="
                    ],
                    capture_output=True,
                    text=True
                )

                elapsed = (
                    ps.stdout.strip()
                    or "N/A"
                )


        print("PID:", pid)
        print("Process running:", running)
        print("Elapsed:", elapsed)


        # ----------------------------------------------------
        # GPU memory
        # ----------------------------------------------------

        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total",
                "--format=csv,noheader,nounits",
                "-i",
                "0"
            ],
            capture_output=True,
            text=True
        )

        print(
            "GPU memory:",
            gpu.stdout.strip(),
            "MiB (used,total)"
        )


        # ----------------------------------------------------
        # Log
        # ----------------------------------------------------

        print(
            "\n----- CURRENT STAGE -----"
        )

        lines = []

        if LOG.exists():

            lines = LOG.read_text(
                errors="replace"
            ).splitlines()

            useful = [
                line
                for line in lines
                if (
                    "[STAGE" in line
                    or "After CSC" in line
                    or "After indices" in line
                    or "After model" in line
                    or "After forward" in line
                    or "After optimizer" in line
                    or "Training seconds:" in line
                    or "Validation AUPRC:" in line
                    or "Validation AUROC:" in line
                    or "Classification:" in line
                    or "STEP 27G GATE" in line
                    or line.startswith(
                        "PASS — T-Social hidden=32"
                    )
                    or line.startswith(
                        "FAIL — T-Social hidden=32"
                    )
                )
            ]

            if useful:

                for line in useful[-22:]:
                    print(line)

            else:

                print(
                    "Loading graph / preparing CSC..."
                )

        else:

            print(
                "Waiting for log..."
            )


        # ----------------------------------------------------
        # Finished
        # ----------------------------------------------------

        if not running:

            print(
                "\n============================================"
            )

            print(
                "Step 27G process ended."
            )

            if lines:

                print(
                    "\n----- FINAL LOG -----"
                )

                for line in lines[-55:]:
                    print(line)

            break


        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(3)


except KeyboardInterrupt:

    print(
        "\nMonitor stopped."
    )

    print(
        "Hidden-32 smoke continues in background."
    )

===== STEP 27H — LIVE T-SOCIAL HIDDEN-32 SMOKE =====
PID: 344
Process running: False
Elapsed: N/A
GPU memory: 0, 15360 MiB (used,total)

----- CURRENT STAGE -----
[STAGE 1/9] Loading canonical T-Social...
[STAGE 1/9] PASS
[STAGE 2/9] Materializing CSC format on CPU...
[STAGE 2/9] PASS
[STAGE 3/9] Loading TR40/validation split...
[STAGE 3/9] PASS
[STAGE 4/9] Moving CSC-only graph to GPU...
[STAGE 4/9] PASS
After CSC graph transfer | allocated=2276.1 MB | reserved=2278.0 MB | peak=2276.1 MB
[STAGE 5/9] Preparing features, labels and indices...
[STAGE 5/9] PASS
After indices | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 6/9] Constructing BWGNN...
[STAGE 6/9] PASS
After model | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 7/9] Starting full-graph training forward...
[STAGE 7/9] Forward complete.
After forward | allocated=8785.5 MB | reserved=10198.0 MB | peak=10859.7 MB
[STAGE 7/9] Loss complete. Starting backward...
===== STEP 27G GATE =====
FAIL —

## Step 27I — T-Social Hidden-32 Compatibility Decision

After hidden=64 full-graph BWGNN failed with CUDA out-of-memory, Step 27G
tested a lower-memory hidden dimension of 32 using the CSC storage adaptation.

### Hidden-32 result

Configuration:

- Hidden dimension: 32
- Wavelet order: 2
- Learning rate: 0.01
- Split: TR40
- Split seed: 2
- Training seed: 2
- Full T-Social graph
- CSC sparse format
- Tesla T4 GPU 0

Observed execution:

- Graph load: PASS
- CSC materialization: PASS
- GPU transfer: PASS
- BWGNN construction: PASS
- Training forward: PASS
- Loss calculation: PASS
- Training backward: **CUDA OOM**

The forward pass reached approximately 10.86 GB peak allocated GPU memory.
Backward propagation subsequently failed while DGL executed sparse
message-passing gradients.

### Compatibility decision

`hidden=32, order=2` can perform a forward pass but cannot complete full-graph
training on the controlled Tesla T4.

This is recorded as a hardware/model compatibility limitation and is not a
performance result.

The test set was not accessed.

The next compatibility test reduces only the hidden dimension to 16.

In [43]:
# ============================================================
# STEP 27I — RECORD HIDDEN-32 INFEASIBILITY
# ============================================================

from pathlib import Path
import json

WORK = Path("/kaggle/working/comp8851_bwgnn")

OUT = (
    WORK /
    "results/smoke/tsocial/"
    "tsocial_hidden32_compatibility_decision.json"
)

OUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

record = {
    "dataset": "T-Social",
    "model": "BWGNN",
    "environment": "Tesla T4 15 GB, GPU 0",
    "full_graph": True,
    "sparse_format": "CSC",

    "configuration": {
        "hidden": 32,
        "order": 2,
        "learning_rate": 0.01
    },

    "graph_load": "PASS",
    "csc_materialization": "PASS",
    "gpu_transfer": "PASS",
    "model_construction": "PASS",
    "training_forward": "PASS",
    "loss_calculation": "PASS",
    "training_backward": "FAIL",

    "failure": "CUDA OOM during DGL sparse backward gspmm",

    "forward_peak_gpu_memory_mb": 10859.7,

    "compatibility_decision":
        "hidden=32 forward is feasible but full training is infeasible "
        "on the controlled Tesla T4.",

    "test_accessed": False
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    )
)

print("===== STEP 27I GATE =====")
print("PASS — hidden=32 training infeasibility recorded.")
print("Test set accessed: NO")
print("Saved:", OUT)

===== STEP 27I GATE =====
PASS — hidden=32 training infeasibility recorded.
Test set accessed: NO
Saved: /kaggle/working/comp8851_bwgnn/results/smoke/tsocial/tsocial_hidden32_compatibility_decision.json


## Step 27J — T-Social Hidden-16 Full-Graph Compatibility Smoke

Because hidden=64 failed during the forward pass and hidden=32 failed during
backward propagation, this step tests a further lower-memory full-graph
configuration.

### Compatibility configuration

- Hidden dimension: **16**
- Wavelet order: 2
- Learning rate: 0.01
- Split: TR40
- Split seed: 2
- Training seed: 2
- Epochs: 1
- Sparse format: CSC
- GPU: Tesla T4 GPU 0

Only the hidden dimension is changed.

The following remain unchanged:

- canonical T-Social dataset,
- all 5,781,065 nodes,
- all 146,211,016 edges,
- feature and label data,
- Step 27A controlled split,
- full-graph message passing,
- Adam optimizer,
- wavelet order 2,
- learning rate 0.01.

This remains a compatibility test rather than hyperparameter tuning.

The test set is not accessed.

If hidden=16 completes forward, backward, optimizer update and validation,
it will establish a trainable full-graph BWGNN configuration for T-Social on
the controlled Tesla T4.

In [44]:
# ============================================================
# STEP 27J-1 — CREATE HIDDEN-16 CSC SMOKE RUNNER
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

SOURCE = (
    WORK /
    "adapters/tsocial_bwgnn_h32_csc_smoke.py"
)

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h16_csc_smoke.py"
)

assert SOURCE.exists()

text = SOURCE.read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# Change only hidden dimension and identifying output labels.
# ------------------------------------------------------------

text = text.replace(
    "HIDDEN = 32",
    "HIDDEN = 16"
)

text = text.replace(
    "tsocial_bwgnn_h32_csc_smoke.json",
    "tsocial_bwgnn_h16_csc_smoke.json"
)

text = text.replace(
    "===== STEP 27G — T-SOCIAL HIDDEN-32 CSC SMOKE =====",
    "===== STEP 27J — T-SOCIAL HIDDEN-16 CSC SMOKE ====="
)

text = text.replace(
    "===== STEP 27G GATE =====",
    "===== STEP 27J GATE ====="
)

text = text.replace(
    "hidden-32 CSC full-graph compatibility smoke",
    "hidden-16 CSC full-graph compatibility smoke"
)

text = text.replace(
    "PASS — T-Social hidden=32 full-graph BWGNN completed on Tesla T4.",
    "PASS — T-Social hidden=16 full-graph BWGNN completed on Tesla T4."
)

text = text.replace(
    "FAIL — T-Social hidden=32 compatibility smoke did not complete.",
    "FAIL — T-Social hidden=16 compatibility smoke did not complete."
)

RUNNER.write_text(
    text,
    encoding="utf-8"
)

# Sanity gates
created = RUNNER.read_text()

assert "HIDDEN = 16" in created
assert "HIDDEN = 32" not in created
assert "STEP 27J GATE" in created

print("===== STEP 27J-1 =====")
print("PASS — hidden=16 compatibility runner created.")
print("Runner:", RUNNER)

===== STEP 27J-1 =====
PASS — hidden=16 compatibility runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/tsocial_bwgnn_h16_csc_smoke.py


In [45]:
# ============================================================
# STEP 27J-2 — LAUNCH HIDDEN-16 T-SOCIAL SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h16_csc_smoke.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = WORK / "tsocial_step27j.log"
PID_FILE = WORK / "tsocial_step27j.pid"

assert RUNNER.exists()
assert PYTHON.exists()


check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[t]social_bwgnn_h16_csc_smoke.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "Step 27J is already running:\n"
        + check.stdout
    )


env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(map(str, cuda))
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


log_handle = open(
    LOG,
    "a",
    buffering=1
)

process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log_handle.close()

print("===== STEP 27J STARTED =====")
print("PID:", process.pid)
print("Compatibility config: hidden=16, order=2, lr=0.01")
print("Run Step 27K now for LIVE progress.")

===== STEP 27J STARTED =====
PID: 366
Compatibility config: hidden=16, order=2, lr=0.01
Run Step 27K now for LIVE progress.


### Step 27K — Live T-Social Hidden-16 Compatibility Monitor

This cell provides continuous progress for Step 27J.

It refreshes every 3 seconds and displays:

- process status,
- elapsed execution time,
- current compatibility stage,
- GPU memory,
- forward/backward progress,
- validation results,
- final compatibility gate.

Stopping this monitor does not stop the background Step 27J process.

In [54]:
# ============================================================
# STEP 27K — LIVE HIDDEN-16 T-SOCIAL MONITOR
# ============================================================

from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "tsocial_step27j.pid"
LOG = WORK / "tsocial_step27j.log"

try:

    while True:

        clear_output(wait=True)

        print(
            "===== STEP 27K — LIVE T-SOCIAL HIDDEN-16 SMOKE ====="
        )

        # ====================================================
        # PROCESS
        # ====================================================

        pid = None
        running = False
        elapsed = "N/A"

        if PID_FILE.exists():

            pid = int(
                PID_FILE.read_text().strip()
            )

            cmd = Path(
                f"/proc/{pid}/cmdline"
            )

            running = (
                cmd.exists()
                and b"tsocial_bwgnn_h16_csc_smoke.py"
                in cmd.read_bytes()
            )

            if running:

                ps = subprocess.run(
                    [
                        "ps",
                        "-p",
                        str(pid),
                        "-o",
                        "etime="
                    ],
                    capture_output=True,
                    text=True
                )

                elapsed = (
                    ps.stdout.strip()
                    or "N/A"
                )


        print("PID:", pid)
        print("Process running:", running)
        print("Elapsed:", elapsed)


        # ====================================================
        # GPU MEMORY
        # ====================================================

        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total",
                "--format=csv,noheader,nounits",
                "-i",
                "0"
            ],
            capture_output=True,
            text=True
        )

        print(
            "GPU memory:",
            gpu.stdout.strip(),
            "MiB (used,total)"
        )


        # ====================================================
        # LOG PROGRESS
        # ====================================================

        print(
            "\n----- CURRENT STAGE -----"
        )

        lines = []

        if LOG.exists():

            lines = LOG.read_text(
                errors="replace"
            ).splitlines()

            useful = [
                line
                for line in lines
                if (
                    "[STAGE" in line
                    or "After CSC" in line
                    or "After indices" in line
                    or "After model" in line
                    or "After forward" in line
                    or "After optimizer" in line
                    or "Training seconds:" in line
                    or "Validation AUPRC:" in line
                    or "Validation AUROC:" in line
                    or "Classification:" in line
                    or "STEP 27J GATE" in line
                    or line.startswith(
                        "PASS — T-Social hidden=16"
                    )
                    or line.startswith(
                        "FAIL — T-Social hidden=16"
                    )
                )
            ]

            if useful:

                for line in useful[-24:]:
                    print(line)

            else:

                print(
                    "Loading graph / preparing CSC..."
                )

        else:

            print(
                "Waiting for log..."
            )


        # ====================================================
        # FINISHED
        # ====================================================

        if not running:

            print(
                "\n============================================"
            )

            print(
                "Step 27J process ended."
            )

            if lines:

                print(
                    "\n----- FINAL LOG -----"
                )

                for line in lines[-60:]:
                    print(line)

            break


        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(3)


except KeyboardInterrupt:

    print(
        "\nLive monitor stopped."
    )

    print(
        "Hidden-16 compatibility smoke continues in background."
    )

===== STEP 27K — LIVE T-SOCIAL HIDDEN-16 SMOKE =====
PID: 366
Process running: False
Elapsed: N/A
GPU memory: 0, 15360 MiB (used,total)

----- CURRENT STAGE -----
[STAGE 1/9] Loading canonical T-Social...
[STAGE 1/9] PASS
[STAGE 2/9] Materializing CSC format on CPU...
[STAGE 2/9] PASS
[STAGE 3/9] Loading TR40/validation split...
[STAGE 3/9] PASS
[STAGE 4/9] Moving CSC-only graph to GPU...
[STAGE 4/9] PASS
After CSC graph transfer | allocated=2276.1 MB | reserved=2278.0 MB | peak=2276.1 MB
[STAGE 5/9] Preparing features, labels and indices...
[STAGE 5/9] PASS
After indices | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 6/9] Constructing BWGNN...
[STAGE 6/9] PASS
After model | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 7/9] Starting full-graph training forward...
[STAGE 7/9] Forward complete.
After forward | allocated=5960.6 MB | reserved=8462.0 MB | peak=6975.4 MB
[STAGE 7/9] Loss complete. Starting backward...
===== STEP 27J GATE =====
FAIL — T

## Step 27L — T-Social Hidden-16 Compatibility Decision

Step 27J tested full-graph BWGNN on T-Social with hidden dimension 16 after
hidden dimensions 64 and 32 were found infeasible on the controlled Tesla T4.

### Hidden-16 result

Configuration:

- Hidden dimension: 16
- Wavelet order: 2
- Learning rate: 0.01
- Split: TR40
- Split seed: 2
- Training seed: 2
- Sparse representation: CSC
- Full T-Social graph
- Tesla T4 GPU 0

Observed execution:

- Canonical graph load: PASS
- CSC materialization: PASS
- GPU graph transfer: PASS
- Model construction: PASS
- Training forward: PASS
- Loss calculation: PASS
- Training backward: **CUDA OOM**

The forward pass reached approximately 6.98 GB peak allocated GPU memory.
During backward propagation, DGL sparse message passing required an additional
allocation of approximately 1.09 GiB and exceeded available T4 memory.

### Compatibility decision

`hidden=16, order=2` is not trainable using full-graph BWGNN on the controlled
Tesla T4.

This remains a hardware/model compatibility finding rather than a performance
result.

The test set was not accessed.

The next compatibility test reduces only the hidden dimension to 8.

In [55]:
# ============================================================
# STEP 27L — RECORD HIDDEN-16 INFEASIBILITY
# ============================================================

from pathlib import Path
import json

WORK = Path("/kaggle/working/comp8851_bwgnn")

OUT = (
    WORK /
    "results/smoke/tsocial/"
    "tsocial_hidden16_compatibility_decision.json"
)

OUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

record = {
    "dataset": "T-Social",
    "model": "BWGNN",
    "environment": "Tesla T4 15 GB, GPU 0",
    "full_graph": True,
    "sparse_format": "CSC",

    "configuration": {
        "hidden": 16,
        "order": 2,
        "learning_rate": 0.01
    },

    "graph_load": "PASS",
    "csc_materialization": "PASS",
    "gpu_transfer": "PASS",
    "model_construction": "PASS",
    "training_forward": "PASS",
    "loss_calculation": "PASS",
    "training_backward": "FAIL",

    "failure":
        "CUDA OOM during DGL sparse backward gspmm",

    "forward_peak_gpu_memory_mb": 6975.4,

    "compatibility_decision":
        "hidden=16 full-graph BWGNN cannot complete training "
        "on the controlled Tesla T4.",

    "test_accessed": False
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    )
)

print("===== STEP 27L GATE =====")
print("PASS — hidden=16 incompatibility recorded.")
print("Test set accessed: NO")
print("Saved:", OUT)

===== STEP 27L GATE =====
PASS — hidden=16 incompatibility recorded.
Test set accessed: NO
Saved: /kaggle/working/comp8851_bwgnn/results/smoke/tsocial/tsocial_hidden16_compatibility_decision.json


## Step 27M — T-Social Hidden-8 Full-Graph Compatibility Smoke

The previous full-graph compatibility tests established that:

- hidden=64 fails during forward propagation,
- hidden=32 completes forward but fails during backward propagation,
- hidden=16 completes forward but fails during backward propagation.

This step tests the next lower-memory configuration.

### Configuration

- Hidden dimension: **8**
- Wavelet order: 2
- Learning rate: 0.01
- Split: TR40
- Split seed: 2
- Training seed: 2
- Epochs: 1
- Sparse representation: CSC
- Full graph
- Tesla T4 GPU 0

Only the hidden dimension is changed.

The graph, split, labels, features, optimizer, learning rate and wavelet order
remain unchanged.

This is still a compatibility test, not hyperparameter tuning.

The test set is not accessed.

A successful result requires completion of:

1. forward pass,
2. loss calculation,
3. backward pass,
4. optimizer update,
5. validation forward pass.

In [56]:
# ============================================================
# STEP 27M-1 — CREATE HIDDEN-8 CSC SMOKE
# ============================================================

from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

SOURCE = (
    WORK /
    "adapters/tsocial_bwgnn_h16_csc_smoke.py"
)

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h8_csc_smoke.py"
)

assert SOURCE.exists()

text = SOURCE.read_text(
    encoding="utf-8"
)

text = text.replace(
    "HIDDEN = 16",
    "HIDDEN = 8"
)

text = text.replace(
    "tsocial_bwgnn_h16_csc_smoke.json",
    "tsocial_bwgnn_h8_csc_smoke.json"
)

text = text.replace(
    "===== STEP 27J — T-SOCIAL HIDDEN-16 CSC SMOKE =====",
    "===== STEP 27M — T-SOCIAL HIDDEN-8 CSC SMOKE ====="
)

text = text.replace(
    "===== STEP 27J GATE =====",
    "===== STEP 27M GATE ====="
)

text = text.replace(
    "hidden-16 CSC full-graph compatibility smoke",
    "hidden-8 CSC full-graph compatibility smoke"
)

text = text.replace(
    "PASS — T-Social hidden=16 full-graph BWGNN completed on Tesla T4.",
    "PASS — T-Social hidden=8 full-graph BWGNN completed on Tesla T4."
)

text = text.replace(
    "FAIL — T-Social hidden=16 compatibility smoke did not complete.",
    "FAIL — T-Social hidden=8 compatibility smoke did not complete."
)

RUNNER.write_text(
    text,
    encoding="utf-8"
)

created = RUNNER.read_text()

assert "HIDDEN = 8" in created
assert "HIDDEN = 16" not in created
assert "STEP 27M GATE" in created

print("===== STEP 27M-1 =====")
print("PASS — hidden=8 compatibility runner created.")
print("Runner:", RUNNER)

===== STEP 27M-1 =====
PASS — hidden=8 compatibility runner created.
Runner: /kaggle/working/comp8851_bwgnn/adapters/tsocial_bwgnn_h8_csc_smoke.py


In [57]:
# ============================================================
# STEP 27M-2 — LAUNCH HIDDEN-8 T-SOCIAL SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

RUNNER = (
    WORK /
    "adapters/tsocial_bwgnn_h8_csc_smoke.py"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

LOG = WORK / "tsocial_step27m.log"
PID_FILE = WORK / "tsocial_step27m.pid"

assert RUNNER.exists()
assert PYTHON.exists()

check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -ef | grep '[t]social_bwgnn_h8_csc_smoke.py' || true"
    ],
    capture_output=True,
    text=True
)

if check.stdout.strip():

    raise RuntimeError(
        "Step 27M is already running:\n"
        + check.stdout
    )

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(map(str, cuda))
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)

log_handle = open(
    LOG,
    "a",
    buffering=1
)

process = subprocess.Popen(
    [
        str(PYTHON),
        str(RUNNER)
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

PID_FILE.write_text(
    str(process.pid)
)

log_handle.close()

print("===== STEP 27M STARTED =====")
print("PID:", process.pid)
print("Compatibility config: hidden=8, order=2, lr=0.01")
print("Run Step 27N now for LIVE progress.")

===== STEP 27M STARTED =====
PID: 400
Compatibility config: hidden=8, order=2, lr=0.01
Run Step 27N now for LIVE progress.


### Step 27N — Live T-Social Hidden-8 Compatibility Monitor

This monitor refreshes every 3 seconds and shows the active Step 27M stage,
GPU memory usage and final compatibility result.

This cell only monitors the experiment. It does not restart or modify it.

In [58]:
# ============================================================
# STEP 27N — LIVE HIDDEN-8 T-SOCIAL MONITOR
# ============================================================

from pathlib import Path
from IPython.display import clear_output
import subprocess
import time

WORK = Path("/kaggle/working/comp8851_bwgnn")

PID_FILE = WORK / "tsocial_step27m.pid"
LOG = WORK / "tsocial_step27m.log"

try:

    while True:

        clear_output(
            wait=True
        )

        print(
            "===== STEP 27N — LIVE T-SOCIAL HIDDEN-8 SMOKE ====="
        )

        pid = None
        running = False
        elapsed = "N/A"

        if PID_FILE.exists():

            pid = int(
                PID_FILE.read_text().strip()
            )

            cmd = Path(
                f"/proc/{pid}/cmdline"
            )

            running = (
                cmd.exists()
                and b"tsocial_bwgnn_h8_csc_smoke.py"
                in cmd.read_bytes()
            )

            if running:

                ps = subprocess.run(
                    [
                        "ps",
                        "-p",
                        str(pid),
                        "-o",
                        "etime="
                    ],
                    capture_output=True,
                    text=True
                )

                elapsed = (
                    ps.stdout.strip()
                    or "N/A"
                )

        print("PID:", pid)
        print("Process running:", running)
        print("Elapsed:", elapsed)

        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total",
                "--format=csv,noheader,nounits",
                "-i",
                "0"
            ],
            capture_output=True,
            text=True
        )

        print(
            "GPU memory:",
            gpu.stdout.strip(),
            "MiB (used,total)"
        )

        print(
            "\n----- CURRENT STAGE -----"
        )

        lines = []

        if LOG.exists():

            lines = LOG.read_text(
                errors="replace"
            ).splitlines()

            useful = [
                line
                for line in lines
                if (
                    "[STAGE" in line
                    or "After CSC" in line
                    or "After indices" in line
                    or "After model" in line
                    or "After forward" in line
                    or "After optimizer" in line
                    or "Training seconds:" in line
                    or "Validation AUPRC:" in line
                    or "Validation AUROC:" in line
                    or "Classification:" in line
                    or "STEP 27M GATE" in line
                    or line.startswith(
                        "PASS — T-Social hidden=8"
                    )
                    or line.startswith(
                        "FAIL — T-Social hidden=8"
                    )
                )
            ]

            if useful:

                for line in useful[-25:]:
                    print(line)

            else:

                print(
                    "Loading graph / preparing CSC..."
                )

        else:

            print(
                "Waiting for log..."
            )

        if not running:

            print(
                "\n============================================"
            )

            print(
                "Step 27M process ended."
            )

            if lines:

                print(
                    "\n----- FINAL LOG -----"
                )

                for line in lines[-60:]:
                    print(line)

            break

        print(
            "\nLive refresh in 3 seconds..."
        )

        time.sleep(3)

except KeyboardInterrupt:

    print(
        "\nMonitor stopped."
    )

    print(
        "Hidden-8 smoke continues in the background."
    )

===== STEP 27N — LIVE T-SOCIAL HIDDEN-8 SMOKE =====
PID: 400
Process running: False
Elapsed: N/A
GPU memory: 0, 15360 MiB (used,total)

----- CURRENT STAGE -----
[STAGE 1/9] Loading canonical T-Social...
[STAGE 1/9] PASS
[STAGE 2/9] Materializing CSC format on CPU...
[STAGE 2/9] PASS
[STAGE 3/9] Loading TR40/validation split...
[STAGE 3/9] PASS
[STAGE 4/9] Moving CSC-only graph to GPU...
[STAGE 4/9] PASS
After CSC graph transfer | allocated=2276.1 MB | reserved=2278.0 MB | peak=2276.1 MB
[STAGE 5/9] Preparing features, labels and indices...
[STAGE 5/9] PASS
After indices | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 6/9] Constructing BWGNN...
[STAGE 6/9] PASS
After model | allocated=3027.6 MB | reserved=3064.0 MB | peak=3047.8 MB
[STAGE 7/9] Starting full-graph training forward...
[STAGE 7/9] Forward complete.
After forward | allocated=4550.0 MB | reserved=5820.0 MB | peak=5034.4 MB
[STAGE 7/9] Loss complete. Starting backward...
===== STEP 27M GATE =====
FAIL — T-

## Step 27O — Final T-Social BWGNN Compatibility Decision

T-Social was evaluated for full-graph BWGNN compatibility on the controlled
Tesla T4 environment before tuning.

### Canonical dataset

- Nodes: 5,781,065
- Edges: 146,211,016
- Features: 10
- Normal nodes: 5,606,785
- Fraud nodes: 174,280

### Compatibility tests

The following full-graph configurations were tested using the canonical graph,
the Step 27A controlled split and GPU 0.

| Hidden | Order | Result |
|---|---|---|
| 64 | 2 | CUDA OOM during forward message passing |
| 32 | 2 | Forward PASS; CUDA OOM during backward |
| 16 | 2 | Forward PASS; CUDA OOM during backward |
| 8 | 2 | Forward PASS; CUDA OOM during backward |

A CSC storage-format adaptation was also tested to avoid unnecessary sparse
format conversion on GPU. This successfully reduced the format-conversion
problem but did not make full-graph training feasible.

### Final compatibility classification

**BWGNN full-graph training on T-Social is infeasible on the controlled
15 GB Tesla T4 environment.**

The limiting factor is the memory required by DGL sparse message passing and
its backward/reverse-graph operations over the 146-million-edge graph, rather
than BWGNN parameter count alone.

No graph sampling, edge removal, node removal or test-guided configuration
change was introduced merely to force the model to run.

Therefore:

- T-Social tuning is not performed.
- T-Social final training runs are not performed.
- Performance metrics are reported as unavailable for BWGNN/T-Social under
  this controlled hardware configuration.
- The compatibility/OOM evidence is retained.
- The test set was never evaluated.

This is recorded as a model-dataset-hardware compatibility result rather than
a failed benchmark implementation.

In [59]:
# ============================================================
# STEP 27O — SAVE FINAL T-SOCIAL COMPATIBILITY DECISION
# ============================================================

from pathlib import Path
import json

WORK = Path("/kaggle/working/comp8851_bwgnn")

OUT = (
    WORK /
    "results/compatibility/tsocial/"
    "bwgnn_tsocial_final_compatibility.json"
)

OUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

record = {
    "dataset": "T-Social",
    "model": "BWGNN",
    "hardware": "Tesla T4 ~15 GB, GPU 0",
    "execution_mode": "full-graph",
    "nodes": 5781065,
    "edges": 146211016,
    "features": 10,

    "attempts": [
        {
            "hidden": 64,
            "order": 2,
            "result": "CUDA OOM during forward"
        },
        {
            "hidden": 32,
            "order": 2,
            "result": "forward PASS; CUDA OOM during backward"
        },
        {
            "hidden": 16,
            "order": 2,
            "result": "forward PASS; CUDA OOM during backward"
        },
        {
            "hidden": 8,
            "order": 2,
            "result": "forward PASS; CUDA OOM during backward"
        }
    ],

    "csc_storage_retry": True,

    "dominant_failure":
        "DGL sparse message-passing/backward memory requirement "
        "on 146,211,016-edge full graph",

    "compatibility":
        "INFEASIBLE on controlled Tesla T4",

    "tuning_performed": False,
    "final_runs_performed": False,
    "sampling_used": False,
    "graph_reduced": False,
    "test_accessed": False,

    "reporting":
        "BWGNN/T-Social performance unavailable under controlled "
        "full-graph T4 configuration."
}

OUT.write_text(
    json.dumps(
        record,
        indent=2
    )
)

print("===== STEP 27O GATE =====")
print("PASS — T-Social compatibility decision finalized.")
print("Classification: INFEASIBLE ON CONTROLLED T4")
print("Tuning required: NO")
print("Final 12 runs required: NO")
print("Test accessed: NO")
print("Move to next dataset: YES")

===== STEP 27O GATE =====
PASS — T-Social compatibility decision finalized.
Classification: INFEASIBLE ON CONTROLLED T4
Tuning required: NO
Final 12 runs required: NO
Test accessed: NO
Move to next dataset: YES


## Step 27P — Package T-Social BWGNN Compatibility Evidence

T-Social full-graph BWGNN was found infeasible on the controlled Tesla T4.

This evidence package preserves:

- canonical controlled split,
- exact BWGNN GPU adapter,
- original hidden-64 smoke runner,
- CSC compatibility runner,
- hidden-32, hidden-16 and hidden-8 compatibility runners,
- execution logs for every compatibility attempt,
- individual compatibility decision records where available,
- final T-Social compatibility classification,
- evidence manifest.

### Final compatibility result

| Hidden | Order | Result |
|---|---:|---|
| 64 | 2 | CUDA OOM during forward |
| 32 | 2 | Forward PASS; CUDA OOM during backward |
| 16 | 2 | Forward PASS; CUDA OOM during backward |
| 8 | 2 | Forward PASS; CUDA OOM during backward |

CSC sparse storage was also tested. It prevented the initial lazy GPU sparse
conversion but full-graph training still exceeded the 15 GB Tesla T4 memory
limit.

No graph sampling, node removal, edge removal or test-guided workaround was
used.

The test set was not evaluated.

This package records T-Social as a model-dataset-hardware compatibility result,
not as a performance result.

In [61]:
# ============================================================
# STEP 27P — PACKAGE T-SOCIAL COMPATIBILITY EVIDENCE
# ============================================================

from pathlib import Path
import json
import zipfile
import hashlib
import datetime

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

RESULT_DIR = (
    WORK /
    "results/compatibility/tsocial"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SMOKE_DIR = (
    WORK /
    "results/smoke/tsocial"
)

ADAPTER_DIR = (
    WORK /
    "adapters"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "tsocial_seed2_nested_splits.npz"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "bwgnn_tsocial_compatibility_evidence_20260909.zip"
)


print(
    "===== STEP 27P — T-SOCIAL EVIDENCE PACKAGING =====",
    flush=True
)


# ============================================================
# 1. FINAL COMPATIBILITY DECISION
# ============================================================

FINAL_DECISION = (
    RESULT_DIR /
    "bwgnn_tsocial_final_compatibility.json"
)

final_record = {

    "dataset":
        "T-Social",

    "model":
        "BWGNN",

    "hardware":
        "Tesla T4 ~15 GB, GPU 0",

    "execution_mode":
        "full-graph",

    "canonical_dataset": {

        "nodes":
            5781065,

        "edges":
            146211016,

        "features":
            10,

        "normal":
            5606785,

        "fraud":
            174280,

        "sha256":
            "8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0"
    },

    "controlled_split": {

        "split_seed":
            2,

        "TR40":
            2312426,

        "TR30":
            1734319,

        "TR20":
            1156212,

        "TR10":
            578106,

        "validation":
            1156213,

        "test":
            2312426
    },

    "compatibility_attempts": [

        {
            "hidden":
                64,

            "order":
                2,

            "learning_rate":
                0.01,

            "storage":
                "original",

            "result":
                "CUDA OOM during first forward pass"
        },

        {
            "hidden":
                64,

            "order":
                2,

            "learning_rate":
                0.01,

            "storage":
                "CSC",

            "result":
                "CUDA OOM during forward DGL/CuSPARSE message passing"
        },

        {
            "hidden":
                32,

            "order":
                2,

            "learning_rate":
                0.01,

            "storage":
                "CSC",

            "forward":
                "PASS",

            "result":
                "CUDA OOM during backward DGL sparse gspmm"
        },

        {
            "hidden":
                16,

            "order":
                2,

            "learning_rate":
                0.01,

            "storage":
                "CSC",

            "forward":
                "PASS",

            "result":
                "CUDA OOM during backward DGL sparse gspmm"
        },

        {
            "hidden":
                8,

            "order":
                2,

            "learning_rate":
                0.01,

            "storage":
                "CSC",

            "forward":
                "PASS",

            "result":
                "CUDA OOM during backward DGL reverse sparse operation"
        }
    ],

    "dominant_limitation":
        (
            "Memory required by full-graph DGL sparse message passing "
            "and backward/reverse-graph operations over "
            "146,211,016 edges."
        ),

    "final_classification":
        "INFEASIBLE ON CONTROLLED TESLA T4",

    "tuning_performed":
        False,

    "final_12_runs_performed":
        False,

    "sampling_used":
        False,

    "graph_reduced":
        False,

    "test_evaluated":
        False,

    "performance_metrics":
        "UNAVAILABLE under controlled full-graph T4 configuration"
}


FINAL_DECISION.write_text(
    json.dumps(
        final_record,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "PASS — final compatibility decision created.",
    flush=True
)


# ============================================================
# 2. VERIFY CONTROLLED SPLIT
# ============================================================

assert SPLIT.exists(), (
    f"Missing T-Social split: {SPLIT}"
)

print(
    "PASS — controlled T-Social split found.",
    flush=True
)


# ============================================================
# 3. EXPECTED RUNNERS + LOGS
# ============================================================

runner_files = [

    ADAPTER_DIR /
    "BWGNN_gpu.py",

    ADAPTER_DIR /
    "tsocial_bwgnn_smoke.py",

    ADAPTER_DIR /
    "tsocial_bwgnn_csc_smoke.py",

    ADAPTER_DIR /
    "tsocial_bwgnn_h32_csc_smoke.py",

    ADAPTER_DIR /
    "tsocial_bwgnn_h16_csc_smoke.py",

    ADAPTER_DIR /
    "tsocial_bwgnn_h8_csc_smoke.py"
]


log_files = [

    WORK /
    "tsocial_step27b.log",

    WORK /
    "tsocial_step27d.log",

    WORK /
    "tsocial_step27g.log",

    WORK /
    "tsocial_step27j.log",

    WORK /
    "tsocial_step27m.log"
]


print(
    "\n===== RUNNER CHECK =====",
    flush=True
)

for path in runner_files:

    print(
        path.name,
        "PASS" if path.exists() else "MISSING",
        flush=True
    )


print(
    "\n===== LOG CHECK =====",
    flush=True
)

for path in log_files:

    print(
        path.name,
        "PASS" if path.exists() else "MISSING",
        flush=True
    )


# Core evidence must exist.

for path in runner_files:

    assert path.exists(), (
        f"Required runner missing: {path}"
    )


for path in log_files:

    assert path.exists(), (
        f"Required execution log missing: {path}"
    )


# ============================================================
# 4. VERIFY OOM EVIDENCE
# ============================================================

print(
    "\n===== FAILURE-EVIDENCE CHECK =====",
    flush=True
)


for log in log_files:

    text = log.read_text(
        errors="replace"
    ).lower()

    assert (
        "out of memory" in text
        or "cudaoutofmemory" in text
    ), (
        f"Expected OOM evidence not found in {log.name}"
    )

    print(
        log.name,
        "CUDA OOM evidence: PASS",
        flush=True
    )


# ============================================================
# 5. OPTIONAL INDIVIDUAL DECISION RECORDS
# ============================================================

optional_decisions = [

    SMOKE_DIR /
    "tsocial_hidden64_compatibility_decision.json",

    SMOKE_DIR /
    "tsocial_hidden32_compatibility_decision.json",

    SMOKE_DIR /
    "tsocial_hidden16_compatibility_decision.json"
]


existing_optional = [
    p
    for p in optional_decisions
    if p.exists()
]


print(
    "\nIndividual compatibility records:",
    len(existing_optional),
    flush=True
)


# ============================================================
# 6. MANIFEST
# ============================================================

MANIFEST = (
    RESULT_DIR /
    "MANIFEST.json"
)

manifest = {

    "created_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),

    "dataset":
        "T-Social",

    "model":
        "BWGNN",

    "purpose":
        "model-dataset-hardware compatibility evidence",

    "final_result":
        "INFEASIBLE ON CONTROLLED TESLA T4",

    "hardware":
        "Tesla T4 ~15 GB, GPU 0",

    "attempted_hidden_dimensions":
        [
            64,
            32,
            16,
            8
        ],

    "wavelet_order":
        2,

    "full_graph":
        True,

    "csc_storage_tested":
        True,

    "sampling_used":
        False,

    "graph_reduced":
        False,

    "tuning_performed":
        False,

    "final_runs_performed":
        False,

    "test_accessed":
        False,

    "checkpoints_included":
        False,

    "contents": {

        "split":
            SPLIT.name,

        "runner_count":
            len(runner_files),

        "log_count":
            len(log_files),

        "individual_decision_records":
            len(existing_optional),

        "final_decision":
            FINAL_DECISION.name
    }
}


MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 7. PACKAGE
# ============================================================

print(
    "\nCreating ZIP...",
    flush=True
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # --------------------------------------------------------
    # Manifest / final decision
    # --------------------------------------------------------

    z.write(
        MANIFEST,
        "manifest/MANIFEST.json"
    )

    z.write(
        FINAL_DECISION,
        (
            "results/"
            "bwgnn_tsocial_final_compatibility.json"
        )
    )


    # --------------------------------------------------------
    # Exact controlled split
    # --------------------------------------------------------

    z.write(
        SPLIT,
        (
            "splits/"
            "tsocial_seed2_nested_splits.npz"
        )
    )


    # --------------------------------------------------------
    # Runners / adapters
    # --------------------------------------------------------

    for path in runner_files:

        z.write(
            path,
            f"adapters/{path.name}"
        )


    # --------------------------------------------------------
    # Logs
    # --------------------------------------------------------

    for path in log_files:

        z.write(
            path,
            f"logs/{path.name}"
        )


    # --------------------------------------------------------
    # Individual compatibility decisions
    # --------------------------------------------------------

    for path in existing_optional:

        z.write(
            path,
            f"results/individual/{path.name}"
        )


# ============================================================
# 8. ZIP SHA256
# ============================================================

h = hashlib.sha256()

with ZIP_PATH.open(
    "rb"
) as f:

    for block in iter(
        lambda: f.read(
            8 * 1024 * 1024
        ),
        b""
    ):

        h.update(
            block
        )


zip_sha = h.hexdigest()


# ============================================================
# 9. FINAL VERIFICATION
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    names = z.namelist()


assert (
    "manifest/MANIFEST.json"
    in names
)

assert (
    "results/bwgnn_tsocial_final_compatibility.json"
    in names
)

assert (
    "splits/tsocial_seed2_nested_splits.npz"
    in names
)

assert sum(
    name.startswith(
        "logs/"
    )
    for name in names
) == 5


print(
    "\n===== STEP 27P GATE =====",
    flush=True
)

print(
    "PASS — T-Social compatibility evidence packaged.",
    flush=True
)

print(
    "Compatibility: INFEASIBLE ON CONTROLLED T4",
    flush=True
)

print(
    "Attempted hidden sizes: 64, 32, 16, 8",
    flush=True
)

print(
    "Execution logs: 5/5",
    flush=True
)

print(
    "Controlled split included: YES",
    flush=True
)

print(
    "BWGNN adapter/runners included: YES",
    flush=True
)

print(
    "Checkpoints included: NO",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

print(
    "\nZIP:",
    ZIP_PATH,
    flush=True
)

print(
    "ZIP SHA256:",
    zip_sha,
    flush=True
)

print(
    "ZIP size:",
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB",
    flush=True
)

===== STEP 27P — T-SOCIAL EVIDENCE PACKAGING =====
PASS — final compatibility decision created.
PASS — controlled T-Social split found.

===== RUNNER CHECK =====
BWGNN_gpu.py PASS
tsocial_bwgnn_smoke.py PASS
tsocial_bwgnn_csc_smoke.py PASS
tsocial_bwgnn_h32_csc_smoke.py PASS
tsocial_bwgnn_h16_csc_smoke.py PASS
tsocial_bwgnn_h8_csc_smoke.py PASS

===== LOG CHECK =====
tsocial_step27b.log PASS
tsocial_step27d.log PASS
tsocial_step27g.log PASS
tsocial_step27j.log PASS
tsocial_step27m.log PASS

===== FAILURE-EVIDENCE CHECK =====
tsocial_step27b.log CUDA OOM evidence: PASS
tsocial_step27d.log CUDA OOM evidence: PASS
tsocial_step27g.log CUDA OOM evidence: PASS
tsocial_step27j.log CUDA OOM evidence: PASS
tsocial_step27m.log CUDA OOM evidence: PASS

Individual compatibility records: 3

Creating ZIP...

===== STEP 27P GATE =====
PASS — T-Social compatibility evidence packaged.
Compatibility: INFEASIBLE ON CONTROLLED T4
Attempted hidden sizes: 64, 32, 16, 8
Execution logs: 5/5
Controlled split i

In [2]:
# ============================================================
# DOWNLOAD T-SOCIAL EVIDENCE ZIP
# ============================================================

from pathlib import Path
from IPython.display import FileLink, display

ZIP = Path(
    "/kaggle/working/"
    "bwgnn_tsocial_compatibility_evidence_20260909.zip"
)

print("Exists:", ZIP.exists())
print("Size:", f"{ZIP.stat().st_size / (1024**2):.2f} MB")

display(
    FileLink(
        str(ZIP),
        result_html_prefix="<b>Download T-Social evidence:</b> "
    )
)

Exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/bwgnn_tsocial_compatibility_evidence_20260909.zip'

## Step 28A — BWGNN × FDCompCN Compatibility Audit

FDCompCN is not a native BWGNN repository dataset.

The current COMP8851 protocol therefore requires a compatibility decision
before any training is attempted.

### Goal

Determine whether FDCompCN can be used with BWGNN through a minimal,
architecture-preserving input adapter.

This step checks:

- whether the frozen BWGNN repository contains any FDCompCN path,
- FDCompCN node and relation structure,
- feature and label availability,
- whether a homogeneous adapter would preserve all nodes and edges,
- whether relation semantics would be discarded by BWGNN.

No model training, validation or test evaluation is performed.

### Decision rule

- **Native** — official repository directly supports FDCompCN.
- **Input adapter** — only representation translation is required and the
  graph semantics remain suitable.
- **Conditional** — technically runnable only with a documented semantic
  compromise that prevents direct equivalence with native FDCompCN models.
- **Not eligible** — making it run would require architecture or graph
  redesign.

The test split is not accessed.

In [3]:
# ============================================================
# STEP 28A — BWGNN × FDCOMPCN COMPATIBILITY AUDIT
# NO TRAINING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

REPO = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

DATASET_PY = REPO / "dataset.py"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

PYTHON = (
    WORK /
    "envs/bwgnn-author/bin/python"
)

SITE = (
    WORK /
    "envs/bwgnn-author/lib/python3.9/site-packages"
)

assert DATASET_PY.exists(), DATASET_PY
assert DATA.exists(), DATA
assert PYTHON.exists(), PYTHON


print(
    "===== STEP 28A — BWGNN × FDCOMPCN COMPATIBILITY =====",
    flush=True
)


# ============================================================
# 1. OFFICIAL BWGNN SOURCE SUPPORT
# ============================================================

source = DATASET_PY.read_text(
    errors="replace"
).lower()


fd_terms = [
    "fdcomp",
    "fdcompcn",
    "comp.dgl"
]

native_hits = [
    term
    for term in fd_terms
    if term in source
]


print(
    "\n[1/4] Frozen BWGNN repository support",
    flush=True
)

print(
    "FDCompCN references:",
    native_hits if native_hits else "NONE",
    flush=True
)

if native_hits:
    print(
        "Repository classification: POSSIBLE NATIVE PATH",
        flush=True
    )
else:
    print(
        "Repository classification: NOT NATIVE",
        flush=True
    )


# ============================================================
# 2. AUTHOR ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(map(str, cuda))
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# 3. FDCOMPCN STRUCTURE + HOMOGENEOUS-ADAPTER AUDIT
# ============================================================

script = r'''
import sys
import dgl
import torch

from dgl.data.utils import load_graphs

DATA = sys.argv[1]

print(
    "\n[2/4] Loading canonical FDCompCN...",
    flush=True
)

graphs, _ = load_graphs(DATA)

g = graphs[0]

print(
    "Nodes:",
    g.num_nodes(),
    flush=True
)

print(
    "Edges:",
    g.num_edges(),
    flush=True
)

print(
    "Node types:",
    g.ntypes,
    flush=True
)

print(
    "Canonical edge types:",
    g.canonical_etypes,
    flush=True
)

print(
    "Edge relation names:",
    g.etypes,
    flush=True
)

print(
    "Node data keys:",
    list(g.ndata.keys()),
    flush=True
)

assert g.num_nodes() == 5317
assert g.num_edges() == 30752


# ------------------------------------------------------------
# Feature / label verification
# ------------------------------------------------------------

feature_key = None

for key in [
    "feature",
    "feat",
    "features"
]:
    if key in g.ndata:
        feature_key = key
        break


label_key = None

for key in [
    "label",
    "labels"
]:
    if key in g.ndata:
        label_key = key
        break


print(
    "\n[3/4] Feature / label compatibility",
    flush=True
)

print(
    "Feature key:",
    feature_key,
    flush=True
)

print(
    "Label key:",
    label_key,
    flush=True
)


if feature_key is None:
    raise RuntimeError(
        "No compatible node feature tensor found."
    )

if label_key is None:
    raise RuntimeError(
        "No compatible node label tensor found."
    )


x = g.ndata[
    feature_key
]

raw_y = g.ndata[
    label_key
]


print(
    "Feature shape:",
    tuple(x.shape),
    flush=True
)

print(
    "Raw label shape:",
    tuple(raw_y.shape),
    flush=True
)


assert tuple(x.shape) == (
    5317,
    57
)


if raw_y.ndim == 2:

    y = raw_y.argmax(
        1
    )

else:

    y = raw_y.reshape(
        -1
    )


normal = int(
    (y == 0).sum()
)

fraud = int(
    (y == 1).sum()
)


print(
    "Normal:",
    normal,
    flush=True
)

print(
    "Fraud:",
    fraud,
    flush=True
)


assert normal == 4758
assert fraud == 559


# ------------------------------------------------------------
# Relation structure
# ------------------------------------------------------------

print(
    "\n[4/4] Homogeneous representation test",
    flush=True
)


relation_count = len(
    g.canonical_etypes
)

print(
    "Relation count:",
    relation_count,
    flush=True
)


# This does NOT train BWGNN.
# It only verifies whether DGL can represent all relations in
# one homogeneous graph without dropping nodes or edges.

hg = dgl.to_homogeneous(
    g
)


print(
    "Homogeneous nodes:",
    hg.num_nodes(),
    flush=True
)

print(
    "Homogeneous edges:",
    hg.num_edges(),
    flush=True
)


assert hg.num_nodes() == g.num_nodes()
assert hg.num_edges() == g.num_edges()


print(
    "All nodes preserved: YES",
    flush=True
)

print(
    "All stored edges preserved: YES",
    flush=True
)


# DGL stores original relation IDs in _TYPE after conversion.
# BWGNN itself does not consume those relation IDs.

print(
    "Relation-ID field present after conversion:",
    dgl.ETYPE in hg.edata,
    flush=True
)


print(
    "\nIMPORTANT:",
    flush=True
)

print(
    "A homogeneous adapter can preserve all nodes and edges, "
    "but standard BWGNN will treat the combined adjacency as "
    "one relation and will not perform relation-specific "
    "message passing.",
    flush=True
)
'''


p = subprocess.run(
    [
        str(PYTHON),
        "-c",
        script,
        str(DATA)
    ],
    env=env,
    text=True,
    capture_output=True
)


print(
    p.stdout,
    flush=True
)


if p.returncode != 0:

    print(
        p.stderr,
        flush=True
    )

    raise RuntimeError(
        f"FDCompCN audit failed: {p.returncode}"
    )


# ============================================================
# 4. PROVISIONAL DECISION
# ============================================================

if native_hits:

    decision = "NATIVE"

else:

    decision = "CONDITIONAL INPUT ADAPTER"


print(
    "\n===== STEP 28A GATE =====",
    flush=True
)

print(
    "PASS — FDCompCN compatibility audit completed.",
    flush=True
)

print(
    "BWGNN native FDCompCN support:",
    "YES" if native_hits else "NO",
    flush=True
)

print(
    "Homogeneous representation possible: YES",
    flush=True
)

print(
    "Nodes/edges preserved by representation conversion: YES",
    flush=True
)

print(
    "Relation-specific semantics used by standard BWGNN: NO",
    flush=True
)

print(
    "Provisional compatibility:",
    decision,
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

AssertionError: /kaggle/working/Rethinking-Anomaly-Detection/dataset.py

In [4]:
from pathlib import Path

print("WORK:", Path("/kaggle/working/comp8851_bwgnn").exists())
print("REPO:", Path("/kaggle/working/Rethinking-Anomaly-Detection").exists())
print(
    "ENV:",
    Path(
        "/kaggle/working/comp8851_bwgnn/"
        "envs/bwgnn-author/bin/python"
    ).exists()
)

print(
    "FDCompCN:",
    Path(
        "/kaggle/input/datasets/pathikahmed0007/"
        "fdcompcn/comp.dgl"
    ).exists()
)

WORK: False
REPO: False
ENV: False
FDCompCN: True


In [5]:
# ============================================================
# STEP 28R — FAST BWGNN RECOVERY + FDCOMPCN COMPATIBILITY AUDIT
# Fresh Kaggle runtime
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
REPO = Path("/kaggle/working/Rethinking-Anomaly-Detection")
ENV = WORK / "envs/bwgnn-author"
PY = ENV / "bin/python"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

COMMIT = "de0631f039bbd19c1890b483cc01f1007f596af7"

WORK.mkdir(parents=True, exist_ok=True)

def run(cmd, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)

    p = subprocess.Popen(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    for line in p.stdout:
        print(line.rstrip(), flush=True)

    rc = p.wait()

    if rc != 0:
        raise RuntimeError(
            f"Command failed with return code {rc}"
        )


print("===== STEP 28R — FAST RECOVERY =====", flush=True)

# ============================================================
# 1. UV
# ============================================================

print("\n[RECOVERY 1/5] Preparing uv...", flush=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "uv"
    ],
    check=True
)

print("[RECOVERY 1/5] PASS", flush=True)


# ============================================================
# 2. FROZEN BWGNN REPOSITORY
# ============================================================

print("\n[RECOVERY 2/5] Restoring frozen BWGNN repository...", flush=True)

if not REPO.exists():

    run([
        "git",
        "clone",
        "https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git",
        str(REPO)
    ])

run([
    "git",
    "-C",
    str(REPO),
    "checkout",
    "--force",
    COMMIT
])

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()

assert head == COMMIT

print("Frozen commit:", head, flush=True)
print("[RECOVERY 2/5] PASS", flush=True)


# ============================================================
# 3. PYTHON 3.9 ENVIRONMENT
# ============================================================

print("\n[RECOVERY 3/5] Creating Python 3.9 environment...", flush=True)

if not PY.exists():

    run([
        "uv",
        "venv",
        str(ENV),
        "--python",
        "3.9"
    ])

print("[RECOVERY 3/5] PASS", flush=True)


# ============================================================
# 4. AUTHOR-COMPATIBLE TORCH + DGL
# ============================================================

print(
    "\n[RECOVERY 4/5] Installing BWGNN author-compatible packages...",
    flush=True
)

# Keep this minimal. FDCompCN compatibility audit is CPU-side,
# but we restore the same Torch/DGL generation used by BWGNN.

run([
    "uv",
    "pip",
    "install",
    "--python",
    str(PY),

    "numpy==1.23.5",
    "scipy==1.9.3",
    "scikit-learn==1.1.3",
    "sympy==1.10.1",

    "torch==1.9.0+cu111",

    "--find-links",
    "https://download.pytorch.org/whl/torch_stable.html"
])

run([
    "uv",
    "pip",
    "install",
    "--python",
    str(PY),

    "dgl-cu111==0.8.1",

    "--find-links",
    "https://data.dgl.ai/wheels/repo.html"
])

print("[RECOVERY 4/5] PASS", flush=True)


# ============================================================
# 5. ENVIRONMENT VERIFICATION
# ============================================================

print("\n[RECOVERY 5/5] Verifying environment...", flush=True)

verify = r'''
import torch
import dgl
import numpy as np

print("Python environment: PASS")
print("Torch:", torch.__version__)
print("DGL:", dgl.__version__)
print("NumPy:", np.__version__)
'''

run([
    str(PY),
    "-c",
    verify
])

print("[RECOVERY 5/5] PASS", flush=True)


# ============================================================
# FDCOMPCN COMPATIBILITY AUDIT
# ============================================================

print(
    "\n\n===== STEP 28A — BWGNN × FDCOMPCN COMPATIBILITY AUDIT =====",
    flush=True
)

assert DATA.exists(), DATA


# ------------------------------------------------------------
# A. Native BWGNN repository support
# ------------------------------------------------------------

dataset_py = REPO / "dataset.py"

assert dataset_py.exists()

source = dataset_py.read_text(
    errors="replace"
).lower()

terms = [
    "fdcomp",
    "fdcompcn",
    "comp.dgl"
]

hits = [
    x for x in terms
    if x in source
]

print(
    "\n[AUDIT 1/3] Native repository support:",
    "FOUND" if hits else "NONE",
    flush=True
)

if hits:
    print("Matches:", hits, flush=True)
else:
    print("BWGNN native FDCompCN path: NO", flush=True)


# ------------------------------------------------------------
# B. Graph structure
# ------------------------------------------------------------

audit = r'''
import sys
import dgl

from dgl.data.utils import load_graphs

DATA = sys.argv[1]

print(
    "\n[AUDIT 2/3] Loading FDCompCN...",
    flush=True
)

graphs, _ = load_graphs(DATA)
g = graphs[0]

print("Nodes:", g.num_nodes(), flush=True)
print("Edges:", g.num_edges(), flush=True)
print("Node types:", g.ntypes, flush=True)
print("Relations:", g.etypes, flush=True)
print("Canonical relations:", g.canonical_etypes, flush=True)
print("Node-data keys:", list(g.ndata.keys()), flush=True)

assert g.num_nodes() == 5317
assert g.num_edges() == 30752


# Feature key
feature_key = None

for key in ["feature", "feat", "features"]:
    if key in g.ndata:
        feature_key = key
        break


# Label key
label_key = None

for key in ["label", "labels"]:
    if key in g.ndata:
        label_key = key
        break


assert feature_key is not None
assert label_key is not None

x = g.ndata[feature_key]
raw_y = g.ndata[label_key]

print("Feature key:", feature_key, flush=True)
print("Feature shape:", tuple(x.shape), flush=True)
print("Label key:", label_key, flush=True)
print("Raw label shape:", tuple(raw_y.shape), flush=True)

assert tuple(x.shape) == (5317, 57)

if raw_y.ndim == 2:
    y = raw_y.argmax(1)
else:
    y = raw_y.reshape(-1)

normal = int((y == 0).sum())
fraud = int((y == 1).sum())

print("Normal:", normal, flush=True)
print("Fraud:", fraud, flush=True)

assert normal == 4758
assert fraud == 559


# ------------------------------------------------------------
# C. Representation-only homogeneous conversion
# ------------------------------------------------------------

print(
    "\n[AUDIT 3/3] Testing representation-only homogeneous conversion...",
    flush=True
)

hg = dgl.to_homogeneous(g)

print("Homogeneous nodes:", hg.num_nodes(), flush=True)
print("Homogeneous edges:", hg.num_edges(), flush=True)

assert hg.num_nodes() == g.num_nodes()
assert hg.num_edges() == g.num_edges()

print("All nodes preserved: YES", flush=True)
print("All stored edges preserved: YES", flush=True)

print(
    "Original relation count:",
    len(g.canonical_etypes),
    flush=True
)

print(
    "Standard BWGNN relation-specific message passing: NO",
    flush=True
)

print(
    "\nAUDIT COMPLETE",
    flush=True
)
'''

run([
    str(PY),
    "-c",
    audit,
    str(DATA)
])


# ============================================================
# FINAL PROVISIONAL CLASSIFICATION
# ============================================================

print(
    "\n===== STEP 28A GATE =====",
    flush=True
)

print(
    "PASS — BWGNN × FDCompCN compatibility audit completed.",
    flush=True
)

print(
    "Native BWGNN FDCompCN support:",
    "YES" if hits else "NO",
    flush=True
)

print(
    "Representation-only homogeneous adapter possible: YES",
    flush=True
)

print(
    "All nodes/edges preserved: YES",
    flush=True
)

print(
    "FDCompCN relation-specific semantics retained by standard BWGNN: NO",
    flush=True
)

print(
    "Provisional classification: CONDITIONAL INPUT ADAPTER",
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

===== STEP 28R — FAST RECOVERY =====

[RECOVERY 1/5] Preparing uv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 65.7 MB/s eta 0:00:00
[RECOVERY 1/5] PASS

[RECOVERY 2/5] Restoring frozen BWGNN repository...

$ git clone https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git /kaggle/working/Rethinking-Anomaly-Detection
Cloning into '/kaggle/working/Rethinking-Anomaly-Detection'...

$ git -C /kaggle/working/Rethinking-Anomaly-Detection checkout --force de0631f039bbd19c1890b483cc01f1007f596af7
Note: switching to 'de0631f039bbd19c1890b483cc01f1007f596af7'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation wi

RuntimeError: Command failed with return code 1

In [6]:
# ============================================================
# STEP 28R-FIX — RESTORE CUDA 11 RUNTIME LIBRARIES
# Continue from failed Recovery 5/5
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

ENV = WORK / "envs/bwgnn-author"
PY = ENV / "bin/python"

SITE = (
    ENV /
    "lib/python3.9/site-packages"
)

print("===== STEP 28R-FIX — CUDA LIBRARY REPAIR =====", flush=True)


# ============================================================
# 1. INSTALL ONLY THE MISSING CUDA RUNTIME LIBRARIES
# ============================================================

print("\n[FIX 1/3] Restoring CUDA runtime libraries...", flush=True)

cmd = [
    "uv",
    "pip",
    "install",
    "--python",
    str(PY),

    "nvidia-cuda-runtime-cu11",
    "nvidia-cublas-cu11",
    "nvidia-cusparse-cu11"
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in p.stdout:
    print(line.rstrip(), flush=True)

rc = p.wait()

if rc != 0:
    raise RuntimeError(
        f"CUDA library install failed: {rc}"
    )

print("[FIX 1/3] PASS", flush=True)


# ============================================================
# 2. BUILD LD_LIBRARY_PATH
# ============================================================

print("\n[FIX 2/3] Configuring library path...", flush=True)

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

for d in cuda_dirs:
    print(
        d,
        "FOUND" if d.exists() else "MISSING",
        flush=True
    )

assert all(
    d.exists()
    for d in cuda_dirs
)

runtime_env = os.environ.copy()

runtime_env["CUDA_VISIBLE_DEVICES"] = "0"
runtime_env["DGLBACKEND"] = "pytorch"
runtime_env["PYTHONUNBUFFERED"] = "1"

runtime_env["LD_LIBRARY_PATH"] = (
    ":".join(
        str(d)
        for d in cuda_dirs
    )
    + ":"
    + runtime_env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)

print("[FIX 2/3] PASS", flush=True)


# ============================================================
# 3. VERIFY TORCH + DGL + GPU
# ============================================================

print("\n[FIX 3/3] Verifying recovered BWGNN environment...", flush=True)

verify = r'''
import torch
import dgl
import numpy as np

print("Python environment: PASS")
print("Torch:", torch.__version__)
print("DGL:", dgl.__version__)
print("NumPy:", np.__version__)

print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))

# Tiny DGL CUDA gate
g = dgl.graph(
    ([0, 1], [1, 0])
).to("cuda:0")

print(
    "DGL CUDA graph:",
    g.device
)

print("DGL CUDA verification: PASS")
'''

p = subprocess.run(
    [
        str(PY),
        "-c",
        verify
    ],
    env=runtime_env,
    text=True,
    capture_output=True
)

print(p.stdout, flush=True)

if p.returncode != 0:
    print(p.stderr, flush=True)
    raise RuntimeError(
        f"Environment verification failed: {p.returncode}"
    )

print("===== STEP 28R-FIX GATE =====")
print("PASS — BWGNN CUDA environment recovered.")

===== STEP 28R-FIX — CUDA LIBRARY REPAIR =====

[FIX 1/3] Restoring CUDA runtime libraries...
Using Python 3.9.25 environment at: comp8851_bwgnn/envs/bwgnn-author
Resolved 3 packages in 72ms
Prepared 3 packages in 4.66s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 3 packages in 855ms
 + nvidia-cublas-cu11==11.11.3.6
 + nvidia-cuda-runtime-cu11==11.8.89
 + nvidia-cusparse-cu11==11.7.5.86
[FIX 1/3] PASS

[FIX 2/3] Configuring library path...
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cuda_runtime/lib FOUND
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cublas/lib FOUND
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cusparse/lib FOUND
[FIX 2/3] PASS

[FIX 3/3] Verifying recovered BWGNN env

## Step 28A — BWGNN × FDCompCN Fast Compatibility Audit

FDCompCN is not a native dataset in the official BWGNN repository, so this
model–dataset pair requires an input-adapter feasibility decision before
training.

This audit checks whether FDCompCN can be translated to the same homogeneous
graph representation already used by BWGNN for its author-supported
multi-relation datasets.

No training, validation selection or test evaluation is performed.

In [7]:
# ============================================================
# STEP 28A — FAST BWGNN × FDCOMPCN COMPATIBILITY AUDIT
# NO TRAINING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
REPO = Path("/kaggle/working/Rethinking-Anomaly-Detection")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

assert PY.exists()
assert REPO.exists()
assert DATA.exists()

print("===== STEP 28A — FDCOMPCN COMPATIBILITY =====", flush=True)


# ============================================================
# 1. CHECK FROZEN BWGNN SOURCE
# ============================================================

dataset_py = REPO / "dataset.py"
main_py = REPO / "main.py"

dataset_text = dataset_py.read_text(errors="replace").lower()
main_text = main_py.read_text(errors="replace").lower()

native_fd = any(
    term in dataset_text
    for term in ["fdcomp", "fdcompcn", "comp.dgl"]
)

existing_homo = (
    "to_homogeneous" in dataset_text
    or "to_homogeneous" in main_text
)

print("\n[1/3] Frozen BWGNN source", flush=True)
print("Native FDCompCN path:", native_fd, flush=True)
print(
    "Existing BWGNN homogeneous-conversion path:",
    existing_homo,
    flush=True
)


# ============================================================
# 2. AUTHOR ENV
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# 3. DATA + REPRESENTATION AUDIT
# ============================================================

audit = r'''
import sys
import dgl
import torch
from dgl.data.utils import load_graphs

DATA = sys.argv[1]

print("\n[2/3] Loading canonical FDCompCN...", flush=True)

graphs, _ = load_graphs(DATA)
g = graphs[0]

print("Nodes:", g.num_nodes(), flush=True)
print("Edges:", g.num_edges(), flush=True)
print("Node types:", g.ntypes, flush=True)
print("Relations:", g.etypes, flush=True)
print("Canonical etypes:", g.canonical_etypes, flush=True)
print("Node data keys:", list(g.ndata.keys()), flush=True)

assert g.num_nodes() == 5317
assert g.num_edges() == 30752


# ------------------------------------------------------------
# Features / labels
# ------------------------------------------------------------

feature_key = next(
    (k for k in ["feature", "feat", "features"] if k in g.ndata),
    None
)

label_key = next(
    (k for k in ["label", "labels"] if k in g.ndata),
    None
)

assert feature_key is not None
assert label_key is not None

x = g.ndata[feature_key]
raw_y = g.ndata[label_key]

if raw_y.ndim == 2:
    y = raw_y.argmax(1)
else:
    y = raw_y.reshape(-1)

normal = int((y == 0).sum().item())
fraud = int((y == 1).sum().item())

print("Feature key:", feature_key, flush=True)
print("Feature shape:", tuple(x.shape), flush=True)
print("Label key:", label_key, flush=True)
print("Normal:", normal, flush=True)
print("Fraud:", fraud, flush=True)

assert tuple(x.shape) == (5317, 57)
assert normal == 4758
assert fraud == 559


# ------------------------------------------------------------
# Homogeneous translation only
# ------------------------------------------------------------

print(
    "\n[3/3] Testing BWGNN-style homogeneous representation...",
    flush=True
)

hg = dgl.to_homogeneous(
    g,
    ndata=[feature_key, label_key]
)

print("Converted nodes:", hg.num_nodes(), flush=True)
print("Converted edges:", hg.num_edges(), flush=True)
print("Converted feature shape:", tuple(hg.ndata[feature_key].shape), flush=True)

assert hg.num_nodes() == 5317
assert hg.num_edges() == 30752
assert tuple(hg.ndata[feature_key].shape) == (5317, 57)

print("Nodes preserved: YES", flush=True)
print("Edges preserved: YES", flush=True)
print("Features preserved: YES", flush=True)
print("Labels preserved: YES", flush=True)
'''

p = subprocess.run(
    [
        str(PY),
        "-c",
        audit,
        str(DATA)
    ],
    env=env,
    text=True,
    capture_output=True
)

print(p.stdout, flush=True)

if p.returncode != 0:
    print(p.stderr, flush=True)
    raise RuntimeError(f"Audit failed: {p.returncode}")


# ============================================================
# FINAL DECISION
# ============================================================

if native_fd:
    decision = "NATIVE"

elif existing_homo:
    decision = "INPUT ADAPTER ELIGIBLE"

else:
    decision = "CONDITIONAL — MANUAL REVIEW REQUIRED"


print("\n===== STEP 28A GATE =====", flush=True)

print(
    "PASS — BWGNN × FDCompCN compatibility audit complete.",
    flush=True
)

print("Native repository path:", "YES" if native_fd else "NO", flush=True)

print(
    "Existing BWGNN homogeneous mechanism:",
    "YES" if existing_homo else "NO",
    flush=True
)

print("Nodes/edges/features/labels preserved: YES", flush=True)

print("Compatibility decision:", decision, flush=True)

print("Training performed: NO", flush=True)
print("Test accessed: NO", flush=True)

===== STEP 28A — FDCOMPCN COMPATIBILITY =====

[1/3] Frozen BWGNN source
Native FDCompCN path: False
Existing BWGNN homogeneous-conversion path: True

[2/3] Loading canonical FDCompCN...
Nodes: 5317
Edges: 30752
Node types: ['company']
Relations: ['homo', 'invest_bc2bc', 'provide_bc2bc', 'sale_bc2bc']
Canonical etypes: [('company', 'homo', 'company'), ('company', 'invest_bc2bc', 'company'), ('company', 'provide_bc2bc', 'company'), ('company', 'sale_bc2bc', 'company')]
Node data keys: ['train_mask', 'test_mask', 'valid_mask', 'label', 'feature']
Feature key: feature
Feature shape: (5317, 57)
Label key: label
Normal: 4758
Fraud: 559

[3/3] Testing BWGNN-style homogeneous representation...
Converted nodes: 5317
Converted edges: 30752
Converted feature shape: (5317, 57)
Nodes preserved: YES
Edges preserved: YES
Features preserved: YES
Labels preserved: YES


===== STEP 28A GATE =====
PASS — BWGNN × FDCompCN compatibility audit complete.
Native repository path: NO
Existing BWGNN homogeneous

## Step 28B — BWGNN × FDCompCN Input-Adapter Smoke Test

Step 28A classified FDCompCN as **INPUT ADAPTER ELIGIBLE**.

FDCompCN is converted using the same homogeneous representation mechanism
already present in BWGNN. All nodes, edges, features and labels are retained.

For the controlled benchmark:

- split seed: 2
- training split: TR40
- training seed: 2
- hidden dimension: 64
- wavelet order: 2
- learning rate: 0.01
- epochs: 1
- GPU: Tesla T4, GPU 0

The controlled nested split is recreated because the Kaggle runtime restarted.

The smoke test evaluates validation only. The test set is not accessed.

In [8]:
# ============================================================
# STEP 28B — FDCOMPCN ADAPTER + SPLIT + 1-EPOCH BWGNN SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")
REPO = Path("/kaggle/working/Rethinking-Anomaly-Detection")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

SPLIT_DIR = WORK / "shared/splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = (
    SPLIT_DIR /
    "fdcompcn_seed2_nested_splits.npz"
)

ADAPTER_DIR = WORK / "adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

BWGNN_ADAPTER = (
    ADAPTER_DIR /
    "BWGNN_gpu.py"
)

assert PY.exists()
assert REPO.exists()
assert DATA.exists()

print(
    "===== STEP 28B — FDCOMPCN BWGNN SMOKE =====",
    flush=True
)


# ============================================================
# 1. REBUILD EXACT BWGNN GPU ADAPTER
# ============================================================

print(
    "\n[1/5] Restoring BWGNN GPU adapter...",
    flush=True
)

source_candidates = list(
    REPO.rglob("BWGNN.py")
)

assert source_candidates, (
    "Could not find BWGNN.py in frozen repository."
)

SOURCE = source_candidates[0]

text = SOURCE.read_text(
    encoding="utf-8"
)

original = text

# The frozen implementation contains CPU-created zero tensors.
# Make only those allocations device-aware.

patterns = [
    (
        "torch.zeros([in_feat.shape[0], 1])",
        "torch.zeros([in_feat.shape[0], 1], device=in_feat.device)"
    ),
    (
        "torch.zeros((in_feat.shape[0], 1))",
        "torch.zeros((in_feat.shape[0], 1), device=in_feat.device)"
    ),
    (
        "torch.zeros([in_feat.shape[0], self._in_feats])",
        "torch.zeros([in_feat.shape[0], self._in_feats], device=in_feat.device)"
    ),
    (
        "torch.zeros((in_feat.shape[0], self._in_feats))",
        "torch.zeros((in_feat.shape[0], self._in_feats), device=in_feat.device)"
    )
]

replacement_count = 0

for old, new in patterns:

    count = text.count(old)

    if count:

        text = text.replace(
            old,
            new
        )

        replacement_count += count


# If exact literals differ slightly, use the previously validated
# generic device patch only for torch.zeros lines involving in_feat.

if replacement_count != 4:

    import re

    text = original

    lines = text.splitlines()

    new_lines = []

    replacement_count = 0

    for line in lines:

        if (
            "torch.zeros" in line
            and "in_feat" in line
            and "device=" not in line
        ):

            closing = line.rfind(")")

            if closing != -1:

                line = (
                    line[:closing]
                    + ", device=in_feat.device"
                    + line[closing:]
                )

                replacement_count += 1

        new_lines.append(line)

    text = "\n".join(new_lines) + "\n"


assert replacement_count == 4, (
    f"Expected exactly 4 GPU-device patches; found {replacement_count}"
)

BWGNN_ADAPTER.write_text(
    text,
    encoding="utf-8"
)

print(
    "Frozen source:",
    SOURCE,
    flush=True
)

print(
    "GPU-device replacements:",
    replacement_count,
    flush=True
)

print(
    "[1/5] PASS",
    flush=True
)


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# 2–5. SPLIT + ADAPTER + SMOKE
# ============================================================

script = r'''
import sys
import time
import random
import numpy as np
import torch
import torch.nn.functional as F
import dgl

from dgl.data.utils import load_graphs

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

DATA = sys.argv[1]
SPLIT_PATH = sys.argv[2]
ADAPTER_DIR = sys.argv[3]

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


SEED = 2
HIDDEN = 64
ORDER = 2
LR = 0.01

DEVICE = torch.device(
    "cuda:0"
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. LOAD DATA
# ============================================================

print(
    "\n[2/5] Loading canonical FDCompCN...",
    flush=True
)

graphs, _ = load_graphs(DATA)

g = graphs[0]

assert g.num_nodes() == 5317
assert g.num_edges() == 30752

x = g.ndata["feature"]

raw_y = g.ndata["label"]

if raw_y.ndim == 2:

    y_cpu = (
        raw_y
        .argmax(1)
        .long()
    )

else:

    y_cpu = (
        raw_y
        .reshape(-1)
        .long()
    )


assert int(
    (y_cpu == 0).sum()
) == 4758

assert int(
    (y_cpu == 1).sum()
) == 559


print(
    "Nodes:",
    g.num_nodes(),
    flush=True
)

print(
    "Edges:",
    g.num_edges(),
    flush=True
)

print(
    "Features:",
    tuple(x.shape),
    flush=True
)

print(
    "[2/5] PASS",
    flush=True
)


# ============================================================
# 3. RECREATE CONTROLLED NESTED SPLIT
# ============================================================

print(
    "\n[3/5] Recreating split seed 2...",
    flush=True
)

labels_np = (
    y_cpu
    .cpu()
    .numpy()
)

ids = np.arange(
    len(labels_np),
    dtype=np.int64
)


remaining, test = train_test_split(
    ids,
    test_size=0.40,
    stratify=labels_np,
    random_state=SEED,
    shuffle=True
)


tr40, val = train_test_split(
    remaining,
    test_size=(1/3),
    stratify=labels_np[remaining],
    random_state=SEED,
    shuffle=True
)


tr30, _ = train_test_split(
    tr40,
    train_size=0.75,
    stratify=labels_np[tr40],
    random_state=SEED,
    shuffle=True
)


tr20, _ = train_test_split(
    tr30,
    train_size=(2/3),
    stratify=labels_np[tr30],
    random_state=SEED,
    shuffle=True
)


tr10, _ = train_test_split(
    tr20,
    train_size=0.50,
    stratify=labels_np[tr20],
    random_state=SEED,
    shuffle=True
)


splits = {
    "TR40": np.sort(tr40),
    "TR30": np.sort(tr30),
    "TR20": np.sort(tr20),
    "TR10": np.sort(tr10),
    "val": np.sort(val),
    "test": np.sort(test)
}


expected_sizes = {
    "TR40": 2126,
    "TR30": 1594,
    "TR20": 1062,
    "TR10": 531,
    "val": 1064,
    "test": 2127
}


expected_counts = {
    "TR40": (1903, 223),
    "TR30": (1427, 167),
    "TR20": (951, 111),
    "TR10": (475, 56),
    "val": (952, 112),
    "test": (1903, 224)
}


for name, expected in expected_sizes.items():

    assert len(
        splits[name]
    ) == expected

    ids_here = splits[name]

    normal = int(
        (labels_np[ids_here] == 0).sum()
    )

    fraud = int(
        (labels_np[ids_here] == 1).sum()
    )

    assert (
        normal,
        fraud
    ) == expected_counts[name]

    print(
        f"{name:<5} "
        f"{len(ids_here):>4} | "
        f"normal={normal} | fraud={fraud}",
        flush=True
    )


assert set(
    splits["TR10"]
) <= set(
    splits["TR20"]
)

assert set(
    splits["TR20"]
) <= set(
    splits["TR30"]
)

assert set(
    splits["TR30"]
) <= set(
    splits["TR40"]
)


np.savez_compressed(
    SPLIT_PATH,
    **splits,
    seed=np.array(
        [SEED],
        dtype=np.int64
    )
)


print(
    "Nested split: PASS",
    flush=True
)

print(
    "[3/5] PASS",
    flush=True
)


# ============================================================
# 4. BWGNN INPUT ADAPTER
# ============================================================

print(
    "\n[4/5] Applying BWGNN homogeneous input adapter...",
    flush=True
)


# Preserve feature + label tensors while combining relations.

hg = dgl.to_homogeneous(
    g,
    ndata=[
        "feature",
        "label"
    ]
)


assert hg.num_nodes() == 5317
assert hg.num_edges() == 30752


# Match BWGNN's existing homogeneous execution semantics.

hg = dgl.add_self_loop(
    hg
)


print(
    "Before self-loops:",
    30752,
    flush=True
)

print(
    "After self-loops:",
    hg.num_edges(),
    flush=True
)

assert hg.num_edges() == (
    30752 + 5317
)


print(
    "All original nodes/edges retained: YES",
    flush=True
)

print(
    "[4/5] PASS",
    flush=True
)


# ============================================================
# 5. ONE-EPOCH GPU SMOKE
# ============================================================

print(
    "\n[5/5] Starting 1-epoch BWGNN smoke...",
    flush=True
)

print(
    "Test set accessed: NO",
    flush=True
)


hg = hg.to(
    DEVICE
)


features = (
    hg.ndata["feature"]
    .float()
)


raw_labels = hg.ndata[
    "label"
]


if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .long()
    )

else:

    labels = (
        raw_labels
        .reshape(-1)
        .long()
    )


train_ids = torch.tensor(
    splits["TR40"],
    dtype=torch.long,
    device=DEVICE
)


val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)


train_y = labels[
    train_ids
]


train_normal = int(
    (train_y == 0).sum().item()
)

train_fraud = int(
    (train_y == 1).sum().item()
)


class_weight = torch.tensor(
    [
        1.0,
        train_normal / train_fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)


model = BWGNN(
    in_feats=57,
    h_feats=HIDDEN,
    num_classes=2,
    graph=hg,
    d=ORDER
).to(
    DEVICE
)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.999),
    eps=1e-8
)


torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# TRAINING-ONLY TIMING
# ------------------------------------------------------------

model.train()

torch.cuda.synchronize()

start = time.perf_counter()


logits = model(
    features
)


loss = F.cross_entropy(
    logits[train_ids],
    labels[train_ids],
    weight=class_weight
)


optimizer.zero_grad()

loss.backward()

optimizer.step()


torch.cuda.synchronize()

train_seconds = (
    time.perf_counter()
    - start
)


training_loss = float(
    loss.item()
)


# ------------------------------------------------------------
# VALIDATION — separate timing
# ------------------------------------------------------------

del logits

model.eval()

torch.cuda.synchronize()

val_start = time.perf_counter()


with torch.no_grad():

    logits = model(
        features
    )


torch.cuda.synchronize()


val_seconds = (
    time.perf_counter()
    - val_start
)


prob = (
    torch.softmax(
        logits,
        dim=1
    )[val_ids, 1]
    .cpu()
    .numpy()
)


val_y = (
    labels[val_ids]
    .cpu()
    .numpy()
)


auprc = float(
    average_precision_score(
        val_y,
        prob
    )
)


auroc = float(
    roc_auc_score(
        val_y,
        prob
    )
)


peak_mb = float(
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


print(
    "Training loss:",
    f"{training_loss:.6f}",
    flush=True
)

print(
    "Training-only seconds:",
    f"{train_seconds:.6f}",
    flush=True
)

print(
    "Validation seconds:",
    f"{val_seconds:.6f}",
    flush=True
)

print(
    "Validation AUPRC:",
    f"{auprc:.6f}",
    flush=True
)

print(
    "Validation AUROC:",
    f"{auroc:.6f}",
    flush=True
)

print(
    "Peak GPU memory MB:",
    f"{peak_mb:.2f}",
    flush=True
)


print(
    "[5/5] PASS",
    flush=True
)


print(
    "\n===== STEP 28B GATE =====",
    flush=True
)

print(
    "PASS — BWGNN × FDCompCN input adapter completed 1 training epoch.",
    flush=True
)

print(
    "Compatibility: INPUT ADAPTER ELIGIBLE",
    flush=True
)

print(
    "Split recreated and verified: YES",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)
'''


p = subprocess.run(
    [
        str(PY),
        "-c",
        script,
        str(DATA),
        str(SPLIT),
        str(ADAPTER_DIR)
    ],
    env=env,
    text=True,
    capture_output=True
)


print(
    p.stdout,
    flush=True
)


if p.returncode != 0:

    print(
        p.stderr,
        flush=True
    )

    raise RuntimeError(
        f"Step 28B failed: {p.returncode}"
    )

===== STEP 28B — FDCOMPCN BWGNN SMOKE =====

[1/5] Restoring BWGNN GPU adapter...
Frozen source: /kaggle/working/Rethinking-Anomaly-Detection/BWGNN.py
GPU-device replacements: 4
[1/5] PASS

[2/5] Loading canonical FDCompCN...
Nodes: 5317
Edges: 30752
Features: (5317, 57)
[2/5] PASS

[3/5] Recreating split seed 2...
TR40  2126 | normal=1903 | fraud=223
TR30  1594 | normal=1427 | fraud=167
TR20  1062 | normal=951 | fraud=111
TR10   531 | normal=475 | fraud=56
val   1064 | normal=952 | fraud=112
test  2127 | normal=1903 | fraud=224
Nested split: PASS
[3/5] PASS

[4/5] Applying BWGNN homogeneous input adapter...
Before self-loops: 30752
After self-loops: 36069
All original nodes/edges retained: YES
[4/5] PASS

[5/5] Starting 1-epoch BWGNN smoke...
Test set accessed: NO
Training loss: 0.694459
Training-only seconds: 0.035640
Validation seconds: 0.003747
Validation AUPRC: 0.128479
Validation AUROC: 0.581262
Peak GPU memory MB: 16.80
[5/5] PASS

===== STEP 28B GATE =====
PASS — BWGNN × FDComp

## Step 29 — BWGNN × FDCompCN TR40 Unified Tuning

FDCompCN passed the architecture-preserving BWGNN input-adapter smoke test.

This step performs the controlled unified tuning stage.

### Fixed protocol

- Dataset: FDCompCN
- Split: TR40 only
- Split seed: 2
- Training seed: 2
- Maximum trials: 12
- Maximum epochs per trial: 100
- Early stopping patience: 20
- Optimizer: Adam
  - beta1 = 0.9
  - beta2 = 0.999
  - eps = 1e-8
- Selection metric: validation AUPRC
- Test set: not accessed

### Search

The 12-trial budget covers BWGNN-exposed hidden dimension and wavelet order,
together with learning rate and weight decay.

The configuration with the highest validation AUPRC is frozen for the later
TR40/TR30/TR20/TR10 final runs.

No test metric is used for tuning or checkpoint selection.

In [11]:
# ============================================================
# STEP 29-FIX — INSTALL PANDAS ONLY
# ============================================================

from pathlib import Path
import subprocess

PY = Path(
    "/kaggle/working/comp8851_bwgnn/"
    "envs/bwgnn-author/bin/python"
)

print("Installing pandas...", flush=True)

subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        str(PY),
        "pandas==1.5.3"
    ],
    check=True
)

print("PASS — pandas installed.")

Installing pandas...


Using Python 3.9.25 environment at: comp8851_bwgnn/envs/bwgnn-author
Resolved 5 packages in 152ms
Prepared 4 packages in 277ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.


PASS — pandas installed.


Installed 4 packages in 207ms
 + pandas==1.5.3
 + python-dateutil==2.9.0.post0
 + pytz==2026.3.post1
 + six==1.17.0


In [12]:
# ============================================================
# STEP 29 — FDCOMPCN BWGNN TR40 TUNING
# LIVE FOREGROUND PROGRESS
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "fdcompcn_seed2_nested_splits.npz"
)

ADAPTER_DIR = WORK / "adapters"

RESULT_DIR = (
    WORK /
    "results/unified/fdcompcn/tuning"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PY.exists()
assert DATA.exists()
assert SPLIT.exists()
assert (ADAPTER_DIR / "BWGNN_gpu.py").exists()


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# TUNING SCRIPT
# ============================================================

script = r'''
import sys
import gc
import json
import time
import random

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import dgl
from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


DATA = sys.argv[1]
SPLIT_PATH = sys.argv[2]
ADAPTER_DIR = sys.argv[3]
RESULT_DIR = Path(sys.argv[4])

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DEVICE = torch.device("cuda:0")

TRAIN_SEED = 2

MAX_EPOCHS = 100
PATIENCE = 20


# ============================================================
# 12 FIXED TRIALS
# ============================================================
#
# LR is within 1e-4 ... 1e-2.
# Weight decay is from the allowed protocol set.
# Hidden dimensions are exposed by BWGNN.
# Orders 1/2/3 are paper/model-supported BWGNN choices.
#
# Exactly 12 attempts — no adaptive expansion.
# ============================================================

TRIALS = [

    # author-like region
    {"hidden": 64,  "order": 2, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 0.0},

    # order variation
    {"hidden": 64,  "order": 1, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 64,  "order": 3, "lr": 1e-2, "weight_decay": 0.0},

    # lower hidden
    {"hidden": 32,  "order": 2, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 32,  "order": 3, "lr": 5e-3, "weight_decay": 0.0},

    # larger hidden
    {"hidden": 128, "order": 2, "lr": 5e-3, "weight_decay": 0.0},
    {"hidden": 128, "order": 3, "lr": 1e-3, "weight_decay": 0.0},

    # regularisation
    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 1e-5},
    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 1e-4},
    {"hidden": 64,  "order": 3, "lr": 1e-3, "weight_decay": 1e-4},
    {"hidden": 32,  "order": 2, "lr": 1e-3, "weight_decay": 1e-3},
]


assert len(TRIALS) == 12


# ============================================================
# LOAD DATA ONCE
# ============================================================

print(
    "===== STEP 29 — FDCOMPCN TR40 TUNING =====",
    flush=True
)

print(
    "Loading FDCompCN once...",
    flush=True
)

graphs, _ = load_graphs(
    DATA
)

g = graphs[0]

assert g.num_nodes() == 5317
assert g.num_edges() == 30752


# Same approved representation adapter from Step 28B.

g = dgl.to_homogeneous(
    g,
    ndata=[
        "feature",
        "label"
    ]
)

g = dgl.add_self_loop(
    g
)

assert g.num_nodes() == 5317
assert g.num_edges() == 36069


g = g.to(
    DEVICE
)


features = (
    g.ndata["feature"]
    .float()
)


raw_labels = g.ndata[
    "label"
]


if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .long()
    )

else:

    labels = (
        raw_labels
        .reshape(-1)
        .long()
    )


splits = np.load(
    SPLIT_PATH
)


train_ids = torch.tensor(
    splits["TR40"],
    dtype=torch.long,
    device=DEVICE
)


val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)


train_y = labels[
    train_ids
]


n_normal = int(
    (train_y == 0)
    .sum()
    .item()
)

n_fraud = int(
    (train_y == 1)
    .sum()
    .item()
)


class_weight = torch.tensor(
    [
        1.0,
        n_normal / n_fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)


val_y_cpu = (
    labels[val_ids]
    .detach()
    .cpu()
    .numpy()
)


print(
    "Graph:",
    g.num_nodes(),
    "nodes |",
    g.num_edges(),
    "edges",
    flush=True
)

print(
    "TR40:",
    len(train_ids),
    "| VAL:",
    len(val_ids),
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

print(
    "Trials:",
    len(TRIALS),
    flush=True
)


# ============================================================
# TUNING
# ============================================================

all_results = []

overall_start = time.perf_counter()


for trial_no, config in enumerate(
    TRIALS,
    start=1
):

    # --------------------------------------------------------
    # Deterministic reset
    # --------------------------------------------------------

    random.seed(
        TRAIN_SEED
    )

    np.random.seed(
        TRAIN_SEED
    )

    torch.manual_seed(
        TRAIN_SEED
    )

    torch.cuda.manual_seed_all(
        TRAIN_SEED
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    gc.collect()
    torch.cuda.empty_cache()


    print(
        "\n"
        + "=" * 65,
        flush=True
    )

    print(
        f"START TRIAL {trial_no:02d}/12 | "
        f"hidden={config['hidden']} | "
        f"order={config['order']} | "
        f"lr={config['lr']} | "
        f"wd={config['weight_decay']}",
        flush=True
    )


    model = BWGNN(
        in_feats=57,
        h_feats=config["hidden"],
        num_classes=2,
        graph=g,
        d=config["order"]
    ).to(
        DEVICE
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config[
            "weight_decay"
        ],
        betas=(
            0.9,
            0.999
        ),
        eps=1e-8
    )


    best_val_auprc = -1.0
    best_val_auroc = None
    best_epoch = None

    patience_counter = 0

    trial_start = time.perf_counter()


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.train()

        logits = model(
            features
        )


        loss = F.cross_entropy(
            logits[train_ids],
            labels[train_ids],
            weight=class_weight
        )


        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        # ----------------------------------------------------
        # VALIDATION ONLY
        # ----------------------------------------------------

        model.eval()


        with torch.no_grad():

            val_logits = model(
                features
            )


            val_prob = (
                torch.softmax(
                    val_logits,
                    dim=1
                )[val_ids, 1]
                .detach()
                .cpu()
                .numpy()
            )


        val_auprc = float(
            average_precision_score(
                val_y_cpu,
                val_prob
            )
        )


        val_auroc = float(
            roc_auc_score(
                val_y_cpu,
                val_prob
            )
        )


        # ----------------------------------------------------
        # CHECKPOINT SELECTION BY VAL AUPRC ONLY
        # ----------------------------------------------------

        improved = (
            val_auprc
            > best_val_auprc
            + 1e-12
        )


        if improved:

            best_val_auprc = val_auprc
            best_val_auroc = val_auroc
            best_epoch = epoch

            patience_counter = 0


            torch.save(
                {
                    "trial":
                        trial_no,

                    "epoch":
                        epoch,

                    "config":
                        config,

                    "model_state_dict":
                        model.state_dict(),

                    "val_auprc":
                        val_auprc,

                    "val_auroc":
                        val_auroc
                },
                RESULT_DIR /
                f"trial_{trial_no:02d}_best.pt"
            )


        else:

            patience_counter += 1


        # ----------------------------------------------------
        # LIVE PROGRESS
        # ----------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
            or improved
            or patience_counter >= PATIENCE
        ):

            print(
                f"trial={trial_no:02d} | "
                f"epoch={epoch:03d} | "
                f"loss={loss.item():.5f} | "
                f"valAUPRC={val_auprc:.6f} | "
                f"best={best_val_auprc:.6f}"
                f"@{best_epoch} | "
                f"patience={patience_counter}/{PATIENCE}",
                flush=True
            )


        if patience_counter >= PATIENCE:

            print(
                f"EARLY STOP trial {trial_no:02d} "
                f"at epoch {epoch}.",
                flush=True
            )

            break


    # ========================================================
    # TRIAL RESULT
    # ========================================================

    trial_seconds = (
        time.perf_counter()
        - trial_start
    )


    record = {

        "trial":
            trial_no,

        "hidden":
            config["hidden"],

        "order":
            config["order"],

        "lr":
            config["lr"],

        "weight_decay":
            config["weight_decay"],

        "best_epoch":
            best_epoch,

        "best_val_auprc":
            best_val_auprc,

        "best_val_auroc":
            best_val_auroc,

        "completed_epochs":
            epoch,

        "trial_seconds":
            trial_seconds,

        "selection_metric":
            "validation AUPRC",

        "test_accessed":
            False
    }


    all_results.append(
        record
    )


    print(
        f"DONE TRIAL {trial_no:02d}/12 | "
        f"best valAUPRC={best_val_auprc:.6f} | "
        f"epoch={best_epoch} | "
        f"time={trial_seconds:.2f}s",
        flush=True
    )


    # Keep a resumable summary after every trial.

    pd.DataFrame(
        all_results
    ).to_csv(
        RESULT_DIR /
        "fdcompcn_tuning_trials.csv",
        index=False
    )


    (
        RESULT_DIR /
        "fdcompcn_tuning_trials.json"
    ).write_text(
        json.dumps(
            all_results,
            indent=2
        )
    )


# ============================================================
# WINNER
# ============================================================

winner = max(
    all_results,
    key=lambda r:
        r["best_val_auprc"]
)


frozen = {

    "dataset":
        "FDCompCN",

    "model":
        "BWGNN",

    "mode":
        "unified",

    "compatibility":
        "INPUT ADAPTER ELIGIBLE",

    "input_adapter":
        "DGL homogeneous conversion + self-loops",

    "split":
        "TR40",

    "split_seed":
        2,

    "training_seed":
        2,

    "selection_metric":
        "validation AUPRC",

    "max_trials":
        12,

    "max_epochs":
        100,

    "patience":
        20,

    "winner":
        winner,

    "test_accessed":
        False
}


(
    RESULT_DIR /
    "fdcompcn_frozen_config.json"
).write_text(
    json.dumps(
        frozen,
        indent=2
    )
)


overall_seconds = (
    time.perf_counter()
    - overall_start
)


print(
    "\n"
    + "=" * 65,
    flush=True
)

print(
    "===== STEP 29 TUNING WINNER =====",
    flush=True
)

print(
    "Trial:",
    winner["trial"],
    flush=True
)

print(
    "Hidden:",
    winner["hidden"],
    flush=True
)

print(
    "Order:",
    winner["order"],
    flush=True
)

print(
    "Learning rate:",
    winner["lr"],
    flush=True
)

print(
    "Weight decay:",
    winner["weight_decay"],
    flush=True
)

print(
    "Best epoch:",
    winner["best_epoch"],
    flush=True
)

print(
    "Best validation AUPRC:",
    f"{winner['best_val_auprc']:.6f}",
    flush=True
)

print(
    "Validation AUROC at winner:",
    f"{winner['best_val_auroc']:.6f}",
    flush=True
)

print(
    "Total tuning seconds:",
    f"{overall_seconds:.2f}",
    flush=True
)


print(
    "\n===== STEP 29 GATE =====",
    flush=True
)

print(
    "PASS — FDCompCN TR40 tuning completed.",
    flush=True
)

print(
    "Trials completed: 12/12",
    flush=True
)

print(
    "Configuration frozen: YES",
    flush=True
)

print(
    "Selection used validation AUPRC only: YES",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)
'''


# ============================================================
# STREAM OUTPUT LIVE
# ============================================================

p = subprocess.Popen(
    [
        str(PY),
        "-c",
        script,
        str(DATA),
        str(SPLIT),
        str(ADAPTER_DIR),
        str(RESULT_DIR)
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in p.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = p.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 29 tuning failed with return code {rc}"
    )

===== STEP 29 — FDCOMPCN TR40 TUNING =====
Loading FDCompCN once...
Graph: 5317 nodes | 36069 edges
TR40: 2126 | VAL: 1064
Test accessed: NO
Trials: 12

START TRIAL 01/12 | hidden=64 | order=2 | lr=0.01 | wd=0.0
trial=01 | epoch=001 | loss=0.69446 | valAUPRC=0.128479 | best=0.128479@1 | patience=0/20
trial=01 | epoch=002 | loss=0.69267 | valAUPRC=0.135973 | best=0.135973@2 | patience=0/20
trial=01 | epoch=005 | loss=0.67540 | valAUPRC=0.136184 | best=0.136184@5 | patience=0/20
trial=01 | epoch=009 | loss=0.66347 | valAUPRC=0.138347 | best=0.138347@9 | patience=0/20
trial=01 | epoch=010 | loss=0.66599 | valAUPRC=0.139325 | best=0.139325@10 | patience=0/20
trial=01 | epoch=011 | loss=0.66019 | valAUPRC=0.139438 | best=0.139438@11 | patience=0/20
trial=01 | epoch=012 | loss=0.65973 | valAUPRC=0.143696 | best=0.143696@12 | patience=0/20
trial=01 | epoch=014 | loss=0.65471 | valAUPRC=0.148570 | best=0.148570@14 | patience=0/20
trial=01 | epoch=015 | loss=0.65473 | valAUPRC=0.152318 | best=0

## Step 30 — BWGNN × FDCompCN Final Unified Runs

Step 29 selected and froze the FDCompCN BWGNN configuration using validation
AUPRC only.

### Frozen configuration

- Hidden dimension: 64
- Wavelet order: 1
- Learning rate: 0.01
- Weight decay: 0
- Maximum epochs: 100
- Early stopping patience: 20

### Controlled final grid

Training ratios:

- TR40
- TR30
- TR20
- TR10

Training seeds:

- 2
- 42
- 72

Total final runs: **12**

### Per-run procedure

For each run:

1. train using only the selected training subset,
2. checkpoint using validation AUPRC,
3. reload the best validation checkpoint,
4. select classification threshold on validation Macro-F1,
5. evaluate the fixed test set once,
6. record predictive metrics,
7. record raw training-only time for every epoch,
8. record total training time,
9. record mean and standard deviation of epoch training time,
10. record validation time separately,
11. measure full-graph inference latency,
12. record peak GPU memory.

The test set is never used for training, checkpoint selection,
hyperparameter tuning or threshold selection.

In [13]:
# ============================================================
# STEP 30 — FDCOMPCN FINAL 12 UNIFIED RUNS
# 4 RATIOS × 3 SEEDS
# LIVE PROGRESS + COMPLETE TIMING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

DATA = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "fdcompcn_seed2_nested_splits.npz"
)

ADAPTER_DIR = WORK / "adapters"

RESULT_DIR = (
    WORK /
    "results/unified/fdcompcn/final"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PY.exists()
assert DATA.exists()
assert SPLIT.exists()
assert (ADAPTER_DIR / "BWGNN_gpu.py").exists()


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# FINAL RUNNER
# ============================================================

script = r'''
import sys
import gc
import csv
import json
import time
import random
import math

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import dgl

from dgl.data.utils import load_graphs

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


DATA = sys.argv[1]
SPLIT_PATH = sys.argv[2]
ADAPTER_DIR = sys.argv[3]
RESULT_DIR = Path(sys.argv[4])

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


# ============================================================
# FROZEN CONFIG FROM STEP 29
# ============================================================

HIDDEN = 64
ORDER = 1
LR = 0.01
WEIGHT_DECAY = 0.0

MAX_EPOCHS = 100
PATIENCE = 20

TRAINING_SEEDS = [
    2,
    42,
    72
]

RATIOS = [
    "TR40",
    "TR30",
    "TR20",
    "TR10"
]

DEVICE = torch.device(
    "cuda:0"
)


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

RUN_DIR = RESULT_DIR / "runs"
EPOCH_DIR = RESULT_DIR / "epoch_times"
CHECKPOINT_DIR = RESULT_DIR / "checkpoints"

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EPOCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# HELPERS
# ============================================================

def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def binary_metrics(y_true, probability, threshold):

    pred = (
        probability >= threshold
    ).astype(np.int64)

    auprc = float(
        average_precision_score(
            y_true,
            probability
        )
    )

    auroc = float(
        roc_auc_score(
            y_true,
            probability
        )
    )

    precision = float(
        precision_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    recall = float(
        recall_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    fraud_f1 = float(
        f1_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    macro_f1 = float(
        f1_score(
            y_true,
            pred,
            average="macro",
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    gmean = float(
        math.sqrt(
            sensitivity
            * specificity
        )
    )

    return {
        "auprc":
            auprc,

        "auroc":
            auroc,

        "fraud_precision":
            precision,

        "fraud_recall":
            recall,

        "fraud_f1":
            fraud_f1,

        "macro_f1":
            macro_f1,

        "gmean":
            gmean,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp)
    }


def choose_validation_threshold(
    y_true,
    probability
):

    candidates = []

    for threshold in np.arange(
        0.01,
        1.00,
        0.01
    ):

        pred = (
            probability >= threshold
        ).astype(np.int64)

        macro = float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0
            )
        )

        recall = float(
            recall_score(
                y_true,
                pred,
                pos_label=1,
                zero_division=0
            )
        )

        candidates.append(
            (
                macro,
                recall,
                -abs(
                    threshold - 0.5
                ),
                float(threshold)
            )
        )

    winner = max(
        candidates
    )

    return {
        "threshold":
            winner[3],

        "validation_macro_f1":
            winner[0],

        "validation_fraud_recall":
            winner[1]
    }


# ============================================================
# LOAD GRAPH ONCE
# ============================================================

print(
    "===== STEP 30 — FDCOMPCN FINAL 12 RUNS =====",
    flush=True
)

print(
    "Loading canonical FDCompCN...",
    flush=True
)

graphs, _ = load_graphs(
    DATA
)

graph = graphs[0]

assert graph.num_nodes() == 5317
assert graph.num_edges() == 30752


# Approved Step 28 adapter.

graph = dgl.to_homogeneous(
    graph,
    ndata=[
        "feature",
        "label"
    ]
)

graph = dgl.add_self_loop(
    graph
)

assert graph.num_nodes() == 5317
assert graph.num_edges() == 36069


graph = graph.to(
    DEVICE
)


features = (
    graph.ndata["feature"]
    .float()
)


raw_labels = graph.ndata[
    "label"
]


if raw_labels.ndim == 2:

    labels = (
        raw_labels
        .argmax(1)
        .long()
    )

else:

    labels = (
        raw_labels
        .reshape(-1)
        .long()
    )


splits = np.load(
    SPLIT_PATH
)


val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)

test_ids = torch.tensor(
    splits["test"],
    dtype=torch.long,
    device=DEVICE
)


val_y_cpu = (
    labels[val_ids]
    .cpu()
    .numpy()
)

test_y_cpu = (
    labels[test_ids]
    .cpu()
    .numpy()
)


print(
    "Graph:",
    graph.num_nodes(),
    "nodes |",
    graph.num_edges(),
    "edges",
    flush=True
)

print(
    "Frozen config:",
    f"hidden={HIDDEN}, "
    f"order={ORDER}, "
    f"lr={LR}, "
    f"wd={WEIGHT_DECAY}",
    flush=True
)

print(
    "Final grid: 4 ratios × 3 seeds = 12 runs",
    flush=True
)


# ============================================================
# FINAL GRID
# ============================================================

all_results = []

run_number = 0


for ratio in RATIOS:

    for seed in TRAINING_SEEDS:

        run_number += 1

        run_id = (
            f"fdcompcn_"
            f"{ratio.lower()}_"
            f"seed{seed}"
        )

        print(
            "\n"
            + "=" * 72,
            flush=True
        )

        print(
            f"START RUN {run_number:02d}/12 | "
            f"{ratio} | seed={seed}",
            flush=True
        )


        # ====================================================
        # REPRODUCIBILITY
        # ====================================================

        seed_everything(
            seed
        )

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


        train_ids = torch.tensor(
            splits[ratio],
            dtype=torch.long,
            device=DEVICE
        )


        train_y = labels[
            train_ids
        ]


        train_normal = int(
            (train_y == 0)
            .sum()
            .item()
        )

        train_fraud = int(
            (train_y == 1)
            .sum()
            .item()
        )


        class_weight = torch.tensor(
            [
                1.0,
                train_normal / train_fraud
            ],
            dtype=torch.float32,
            device=DEVICE
        )


        model = BWGNN(
            in_feats=57,
            h_feats=HIDDEN,
            num_classes=2,
            graph=graph,
            d=ORDER
        ).to(
            DEVICE
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            betas=(
                0.9,
                0.999
            ),
            eps=1e-8
        )


        checkpoint = (
            CHECKPOINT_DIR /
            f"{run_id}_best.pt"
        )


        # ====================================================
        # TRAINING
        # ====================================================

        best_val_auprc = -1.0
        best_epoch = None

        patience_counter = 0

        epoch_records = []

        validation_seconds_total = 0.0

        whole_run_start = (
            time.perf_counter()
        )


        for epoch in range(
            1,
            MAX_EPOCHS + 1
        ):

            # -----------------------------------------------
            # TRAINING-ONLY TIMER
            # -----------------------------------------------

            model.train()

            torch.cuda.synchronize()

            epoch_start = (
                time.perf_counter()
            )


            logits = model(
                features
            )


            loss = F.cross_entropy(
                logits[train_ids],
                labels[train_ids],
                weight=class_weight
            )


            optimizer.zero_grad()

            loss.backward()

            optimizer.step()


            torch.cuda.synchronize()


            epoch_train_seconds = (
                time.perf_counter()
                - epoch_start
            )


            # -----------------------------------------------
            # VALIDATION — SEPARATE TIMER
            # -----------------------------------------------

            model.eval()

            torch.cuda.synchronize()

            validation_start = (
                time.perf_counter()
            )


            with torch.no_grad():

                val_logits = model(
                    features
                )


            torch.cuda.synchronize()


            validation_seconds = (
                time.perf_counter()
                - validation_start
            )


            validation_seconds_total += (
                validation_seconds
            )


            val_prob = (
                torch.softmax(
                    val_logits,
                    dim=1
                )[val_ids, 1]
                .detach()
                .cpu()
                .numpy()
            )


            val_auprc = float(
                average_precision_score(
                    val_y_cpu,
                    val_prob
                )
            )


            improved = (
                val_auprc
                > best_val_auprc
                + 1e-12
            )


            if improved:

                best_val_auprc = (
                    val_auprc
                )

                best_epoch = epoch

                patience_counter = 0


                torch.save(
                    {
                        "epoch":
                            epoch,

                        "ratio":
                            ratio,

                        "seed":
                            seed,

                        "hidden":
                            HIDDEN,

                        "order":
                            ORDER,

                        "lr":
                            LR,

                        "weight_decay":
                            WEIGHT_DECAY,

                        "val_auprc":
                            val_auprc,

                        "model_state_dict":
                            model.state_dict()
                    },
                    checkpoint
                )


            else:

                patience_counter += 1


            epoch_records.append(
                {
                    "run_id":
                        run_id,

                    "ratio":
                        ratio,

                    "seed":
                        seed,

                    "epoch":
                        epoch,

                    "training_loss":
                        float(
                            loss.item()
                        ),

                    "train_seconds":
                        float(
                            epoch_train_seconds
                        ),

                    "validation_seconds":
                        float(
                            validation_seconds
                        ),

                    "val_auprc":
                        val_auprc,

                    "best_val_auprc":
                        best_val_auprc,

                    "patience_counter":
                        patience_counter
                }
            )


            if (
                epoch == 1
                or epoch % 10 == 0
                or improved
                or patience_counter >= PATIENCE
            ):

                print(
                    f"run={run_number:02d}/12 | "
                    f"{ratio} seed={seed} | "
                    f"epoch={epoch:03d} | "
                    f"loss={loss.item():.5f} | "
                    f"valAUPRC={val_auprc:.6f} | "
                    f"best={best_val_auprc:.6f}"
                    f"@{best_epoch} | "
                    f"patience={patience_counter}/20",
                    flush=True
                )


            if patience_counter >= PATIENCE:

                print(
                    f"EARLY STOP | "
                    f"{ratio} seed={seed} | "
                    f"epoch={epoch}",
                    flush=True
                )

                break


        # ====================================================
        # RAW EPOCH TIMING CSV
        # ====================================================

        epoch_csv = (
            EPOCH_DIR /
            f"{run_id}_epoch_times.csv"
        )


        with epoch_csv.open(
            "w",
            newline=""
        ) as handle:

            writer = csv.DictWriter(
                handle,
                fieldnames=list(
                    epoch_records[0].keys()
                )
            )

            writer.writeheader()

            writer.writerows(
                epoch_records
            )


        # ====================================================
        # RESTORE BEST VALIDATION CHECKPOINT
        # ====================================================

        saved = torch.load(
            checkpoint,
            map_location=DEVICE
        )

        model.load_state_dict(
            saved[
                "model_state_dict"
            ]
        )

        model.eval()


        # ====================================================
        # VALIDATION SCORES FOR THRESHOLD
        # ====================================================

        with torch.no_grad():

            val_logits = model(
                features
            )

            val_prob = (
                torch.softmax(
                    val_logits,
                    dim=1
                )[val_ids, 1]
                .detach()
                .cpu()
                .numpy()
            )


        threshold_info = (
            choose_validation_threshold(
                val_y_cpu,
                val_prob
            )
        )


        threshold = (
            threshold_info[
                "threshold"
            ]
        )


        # ====================================================
        # TEST — ONCE AFTER ALL CHOICES FROZEN
        # ====================================================

        with torch.no_grad():

            test_logits = model(
                features
            )

            test_prob = (
                torch.softmax(
                    test_logits,
                    dim=1
                )[test_ids, 1]
                .detach()
                .cpu()
                .numpy()
            )


        test_metrics = (
            binary_metrics(
                test_y_cpu,
                test_prob,
                threshold
            )
        )


        # ====================================================
        # FULL-GRAPH INFERENCE LATENCY
        # ====================================================

        model.eval()


        with torch.no_grad():

            # 3 warmup forwards
            for _ in range(3):

                _ = model(
                    features
                )


            latency_ms = []


            for _ in range(10):

                torch.cuda.synchronize()

                latency_start = (
                    time.perf_counter()
                )


                _ = model(
                    features
                )


                torch.cuda.synchronize()


                latency_ms.append(
                    (
                        time.perf_counter()
                        - latency_start
                    )
                    * 1000.0
                )


        inference_mean_ms = float(
            np.mean(
                latency_ms
            )
        )

        inference_std_ms = float(
            np.std(
                latency_ms,
                ddof=1
            )
        )


        # ====================================================
        # TIMING SUMMARIES
        # ====================================================

        epoch_train_array = np.array(
            [
                x["train_seconds"]
                for x in epoch_records
            ],
            dtype=float
        )


        total_train_seconds = float(
            epoch_train_array.sum()
        )

        mean_epoch_train_seconds = float(
            epoch_train_array.mean()
        )

        std_epoch_train_seconds = float(
            epoch_train_array.std(
                ddof=1
            )
            if len(epoch_train_array) > 1
            else 0.0
        )


        whole_run_seconds = float(
            time.perf_counter()
            - whole_run_start
        )


        peak_gpu_memory_mb = float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )


        # ====================================================
        # SAVE RUN JSON
        # ====================================================

        record = {

            "run_id":
                run_id,

            "dataset":
                "FDCompCN",

            "model":
                "BWGNN",

            "mode":
                "unified",

            "compatibility":
                "INPUT ADAPTER ELIGIBLE",

            "ratio":
                ratio,

            "training_seed":
                seed,

            "split_seed":
                2,

            "hidden":
                HIDDEN,

            "order":
                ORDER,

            "lr":
                LR,

            "weight_decay":
                WEIGHT_DECAY,

            "completed_epochs":
                len(epoch_records),

            "best_epoch":
                best_epoch,

            "best_val_auprc":
                best_val_auprc,

            "validation_threshold":
                threshold,

            "validation_threshold_macro_f1":
                threshold_info[
                    "validation_macro_f1"
                ],

            "validation_threshold_fraud_recall":
                threshold_info[
                    "validation_fraud_recall"
                ],

            "test_auprc":
                test_metrics["auprc"],

            "test_auroc":
                test_metrics["auroc"],

            "test_fraud_precision":
                test_metrics[
                    "fraud_precision"
                ],

            "test_fraud_recall":
                test_metrics[
                    "fraud_recall"
                ],

            "test_fraud_f1":
                test_metrics[
                    "fraud_f1"
                ],

            "test_macro_f1":
                test_metrics[
                    "macro_f1"
                ],

            "test_gmean":
                test_metrics[
                    "gmean"
                ],

            "tn":
                test_metrics["tn"],

            "fp":
                test_metrics["fp"],

            "fn":
                test_metrics["fn"],

            "tp":
                test_metrics["tp"],

            "total_train_seconds":
                total_train_seconds,

            "mean_epoch_train_seconds":
                mean_epoch_train_seconds,

            "std_epoch_train_seconds":
                std_epoch_train_seconds,

            "validation_seconds_total":
                validation_seconds_total,

            "whole_run_seconds":
                whole_run_seconds,

            "inference_latency_mean_ms":
                inference_mean_ms,

            "inference_latency_std_ms":
                inference_std_ms,

            "inference_warmups":
                3,

            "inference_repetitions":
                10,

            "peak_gpu_memory_mb":
                peak_gpu_memory_mb,

            "test_used_for_selection":
                False
        }


        (
            RUN_DIR /
            f"{run_id}.json"
        ).write_text(
            json.dumps(
                record,
                indent=2
            )
        )


        all_results.append(
            record
        )


        print(
            f"DONE RUN {run_number:02d}/12 | "
            f"{ratio} seed={seed} | "
            f"AUPRC={record['test_auprc']:.6f} | "
            f"AUROC={record['test_auroc']:.6f} | "
            f"MacroF1={record['test_macro_f1']:.6f} | "
            f"GMean={record['test_gmean']:.6f} | "
            f"epoch={best_epoch} | "
            f"train={total_train_seconds:.3f}s | "
            f"latency={inference_mean_ms:.3f}ms",
            flush=True
        )


        del model
        del optimizer

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# SAVE ALL-RUN CSV
# ============================================================

summary_csv = (
    RESULT_DIR /
    "fdcompcn_final_all_runs.csv"
)


with summary_csv.open(
    "w",
    newline=""
) as handle:

    writer = csv.DictWriter(
        handle,
        fieldnames=list(
            all_results[0].keys()
        )
    )

    writer.writeheader()

    writer.writerows(
        all_results
    )


(
    RESULT_DIR /
    "fdcompcn_final_all_runs.json"
).write_text(
    json.dumps(
        all_results,
        indent=2
    )
)


# ============================================================
# AGGREGATE BY TRAINING RATIO
# ============================================================

summary = {}


for ratio in RATIOS:

    rows = [
        r
        for r in all_results
        if r["ratio"] == ratio
    ]


    fields = [
        "test_auprc",
        "test_auroc",
        "test_fraud_precision",
        "test_fraud_recall",
        "test_fraud_f1",
        "test_macro_f1",
        "test_gmean",
        "mean_epoch_train_seconds",
        "total_train_seconds",
        "inference_latency_mean_ms",
        "peak_gpu_memory_mb"
    ]


    summary[ratio] = {}


    for field in fields:

        values = np.array(
            [
                row[field]
                for row in rows
            ],
            dtype=float
        )


        summary[ratio][field] = {
            "mean":
                float(
                    values.mean()
                ),

            "std":
                float(
                    values.std(
                        ddof=1
                    )
                )
        }


(
    RESULT_DIR /
    "fdcompcn_final_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# PRINT SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 72,
    flush=True
)

print(
    "===== STEP 30 FINAL SUMMARY =====",
    flush=True
)


for ratio in RATIOS:

    x = summary[
        ratio
    ]

    print(
        f"\n{ratio}",
        flush=True
    )

    print(
        "AUPRC:",
        f"{x['test_auprc']['mean']:.6f} "
        f"± {x['test_auprc']['std']:.6f}",
        flush=True
    )

    print(
        "AUROC:",
        f"{x['test_auroc']['mean']:.6f} "
        f"± {x['test_auroc']['std']:.6f}",
        flush=True
    )

    print(
        "Macro-F1:",
        f"{x['test_macro_f1']['mean']:.6f} "
        f"± {x['test_macro_f1']['std']:.6f}",
        flush=True
    )

    print(
        "Fraud Recall:",
        f"{x['test_fraud_recall']['mean']:.6f} "
        f"± {x['test_fraud_recall']['std']:.6f}",
        flush=True
    )

    print(
        "G-Mean:",
        f"{x['test_gmean']['mean']:.6f} "
        f"± {x['test_gmean']['std']:.6f}",
        flush=True
    )

    print(
        "Mean epoch training:",
        f"{x['mean_epoch_train_seconds']['mean']:.6f}s "
        f"± {x['mean_epoch_train_seconds']['std']:.6f}",
        flush=True
    )

    print(
        "Total training:",
        f"{x['total_train_seconds']['mean']:.6f}s "
        f"± {x['total_train_seconds']['std']:.6f}",
        flush=True
    )

    print(
        "Inference latency:",
        f"{x['inference_latency_mean_ms']['mean']:.6f}ms "
        f"± {x['inference_latency_mean_ms']['std']:.6f}",
        flush=True
    )

    print(
        "Peak GPU memory:",
        f"{x['peak_gpu_memory_mb']['mean']:.2f}MB "
        f"± {x['peak_gpu_memory_mb']['std']:.2f}",
        flush=True
    )


# ============================================================
# FINAL GATES
# ============================================================

assert len(
    all_results
) == 12


assert len(
    list(
        RUN_DIR.glob(
            "*.json"
        )
    )
) == 12


assert len(
    list(
        EPOCH_DIR.glob(
            "*_epoch_times.csv"
        )
    )
) == 12


print(
    "\n===== STEP 30 GATE =====",
    flush=True
)

print(
    "PASS — FDCompCN final unified benchmark completed.",
    flush=True
)

print(
    "Final runs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)

print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Training timing excludes validation: YES",
    flush=True
)

print(
    "Validation threshold selection only: YES",
    flush=True
)

print(
    "Test used for tuning/checkpoint/threshold selection: NO",
    flush=True
)

print(
    "Inference latency recorded: YES",
    flush=True
)

print(
    "Peak GPU memory recorded: YES",
    flush=True
)
'''


# ============================================================
# STREAM LIVE
# ============================================================

p = subprocess.Popen(
    [
        str(PY),
        "-c",
        script,
        str(DATA),
        str(SPLIT),
        str(ADAPTER_DIR),
        str(RESULT_DIR)
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in p.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = p.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 30 failed with return code {rc}"
    )

===== STEP 30 — FDCOMPCN FINAL 12 RUNS =====
Loading canonical FDCompCN...
Graph: 5317 nodes | 36069 edges
Frozen config: hidden=64, order=1, lr=0.01, wd=0.0
Final grid: 4 ratios × 3 seeds = 12 runs

START RUN 01/12 | TR40 | seed=2
run=01/12 | TR40 seed=2 | epoch=001 | loss=0.69953 | valAUPRC=0.125051 | best=0.125051@1 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=004 | loss=0.68557 | valAUPRC=0.128273 | best=0.128273@4 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=005 | loss=0.68085 | valAUPRC=0.131677 | best=0.131677@5 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=007 | loss=0.67436 | valAUPRC=0.132451 | best=0.132451@7 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=008 | loss=0.67541 | valAUPRC=0.134422 | best=0.134422@8 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=009 | loss=0.67035 | valAUPRC=0.145929 | best=0.145929@9 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=010 | loss=0.66909 | valAUPRC=0.143606 | best=0.145929@9 | patience=1/20
run=01/12 | TR40 seed=2 | epoch=0

## Step 31 — Package BWGNN × FDCompCN Evidence

This package preserves the complete FDCompCN BWGNN benchmark evidence.

Included:

- compatibility decision,
- frozen BWGNN commit information,
- GPU adapter,
- exact controlled split,
- Step 29 tuning trial summary,
- frozen winning configuration,
- 12 final run JSON records,
- 12 raw per-epoch timing CSVs,
- final aggregate results,
- manifest.

The model checkpoints are intentionally excluded from the evidence package.

### Final status

- Compatibility: Input Adapter Eligible
- Tuning: 12/12 trials complete
- Final runs: 12/12 complete
- Raw epoch timing: 12/12
- Inference latency: recorded
- Peak GPU memory: recorded
- Test-set tuning: none

In [14]:
# ============================================================
# STEP 31 — PACKAGE COMPLETE FDCOMPCN EVIDENCE
# ============================================================

from pathlib import Path
import json
import zipfile
import hashlib
import datetime

WORK = Path("/kaggle/working/comp8851_bwgnn")

REPO = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "fdcompcn_seed2_nested_splits.npz"
)

ADAPTER = (
    WORK /
    "adapters/BWGNN_gpu.py"
)

TUNING_DIR = (
    WORK /
    "results/unified/fdcompcn/tuning"
)

FINAL_DIR = (
    WORK /
    "results/unified/fdcompcn/final"
)

RUN_DIR = FINAL_DIR / "runs"
EPOCH_DIR = FINAL_DIR / "epoch_times"

EVIDENCE_DIR = (
    WORK /
    "results/evidence/fdcompcn"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "bwgnn_fdcompcn_complete_evidence_20260910.zip"
)


print(
    "===== STEP 31 — FDCOMPCN EVIDENCE PACKAGING =====",
    flush=True
)


# ============================================================
# 1. VERIFY REQUIRED FILES
# ============================================================

required = [

    SPLIT,

    ADAPTER,

    TUNING_DIR /
    "fdcompcn_tuning_trials.csv",

    TUNING_DIR /
    "fdcompcn_tuning_trials.json",

    TUNING_DIR /
    "fdcompcn_frozen_config.json",

    FINAL_DIR /
    "fdcompcn_final_all_runs.csv",

    FINAL_DIR /
    "fdcompcn_final_all_runs.json",

    FINAL_DIR /
    "fdcompcn_final_summary.json"
]


for path in required:

    assert path.exists(), (
        f"Missing required evidence: {path}"
    )


run_jsons = sorted(
    RUN_DIR.glob("*.json")
)

epoch_csvs = sorted(
    EPOCH_DIR.glob(
        "*_epoch_times.csv"
    )
)


assert len(run_jsons) == 12, (
    f"Expected 12 run JSONs, found {len(run_jsons)}"
)

assert len(epoch_csvs) == 12, (
    f"Expected 12 epoch CSVs, found {len(epoch_csvs)}"
)


print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)


# ============================================================
# 2. FROZEN REPOSITORY COMMIT
# ============================================================

import subprocess

commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


assert (
    commit
    == "de0631f039bbd19c1890b483cc01f1007f596af7"
)


print(
    "Frozen BWGNN commit:",
    commit,
    flush=True
)


# ============================================================
# 3. COMPATIBILITY DECISION RECORD
# ============================================================

COMPAT = (
    EVIDENCE_DIR /
    "fdcompcn_compatibility_decision.json"
)

compatibility = {

    "dataset":
        "FDCompCN",

    "model":
        "BWGNN",

    "native_repository_support":
        False,

    "classification":
        "INPUT ADAPTER ELIGIBLE",

    "adapter":
        (
            "DGL heterogeneous-to-homogeneous representation "
            "plus self-loops, matching existing BWGNN homogeneous "
            "execution semantics"
        ),

    "canonical_nodes":
        5317,

    "canonical_edges":
        30752,

    "adapted_edges_with_self_loops":
        36069,

    "features":
        57,

    "normal_nodes":
        4758,

    "fraud_nodes":
        559,

    "original_relations": [
        "homo",
        "invest_bc2bc",
        "provide_bc2bc",
        "sale_bc2bc"
    ],

    "nodes_preserved":
        True,

    "original_edges_preserved":
        True,

    "features_preserved":
        True,

    "labels_preserved":
        True,

    "architecture_changed":
        False,

    "smoke_test":
        "PASS",

    "test_accessed_during_compatibility":
        False
}


COMPAT.write_text(
    json.dumps(
        compatibility,
        indent=2
    )
)


# ============================================================
# 4. ENVIRONMENT RECORD
# ============================================================

ENV_RECORD = (
    EVIDENCE_DIR /
    "fdcompcn_environment.json"
)

environment = {

    "repository":
        "https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git",

    "commit":
        commit,

    "python":
        "3.9.25",

    "torch":
        "1.9.0+cu111",

    "dgl":
        "0.8.1",

    "numpy":
        "1.23.5",

    "gpu":
        "Tesla T4",

    "controlled_gpu":
        0,

    "cuda_visible_devices":
        "0",

    "adapter":
        "BWGNN_gpu.py",

    "gpu_patch":
        "4 torch.zeros allocations made device-aware"
}


ENV_RECORD.write_text(
    json.dumps(
        environment,
        indent=2
    )
)


# ============================================================
# 5. READ FROZEN CONFIG + FINAL SUMMARY
# ============================================================

frozen = json.loads(
    (
        TUNING_DIR /
        "fdcompcn_frozen_config.json"
    ).read_text()
)

summary = json.loads(
    (
        FINAL_DIR /
        "fdcompcn_final_summary.json"
    ).read_text()
)


winner = frozen["winner"]


assert winner["trial"] == 3
assert winner["hidden"] == 64
assert winner["order"] == 1
assert winner["lr"] == 0.01
assert winner["weight_decay"] == 0.0


# ============================================================
# 6. MANIFEST
# ============================================================

MANIFEST = (
    EVIDENCE_DIR /
    "MANIFEST.json"
)

manifest = {

    "created_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),

    "dataset":
        "FDCompCN",

    "model":
        "BWGNN",

    "benchmark_mode":
        "unified controlled",

    "compatibility":
        "INPUT ADAPTER ELIGIBLE",

    "repository_commit":
        commit,

    "split_seed":
        2,

    "training_seeds":
        [
            2,
            42,
            72
        ],

    "training_ratios":
        [
            "TR40",
            "TR30",
            "TR20",
            "TR10"
        ],

    "tuning_trials":
        12,

    "frozen_configuration": {

        "hidden":
            winner["hidden"],

        "order":
            winner["order"],

        "learning_rate":
            winner["lr"],

        "weight_decay":
            winner["weight_decay"],

        "best_tuning_epoch":
            winner["best_epoch"],

        "best_validation_auprc":
            winner["best_val_auprc"]
    },

    "final_runs":
        12,

    "raw_epoch_timing_files":
        12,

    "training_timing_excludes_validation":
        True,

    "inference_latency_recorded":
        True,

    "peak_gpu_memory_recorded":
        True,

    "threshold_selected_on_validation":
        True,

    "test_used_for_tuning":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_selection":
        False,

    "checkpoints_included":
        False,

    "results_summary":
        summary
}


MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


# ============================================================
# 7. CREATE ZIP
# ============================================================

print(
    "\nCreating ZIP...",
    flush=True
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # --------------------------------------------------------
    # Manifest / compatibility / environment
    # --------------------------------------------------------

    z.write(
        MANIFEST,
        "manifest/MANIFEST.json"
    )

    z.write(
        COMPAT,
        "compatibility/fdcompcn_compatibility_decision.json"
    )

    z.write(
        ENV_RECORD,
        "environment/fdcompcn_environment.json"
    )


    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    z.write(
        SPLIT,
        "splits/fdcompcn_seed2_nested_splits.npz"
    )


    # --------------------------------------------------------
    # Adapter
    # --------------------------------------------------------

    z.write(
        ADAPTER,
        "adapters/BWGNN_gpu.py"
    )


    # --------------------------------------------------------
    # Tuning
    # --------------------------------------------------------

    for name in [
        "fdcompcn_tuning_trials.csv",
        "fdcompcn_tuning_trials.json",
        "fdcompcn_frozen_config.json"
    ]:

        path = TUNING_DIR / name

        z.write(
            path,
            f"tuning/{name}"
        )


    # --------------------------------------------------------
    # Final aggregate results
    # --------------------------------------------------------

    for name in [
        "fdcompcn_final_all_runs.csv",
        "fdcompcn_final_all_runs.json",
        "fdcompcn_final_summary.json"
    ]:

        path = FINAL_DIR / name

        z.write(
            path,
            f"final/{name}"
        )


    # --------------------------------------------------------
    # Individual run records
    # --------------------------------------------------------

    for path in run_jsons:

        z.write(
            path,
            f"final/runs/{path.name}"
        )


    # --------------------------------------------------------
    # Raw per-epoch timing
    # --------------------------------------------------------

    for path in epoch_csvs:

        z.write(
            path,
            f"final/epoch_times/{path.name}"
        )


# ============================================================
# 8. ZIP SHA256
# ============================================================

h = hashlib.sha256()

with ZIP_PATH.open("rb") as f:

    for block in iter(
        lambda: f.read(
            8 * 1024 * 1024
        ),
        b""
    ):

        h.update(block)


zip_sha = h.hexdigest()


# ============================================================
# 9. VERIFY ZIP CONTENT
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    names = z.namelist()


assert (
    "manifest/MANIFEST.json"
    in names
)

assert (
    "tuning/fdcompcn_frozen_config.json"
    in names
)

assert sum(
    x.startswith(
        "final/runs/"
    )
    for x in names
) == 12

assert sum(
    x.startswith(
        "final/epoch_times/"
    )
    for x in names
) == 12


print(
    "\n===== STEP 31 GATE =====",
    flush=True
)

print(
    "PASS — complete FDCompCN evidence packaged.",
    flush=True
)

print(
    "Compatibility: INPUT ADAPTER ELIGIBLE",
    flush=True
)

print(
    "Tuning trials: 12/12",
    flush=True
)

print(
    "Final runs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)

print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Controlled split included: YES",
    flush=True
)

print(
    "Frozen config included: YES",
    flush=True
)

print(
    "BWGNN adapter included: YES",
    flush=True
)

print(
    "Checkpoints included: NO",
    flush=True
)

print(
    "Test-set tuning: NO",
    flush=True
)

print(
    "\nZIP:",
    ZIP_PATH,
    flush=True
)

print(
    "ZIP SHA256:",
    zip_sha,
    flush=True
)

print(
    "ZIP size:",
    f"{ZIP_PATH.stat().st_size/(1024**2):.2f} MB",
    flush=True
)

===== STEP 31 — FDCOMPCN EVIDENCE PACKAGING =====
Run JSONs: 12/12
Raw epoch timing CSVs: 12/12
Frozen BWGNN commit: de0631f039bbd19c1890b483cc01f1007f596af7

Creating ZIP...

===== STEP 31 GATE =====
PASS — complete FDCompCN evidence packaged.
Compatibility: INPUT ADAPTER ELIGIBLE
Tuning trials: 12/12
Final runs: 12/12
Raw epoch timing CSVs: 12/12
Run JSONs: 12/12
Controlled split included: YES
Frozen config included: YES
BWGNN adapter included: YES
Checkpoints included: NO
Test-set tuning: NO

ZIP: /kaggle/working/bwgnn_fdcompcn_complete_evidence_20260910.zip
ZIP SHA256: 82bbf91fec7e4166ef59fb0b065da1cb99432ef80cb5533394f26ae67a2126ac
ZIP size: 0.06 MB


## Step 32A — BWGNN × Elliptic Compatibility Audit

Elliptic is not a native dataset in the official BWGNN repository.

This step checks whether Elliptic can be represented for BWGNN using only an
architecture-preserving input adapter.

The audit verifies:

- canonical file hashes,
- transaction/node alignment,
- feature structure,
- known / unknown label counts,
- timestep range,
- directed edge preservation,
- temporal edge direction,
- whether all unknown-labelled nodes can remain in the graph while only
  known-labelled nodes are used for supervised loss/evaluation.

No training, tuning or test evaluation is performed.

Chronology is preserved; no random split is created in this step.

In [15]:
# ============================================================
# STEP 32A — BWGNN × ELLIPTIC COMPATIBILITY AUDIT
# NO TRAINING
# ============================================================

import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

WORK = Path("/kaggle/working/comp8851_bwgnn")
REPO = Path("/kaggle/working/Rethinking-Anomaly-Detection")

FEATURES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_features.csv"
)

CLASSES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_classes.csv"
)

EDGES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_edgelist.csv"
)

EXPECTED_HASHES = {
    FEATURES:
        "fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0",

    CLASSES:
        "93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493",

    EDGES:
        "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e"
}

print(
    "===== STEP 32A — BWGNN × ELLIPTIC COMPATIBILITY =====",
    flush=True
)


# ============================================================
# 1. FROZEN BWGNN SOURCE
# ============================================================

dataset_py = REPO / "dataset.py"

assert dataset_py.exists()

source = dataset_py.read_text(
    errors="replace"
).lower()

native = (
    "elliptic" in source
)

print(
    "\n[1/6] Frozen BWGNN source",
    flush=True
)

print(
    "Native Elliptic path:",
    native,
    flush=True
)


# ============================================================
# 2. CANONICAL HASHES
# ============================================================

print(
    "\n[2/6] Verifying canonical files...",
    flush=True
)

for path, expected in EXPECTED_HASHES.items():

    assert path.exists(), path

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                8 * 1024 * 1024
            ),
            b""
        ):
            h.update(block)

    actual = h.hexdigest()

    print(
        path.name,
        actual,
        flush=True
    )

    assert actual == expected


print(
    "Canonical hashes: PASS",
    flush=True
)


# ============================================================
# 3. LOAD RAW ELLIPTIC TABLES
# ============================================================

print(
    "\n[3/6] Loading Elliptic tables...",
    flush=True
)


# Feature file has no header.
features = pd.read_csv(
    FEATURES,
    header=None
)

classes = pd.read_csv(
    CLASSES
)

edges = pd.read_csv(
    EDGES
)


print(
    "Feature table shape:",
    features.shape,
    flush=True
)

print(
    "Classes shape:",
    classes.shape,
    flush=True
)

print(
    "Edges shape:",
    edges.shape,
    flush=True
)


assert features.shape == (
    203769,
    167
)


# ------------------------------------------------------------
# Canonical node IDs + timesteps
# ------------------------------------------------------------

tx_ids = (
    features.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

timesteps = (
    features.iloc[:, 1]
    .astype(np.int64)
    .to_numpy()
)


assert len(
    np.unique(tx_ids)
) == 203769


print(
    "Unique transaction IDs:",
    len(
        np.unique(tx_ids)
    ),
    flush=True
)

print(
    "Timestep range:",
    int(timesteps.min()),
    "to",
    int(timesteps.max()),
    flush=True
)


assert int(
    timesteps.min()
) == 1

assert int(
    timesteps.max()
) == 49


print(
    "[3/6] PASS",
    flush=True
)


# ============================================================
# 4. LABEL ALIGNMENT
# ============================================================

print(
    "\n[4/6] Verifying labels...",
    flush=True
)


class_id_col = classes.columns[0]
class_col = classes.columns[1]


class_ids = (
    classes[
        class_id_col
    ]
    .astype(np.int64)
    .to_numpy()
)


assert len(class_ids) == 203769

assert set(
    class_ids
) == set(
    tx_ids
)


raw_class = (
    classes[
        class_col
    ]
    .astype(str)
    .str.strip()
)


unknown = int(
    (
        raw_class.str.lower()
        == "unknown"
    ).sum()
)

fraud = int(
    (
        raw_class
        == "1"
    ).sum()
)

normal = int(
    (
        raw_class
        == "2"
    ).sum()
)

known = (
    fraud
    + normal
)


print(
    "Known labelled:",
    known,
    flush=True
)

print(
    "Unknown:",
    unknown,
    flush=True
)

print(
    "Fraud / illicit:",
    fraud,
    flush=True
)

print(
    "Normal / licit:",
    normal,
    flush=True
)


assert known == 46564
assert unknown == 157205
assert fraud == 4545
assert normal == 42019


print(
    "[4/6] PASS",
    flush=True
)


# ============================================================
# 5. EDGE / NODE / TEMPORAL ALIGNMENT
# ============================================================

print(
    "\n[5/6] Auditing directed graph structure...",
    flush=True
)


src_col = edges.columns[0]
dst_col = edges.columns[1]


src = (
    edges[
        src_col
    ]
    .astype(np.int64)
    .to_numpy()
)

dst = (
    edges[
        dst_col
    ]
    .astype(np.int64)
    .to_numpy()
)


node_set = set(
    tx_ids.tolist()
)


missing_src = int(
    sum(
        x not in node_set
        for x in src
    )
)

missing_dst = int(
    sum(
        x not in node_set
        for x in dst
    )
)


print(
    "Stored directed edges:",
    len(edges),
    flush=True
)

print(
    "Missing source IDs:",
    missing_src,
    flush=True
)

print(
    "Missing destination IDs:",
    missing_dst,
    flush=True
)


assert missing_src == 0
assert missing_dst == 0


# ------------------------------------------------------------
# Map transaction ID -> timestep.
# Check temporal orientation without changing graph.
# ------------------------------------------------------------

time_map = dict(
    zip(
        tx_ids.tolist(),
        timesteps.tolist()
    )
)


src_time = np.fromiter(
    (
        time_map[x]
        for x in src
    ),
    dtype=np.int16,
    count=len(src)
)

dst_time = np.fromiter(
    (
        time_map[x]
        for x in dst
    ),
    dtype=np.int16,
    count=len(dst)
)


backward_edges = int(
    (
        dst_time
        < src_time
    ).sum()
)

same_time_edges = int(
    (
        dst_time
        == src_time
    ).sum()
)

forward_edges = int(
    (
        dst_time
        > src_time
    ).sum()
)


print(
    "Same-timestep edges:",
    same_time_edges,
    flush=True
)

print(
    "Forward-in-time edges:",
    forward_edges,
    flush=True
)

print(
    "Backward-in-time edges:",
    backward_edges,
    flush=True
)


print(
    "[5/6] PASS",
    flush=True
)


# ============================================================
# 6. BWGNN INPUT-ADAPTER FEASIBILITY
# ============================================================

print(
    "\n[6/6] BWGNN input-adapter feasibility",
    flush=True
)


# Elliptic is already a one-node-type homogeneous transaction graph.
# Adapter requirements:
#
# 1. map tx IDs -> contiguous DGL node indices
# 2. retain all directed edges exactly once
# 3. retain all 203,769 nodes
# 4. retain feature rows
# 5. assign -1 to unknown labels
# 6. train/evaluate only known labelled IDs
# 7. preserve chronological split boundaries


print(
    "Graph type: homogeneous transaction graph",
    flush=True
)

print(
    "Relation conversion required: NO",
    flush=True
)

print(
    "Node-ID remapping required: YES",
    flush=True
)

print(
    "Edge direction can be preserved: YES",
    flush=True
)

print(
    "Unknown nodes can remain for message passing: YES",
    flush=True
)

print(
    "Unknown nodes excluded from supervised loss/evaluation: YES",
    flush=True
)

print(
    "Chronological split can be preserved: YES",
    flush=True
)


if native:

    decision = "NATIVE"

else:

    decision = "INPUT ADAPTER ELIGIBLE"


print(
    "\n===== STEP 32A GATE =====",
    flush=True
)

print(
    "PASS — BWGNN × Elliptic compatibility audit completed.",
    flush=True
)

print(
    "Native BWGNN Elliptic support:",
    "YES" if native else "NO",
    flush=True
)

print(
    "All canonical nodes can be preserved: YES",
    flush=True
)

print(
    "All directed edges can be preserved: YES",
    flush=True
)

print(
    "Chronology can be preserved: YES",
    flush=True
)

print(
    "Known/unknown label isolation possible: YES",
    flush=True
)

print(
    "Compatibility decision:",
    decision,
    flush=True
)

print(
    "Training performed: NO",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

===== STEP 32A — BWGNN × ELLIPTIC COMPATIBILITY =====

[1/6] Frozen BWGNN source
Native Elliptic path: False

[2/6] Verifying canonical files...
elliptic_txs_features.csv fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0
elliptic_txs_classes.csv 93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493
elliptic_txs_edgelist.csv a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e
Canonical hashes: PASS

[3/6] Loading Elliptic tables...
Feature table shape: (203769, 167)
Classes shape: (203769, 2)
Edges shape: (234355, 2)
Unique transaction IDs: 203769
Timestep range: 1 to 49
[3/6] PASS

[4/6] Verifying labels...
Known labelled: 46564
Unknown: 157205
Fraud / illicit: 4545
Normal / licit: 42019
[4/6] PASS

[5/6] Auditing directed graph structure...
Stored directed edges: 234355
Missing source IDs: 0
Missing destination IDs: 0
Same-timestep edges: 234355
Forward-in-time edges: 0
Backward-in-time edges: 0
[5/6] PASS

[6/6] BWGNN input-adapter feasibility
G

## Step 32B — Elliptic Chronological Split and BWGNN Smoke Test

Step 32A classified BWGNN × Elliptic as **INPUT ADAPTER ELIGIBLE**.

### Elliptic modeling convention

The model input uses:

- 165 numeric transaction features,
- excluding `txId`,
- excluding `time_step`.

`time_step` is retained only for chronological split construction.

Unknown-labelled transactions remain in the graph for message passing but are
never used in supervised loss, checkpoint selection or evaluation.

### Controlled chronological split

Whole timesteps are used so no timestep is divided between experimental
partitions.

- TR10: labelled nodes at time steps 1–3
- TR20: labelled nodes at time steps 1–7
- TR30: labelled nodes at time steps 1–12
- TR40: labelled nodes at time steps 1–20
- Validation: labelled nodes at time steps 21–31
- Test: labelled nodes at time steps 32–49

This gives nested training availability while keeping validation and test
strictly later in time.

### Smoke configuration

- Training subset: TR40
- Training seed: 2
- Hidden dimension: 64
- Wavelet order: 2
- Learning rate: 0.01
- Epochs: 1
- GPU: Tesla T4, GPU 0

No reverse edges are introduced.

Original directed payment-flow edges are retained, and self-loops are added to
match BWGNN's existing homogeneous execution semantics.

The test set is not evaluated during the smoke test.

In [16]:
# ============================================================
# STEP 32B — ELLIPTIC CHRONOLOGICAL SPLIT + BWGNN SMOKE
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

ADAPTER_DIR = WORK / "adapters"

FEATURES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_features.csv"
)

CLASSES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_classes.csv"
)

EDGES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_edgelist.csv"
)

SPLIT_DIR = WORK / "shared/splits"
SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_PATH = (
    SPLIT_DIR /
    "elliptic_chronological_nested_splits.npz"
)

assert PY.exists()
assert FEATURES.exists()
assert CLASSES.exists()
assert EDGES.exists()
assert (ADAPTER_DIR / "BWGNN_gpu.py").exists()


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# SMOKE SCRIPT
# ============================================================

script = r'''
import sys
import time
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

FEATURES = sys.argv[1]
CLASSES = sys.argv[2]
EDGES = sys.argv[3]
SPLIT_PATH = sys.argv[4]
ADAPTER_DIR = sys.argv[5]

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


DEVICE = torch.device(
    "cuda:0"
)

SEED = 2

HIDDEN = 64
ORDER = 2
LR = 0.01


print(
    "===== STEP 32B — ELLIPTIC BWGNN SMOKE =====",
    flush=True
)


# ============================================================
# 1. LOAD TABLES
# ============================================================

print(
    "\n[1/6] Loading Elliptic tables...",
    flush=True
)

feat_df = pd.read_csv(
    FEATURES,
    header=None
)

class_df = pd.read_csv(
    CLASSES
)

edge_df = pd.read_csv(
    EDGES
)


assert feat_df.shape == (
    203769,
    167
)


tx_ids = (
    feat_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

timesteps = (
    feat_df.iloc[:, 1]
    .astype(np.int64)
    .to_numpy()
)


# 165 modeling features:
# exclude txId and time_step.

x_np = (
    feat_df.iloc[:, 2:]
    .to_numpy(
        dtype=np.float32
    )
)


assert x_np.shape == (
    203769,
    165
)


print(
    "Nodes:",
    len(tx_ids),
    flush=True
)

print(
    "Model feature shape:",
    x_np.shape,
    flush=True
)

print(
    "[1/6] PASS",
    flush=True
)


# ============================================================
# 2. ALIGN LABELS
# ============================================================

print(
    "\n[2/6] Aligning labels...",
    flush=True
)


id_to_idx = {
    int(tx):
        i
    for i, tx in enumerate(tx_ids)
}


labels = np.full(
    len(tx_ids),
    -1,
    dtype=np.int64
)


class_id_col = class_df.columns[0]
class_col = class_df.columns[1]


for tx, raw in zip(
    class_df[class_id_col],
    class_df[class_col]
):

    idx = id_to_idx[
        int(tx)
    ]

    value = str(raw).strip()

    if value == "1":

        labels[idx] = 1

    elif value == "2":

        labels[idx] = 0

    else:

        labels[idx] = -1


known_mask = (
    labels >= 0
)


assert int(
    known_mask.sum()
) == 46564

assert int(
    (labels == 1).sum()
) == 4545

assert int(
    (labels == 0).sum()
) == 42019


print(
    "Known:",
    int(known_mask.sum()),
    flush=True
)

print(
    "Unknown:",
    int((labels < 0).sum()),
    flush=True
)

print(
    "[2/6] PASS",
    flush=True
)


# ============================================================
# 3. CHRONOLOGICAL NESTED SPLIT
# ============================================================

print(
    "\n[3/6] Creating chronological nested split...",
    flush=True
)


def known_ids_between(
    start_t,
    end_t
):

    return np.where(
        known_mask
        & (timesteps >= start_t)
        & (timesteps <= end_t)
    )[0].astype(
        np.int64
    )


splits = {

    "TR10":
        known_ids_between(
            1,
            3
        ),

    "TR20":
        known_ids_between(
            1,
            7
        ),

    "TR30":
        known_ids_between(
            1,
            12
        ),

    "TR40":
        known_ids_between(
            1,
            20
        ),

    "val":
        known_ids_between(
            21,
            31
        ),

    "test":
        known_ids_between(
            32,
            49
        )
}


expected_sizes = {

    "TR10":
        4543,

    "TR20":
        9553,

    "TR30":
        13670,

    "TR40":
        18889,

    "val":
        8726,

    "test":
        18949
}


for name, expected in (
    expected_sizes.items()
):

    actual = len(
        splits[name]
    )

    ids = splits[
        name
    ]

    fraud = int(
        (labels[ids] == 1)
        .sum()
    )

    normal = int(
        (labels[ids] == 0)
        .sum()
    )


    print(
        f"{name:<5} "
        f"{actual:>6} | "
        f"normal={normal:>6} | "
        f"fraud={fraud:>5}",
        flush=True
    )


    assert actual == expected


# ------------------------------------------------------------
# Nesting
# ------------------------------------------------------------

assert set(
    splits["TR10"]
) <= set(
    splits["TR20"]
)

assert set(
    splits["TR20"]
) <= set(
    splits["TR30"]
)

assert set(
    splits["TR30"]
) <= set(
    splits["TR40"]
)


# ------------------------------------------------------------
# Chronological leakage gates
# ------------------------------------------------------------

assert timesteps[
    splits["TR40"]
].max() <= 20

assert timesteps[
    splits["val"]
].min() >= 21

assert timesteps[
    splits["val"]
].max() <= 31

assert timesteps[
    splits["test"]
].min() >= 32


assert not (
    set(
        splits["TR40"]
    )
    & set(
        splits["val"]
    )
)

assert not (
    set(
        splits["TR40"]
    )
    & set(
        splits["test"]
    )
)

assert not (
    set(
        splits["val"]
    )
    & set(
        splits["test"]
    )
)


np.savez_compressed(
    SPLIT_PATH,

    **splits,

    split_seed=np.array(
        [2],
        dtype=np.int64
    ),

    tr10_max_timestep=np.array(
        [3]
    ),

    tr20_max_timestep=np.array(
        [7]
    ),

    tr30_max_timestep=np.array(
        [12]
    ),

    tr40_max_timestep=np.array(
        [20]
    ),

    val_start_timestep=np.array(
        [21]
    ),

    val_end_timestep=np.array(
        [31]
    ),

    test_start_timestep=np.array(
        [32]
    )
)


print(
    "Nested chronology: PASS",
    flush=True
)

print(
    "Train/validation/test temporal leakage: NONE",
    flush=True
)

print(
    "Saved:",
    SPLIT_PATH,
    flush=True
)

print(
    "[3/6] PASS",
    flush=True
)


# ============================================================
# 4. BUILD DGL GRAPH
# ============================================================

print(
    "\n[4/6] Building BWGNN Elliptic input graph...",
    flush=True
)


src_tx = (
    edge_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

dst_tx = (
    edge_df.iloc[:, 1]
    .astype(np.int64)
    .to_numpy()
)


src_idx = np.fromiter(
    (
        id_to_idx[
            int(x)
        ]
        for x in src_tx
    ),
    dtype=np.int64,
    count=len(src_tx)
)

dst_idx = np.fromiter(
    (
        id_to_idx[
            int(x)
        ]
        for x in dst_tx
    ),
    dtype=np.int64,
    count=len(dst_tx)
)


graph = dgl.graph(
    (
        torch.from_numpy(
            src_idx
        ),
        torch.from_numpy(
            dst_idx
        )
    ),
    num_nodes=203769
)


assert graph.num_nodes() == 203769
assert graph.num_edges() == 234355


# Preserve original directed edges.
# Do NOT add reverse edges.

original_edges = (
    graph.num_edges()
)


# Match standard BWGNN homogeneous execution semantics.

graph = dgl.add_self_loop(
    graph
)


assert graph.num_edges() == (
    234355 + 203769
)


graph.ndata[
    "feature"
] = torch.from_numpy(
    x_np
)

graph.ndata[
    "label"
] = torch.from_numpy(
    labels
)


print(
    "Original directed edges:",
    original_edges,
    flush=True
)

print(
    "Reverse edges added: NO",
    flush=True
)

print(
    "Edges after self-loops:",
    graph.num_edges(),
    flush=True
)

print(
    "Unknown nodes retained:",
    int(
        (labels < 0).sum()
    ),
    flush=True
)

print(
    "[4/6] PASS",
    flush=True
)


# ============================================================
# 5. MOVE TO GPU + BUILD MODEL
# ============================================================

print(
    "\n[5/6] Preparing Tesla T4 smoke...",
    flush=True
)


random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

torch.cuda.manual_seed_all(
    SEED
)


graph = graph.to(
    DEVICE
)


features = (
    graph.ndata[
        "feature"
    ].float()
)

labels_gpu = (
    graph.ndata[
        "label"
    ].long()
)


train_ids = torch.tensor(
    splits["TR40"],
    dtype=torch.long,
    device=DEVICE
)

val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)


# Safety: unknown labels never enter supervised subsets.

assert torch.all(
    labels_gpu[
        train_ids
    ] >= 0
)

assert torch.all(
    labels_gpu[
        val_ids
    ] >= 0
)


train_y = labels_gpu[
    train_ids
]


train_normal = int(
    (train_y == 0)
    .sum()
    .item()
)

train_fraud = int(
    (train_y == 1)
    .sum()
    .item()
)


class_weight = torch.tensor(
    [
        1.0,
        train_normal / train_fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)


model = BWGNN(
    in_feats=165,
    h_feats=HIDDEN,
    num_classes=2,
    graph=graph,
    d=ORDER
).to(
    DEVICE
)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=(
        0.9,
        0.999
    ),
    eps=1e-8
)


print(
    "TR40 normal/fraud:",
    train_normal,
    train_fraud,
    flush=True
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
    flush=True
)

print(
    "[5/6] PASS",
    flush=True
)


# ============================================================
# 6. ONE-EPOCH SMOKE
# ============================================================

print(
    "\n[6/6] Starting 1 training epoch...",
    flush=True
)

print(
    "Test set accessed: NO",
    flush=True
)


torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# Training-only timing
# ------------------------------------------------------------

model.train()

torch.cuda.synchronize()

train_start = (
    time.perf_counter()
)


logits = model(
    features
)


loss = F.cross_entropy(
    logits[
        train_ids
    ],
    labels_gpu[
        train_ids
    ],
    weight=class_weight
)


optimizer.zero_grad()

loss.backward()

optimizer.step()


torch.cuda.synchronize()


train_seconds = (
    time.perf_counter()
    - train_start
)


# ------------------------------------------------------------
# Validation timing separately
# ------------------------------------------------------------

model.eval()

torch.cuda.synchronize()

val_start = (
    time.perf_counter()
)


with torch.no_grad():

    val_logits = model(
        features
    )


torch.cuda.synchronize()


val_seconds = (
    time.perf_counter()
    - val_start
)


val_prob = (
    torch.softmax(
        val_logits,
        dim=1
    )[
        val_ids,
        1
    ]
    .cpu()
    .numpy()
)


val_y = (
    labels_gpu[
        val_ids
    ]
    .cpu()
    .numpy()
)


val_auprc = float(
    average_precision_score(
        val_y,
        val_prob
    )
)

val_auroc = float(
    roc_auc_score(
        val_y,
        val_prob
    )
)


peak_mb = float(
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


print(
    "Training loss:",
    f"{loss.item():.6f}",
    flush=True
)

print(
    "Training-only seconds:",
    f"{train_seconds:.6f}",
    flush=True
)

print(
    "Validation seconds:",
    f"{val_seconds:.6f}",
    flush=True
)

print(
    "Validation AUPRC:",
    f"{val_auprc:.6f}",
    flush=True
)

print(
    "Validation AUROC:",
    f"{val_auroc:.6f}",
    flush=True
)

print(
    "Peak GPU memory MB:",
    f"{peak_mb:.2f}",
    flush=True
)


print(
    "[6/6] PASS",
    flush=True
)


print(
    "\n===== STEP 32B GATE =====",
    flush=True
)

print(
    "PASS — BWGNN × Elliptic chronological smoke completed.",
    flush=True
)

print(
    "Compatibility: INPUT ADAPTER ELIGIBLE",
    flush=True
)

print(
    "Model features: 165",
    flush=True
)

print(
    "Unknown nodes retained in graph: YES",
    flush=True
)

print(
    "Unknown nodes used in supervised loss: NO",
    flush=True
)

print(
    "Reverse edges added: NO",
    flush=True
)

print(
    "Chronological split verified: YES",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)
'''


# ============================================================
# RUN WITH LIVE OUTPUT
# ============================================================

p = subprocess.Popen(
    [
        str(PY),
        "-c",
        script,
        str(FEATURES),
        str(CLASSES),
        str(EDGES),
        str(SPLIT_PATH),
        str(ADAPTER_DIR)
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in p.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = p.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 32B failed with return code {rc}"
    )

===== STEP 32B — ELLIPTIC BWGNN SMOKE =====

[1/6] Loading Elliptic tables...
Nodes: 203769
Model feature shape: (203769, 165)
[1/6] PASS

[2/6] Aligning labels...
Known: 46564
Unknown: 157205
[2/6] PASS

[3/6] Creating chronological nested split...
TR10    4543 | normal=  4497 | fraud=   46
TR20    9553 | normal=  9362 | fraud=  191
TR30   13670 | normal= 12999 | fraud=  671
TR40   18889 | normal= 17118 | fraud= 1771
val     8726 | normal=  7437 | fraud= 1289
test   18949 | normal= 17464 | fraud= 1485
Nested chronology: PASS
Train/validation/test temporal leakage: NONE
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/elliptic_chronological_nested_splits.npz
[3/6] PASS

[4/6] Building BWGNN Elliptic input graph...
Original directed edges: 234355
Reverse edges added: NO
Edges after self-loops: 438124
Unknown nodes retained: 157205
[4/6] PASS

[5/6] Preparing Tesla T4 smoke...
TR40 normal/fraud: 17118 1771
GPU: Tesla T4
[5/6] PASS

[6/6] Starting 1 training epoch...
Test set accessed:

## Step 33 — BWGNN × Elliptic TR40 Unified Tuning

Step 32B confirmed that BWGNN can train on the canonical Elliptic graph while:

- preserving all 203,769 transaction nodes,
- retaining all 234,355 original directed edges,
- retaining unknown-labelled nodes for message passing,
- excluding unknown-labelled nodes from supervised loss and evaluation,
- preserving the chronological training/validation/test split.

### Tuning protocol

- Dataset: Elliptic
- Training subset: chronological TR40
- Split seed: 2
- Training seed: 2
- Maximum trials: 12
- Maximum epochs: 100
- Early stopping patience: 20
- Optimizer: Adam
- Selection metric: validation AUPRC
- Test set: not accessed

The winning configuration is frozen for the later TR40/TR30/TR20/TR10
three-seed final runs.

In [17]:
# ============================================================
# STEP 33 — ELLIPTIC BWGNN TR40 TUNING
# LIVE PROGRESS
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

ADAPTER_DIR = WORK / "adapters"

FEATURES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_features.csv"
)

CLASSES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_classes.csv"
)

EDGES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_edgelist.csv"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "elliptic_chronological_nested_splits.npz"
)

RESULT_DIR = (
    WORK /
    "results/unified/elliptic/tuning"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PY.exists()
assert FEATURES.exists()
assert CLASSES.exists()
assert EDGES.exists()
assert SPLIT.exists()
assert (ADAPTER_DIR / "BWGNN_gpu.py").exists()


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# TUNING SCRIPT
# ============================================================

script = r'''
import sys
import gc
import csv
import json
import random
import time

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


FEATURES = sys.argv[1]
CLASSES = sys.argv[2]
EDGES = sys.argv[3]
SPLIT_PATH = sys.argv[4]
ADAPTER_DIR = sys.argv[5]
RESULT_DIR = Path(sys.argv[6])

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


DEVICE = torch.device("cuda:0")

TRAIN_SEED = 2
MAX_EPOCHS = 100
PATIENCE = 20


# ============================================================
# 12 FIXED TRIALS
# ============================================================

TRIALS = [

    {"hidden": 64,  "order": 2, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 0.0},

    {"hidden": 64,  "order": 1, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 64,  "order": 3, "lr": 1e-2, "weight_decay": 0.0},

    {"hidden": 32,  "order": 2, "lr": 1e-2, "weight_decay": 0.0},
    {"hidden": 32,  "order": 3, "lr": 5e-3, "weight_decay": 0.0},

    {"hidden": 128, "order": 2, "lr": 5e-3, "weight_decay": 0.0},
    {"hidden": 128, "order": 3, "lr": 1e-3, "weight_decay": 0.0},

    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 1e-5},
    {"hidden": 64,  "order": 2, "lr": 5e-3, "weight_decay": 1e-4},

    {"hidden": 64,  "order": 3, "lr": 1e-3, "weight_decay": 1e-4},
    {"hidden": 32,  "order": 2, "lr": 1e-3, "weight_decay": 1e-3}
]

assert len(TRIALS) == 12


# ============================================================
# LOAD ELLIPTIC
# ============================================================

print(
    "===== STEP 33 — ELLIPTIC TR40 TUNING =====",
    flush=True
)

print(
    "Loading Elliptic once...",
    flush=True
)

feat_df = pd.read_csv(
    FEATURES,
    header=None
)

class_df = pd.read_csv(
    CLASSES
)

edge_df = pd.read_csv(
    EDGES
)

tx_ids = (
    feat_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

x_np = (
    feat_df.iloc[:, 2:]
    .to_numpy(dtype=np.float32)
)

assert x_np.shape == (203769, 165)


# ============================================================
# LABELS
# ============================================================

id_to_idx = {
    int(tx):
        i
    for i, tx in enumerate(tx_ids)
}

labels_np = np.full(
    len(tx_ids),
    -1,
    dtype=np.int64
)

class_id_col = class_df.columns[0]
class_col = class_df.columns[1]

for tx, raw in zip(
    class_df[class_id_col],
    class_df[class_col]
):

    idx = id_to_idx[int(tx)]
    value = str(raw).strip()

    if value == "1":
        labels_np[idx] = 1

    elif value == "2":
        labels_np[idx] = 0


assert int((labels_np >= 0).sum()) == 46564


# ============================================================
# GRAPH
# ============================================================

src_tx = (
    edge_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

dst_tx = (
    edge_df.iloc[:, 1]
    .astype(np.int64)
    .to_numpy()
)

src_idx = np.fromiter(
    (
        id_to_idx[int(x)]
        for x in src_tx
    ),
    dtype=np.int64,
    count=len(src_tx)
)

dst_idx = np.fromiter(
    (
        id_to_idx[int(x)]
        for x in dst_tx
    ),
    dtype=np.int64,
    count=len(dst_tx)
)

graph = dgl.graph(
    (
        torch.from_numpy(src_idx),
        torch.from_numpy(dst_idx)
    ),
    num_nodes=203769
)

assert graph.num_edges() == 234355

# No reverse edges.
graph = dgl.add_self_loop(
    graph
)

assert graph.num_edges() == 438124

graph.ndata["feature"] = torch.from_numpy(
    x_np
)

graph.ndata["label"] = torch.from_numpy(
    labels_np
)

graph = graph.to(
    DEVICE
)

features = (
    graph.ndata["feature"]
    .float()
)

labels = (
    graph.ndata["label"]
    .long()
)


# ============================================================
# SPLITS
# ============================================================

splits = np.load(
    SPLIT_PATH
)

train_ids = torch.tensor(
    splits["TR40"],
    dtype=torch.long,
    device=DEVICE
)

val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)

assert torch.all(
    labels[train_ids] >= 0
)

assert torch.all(
    labels[val_ids] >= 0
)


train_y = labels[
    train_ids
]

normal = int(
    (train_y == 0)
    .sum()
    .item()
)

fraud = int(
    (train_y == 1)
    .sum()
    .item()
)

class_weight = torch.tensor(
    [
        1.0,
        normal / fraud
    ],
    dtype=torch.float32,
    device=DEVICE
)

val_y_cpu = (
    labels[val_ids]
    .cpu()
    .numpy()
)


print(
    "Graph:",
    graph.num_nodes(),
    "nodes |",
    graph.num_edges(),
    "edges",
    flush=True
)

print(
    "TR40:",
    len(train_ids),
    "| VAL:",
    len(val_ids),
    flush=True
)

print(
    "TR40 normal/fraud:",
    normal,
    fraud,
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)

print(
    "Trials: 12",
    flush=True
)


# ============================================================
# TUNING
# ============================================================

all_results = []

overall_start = time.perf_counter()


for trial_no, config in enumerate(
    TRIALS,
    start=1
):

    random.seed(
        TRAIN_SEED
    )

    np.random.seed(
        TRAIN_SEED
    )

    torch.manual_seed(
        TRAIN_SEED
    )

    torch.cuda.manual_seed_all(
        TRAIN_SEED
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    gc.collect()
    torch.cuda.empty_cache()


    print(
        "\n"
        + "=" * 72,
        flush=True
    )

    print(
        f"START TRIAL {trial_no:02d}/12 | "
        f"hidden={config['hidden']} | "
        f"order={config['order']} | "
        f"lr={config['lr']} | "
        f"wd={config['weight_decay']}",
        flush=True
    )


    model = BWGNN(
        in_feats=165,
        h_feats=config["hidden"],
        num_classes=2,
        graph=graph,
        d=config["order"]
    ).to(
        DEVICE
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
        betas=(0.9, 0.999),
        eps=1e-8
    )


    best_val_auprc = -1.0
    best_val_auroc = None
    best_epoch = None

    patience_counter = 0

    trial_start = time.perf_counter()


    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        logits = model(
            features
        )

        loss = F.cross_entropy(
            logits[train_ids],
            labels[train_ids],
            weight=class_weight
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        with torch.no_grad():

            val_logits = model(
                features
            )

        val_prob = (
            torch.softmax(
                val_logits,
                dim=1
            )[val_ids, 1]
            .detach()
            .cpu()
            .numpy()
        )

        val_auprc = float(
            average_precision_score(
                val_y_cpu,
                val_prob
            )
        )

        val_auroc = float(
            roc_auc_score(
                val_y_cpu,
                val_prob
            )
        )


        improved = (
            val_auprc
            > best_val_auprc
            + 1e-12
        )


        if improved:

            best_val_auprc = val_auprc
            best_val_auroc = val_auroc
            best_epoch = epoch

            patience_counter = 0

            torch.save(
                {
                    "trial":
                        trial_no,

                    "epoch":
                        epoch,

                    "config":
                        config,

                    "val_auprc":
                        val_auprc,

                    "val_auroc":
                        val_auroc,

                    "model_state_dict":
                        model.state_dict()
                },
                RESULT_DIR /
                f"trial_{trial_no:02d}_best.pt"
            )

        else:

            patience_counter += 1


        # ====================================================
        # LIVE PROGRESS
        # ====================================================

        if (
            epoch == 1
            or epoch % 5 == 0
            or improved
            or patience_counter >= PATIENCE
        ):

            print(
                f"trial={trial_no:02d} | "
                f"epoch={epoch:03d} | "
                f"loss={loss.item():.5f} | "
                f"valAUPRC={val_auprc:.6f} | "
                f"best={best_val_auprc:.6f}"
                f"@{best_epoch} | "
                f"patience={patience_counter}/{PATIENCE}",
                flush=True
            )


        if patience_counter >= PATIENCE:

            print(
                f"EARLY STOP trial {trial_no:02d} "
                f"at epoch {epoch}",
                flush=True
            )

            break


    trial_seconds = (
        time.perf_counter()
        - trial_start
    )


    record = {

        "trial":
            trial_no,

        "hidden":
            config["hidden"],

        "order":
            config["order"],

        "lr":
            config["lr"],

        "weight_decay":
            config["weight_decay"],

        "best_epoch":
            best_epoch,

        "best_val_auprc":
            best_val_auprc,

        "best_val_auroc":
            best_val_auroc,

        "completed_epochs":
            epoch,

        "trial_seconds":
            trial_seconds,

        "test_accessed":
            False
    }


    all_results.append(
        record
    )


    print(
        f"DONE TRIAL {trial_no:02d}/12 | "
        f"best valAUPRC={best_val_auprc:.6f} | "
        f"epoch={best_epoch} | "
        f"time={trial_seconds:.2f}s",
        flush=True
    )


    # resumable summaries

    with (
        RESULT_DIR /
        "elliptic_tuning_trials.csv"
    ).open(
        "w",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=list(
                all_results[0].keys()
            )
        )

        writer.writeheader()
        writer.writerows(
            all_results
        )


    (
        RESULT_DIR /
        "elliptic_tuning_trials.json"
    ).write_text(
        json.dumps(
            all_results,
            indent=2
        )
    )


# ============================================================
# WINNER
# ============================================================

winner = max(
    all_results,
    key=lambda x:
        x["best_val_auprc"]
)


frozen = {

    "dataset":
        "Elliptic",

    "model":
        "BWGNN",

    "mode":
        "unified",

    "compatibility":
        "INPUT ADAPTER ELIGIBLE",

    "model_features":
        165,

    "unknown_nodes_retained":
        True,

    "reverse_edges_added":
        False,

    "chronological_split":
        True,

    "split_seed":
        2,

    "training_seed":
        2,

    "selection_metric":
        "validation AUPRC",

    "max_trials":
        12,

    "max_epochs":
        100,

    "patience":
        20,

    "winner":
        winner,

    "test_accessed":
        False
}


(
    RESULT_DIR /
    "elliptic_frozen_config.json"
).write_text(
    json.dumps(
        frozen,
        indent=2
    )
)


total_seconds = (
    time.perf_counter()
    - overall_start
)


print(
    "\n"
    + "=" * 72,
    flush=True
)

print(
    "===== STEP 33 TUNING WINNER =====",
    flush=True
)

print(
    "Trial:",
    winner["trial"],
    flush=True
)

print(
    "Hidden:",
    winner["hidden"],
    flush=True
)

print(
    "Order:",
    winner["order"],
    flush=True
)

print(
    "Learning rate:",
    winner["lr"],
    flush=True
)

print(
    "Weight decay:",
    winner["weight_decay"],
    flush=True
)

print(
    "Best epoch:",
    winner["best_epoch"],
    flush=True
)

print(
    "Best validation AUPRC:",
    f"{winner['best_val_auprc']:.6f}",
    flush=True
)

print(
    "Validation AUROC:",
    f"{winner['best_val_auroc']:.6f}",
    flush=True
)

print(
    "Total tuning seconds:",
    f"{total_seconds:.2f}",
    flush=True
)


print(
    "\n===== STEP 33 GATE =====",
    flush=True
)

print(
    "PASS — Elliptic TR40 tuning completed.",
    flush=True
)

print(
    "Trials completed: 12/12",
    flush=True
)

print(
    "Configuration frozen: YES",
    flush=True
)

print(
    "Selection used validation AUPRC only: YES",
    flush=True
)

print(
    "Chronology preserved: YES",
    flush=True
)

print(
    "Unknown labels used for supervision: NO",
    flush=True
)

print(
    "Test accessed: NO",
    flush=True
)
'''


# ============================================================
# STREAM LIVE OUTPUT
# ============================================================

p = subprocess.Popen(
    [
        str(PY),
        "-c",
        script,
        str(FEATURES),
        str(CLASSES),
        str(EDGES),
        str(SPLIT),
        str(ADAPTER_DIR),
        str(RESULT_DIR)
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in p.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = p.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 33 failed with return code {rc}"
    )

===== STEP 33 — ELLIPTIC TR40 TUNING =====
Loading Elliptic once...
Graph: 203769 nodes | 438124 edges
TR40: 18889 | VAL: 8726
TR40 normal/fraud: 17118 1771
Test accessed: NO
Trials: 12

START TRIAL 01/12 | hidden=64 | order=2 | lr=0.01 | wd=0.0
trial=01 | epoch=001 | loss=0.67704 | valAUPRC=0.425524 | best=0.425524@1 | patience=0/20
trial=01 | epoch=002 | loss=0.59650 | valAUPRC=0.478035 | best=0.478035@2 | patience=0/20
trial=01 | epoch=003 | loss=0.48831 | valAUPRC=0.536794 | best=0.536794@3 | patience=0/20
trial=01 | epoch=005 | loss=0.37793 | valAUPRC=0.438434 | best=0.536794@3 | patience=2/20
trial=01 | epoch=008 | loss=0.21559 | valAUPRC=0.556912 | best=0.556912@8 | patience=0/20
trial=01 | epoch=009 | loss=0.19820 | valAUPRC=0.570969 | best=0.570969@9 | patience=0/20
trial=01 | epoch=010 | loss=0.18321 | valAUPRC=0.569998 | best=0.570969@9 | patience=1/20
trial=01 | epoch=011 | loss=0.16401 | valAUPRC=0.579652 | best=0.579652@11 | patience=0/20
trial=01 | epoch=013 | loss=0.139

## Step 34 — BWGNN × Elliptic Final Unified Runs

Step 33 selected and froze the Elliptic BWGNN configuration using validation
AUPRC only.

### Frozen configuration

- Hidden dimension: 128
- Wavelet order: 2
- Learning rate: 0.005
- Weight decay: 0
- Maximum epochs: 100
- Early stopping patience: 20

### Controlled final grid

Training subsets:

- TR40
- TR30
- TR20
- TR10

Training seeds:

- 2
- 42
- 72

Total final runs: **12**

### Elliptic safeguards

- 165 model features are used.
- `txId` and `time_step` are not model features.
- All unknown-labelled nodes remain in the graph for message passing.
- Unknown-labelled nodes are excluded from supervised loss and evaluation.
- Original directed payment-flow edges are retained.
- No reverse edges are introduced.
- Chronological training, validation and test boundaries remain fixed.

For every run, raw per-epoch training time, separate validation time,
test metrics, validation-selected threshold, inference latency and peak GPU
memory are saved.

In [18]:
# ============================================================
# STEP 34 — ELLIPTIC FINAL 12 UNIFIED RUNS
# 4 RATIOS × 3 SEEDS
# LIVE PROGRESS + RAW EPOCH TIMING
# ============================================================

import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/comp8851_bwgnn")

PY = WORK / "envs/bwgnn-author/bin/python"
SITE = WORK / "envs/bwgnn-author/lib/python3.9/site-packages"

ADAPTER_DIR = WORK / "adapters"

FEATURES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_features.csv"
)

CLASSES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_classes.csv"
)

EDGES = Path(
    "/kaggle/input/datasets/pathikahmed0007/elliptic/"
    "elliptic_txs_edgelist.csv"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "elliptic_chronological_nested_splits.npz"
)

RESULT_DIR = (
    WORK /
    "results/unified/elliptic/final"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PY.exists()
assert FEATURES.exists()
assert CLASSES.exists()
assert EDGES.exists()
assert SPLIT.exists()
assert (ADAPTER_DIR / "BWGNN_gpu.py").exists()


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = "0"
env["DGLBACKEND"] = "pytorch"
env["PYTHONUNBUFFERED"] = "1"

cuda_dirs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib"
]

env["LD_LIBRARY_PATH"] = (
    ":".join(str(x) for x in cuda_dirs)
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


# ============================================================
# FINAL RUNNER
# ============================================================

script = r'''
import sys
import gc
import csv
import json
import math
import random
import time

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import dgl

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


FEATURES = sys.argv[1]
CLASSES = sys.argv[2]
EDGES = sys.argv[3]
SPLIT_PATH = sys.argv[4]
ADAPTER_DIR = sys.argv[5]
RESULT_DIR = Path(sys.argv[6])

sys.path.insert(
    0,
    ADAPTER_DIR
)

from BWGNN_gpu import BWGNN


# ============================================================
# FROZEN STEP 33 CONFIG
# ============================================================

HIDDEN = 128
ORDER = 2
LR = 0.005
WEIGHT_DECAY = 0.0

MAX_EPOCHS = 100
PATIENCE = 20

RATIOS = [
    "TR40",
    "TR30",
    "TR20",
    "TR10"
]

TRAIN_SEEDS = [
    2,
    42,
    72
]

DEVICE = torch.device(
    "cuda:0"
)


# ============================================================
# OUTPUT TREE
# ============================================================

RUN_DIR = RESULT_DIR / "runs"
EPOCH_DIR = RESULT_DIR / "epoch_times"
CHECKPOINT_DIR = RESULT_DIR / "checkpoints"

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EPOCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# HELPERS
# ============================================================

def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def choose_threshold(
    y_true,
    probability
):

    candidates = []

    for threshold in np.arange(
        0.01,
        1.00,
        0.01
    ):

        pred = (
            probability >= threshold
        ).astype(np.int64)

        macro = float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0
            )
        )

        fraud_recall = float(
            recall_score(
                y_true,
                pred,
                pos_label=1,
                zero_division=0
            )
        )

        candidates.append(
            (
                macro,
                fraud_recall,
                -abs(
                    float(threshold)
                    - 0.5
                ),
                float(threshold)
            )
        )

    winner = max(
        candidates
    )

    return {
        "threshold":
            winner[3],

        "validation_macro_f1":
            winner[0],

        "validation_fraud_recall":
            winner[1]
    }


def calculate_metrics(
    y_true,
    probability,
    threshold
):

    pred = (
        probability >= threshold
    ).astype(np.int64)

    auprc = float(
        average_precision_score(
            y_true,
            probability
        )
    )

    auroc = float(
        roc_auc_score(
            y_true,
            probability
        )
    )

    fraud_precision = float(
        precision_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    fraud_recall = float(
        recall_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    fraud_f1 = float(
        f1_score(
            y_true,
            pred,
            pos_label=1,
            zero_division=0
        )
    )

    macro_f1 = float(
        f1_score(
            y_true,
            pred,
            average="macro",
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    gmean = float(
        math.sqrt(
            sensitivity
            * specificity
        )
    )

    return {
        "auprc":
            auprc,

        "auroc":
            auroc,

        "fraud_precision":
            fraud_precision,

        "fraud_recall":
            fraud_recall,

        "fraud_f1":
            fraud_f1,

        "macro_f1":
            macro_f1,

        "gmean":
            gmean,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp)
    }


# ============================================================
# LOAD CANONICAL ELLIPTIC ONCE
# ============================================================

print(
    "===== STEP 34 — ELLIPTIC FINAL 12 RUNS =====",
    flush=True
)

print(
    "Loading canonical Elliptic...",
    flush=True
)


feat_df = pd.read_csv(
    FEATURES,
    header=None
)

class_df = pd.read_csv(
    CLASSES
)

edge_df = pd.read_csv(
    EDGES
)


tx_ids = (
    feat_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)


# txId + time_step excluded.
x_np = (
    feat_df.iloc[:, 2:]
    .to_numpy(
        dtype=np.float32
    )
)

assert x_np.shape == (
    203769,
    165
)


# ============================================================
# LABEL ALIGNMENT
# ============================================================

id_to_idx = {
    int(tx):
        i
    for i, tx in enumerate(tx_ids)
}


labels_np = np.full(
    len(tx_ids),
    -1,
    dtype=np.int64
)


class_id_col = class_df.columns[0]
class_col = class_df.columns[1]


for tx, raw in zip(
    class_df[class_id_col],
    class_df[class_col]
):

    idx = id_to_idx[
        int(tx)
    ]

    value = str(
        raw
    ).strip()

    if value == "1":

        labels_np[idx] = 1

    elif value == "2":

        labels_np[idx] = 0


assert int(
    (labels_np == 1).sum()
) == 4545

assert int(
    (labels_np == 0).sum()
) == 42019

assert int(
    (labels_np < 0).sum()
) == 157205


# ============================================================
# GRAPH
# ============================================================

src_tx = (
    edge_df.iloc[:, 0]
    .astype(np.int64)
    .to_numpy()
)

dst_tx = (
    edge_df.iloc[:, 1]
    .astype(np.int64)
    .to_numpy()
)


src_idx = np.fromiter(
    (
        id_to_idx[
            int(x)
        ]
        for x in src_tx
    ),
    dtype=np.int64,
    count=len(src_tx)
)

dst_idx = np.fromiter(
    (
        id_to_idx[
            int(x)
        ]
        for x in dst_tx
    ),
    dtype=np.int64,
    count=len(dst_tx)
)


graph = dgl.graph(
    (
        torch.from_numpy(
            src_idx
        ),
        torch.from_numpy(
            dst_idx
        )
    ),
    num_nodes=203769
)


assert graph.num_nodes() == 203769
assert graph.num_edges() == 234355


# Preserve directed graph.
# No reverse edges.

graph = dgl.add_self_loop(
    graph
)

assert graph.num_edges() == 438124


graph.ndata[
    "feature"
] = torch.from_numpy(
    x_np
)

graph.ndata[
    "label"
] = torch.from_numpy(
    labels_np
)


graph = graph.to(
    DEVICE
)


features = (
    graph.ndata[
        "feature"
    ]
    .float()
)

labels = (
    graph.ndata[
        "label"
    ]
    .long()
)


# ============================================================
# CONTROLLED SPLIT
# ============================================================

splits = np.load(
    SPLIT_PATH
)


val_ids = torch.tensor(
    splits["val"],
    dtype=torch.long,
    device=DEVICE
)

test_ids = torch.tensor(
    splits["test"],
    dtype=torch.long,
    device=DEVICE
)


assert torch.all(
    labels[
        val_ids
    ] >= 0
)

assert torch.all(
    labels[
        test_ids
    ] >= 0
)


val_y_cpu = (
    labels[
        val_ids
    ]
    .cpu()
    .numpy()
)

test_y_cpu = (
    labels[
        test_ids
    ]
    .cpu()
    .numpy()
)


print(
    "Graph:",
    graph.num_nodes(),
    "nodes |",
    graph.num_edges(),
    "edges",
    flush=True
)

print(
    "Model features: 165",
    flush=True
)

print(
    "Unknown nodes retained:",
    int(
        (labels_np < 0).sum()
    ),
    flush=True
)

print(
    "Reverse edges added: NO",
    flush=True
)

print(
    "VAL:",
    len(val_ids),
    "| TEST:",
    len(test_ids),
    flush=True
)

print(
    "Frozen config:",
    f"hidden={HIDDEN}, "
    f"order={ORDER}, "
    f"lr={LR}, "
    f"wd={WEIGHT_DECAY}",
    flush=True
)

print(
    "Final grid: 12 runs",
    flush=True
)


# ============================================================
# RUN GRID
# ============================================================

all_results = []

run_number = 0


for ratio in RATIOS:

    for seed in TRAIN_SEEDS:

        run_number += 1

        run_id = (
            f"elliptic_"
            f"{ratio.lower()}_"
            f"seed{seed}"
        )


        print(
            "\n"
            + "=" * 76,
            flush=True
        )

        print(
            f"START RUN {run_number:02d}/12 | "
            f"{ratio} | seed={seed}",
            flush=True
        )


        # ====================================================
        # REPRODUCIBILITY
        # ====================================================

        seed_everything(
            seed
        )

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


        train_ids = torch.tensor(
            splits[ratio],
            dtype=torch.long,
            device=DEVICE
        )


        # No unknown labels in supervised training.

        assert torch.all(
            labels[
                train_ids
            ] >= 0
        )


        train_y = labels[
            train_ids
        ]


        train_normal = int(
            (train_y == 0)
            .sum()
            .item()
        )

        train_fraud = int(
            (train_y == 1)
            .sum()
            .item()
        )


        class_weight = torch.tensor(
            [
                1.0,
                train_normal / train_fraud
            ],
            dtype=torch.float32,
            device=DEVICE
        )


        print(
            f"Training nodes={len(train_ids)} | "
            f"normal={train_normal} | "
            f"fraud={train_fraud}",
            flush=True
        )


        # ====================================================
        # MODEL
        # ====================================================

        model = BWGNN(
            in_feats=165,
            h_feats=HIDDEN,
            num_classes=2,
            graph=graph,
            d=ORDER
        ).to(
            DEVICE
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            betas=(
                0.9,
                0.999
            ),
            eps=1e-8
        )


        checkpoint = (
            CHECKPOINT_DIR /
            f"{run_id}_best.pt"
        )


        # ====================================================
        # TRAIN
        # ====================================================

        best_val_auprc = -1.0
        best_epoch = None

        patience_counter = 0

        epoch_records = []

        validation_seconds_total = 0.0

        whole_run_start = (
            time.perf_counter()
        )


        for epoch in range(
            1,
            MAX_EPOCHS + 1
        ):

            # =================================================
            # TRAINING-ONLY TIMER
            # =================================================

            model.train()

            torch.cuda.synchronize()

            train_start = (
                time.perf_counter()
            )


            logits = model(
                features
            )


            loss = F.cross_entropy(
                logits[
                    train_ids
                ],
                labels[
                    train_ids
                ],
                weight=class_weight
            )


            optimizer.zero_grad()

            loss.backward()

            optimizer.step()


            torch.cuda.synchronize()


            train_seconds = (
                time.perf_counter()
                - train_start
            )


            # =================================================
            # VALIDATION — SEPARATE
            # =================================================

            model.eval()

            torch.cuda.synchronize()

            validation_start = (
                time.perf_counter()
            )


            with torch.no_grad():

                val_logits = model(
                    features
                )


            torch.cuda.synchronize()


            validation_seconds = (
                time.perf_counter()
                - validation_start
            )


            validation_seconds_total += (
                validation_seconds
            )


            val_probability = (
                torch.softmax(
                    val_logits,
                    dim=1
                )[
                    val_ids,
                    1
                ]
                .detach()
                .cpu()
                .numpy()
            )


            val_auprc = float(
                average_precision_score(
                    val_y_cpu,
                    val_probability
                )
            )


            improved = (
                val_auprc
                > best_val_auprc
                + 1e-12
            )


            if improved:

                best_val_auprc = (
                    val_auprc
                )

                best_epoch = epoch

                patience_counter = 0


                torch.save(
                    {
                        "ratio":
                            ratio,

                        "seed":
                            seed,

                        "epoch":
                            epoch,

                        "hidden":
                            HIDDEN,

                        "order":
                            ORDER,

                        "lr":
                            LR,

                        "weight_decay":
                            WEIGHT_DECAY,

                        "val_auprc":
                            val_auprc,

                        "model_state_dict":
                            model.state_dict()
                    },
                    checkpoint
                )


            else:

                patience_counter += 1


            epoch_records.append(
                {
                    "run_id":
                        run_id,

                    "ratio":
                        ratio,

                    "seed":
                        seed,

                    "epoch":
                        epoch,

                    "training_loss":
                        float(
                            loss.item()
                        ),

                    "train_seconds":
                        float(
                            train_seconds
                        ),

                    "validation_seconds":
                        float(
                            validation_seconds
                        ),

                    "val_auprc":
                        float(
                            val_auprc
                        ),

                    "best_val_auprc":
                        float(
                            best_val_auprc
                        ),

                    "patience_counter":
                        int(
                            patience_counter
                        )
                }
            )


            # =================================================
            # LIVE PROGRESS
            # =================================================

            if (
                epoch == 1
                or epoch % 10 == 0
                or improved
                or patience_counter >= PATIENCE
            ):

                print(
                    f"run={run_number:02d}/12 | "
                    f"{ratio} seed={seed} | "
                    f"epoch={epoch:03d} | "
                    f"loss={loss.item():.5f} | "
                    f"valAUPRC={val_auprc:.6f} | "
                    f"best={best_val_auprc:.6f}"
                    f"@{best_epoch} | "
                    f"patience={patience_counter}/20",
                    flush=True
                )


            if patience_counter >= PATIENCE:

                print(
                    f"EARLY STOP | "
                    f"{ratio} seed={seed} | "
                    f"epoch={epoch}",
                    flush=True
                )

                break


        # ====================================================
        # SAVE RAW PER-EPOCH TIMING
        # ====================================================

        epoch_csv = (
            EPOCH_DIR /
            f"{run_id}_epoch_times.csv"
        )


        with epoch_csv.open(
            "w",
            newline=""
        ) as handle:

            writer = csv.DictWriter(
                handle,
                fieldnames=list(
                    epoch_records[
                        0
                    ].keys()
                )
            )

            writer.writeheader()

            writer.writerows(
                epoch_records
            )


        # ====================================================
        # RESTORE BEST VALIDATION CHECKPOINT
        # ====================================================

        saved = torch.load(
            checkpoint,
            map_location=DEVICE
        )


        model.load_state_dict(
            saved[
                "model_state_dict"
            ]
        )

        model.eval()


        # ====================================================
        # VALIDATION THRESHOLD
        # ====================================================

        with torch.no_grad():

            val_logits = model(
                features
            )


        val_probability = (
            torch.softmax(
                val_logits,
                dim=1
            )[
                val_ids,
                1
            ]
            .detach()
            .cpu()
            .numpy()
        )


        threshold_result = (
            choose_threshold(
                val_y_cpu,
                val_probability
            )
        )


        threshold = (
            threshold_result[
                "threshold"
            ]
        )


        # ====================================================
        # TEST EVALUATION — AFTER ALL CHOICES FROZEN
        # ====================================================

        with torch.no_grad():

            test_logits = model(
                features
            )


        test_probability = (
            torch.softmax(
                test_logits,
                dim=1
            )[
                test_ids,
                1
            ]
            .detach()
            .cpu()
            .numpy()
        )


        test_metrics = (
            calculate_metrics(
                test_y_cpu,
                test_probability,
                threshold
            )
        )


        # ====================================================
        # FULL-GRAPH INFERENCE LATENCY
        #
        # Team standardization:
        # 3 warmups + 10 synchronized forwards
        # ====================================================

        model.eval()


        with torch.no_grad():

            for _ in range(3):

                _ = model(
                    features
                )


            latency_ms = []


            for _ in range(10):

                torch.cuda.synchronize()

                inference_start = (
                    time.perf_counter()
                )


                _ = model(
                    features
                )


                torch.cuda.synchronize()


                latency_ms.append(
                    (
                        time.perf_counter()
                        - inference_start
                    )
                    * 1000.0
                )


        inference_mean_ms = float(
            np.mean(
                latency_ms
            )
        )

        inference_std_ms = float(
            np.std(
                latency_ms,
                ddof=1
            )
        )


        # ====================================================
        # TIMING SUMMARY
        # ====================================================

        epoch_train_times = np.array(
            [
                row[
                    "train_seconds"
                ]
                for row in epoch_records
            ],
            dtype=float
        )


        total_train_seconds = float(
            epoch_train_times.sum()
        )

        mean_epoch_train_seconds = float(
            epoch_train_times.mean()
        )

        std_epoch_train_seconds = float(
            epoch_train_times.std(
                ddof=1
            )
            if len(
                epoch_train_times
            ) > 1
            else 0.0
        )


        whole_run_seconds = float(
            time.perf_counter()
            - whole_run_start
        )


        peak_gpu_memory_mb = float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )


        # ====================================================
        # RUN RECORD
        # ====================================================

        record = {

            "run_id":
                run_id,

            "dataset":
                "Elliptic",

            "model":
                "BWGNN",

            "mode":
                "unified",

            "compatibility":
                "INPUT ADAPTER ELIGIBLE",

            "chronological_split":
                True,

            "ratio":
                ratio,

            "training_seed":
                seed,

            "split_seed":
                2,

            "model_features":
                165,

            "unknown_nodes_retained":
                True,

            "unknown_labels_used_for_supervision":
                False,

            "reverse_edges_added":
                False,

            "hidden":
                HIDDEN,

            "order":
                ORDER,

            "lr":
                LR,

            "weight_decay":
                WEIGHT_DECAY,

            "completed_epochs":
                len(
                    epoch_records
                ),

            "best_epoch":
                best_epoch,

            "best_val_auprc":
                best_val_auprc,

            "validation_threshold":
                threshold,

            "validation_threshold_macro_f1":
                threshold_result[
                    "validation_macro_f1"
                ],

            "validation_threshold_fraud_recall":
                threshold_result[
                    "validation_fraud_recall"
                ],

            "test_auprc":
                test_metrics[
                    "auprc"
                ],

            "test_auroc":
                test_metrics[
                    "auroc"
                ],

            "test_fraud_precision":
                test_metrics[
                    "fraud_precision"
                ],

            "test_fraud_recall":
                test_metrics[
                    "fraud_recall"
                ],

            "test_fraud_f1":
                test_metrics[
                    "fraud_f1"
                ],

            "test_macro_f1":
                test_metrics[
                    "macro_f1"
                ],

            "test_gmean":
                test_metrics[
                    "gmean"
                ],

            "tn":
                test_metrics[
                    "tn"
                ],

            "fp":
                test_metrics[
                    "fp"
                ],

            "fn":
                test_metrics[
                    "fn"
                ],

            "tp":
                test_metrics[
                    "tp"
                ],

            "total_train_seconds":
                total_train_seconds,

            "mean_epoch_train_seconds":
                mean_epoch_train_seconds,

            "std_epoch_train_seconds":
                std_epoch_train_seconds,

            "validation_seconds_total":
                validation_seconds_total,

            "whole_run_seconds":
                whole_run_seconds,

            "inference_latency_mean_ms":
                inference_mean_ms,

            "inference_latency_std_ms":
                inference_std_ms,

            "inference_warmups":
                3,

            "inference_repetitions":
                10,

            "peak_gpu_memory_mb":
                peak_gpu_memory_mb,

            "test_used_for_training":
                False,

            "test_used_for_checkpoint_selection":
                False,

            "test_used_for_threshold_selection":
                False
        }


        (
            RUN_DIR /
            f"{run_id}.json"
        ).write_text(
            json.dumps(
                record,
                indent=2
            )
        )


        all_results.append(
            record
        )


        # Resumable all-runs CSV after each completed run.

        with (
            RESULT_DIR /
            "elliptic_final_all_runs_partial.csv"
        ).open(
            "w",
            newline=""
        ) as handle:

            writer = csv.DictWriter(
                handle,
                fieldnames=list(
                    all_results[
                        0
                    ].keys()
                )
            )

            writer.writeheader()

            writer.writerows(
                all_results
            )


        print(
            f"DONE RUN {run_number:02d}/12 | "
            f"{ratio} seed={seed} | "
            f"AUPRC={record['test_auprc']:.6f} | "
            f"AUROC={record['test_auroc']:.6f} | "
            f"MacroF1={record['test_macro_f1']:.6f} | "
            f"Recall={record['test_fraud_recall']:.6f} | "
            f"GMean={record['test_gmean']:.6f} | "
            f"bestEpoch={best_epoch} | "
            f"train={total_train_seconds:.3f}s | "
            f"latency={inference_mean_ms:.3f}ms | "
            f"peak={peak_gpu_memory_mb:.1f}MB",
            flush=True
        )


        del model
        del optimizer

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# ALL-RUN OUTPUTS
# ============================================================

assert len(
    all_results
) == 12


all_runs_csv = (
    RESULT_DIR /
    "elliptic_final_all_runs.csv"
)


with all_runs_csv.open(
    "w",
    newline=""
) as handle:

    writer = csv.DictWriter(
        handle,
        fieldnames=list(
            all_results[
                0
            ].keys()
        )
    )

    writer.writeheader()

    writer.writerows(
        all_results
    )


(
    RESULT_DIR /
    "elliptic_final_all_runs.json"
).write_text(
    json.dumps(
        all_results,
        indent=2
    )
)


# ============================================================
# AGGREGATE SUMMARY BY RATIO
# ============================================================

fields = [
    "test_auprc",
    "test_auroc",
    "test_fraud_precision",
    "test_fraud_recall",
    "test_fraud_f1",
    "test_macro_f1",
    "test_gmean",
    "mean_epoch_train_seconds",
    "total_train_seconds",
    "inference_latency_mean_ms",
    "peak_gpu_memory_mb"
]


summary = {}


for ratio in RATIOS:

    rows = [
        row
        for row in all_results
        if row[
            "ratio"
        ] == ratio
    ]


    assert len(rows) == 3


    summary[
        ratio
    ] = {}


    for field in fields:

        values = np.array(
            [
                row[field]
                for row in rows
            ],
            dtype=float
        )


        summary[
            ratio
        ][field] = {

            "mean":
                float(
                    values.mean()
                ),

            "std":
                float(
                    values.std(
                        ddof=1
                    )
                )
        }


(
    RESULT_DIR /
    "elliptic_final_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# PRINT FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 76,
    flush=True
)

print(
    "===== STEP 34 FINAL SUMMARY =====",
    flush=True
)


for ratio in RATIOS:

    s = summary[
        ratio
    ]


    print(
        f"\n{ratio}",
        flush=True
    )


    print(
        "AUPRC:",
        f"{s['test_auprc']['mean']:.6f} "
        f"± {s['test_auprc']['std']:.6f}",
        flush=True
    )


    print(
        "AUROC:",
        f"{s['test_auroc']['mean']:.6f} "
        f"± {s['test_auroc']['std']:.6f}",
        flush=True
    )


    print(
        "Fraud Precision:",
        f"{s['test_fraud_precision']['mean']:.6f} "
        f"± {s['test_fraud_precision']['std']:.6f}",
        flush=True
    )


    print(
        "Fraud Recall:",
        f"{s['test_fraud_recall']['mean']:.6f} "
        f"± {s['test_fraud_recall']['std']:.6f}",
        flush=True
    )


    print(
        "Fraud F1:",
        f"{s['test_fraud_f1']['mean']:.6f} "
        f"± {s['test_fraud_f1']['std']:.6f}",
        flush=True
    )


    print(
        "Macro-F1:",
        f"{s['test_macro_f1']['mean']:.6f} "
        f"± {s['test_macro_f1']['std']:.6f}",
        flush=True
    )


    print(
        "G-Mean:",
        f"{s['test_gmean']['mean']:.6f} "
        f"± {s['test_gmean']['std']:.6f}",
        flush=True
    )


    print(
        "Mean epoch training:",
        f"{s['mean_epoch_train_seconds']['mean']:.6f}s "
        f"± {s['mean_epoch_train_seconds']['std']:.6f}",
        flush=True
    )


    print(
        "Total training:",
        f"{s['total_train_seconds']['mean']:.6f}s "
        f"± {s['total_train_seconds']['std']:.6f}",
        flush=True
    )


    print(
        "Inference latency:",
        f"{s['inference_latency_mean_ms']['mean']:.6f}ms "
        f"± {s['inference_latency_mean_ms']['std']:.6f}",
        flush=True
    )


    print(
        "Peak GPU memory:",
        f"{s['peak_gpu_memory_mb']['mean']:.2f}MB "
        f"± {s['peak_gpu_memory_mb']['std']:.2f}",
        flush=True
    )


# ============================================================
# FINAL GATES
# ============================================================

run_jsons = list(
    RUN_DIR.glob(
        "*.json"
    )
)

epoch_csvs = list(
    EPOCH_DIR.glob(
        "*_epoch_times.csv"
    )
)


assert len(
    run_jsons
) == 12

assert len(
    epoch_csvs
) == 12


print(
    "\n===== STEP 34 GATE =====",
    flush=True
)

print(
    "PASS — Elliptic final unified benchmark completed.",
    flush=True
)

print(
    "Final runs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)

print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Chronology preserved: YES",
    flush=True
)

print(
    "Unknown nodes retained: YES",
    flush=True
)

print(
    "Unknown labels used for supervision: NO",
    flush=True
)

print(
    "Reverse edges added: NO",
    flush=True
)

print(
    "Training timing excludes validation: YES",
    flush=True
)

print(
    "Threshold selected using validation only: YES",
    flush=True
)

print(
    "Test used for training/checkpoint/threshold selection: NO",
    flush=True
)

print(
    "Inference latency recorded: YES",
    flush=True
)

print(
    "Peak GPU memory recorded: YES",
    flush=True
)
'''


# ============================================================
# STREAM LIVE OUTPUT
# ============================================================

p = subprocess.Popen(
    [
        str(PY),
        "-c",
        script,
        str(FEATURES),
        str(CLASSES),
        str(EDGES),
        str(SPLIT),
        str(ADAPTER_DIR),
        str(RESULT_DIR)
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in p.stdout:

    print(
        line.rstrip(),
        flush=True
    )


rc = p.wait()


if rc != 0:

    raise RuntimeError(
        f"Step 34 failed with return code {rc}"
    )

===== STEP 34 — ELLIPTIC FINAL 12 RUNS =====
Loading canonical Elliptic...
Graph: 203769 nodes | 438124 edges
Model features: 165
Unknown nodes retained: 157205
Reverse edges added: NO
VAL: 8726 | TEST: 18949
Frozen config: hidden=128, order=2, lr=0.005, wd=0.0
Final grid: 12 runs

START RUN 01/12 | TR40 | seed=2
Training nodes=18889 | normal=17118 | fraud=1771
run=01/12 | TR40 seed=2 | epoch=001 | loss=0.68431 | valAUPRC=0.546947 | best=0.546947@1 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=002 | loss=0.62886 | valAUPRC=0.585780 | best=0.585780@2 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=008 | loss=0.24628 | valAUPRC=0.634993 | best=0.634993@8 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=009 | loss=0.22342 | valAUPRC=0.649587 | best=0.649587@9 | patience=0/20
run=01/12 | TR40 seed=2 | epoch=010 | loss=0.20344 | valAUPRC=0.630501 | best=0.649587@9 | patience=1/20
run=01/12 | TR40 seed=2 | epoch=019 | loss=0.10513 | valAUPRC=0.649908 | best=0.649908@19 | patience=0/20
run=0

## Step 35 — Package BWGNN × Elliptic Evidence

This package preserves the complete BWGNN × Elliptic controlled benchmark
evidence.

Included:

- compatibility decision,
- canonical Elliptic file hashes,
- frozen BWGNN repository commit,
- BWGNN GPU compatibility adapter,
- chronological nested split,
- tuning trial summary,
- frozen winning configuration,
- 12 final run JSON records,
- 12 raw per-epoch timing CSVs,
- final all-run results,
- aggregate summary,
- environment record,
- evidence manifest.

Model checkpoints are intentionally excluded.

### Final Elliptic status

- Compatibility: Input Adapter Eligible
- Chronology preserved: Yes
- Unknown nodes retained for message passing: Yes
- Unknown labels used for supervision: No
- Reverse edges added: No
- Tuning trials: 12/12
- Final runs: 12/12
- Raw epoch timing CSVs: 12/12
- Test-set tuning: None

In [19]:
# ============================================================
# STEP 35 — PACKAGE COMPLETE BWGNN × ELLIPTIC EVIDENCE
# ============================================================

from pathlib import Path
import subprocess
import hashlib
import json
import zipfile
import datetime

WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

REPO = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

ADAPTER = (
    WORK /
    "adapters/BWGNN_gpu.py"
)

SPLIT = (
    WORK /
    "shared/splits/"
    "elliptic_chronological_nested_splits.npz"
)

TUNING_DIR = (
    WORK /
    "results/unified/elliptic/tuning"
)

FINAL_DIR = (
    WORK /
    "results/unified/elliptic/final"
)

RUN_DIR = (
    FINAL_DIR /
    "runs"
)

EPOCH_DIR = (
    FINAL_DIR /
    "epoch_times"
)

EVIDENCE_DIR = (
    WORK /
    "results/evidence/elliptic"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "bwgnn_elliptic_complete_evidence_20260910.zip"
)


# ============================================================
# CANONICAL RAW DATA
# ============================================================

FEATURES = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_features.csv"
)

CLASSES = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_classes.csv"
)

EDGES = Path(
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_edgelist.csv"
)


EXPECTED_RAW_HASHES = {

    "elliptic_txs_features.csv":
        "fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0",

    "elliptic_txs_classes.csv":
        "93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493",

    "elliptic_txs_edgelist.csv":
        "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e"
}


print(
    "===== STEP 35 — ELLIPTIC EVIDENCE PACKAGING =====",
    flush=True
)


# ============================================================
# HELPER
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    8 * 1024 * 1024
                ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


# ============================================================
# 1. VERIFY CORE FILES
# ============================================================

print(
    "\n[1/7] Verifying required evidence...",
    flush=True
)


required = [

    FEATURES,
    CLASSES,
    EDGES,

    ADAPTER,
    SPLIT,

    TUNING_DIR /
    "elliptic_tuning_trials.csv",

    TUNING_DIR /
    "elliptic_tuning_trials.json",

    TUNING_DIR /
    "elliptic_frozen_config.json",

    FINAL_DIR /
    "elliptic_final_all_runs.csv",

    FINAL_DIR /
    "elliptic_final_all_runs.json",

    FINAL_DIR /
    "elliptic_final_summary.json"
]


for path in required:

    assert path.exists(), (
        f"Missing required evidence: {path}"
    )


run_jsons = sorted(
    RUN_DIR.glob(
        "*.json"
    )
)

epoch_csvs = sorted(
    EPOCH_DIR.glob(
        "*_epoch_times.csv"
    )
)


assert len(
    run_jsons
) == 12

assert len(
    epoch_csvs
) == 12


print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)

print(
    "[1/7] PASS",
    flush=True
)


# ============================================================
# 2. VERIFY RAW DATA HASHES
# ============================================================

print(
    "\n[2/7] Verifying canonical Elliptic hashes...",
    flush=True
)


raw_paths = {

    "elliptic_txs_features.csv":
        FEATURES,

    "elliptic_txs_classes.csv":
        CLASSES,

    "elliptic_txs_edgelist.csv":
        EDGES
}


actual_raw_hashes = {}


for name, path in raw_paths.items():

    actual = sha256_file(
        path
    )

    expected = (
        EXPECTED_RAW_HASHES[
            name
        ]
    )

    assert actual == expected

    actual_raw_hashes[
        name
    ] = actual

    print(
        name,
        actual,
        flush=True
    )


print(
    "[2/7] PASS",
    flush=True
)


# ============================================================
# 3. VERIFY FROZEN BWGNN COMMIT
# ============================================================

print(
    "\n[3/7] Verifying frozen BWGNN source...",
    flush=True
)


commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD"
    ],
    text=True
).strip()


EXPECTED_COMMIT = (
    "de0631f039bbd19c1890b483cc01f1007f596af7"
)


assert commit == EXPECTED_COMMIT


print(
    "Frozen commit:",
    commit,
    flush=True
)

print(
    "[3/7] PASS",
    flush=True
)


# ============================================================
# 4. VERIFY FROZEN CONFIG
# ============================================================

print(
    "\n[4/7] Verifying frozen Elliptic configuration...",
    flush=True
)


frozen = json.loads(
    (
        TUNING_DIR /
        "elliptic_frozen_config.json"
    ).read_text()
)


winner = frozen[
    "winner"
]


assert winner[
    "trial"
] == 7

assert winner[
    "hidden"
] == 128

assert winner[
    "order"
] == 2

assert winner[
    "lr"
] == 0.005

assert winner[
    "weight_decay"
] == 0.0

assert winner[
    "best_epoch"
] == 79


print(
    "Winner trial:",
    winner["trial"],
    flush=True
)

print(
    "Hidden:",
    winner["hidden"],
    flush=True
)

print(
    "Order:",
    winner["order"],
    flush=True
)

print(
    "Learning rate:",
    winner["lr"],
    flush=True
)

print(
    "Weight decay:",
    winner["weight_decay"],
    flush=True
)

print(
    "Tuning best epoch:",
    winner["best_epoch"],
    flush=True
)

print(
    "Best validation AUPRC:",
    winner["best_val_auprc"],
    flush=True
)


print(
    "[4/7] PASS",
    flush=True
)


# ============================================================
# 5. CREATE SUPPORT RECORDS
# ============================================================

print(
    "\n[5/7] Creating manifest records...",
    flush=True
)


split_sha = sha256_file(
    SPLIT
)

adapter_sha = sha256_file(
    ADAPTER
)


# ------------------------------------------------------------
# Compatibility decision
# ------------------------------------------------------------

COMPAT_PATH = (
    EVIDENCE_DIR /
    "elliptic_compatibility_decision.json"
)


compatibility = {

    "dataset":
        "Elliptic",

    "model":
        "BWGNN",

    "native_repository_support":
        False,

    "classification":
        "INPUT ADAPTER ELIGIBLE",

    "canonical_nodes":
        203769,

    "canonical_directed_edges":
        234355,

    "edges_after_self_loops":
        438124,

    "model_features":
        165,

    "time_step_used_as_model_feature":
        False,

    "time_step_used_for_split":
        True,

    "known_labelled_nodes":
        46564,

    "unknown_nodes":
        157205,

    "fraud_nodes":
        4545,

    "normal_nodes":
        42019,

    "unknown_nodes_retained_for_message_passing":
        True,

    "unknown_labels_used_for_supervision":
        False,

    "original_edge_direction_preserved":
        True,

    "reverse_edges_added":
        False,

    "self_loops_added":
        True,

    "architecture_changed":
        False,

    "chronology_preserved":
        True,

    "training_periods": {

        "TR10":
            "time steps 1-3",

        "TR20":
            "time steps 1-7",

        "TR30":
            "time steps 1-12",

        "TR40":
            "time steps 1-20"
    },

    "validation_period":
        "time steps 21-31",

    "test_period":
        "time steps 32-49",

    "split_sizes": {

        "TR10":
            4543,

        "TR20":
            9553,

        "TR30":
            13670,

        "TR40":
            18889,

        "validation":
            8726,

        "test":
            18949
    },

    "smoke_test":
        "PASS",

    "test_accessed_during_compatibility":
        False
}


COMPAT_PATH.write_text(
    json.dumps(
        compatibility,
        indent=2
    )
)


# ------------------------------------------------------------
# Environment record
# ------------------------------------------------------------

ENV_PATH = (
    EVIDENCE_DIR /
    "elliptic_environment.json"
)


environment = {

    "repository":
        (
            "https://github.com/"
            "squareRoot3/"
            "Rethinking-Anomaly-Detection.git"
        ),

    "commit":
        commit,

    "python":
        "3.9.25",

    "torch":
        "1.9.0+cu111",

    "dgl":
        "0.8.1",

    "numpy":
        "1.23.5",

    "gpu":
        "Tesla T4",

    "controlled_gpu":
        0,

    "cuda_visible_devices":
        "0",

    "adapter_file":
        "BWGNN_gpu.py",

    "adapter_sha256":
        adapter_sha,

    "gpu_patch":
        (
            "Four torch.zeros allocations "
            "made device-aware"
        )
}


ENV_PATH.write_text(
    json.dumps(
        environment,
        indent=2
    )
)


# ------------------------------------------------------------
# Dataset provenance
# ------------------------------------------------------------

PROVENANCE_PATH = (
    EVIDENCE_DIR /
    "elliptic_dataset_provenance.json"
)


provenance = {

    "dataset":
        "Elliptic",

    "canonical_paths": {

        "features":
            str(FEATURES),

        "classes":
            str(CLASSES),

        "edges":
            str(EDGES)
    },

    "sha256":
        actual_raw_hashes,

    "nodes":
        203769,

    "directed_edges":
        234355,

    "raw_feature_columns":
        167,

    "model_features":
        165,

    "removed_from_model_input": [
        "txId",
        "time_step"
    ],

    "time_steps":
        "1-49"
}


PROVENANCE_PATH.write_text(
    json.dumps(
        provenance,
        indent=2
    )
)


print(
    "Split SHA256:",
    split_sha,
    flush=True
)

print(
    "Adapter SHA256:",
    adapter_sha,
    flush=True
)

print(
    "[5/7] PASS",
    flush=True
)


# ============================================================
# 6. MANIFEST + ZIP
# ============================================================

print(
    "\n[6/7] Building evidence ZIP...",
    flush=True
)


summary = json.loads(
    (
        FINAL_DIR /
        "elliptic_final_summary.json"
    ).read_text()
)


MANIFEST_PATH = (
    EVIDENCE_DIR /
    "MANIFEST.json"
)


manifest = {

    "created_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),

    "dataset":
        "Elliptic",

    "model":
        "BWGNN",

    "benchmark_mode":
        "unified controlled",

    "compatibility":
        "INPUT ADAPTER ELIGIBLE",

    "repository_commit":
        commit,

    "dataset_hashes":
        actual_raw_hashes,

    "split_file":
        SPLIT.name,

    "split_sha256":
        split_sha,

    "adapter_file":
        ADAPTER.name,

    "adapter_sha256":
        adapter_sha,

    "model_features":
        165,

    "chronology_preserved":
        True,

    "reverse_edges_added":
        False,

    "unknown_nodes_retained":
        True,

    "unknown_labels_used_for_supervision":
        False,

    "split_seed":
        2,

    "training_seeds": [
        2,
        42,
        72
    ],

    "training_ratios": [
        "TR40",
        "TR30",
        "TR20",
        "TR10"
    ],

    "tuning_trials":
        12,

    "frozen_configuration": {

        "hidden":
            winner["hidden"],

        "order":
            winner["order"],

        "learning_rate":
            winner["lr"],

        "weight_decay":
            winner[
                "weight_decay"
            ],

        "best_tuning_epoch":
            winner[
                "best_epoch"
            ],

        "best_validation_auprc":
            winner[
                "best_val_auprc"
            ]
    },

    "final_runs":
        12,

    "raw_epoch_timing_files":
        12,

    "training_timing_excludes_validation":
        True,

    "validation_threshold_only":
        True,

    "test_used_for_tuning":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_selection":
        False,

    "inference_latency_recorded":
        True,

    "peak_gpu_memory_recorded":
        True,

    "checkpoints_included":
        False,

    "results_summary":
        summary
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    z.write(
        MANIFEST_PATH,
        "manifest/MANIFEST.json"
    )


    # --------------------------------------------------------
    # Compatibility / provenance / environment
    # --------------------------------------------------------

    z.write(
        COMPAT_PATH,
        (
            "compatibility/"
            "elliptic_compatibility_decision.json"
        )
    )

    z.write(
        PROVENANCE_PATH,
        (
            "dataset/"
            "elliptic_dataset_provenance.json"
        )
    )

    z.write(
        ENV_PATH,
        (
            "environment/"
            "elliptic_environment.json"
        )
    )


    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    z.write(
        SPLIT,
        (
            "splits/"
            "elliptic_chronological_nested_splits.npz"
        )
    )


    # --------------------------------------------------------
    # Adapter
    # --------------------------------------------------------

    z.write(
        ADAPTER,
        "adapters/BWGNN_gpu.py"
    )


    # --------------------------------------------------------
    # Tuning
    # --------------------------------------------------------

    for name in [

        "elliptic_tuning_trials.csv",
        "elliptic_tuning_trials.json",
        "elliptic_frozen_config.json"

    ]:

        path = (
            TUNING_DIR /
            name
        )

        z.write(
            path,
            f"tuning/{name}"
        )


    # --------------------------------------------------------
    # Final aggregate output
    # --------------------------------------------------------

    for name in [

        "elliptic_final_all_runs.csv",
        "elliptic_final_all_runs.json",
        "elliptic_final_summary.json"

    ]:

        path = (
            FINAL_DIR /
            name
        )

        z.write(
            path,
            f"final/{name}"
        )


    # --------------------------------------------------------
    # Individual final runs
    # --------------------------------------------------------

    for path in run_jsons:

        z.write(
            path,
            (
                "final/runs/"
                f"{path.name}"
            )
        )


    # --------------------------------------------------------
    # Raw per-epoch timing files
    # --------------------------------------------------------

    for path in epoch_csvs:

        z.write(
            path,
            (
                "final/epoch_times/"
                f"{path.name}"
            )
        )


print(
    "[6/7] PASS",
    flush=True
)


# ============================================================
# 7. VERIFY ZIP + HASH
# ============================================================

print(
    "\n[7/7] Verifying ZIP...",
    flush=True
)


zip_sha = sha256_file(
    ZIP_PATH
)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    names = (
        z.namelist()
    )


assert (
    "manifest/MANIFEST.json"
    in names
)

assert (
    "compatibility/"
    "elliptic_compatibility_decision.json"
    in names
)

assert (
    "dataset/"
    "elliptic_dataset_provenance.json"
    in names
)

assert (
    "environment/"
    "elliptic_environment.json"
    in names
)

assert (
    "splits/"
    "elliptic_chronological_nested_splits.npz"
    in names
)

assert (
    "tuning/"
    "elliptic_frozen_config.json"
    in names
)


assert sum(
    name.startswith(
        "final/runs/"
    )
    for name in names
) == 12


assert sum(
    name.startswith(
        "final/epoch_times/"
    )
    for name in names
) == 12


assert not any(
    name.endswith(
        ".pt"
    )
    for name in names
)


print(
    "[7/7] PASS",
    flush=True
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 35 GATE =====",
    flush=True
)

print(
    "PASS — complete Elliptic evidence packaged.",
    flush=True
)

print(
    "Compatibility: INPUT ADAPTER ELIGIBLE",
    flush=True
)

print(
    "Chronology preserved: YES",
    flush=True
)

print(
    "Unknown nodes retained: YES",
    flush=True
)

print(
    "Unknown labels used for supervision: NO",
    flush=True
)

print(
    "Reverse edges added: NO",
    flush=True
)

print(
    "Tuning trials: 12/12",
    flush=True
)

print(
    "Final runs: 12/12",
    flush=True
)

print(
    "Raw epoch timing CSVs: 12/12",
    flush=True
)

print(
    "Run JSONs: 12/12",
    flush=True
)

print(
    "Controlled chronological split included: YES",
    flush=True
)

print(
    "Frozen config included: YES",
    flush=True
)

print(
    "BWGNN GPU adapter included: YES",
    flush=True
)

print(
    "Dataset provenance included: YES",
    flush=True
)

print(
    "Checkpoints included: NO",
    flush=True
)

print(
    "Test-set tuning: NO",
    flush=True
)

print(
    "\nZIP:",
    ZIP_PATH,
    flush=True
)

print(
    "ZIP SHA256:",
    zip_sha,
    flush=True
)

print(
    "ZIP size:",
    f"{ZIP_PATH.stat().st_size/(1024**2):.2f} MB",
    flush=True
)

===== STEP 35 — ELLIPTIC EVIDENCE PACKAGING =====

[1/7] Verifying required evidence...
Run JSONs: 12/12
Raw epoch timing CSVs: 12/12
[1/7] PASS

[2/7] Verifying canonical Elliptic hashes...
elliptic_txs_features.csv fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0
elliptic_txs_classes.csv 93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493
elliptic_txs_edgelist.csv a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e
[2/7] PASS

[3/7] Verifying frozen BWGNN source...
Frozen commit: de0631f039bbd19c1890b483cc01f1007f596af7
[3/7] PASS

[4/7] Verifying frozen Elliptic configuration...
Winner trial: 7
Hidden: 128
Order: 2
Learning rate: 0.005
Weight decay: 0.0
Tuning best epoch: 79
Best validation AUPRC: 0.7142524678075058
[4/7] PASS

[5/7] Creating manifest records...
Split SHA256: 1e7963e9d09935fb33e0df74cfb786cab826d1b95ae10b9592dbc240b48ffdbc
Adapter SHA256: 7ed1ce573857a0f6dc40b77eb001179ace41f2dc9dd69f97da35c52c47da3145
[5/7] PASS

[6/7] Buil

## Step 36 — BWGNN Model-Level Completion Audit and Master Evidence Package

Step 35 completed and packaged the final Elliptic BWGNN experiment.

BWGNN has now been investigated across all six datasets in the controlled
benchmark scope.

### Dataset-level outcome

- YelpChi — unified benchmark complete
- Amazon — unified benchmark complete
- T-Finance — unified benchmark complete with native timing
- T-Social — full-graph BWGNN infeasible on the controlled Tesla T4
- FDCompCN — input-adapter eligible; unified benchmark complete
- Elliptic — input-adapter eligible; chronological unified benchmark complete

### Important evidence distinction

For YelpChi and Amazon:

- the original Step 15 and Step 19 performance results remain authoritative,
- Steps 24B, 25 and 26 provide supplemental computational-efficiency evidence,
- the timing reruns do not replace the original benchmark metrics.

For T-Social:

- the canonical full graph was preserved,
- sampling was not introduced,
- the graph was not reduced,
- the model architecture was not redesigned,
- the controlled T4 infeasibility result is retained rather than forcing an
  incomparable experiment.

For FDCompCN and Elliptic:

- only input representation adapters were used,
- BWGNN's core architecture was not changed.

Elliptic additionally preserves chronological ordering and retains
unknown-labelled nodes only for message passing.

### Purpose of this step

This step performs no training and no test evaluation.

It:

1. verifies the five dataset evidence ZIPs that cover all six datasets,
2. verifies their exact SHA256 identities,
3. checks the required evidence structure,
4. confirms that no model checkpoints are packaged,
5. creates one BWGNN dataset-status index,
6. preserves the five existing evidence ZIPs byte-for-byte,
7. builds one final model-level BWGNN evidence archive.

The resulting package is the closure record for the BWGNN stage of the
benchmark.

In [1]:
# ============================================================
# STEP 36 — BWGNN MODEL-LEVEL COMPLETION AUDIT
# + MASTER EVIDENCE PACKAGE
# ============================================================

from pathlib import Path

import hashlib
import json
import csv
import shutil
import zipfile
import datetime


ROOT = Path(
    "/kaggle/working"
)


# ============================================================
# 1. DATASET EVIDENCE PACKAGES
# ============================================================

PACKAGES = {

    "yelp_amazon":
        ROOT /
        "bwgnn_yelp_amazon_timing_evidence_20260909.zip",

    "tfinance":
        ROOT /
        "bwgnn_tfinance_evidence_20260909.zip",

    "tsocial":
        ROOT /
        "bwgnn_tsocial_compatibility_evidence_20260909.zip",

    "fdcompcn":
        ROOT /
        "bwgnn_fdcompcn_complete_evidence_20260910.zip",

    "elliptic":
        ROOT /
        "bwgnn_elliptic_complete_evidence_20260910.zip"
}


EXPECTED_HASHES = {

    "yelp_amazon":
        "c6f8f891a1cd05742a1338804caf33ef30b1ad3b49ed93daa3f154eab62491de",

    "tfinance":
        "3125fab6c0a3c199d7249dfb50311e35e5cadf9bb3d767523d5a693a0496bff8",

    "tsocial":
        "a50d1b4b4aa0bb6e21b2514c12a2d9d2a302b271aa3521c6cab39fca244d80ce",

    "fdcompcn":
        "82bbf91fec7e4166ef59fb0b065da1cb99432ef80cb5533394f26ae67a2126ac",

    "elliptic":
        "fe63086e0100d81e7191e46b79911d46713b90cf9156d66d4ea8b58a3b19aa26"
}


UPSTREAM_COMMIT = (
    "de0631f039bbd19c1890b483cc01f1007f596af7"
)


print(
    "===== STEP 36 — BWGNN MODEL-LEVEL CLOSURE =====",
    flush=True
)


# ============================================================
# HELPER
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
            f.read(
                8 * 1024 * 1024
            ),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


# ============================================================
# 2. VERIFY PACKAGE IDENTITIES
# ============================================================

print(
    "\n[1/7] Verifying evidence package identities...",
    flush=True
)


verified_hashes = {}


for key, path in PACKAGES.items():

    assert path.exists(), (
        f"Missing evidence package: {path}"
    )

    actual = sha256_file(
        path
    )

    expected = EXPECTED_HASHES[
        key
    ]

    print(
        path.name,
        actual,
        flush=True
    )

    assert actual == expected, (
        f"SHA256 mismatch for {path.name}"
    )

    verified_hashes[
        path.name
    ] = actual


print(
    "[1/7] PASS",
    flush=True
)


# ============================================================
# 3. VERIFY ZIP INTEGRITY + NO CHECKPOINTS
# ============================================================

print(
    "\n[2/7] Verifying ZIP integrity...",
    flush=True
)


for key, path in PACKAGES.items():

    with zipfile.ZipFile(
        path
    ) as z:

        bad = z.testzip()

        assert bad is None, (
            f"Corrupt ZIP member in {path.name}: {bad}"
        )

        names = z.namelist()

        assert not any(
            name.lower().endswith(
                ".pt"
            )
            for name in names
        ), (
            f"Checkpoint unexpectedly packaged in {path.name}"
        )

        print(
            path.name,
            "| files:",
            len(names),
            "| integrity: PASS",
            flush=True
        )


print(
    "[2/7] PASS",
    flush=True
)


# ============================================================
# 4. VERIFY YELPCHI + AMAZON EVIDENCE
# ============================================================

print(
    "\n[3/7] Verifying YelpChi + Amazon...",
    flush=True
)


with zipfile.ZipFile(
    PACKAGES["yelp_amazon"]
) as z:

    names = z.namelist()

    manifest = json.loads(
        z.read(
            "manifest.json"
        )
    )


    assert (
        manifest[
            "yelp_completed_runs"
        ]
        == 12
    )

    assert (
        manifest[
            "amazon_completed_runs"
        ]
        == 12
    )


    yelp_runs = sum(
        name.startswith(
            "yelp/results/runs/"
        )
        and name.endswith(
            ".json"
        )
        for name in names
    )

    amazon_runs = sum(
        name.startswith(
            "amazon/results/runs/"
        )
        and name.endswith(
            ".json"
        )
        for name in names
    )


    yelp_epoch = sum(
        name.startswith(
            "yelp/results/epoch_times/"
        )
        and name.endswith(
            "_epoch_times.csv"
        )
        for name in names
    )

    amazon_epoch = sum(
        name.startswith(
            "amazon/results/epoch_times/"
        )
        and name.endswith(
            "_epoch_times.csv"
        )
        for name in names
    )


    assert yelp_runs == 12
    assert amazon_runs == 12

    assert yelp_epoch == 12
    assert amazon_epoch == 12


print(
    "YelpChi supplemental timing runs: 12/12",
    flush=True
)

print(
    "Amazon supplemental timing runs: 12/12",
    flush=True
)

print(
    "Primary Step 15 / Step 19 results replaced: NO",
    flush=True
)

print(
    "[3/7] PASS",
    flush=True
)


# ============================================================
# 5. VERIFY T-FINANCE
# ============================================================

print(
    "\n[4/7] Verifying T-Finance...",
    flush=True
)


with zipfile.ZipFile(
    PACKAGES["tfinance"]
) as z:

    names = z.namelist()

    config = json.loads(
        z.read(
            "results/"
            "tfinance_frozen_config.json"
        )
    )


    assert (
        config[
            "upstream_commit"
        ]
        == UPSTREAM_COMMIT
    )


    run_count = sum(
        name.startswith(
            "results/runs/"
        )
        and name.endswith(
            ".json"
        )
        for name in names
    )

    epoch_count = sum(
        name.startswith(
            "results/epoch_times/"
        )
        and name.endswith(
            "_epoch_times.csv"
        )
        for name in names
    )


    assert run_count == 12
    assert epoch_count == 12


print(
    "T-Finance final runs: 12/12",
    flush=True
)

print(
    "T-Finance raw epoch timing: 12/12",
    flush=True
)

print(
    "[4/7] PASS",
    flush=True
)


# ============================================================
# 6. VERIFY T-SOCIAL
# ============================================================

print(
    "\n[5/7] Verifying T-Social compatibility evidence...",
    flush=True
)


with zipfile.ZipFile(
    PACKAGES["tsocial"]
) as z:

    names = z.namelist()

    decision = json.loads(
        z.read(
            "results/"
            "bwgnn_tsocial_final_compatibility.json"
        )
    )


    assert (
        decision[
            "final_classification"
        ]
        == "INFEASIBLE ON CONTROLLED TESLA T4"
    )

    assert (
        decision[
            "tuning_performed"
        ]
        is False
    )

    assert (
        decision[
            "final_12_runs_performed"
        ]
        is False
    )

    assert (
        decision[
            "sampling_used"
        ]
        is False
    )

    assert (
        decision[
            "graph_reduced"
        ]
        is False
    )

    assert (
        decision[
            "test_evaluated"
        ]
        is False
    )


    log_count = sum(
        name.startswith(
            "logs/"
        )
        and name.endswith(
            ".log"
        )
        for name in names
    )


    assert log_count == 5


print(
    "Classification: INFEASIBLE ON CONTROLLED TESLA T4",
    flush=True
)

print(
    "Sampling used: NO",
    flush=True
)

print(
    "Graph reduced: NO",
    flush=True
)

print(
    "Test evaluated: NO",
    flush=True
)

print(
    "[5/7] PASS",
    flush=True
)


# ============================================================
# 7. VERIFY FDCOMPCN + ELLIPTIC
# ============================================================

print(
    "\n[6/7] Verifying adapter-eligible datasets...",
    flush=True
)


# ------------------------------------------------------------
# FDCompCN
# ------------------------------------------------------------

with zipfile.ZipFile(
    PACKAGES["fdcompcn"]
) as z:

    names = z.namelist()

    fd_manifest = json.loads(
        z.read(
            "manifest/MANIFEST.json"
        )
    )


    assert (
        fd_manifest[
            "compatibility"
        ]
        == "INPUT ADAPTER ELIGIBLE"
    )

    assert (
        fd_manifest[
            "final_runs"
        ]
        == 12
    )


    fd_runs = sum(
        name.startswith(
            "final/runs/"
        )
        and name.endswith(
            ".json"
        )
        for name in names
    )

    fd_epoch = sum(
        name.startswith(
            "final/epoch_times/"
        )
        and name.endswith(
            "_epoch_times.csv"
        )
        for name in names
    )


    assert fd_runs == 12
    assert fd_epoch == 12


# ------------------------------------------------------------
# Elliptic
# ------------------------------------------------------------

with zipfile.ZipFile(
    PACKAGES["elliptic"]
) as z:

    names = z.namelist()

    ell_manifest = json.loads(
        z.read(
            "manifest/MANIFEST.json"
        )
    )


    assert (
        ell_manifest[
            "compatibility"
        ]
        == "INPUT ADAPTER ELIGIBLE"
    )

    assert (
        ell_manifest[
            "chronology_preserved"
        ]
        is True
    )

    assert (
        ell_manifest[
            "unknown_nodes_retained"
        ]
        is True
    )

    assert (
        ell_manifest[
            "unknown_labels_used_for_supervision"
        ]
        is False
    )

    assert (
        ell_manifest[
            "reverse_edges_added"
        ]
        is False
    )

    assert (
        ell_manifest[
            "final_runs"
        ]
        == 12
    )


    ell_runs = sum(
        name.startswith(
            "final/runs/"
        )
        and name.endswith(
            ".json"
        )
        for name in names
    )

    ell_epoch = sum(
        name.startswith(
            "final/epoch_times/"
        )
        and name.endswith(
            "_epoch_times.csv"
        )
        for name in names
    )


    assert ell_runs == 12
    assert ell_epoch == 12


print(
    "FDCompCN: INPUT ADAPTER ELIGIBLE | runs 12/12",
    flush=True
)

print(
    "Elliptic: INPUT ADAPTER ELIGIBLE | runs 12/12",
    flush=True
)

print(
    "Elliptic chronology preserved: YES",
    flush=True
)

print(
    "[6/7] PASS",
    flush=True
)


# ============================================================
# 8. BUILD MODEL-LEVEL STATUS INDEX
# ============================================================

records = [

    {
        "dataset":
            "YelpChi",

        "repository_support":
            "NATIVE",

        "compatibility":
            "ELIGIBLE",

        "outcome":
            "UNIFIED COMPLETE",

        "primary_performance_source":
            "Step 15",

        "timing_source":
            "Steps 24B/26 supplemental",

        "final_runs":
            12,

        "raw_epoch_timing_files":
            12,

        "chronological":
            False,

        "test_tuning":
            False,

        "note":
            (
                "Primary Step 15 performance unchanged "
                "by timing backfill."
            )
    },

    {
        "dataset":
            "Amazon",

        "repository_support":
            "NATIVE",

        "compatibility":
            "ELIGIBLE",

        "outcome":
            "UNIFIED COMPLETE",

        "primary_performance_source":
            "Step 19",

        "timing_source":
            "Steps 25/26 supplemental",

        "final_runs":
            12,

        "raw_epoch_timing_files":
            12,

        "chronological":
            False,

        "test_tuning":
            False,

        "note":
            (
                "Primary Step 19 performance unchanged "
                "by timing backfill."
            )
    },

    {
        "dataset":
            "T-Finance",

        "repository_support":
            "NATIVE",

        "compatibility":
            "ELIGIBLE",

        "outcome":
            "UNIFIED COMPLETE",

        "primary_performance_source":
            "Step 22B",

        "timing_source":
            "Step 22B native timed final runs",

        "final_runs":
            12,

        "raw_epoch_timing_files":
            12,

        "chronological":
            False,

        "test_tuning":
            False,

        "note":
            "Full timed unified grid complete."
    },

    {
        "dataset":
            "T-Social",

        "repository_support":
            "NATIVE",

        "compatibility":
            "CONTROLLED-HARDWARE INFEASIBLE",

        "outcome":
            "T4 INFEASIBLE",

        "primary_performance_source":
            "UNAVAILABLE",

        "timing_source":
            "Compatibility attempts only",

        "final_runs":
            0,

        "raw_epoch_timing_files":
            0,

        "chronological":
            False,

        "test_tuning":
            False,

        "note":
            (
                "Full-graph BWGNN retained; no sampling "
                "or graph reduction forced."
            )
    },

    {
        "dataset":
            "FDCompCN",

        "repository_support":
            "NO NATIVE CONFIG",

        "compatibility":
            "INPUT ADAPTER ELIGIBLE",

        "outcome":
            "UNIFIED COMPLETE",

        "primary_performance_source":
            "Step 30",

        "timing_source":
            "Step 30",

        "final_runs":
            12,

        "raw_epoch_timing_files":
            12,

        "chronological":
            False,

        "test_tuning":
            False,

        "note":
            (
                "Input adapter preserves nodes, edges, "
                "features and labels."
            )
    },

    {
        "dataset":
            "Elliptic",

        "repository_support":
            "NO NATIVE CONFIG",

        "compatibility":
            "INPUT ADAPTER ELIGIBLE",

        "outcome":
            "UNIFIED COMPLETE",

        "primary_performance_source":
            "Step 34",

        "timing_source":
            "Step 34",

        "final_runs":
            12,

        "raw_epoch_timing_files":
            12,

        "chronological":
            True,

        "test_tuning":
            False,

        "note":
            (
                "Chronology preserved; unknown nodes retained "
                "but never supervised."
            )
    }
]


assert len(
    records
) == 6


primary_final_runs = sum(
    row[
        "final_runs"
    ]
    for row in records
)


raw_epoch_files = sum(
    row[
        "raw_epoch_timing_files"
    ]
    for row in records
)


assert primary_final_runs == 60
assert raw_epoch_files == 60


# ============================================================
# 9. CREATE MASTER EVIDENCE DIRECTORY
# ============================================================

OUT_DIR = (
    ROOT /
    "bwgnn_complete_model_evidence"
)


if OUT_DIR.exists():

    shutil.rmtree(
        OUT_DIR
    )


(
    OUT_DIR /
    "dataset_packages"
).mkdir(
    parents=True,
    exist_ok=True
)


# Preserve existing packages byte-for-byte.

for key, path in PACKAGES.items():

    shutil.copy2(
        path,
        OUT_DIR /
        "dataset_packages" /
        path.name
    )


# ============================================================
# 10. SAVE STATUS CSV + JSON
# ============================================================

STATUS_CSV = (
    OUT_DIR /
    "bwgnn_dataset_status.csv"
)


with STATUS_CSV.open(
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            records[0].keys()
        )
    )

    writer.writeheader()

    writer.writerows(
        records
    )


STATUS_JSON = (
    OUT_DIR /
    "bwgnn_dataset_status.json"
)


STATUS_JSON.write_text(
    json.dumps(
        records,
        indent=2
    )
)


# ============================================================
# 11. MASTER MANIFEST
# ============================================================

manifest = {

    "created_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),

    "model":
        "BWGNN",

    "repository":
        (
            "https://github.com/"
            "squareRoot3/"
            "Rethinking-Anomaly-Detection.git"
        ),

    "repository_commit":
        UPSTREAM_COMMIT,

    "controlled_hardware":
        "Tesla T4, GPU 0",

    "benchmark_scope": [
        "YelpChi",
        "Amazon",
        "T-Finance",
        "T-Social",
        "FDCompCN",
        "Elliptic"
    ],

    "datasets_audited":
        6,

    "unified_performance_complete_datasets":
        5,

    "controlled_hardware_infeasible_datasets": [
        "T-Social"
    ],

    "input_adapter_datasets": [
        "FDCompCN",
        "Elliptic"
    ],

    "primary_final_runs":
        primary_final_runs,

    "raw_epoch_timing_files_for_performance_complete_datasets":
        raw_epoch_files,

    "supplemental_timing_reruns":
        24,

    "split_seed":
        2,

    "training_seeds": [
        2,
        42,
        72
    ],

    "training_ratios": [
        "TR40",
        "TR30",
        "TR20",
        "TR10"
    ],

    "test_set_tuning":
        False,

    "checkpoints_included":
        False,

    "dataset_package_sha256":
        verified_hashes,

    "dataset_status_file":
        "bwgnn_dataset_status.json",

    "notes": {

        "YelpChi":
            (
                "Primary Step 15 performance remains "
                "authoritative; Step 24B/26 timing reruns "
                "are supplemental only."
            ),

        "Amazon":
            (
                "Primary Step 19 performance remains "
                "authoritative; Step 25/26 timing reruns "
                "are supplemental only."
            ),

        "T-Social":
            (
                "Documented infeasibility is retained as "
                "a valid compatibility result rather than "
                "altering the full-graph model or controlled "
                "hardware."
            )
    }
}


MANIFEST_PATH = (
    OUT_DIR /
    "MANIFEST.json"
)


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


# ============================================================
# 12. BUILD MASTER ZIP
# ============================================================

print(
    "\n[7/7] Building BWGNN master evidence ZIP...",
    flush=True
)


MASTER_ZIP = (
    ROOT /
    "bwgnn_complete_model_evidence_20260910.zip"
)


if MASTER_ZIP.exists():

    MASTER_ZIP.unlink()


# Dataset packages are already ZIP-compressed.
# ZIP_STORED preserves them efficiently and byte-for-byte.

with zipfile.ZipFile(
    MASTER_ZIP,
    "w",
    compression=zipfile.ZIP_STORED
) as z:

    for path in sorted(
        OUT_DIR.rglob(
            "*"
        )
    ):

        if path.is_file():

            z.write(
                path,
                arcname=str(
                    path.relative_to(
                        OUT_DIR
                    )
                )
            )


# ============================================================
# 13. VERIFY MASTER PACKAGE
# ============================================================

with zipfile.ZipFile(
    MASTER_ZIP
) as z:

    assert z.testzip() is None

    names = set(
        z.namelist()
    )


    assert (
        "MANIFEST.json"
        in names
    )

    assert (
        "bwgnn_dataset_status.csv"
        in names
    )

    assert (
        "bwgnn_dataset_status.json"
        in names
    )


    # Verify all nested evidence packages remain exact.

    for key, path in PACKAGES.items():

        archive_name = (
            "dataset_packages/"
            + path.name
        )

        assert (
            archive_name
            in names
        )

        embedded_hash = hashlib.sha256(
            z.read(
                archive_name
            )
        ).hexdigest()

        assert (
            embedded_hash
            == EXPECTED_HASHES[
                key
            ]
        )


master_sha = sha256_file(
    MASTER_ZIP
)


print(
    "[7/7] PASS",
    flush=True
)


# ============================================================
# FINAL GATE
# ============================================================

print(
    "\n===== STEP 36 GATE =====",
    flush=True
)

print(
    "PASS — BWGNN model-level evidence closure completed.",
    flush=True
)

print(
    "Datasets audited: 6/6",
    flush=True
)

print(
    "Unified performance-complete datasets: 5/6",
    flush=True
)

print(
    "Primary final unified runs: 60",
    flush=True
)

print(
    "Raw epoch timing files: 60",
    flush=True
)

print(
    "YelpChi/Amazon supplemental timing reruns: 24",
    flush=True
)

print(
    "T-Social controlled T4 infeasibility retained: YES",
    flush=True
)

print(
    "Sampling forced for T-Social: NO",
    flush=True
)

print(
    "Graph reduction forced for T-Social: NO",
    flush=True
)

print(
    "FDCompCN input adapter: VERIFIED",
    flush=True
)

print(
    "Elliptic input adapter: VERIFIED",
    flush=True
)

print(
    "Elliptic chronology preserved: YES",
    flush=True
)

print(
    "Test-set tuning: NO",
    flush=True
)

print(
    "Checkpoints included: NO",
    flush=True
)

print(
    "\nMASTER ZIP:",
    MASTER_ZIP,
    flush=True
)

print(
    "MASTER ZIP SHA256:",
    master_sha,
    flush=True
)

print(
    "MASTER ZIP size:",
    f"{MASTER_ZIP.stat().st_size/(1024**2):.2f} MB",
    flush=True
)

===== STEP 36 — BWGNN MODEL-LEVEL CLOSURE =====

[1/7] Verifying evidence package identities...


AssertionError: Missing evidence package: /kaggle/working/bwgnn_yelp_amazon_timing_evidence_20260909.zip

## Step 36A — Resolve BWGNN Evidence Package Locations

The previous Step 36 attempt stopped because earlier evidence archives were
not present in `/kaggle/working`.

This is a storage-path issue only. No experiment is rerun.

This gate searches both:

- `/kaggle/working`
- `/kaggle/input`

and identifies each evidence archive by its frozen SHA256 rather than relying
on its exact filename or directory.

This also protects against Kaggle renaming files or placing them inside a
dataset subdirectory.

In [5]:
# ============================================================
# STEP 36A — RESOLVE BWGNN EVIDENCE PACKAGE LOCATIONS
# ============================================================

from pathlib import Path
import hashlib


EXPECTED = {

    "yelp_amazon": {
        "name":
            "bwgnn_yelp_amazon_timing_evidence_20260909.zip",

        "sha256":
            "c6f8f891a1cd05742a1338804caf33ef30b1ad3b49ed93daa3f154eab62491de"
    },

    "tfinance": {
        "name":
            "bwgnn_tfinance_evidence_20260909.zip",

        "sha256":
            "3125fab6c0a3c199d7249dfb50311e35e5cadf9bb3d767523d5a693a0496bff8"
    },

    "tsocial": {
        "name":
            "bwgnn_tsocial_compatibility_evidence_20260909.zip",

        "sha256":
            "a50d1b4b4aa0bb6e21b2514c12a2d9d2a302b271aa3521c6cab39fca244d80ce"
    },

    "fdcompcn": {
        "name":
            "bwgnn_fdcompcn_complete_evidence_20260910.zip",

        "sha256":
            "82bbf91fec7e4166ef59fb0b065da1cb99432ef80cb5533394f26ae67a2126ac"
    },

    "elliptic": {
        "name":
            "bwgnn_elliptic_complete_evidence_20260910.zip",

        "sha256":
            "fe63086e0100d81e7191e46b79911d46713b90cf9156d66d4ea8b58a3b19aa26"
    }
}


SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):

            h.update(block)

    return h.hexdigest()


print(
    "===== STEP 36A — RESOLVE EVIDENCE LOCATIONS ====="
)


# ------------------------------------------------------------
# Collect ZIP candidates
# ------------------------------------------------------------

zip_candidates = []


for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    zip_candidates.extend(
        p
        for p in root.rglob("*.zip")
        if p.is_file()
    )


print(
    "ZIP candidates found:",
    len(zip_candidates)
)


# ------------------------------------------------------------
# First try filename matching, then verify SHA.
# If Kaggle renamed the archive, fall back to SHA scan.
# ------------------------------------------------------------

PACKAGES = {}

used_paths = set()


for key, spec in EXPECTED.items():

    expected_name = spec["name"]
    expected_sha = spec["sha256"]

    stem = Path(expected_name).stem


    # ----------------------------------------
    # Fast candidate search by filename stem
    # Handles names such as:
    # file.zip
    # file (1).zip
    # ----------------------------------------

    likely = [

        p for p in zip_candidates

        if (
            p.name == expected_name
            or
            p.stem == stem
            or
            p.stem.startswith(stem + " (")
            or
            p.stem.startswith(stem + "_")
        )

    ]


    match = None


    for p in likely:

        actual_sha = sha256_file(p)

        if actual_sha == expected_sha:

            match = p
            break


    # ----------------------------------------
    # Hash fallback
    # ----------------------------------------

    if match is None:

        for p in zip_candidates:

            if p in used_paths:
                continue

            actual_sha = sha256_file(p)

            if actual_sha == expected_sha:

                match = p
                break


    if match is not None:

        PACKAGES[key] = match

        used_paths.add(match)

        print(
            f"{key:12s} PASS"
        )

        print(
            "  PATH:",
            match
        )

        print(
            "  SHA256:",
            expected_sha
        )

    else:

        print(
            f"{key:12s} MISSING"
        )

        print(
            "  Expected archive:",
            expected_name
        )

        print(
            "  Expected SHA256:",
            expected_sha
        )


# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

missing = [

    key
    for key in EXPECTED
    if key not in PACKAGES

]


print(
    "\n===== STEP 36A GATE ====="
)


if missing:

    print(
        "FAIL — required completed evidence archives are "
        "not available in the current Kaggle runtime."
    )

    print(
        "Missing:",
        ", ".join(missing)
    )

    print(
        "\nNo training should be rerun."
    )

    print(
        "Upload/add the missing ZIP archive(s) as Kaggle "
        "notebook inputs, then rerun Step 36A."
    )

else:

    print(
        "PASS — all five evidence packages resolved by SHA256."
    )

    print(
        "Evidence packages represented: 5/5"
    )

    print(
        "Datasets represented: 6/6"
    )

    print(
        "Training rerun required: NO"
    )

    print(
        "Ready for Step 36 model-level closure: YES"
    )

===== STEP 36A — RESOLVE EVIDENCE LOCATIONS =====
ZIP candidates found: 0
yelp_amazon  MISSING
  Expected archive: bwgnn_yelp_amazon_timing_evidence_20260909.zip
  Expected SHA256: c6f8f891a1cd05742a1338804caf33ef30b1ad3b49ed93daa3f154eab62491de
tfinance     MISSING
  Expected archive: bwgnn_tfinance_evidence_20260909.zip
  Expected SHA256: 3125fab6c0a3c199d7249dfb50311e35e5cadf9bb3d767523d5a693a0496bff8
tsocial      MISSING
  Expected archive: bwgnn_tsocial_compatibility_evidence_20260909.zip
  Expected SHA256: a50d1b4b4aa0bb6e21b2514c12a2d9d2a302b271aa3521c6cab39fca244d80ce
fdcompcn     MISSING
  Expected archive: bwgnn_fdcompcn_complete_evidence_20260910.zip
  Expected SHA256: 82bbf91fec7e4166ef59fb0b065da1cb99432ef80cb5533394f26ae67a2126ac
elliptic     MISSING
  Expected archive: bwgnn_elliptic_complete_evidence_20260910.zip
  Expected SHA256: fe63086e0100d81e7191e46b79911d46713b90cf9156d66d4ea8b58a3b19aa26

===== STEP 36A GATE =====
FAIL — required completed evidence archives are 

## Step 36B — Restore Completed BWGNN Evidence Packages

The current Kaggle runtime does not contain the previously completed evidence
archives.

No model training or evaluation is repeated.

A verified restore bundle containing all five completed evidence packages is
mounted as a Kaggle input.

This step:

1. identifies the restore bundle by SHA256,
2. verifies the outer archive,
3. extracts the five original evidence ZIPs to `/kaggle/working`,
4. verifies every extracted package against its frozen SHA256,
5. restores the evidence required for the Step 36 model-level closure.

The inner evidence archives must remain byte-for-byte identical to the
previously completed packages.

In [6]:
# ============================================================
# STEP 36B — RESTORE BWGNN COMPLETED EVIDENCE PACKAGES
# ============================================================

from pathlib import Path
import hashlib
import zipfile
import shutil


EXPECTED_BUNDLE_SHA = (
    "dbaa81cb28257a78bbca658e2707e882d074b3eadf953b6a318a076475513bed"
)


EXPECTED_PACKAGES = {

    "bwgnn_yelp_amazon_timing_evidence_20260909.zip":
        "c6f8f891a1cd05742a1338804caf33ef30b1ad3b49ed93daa3f154eab62491de",

    "bwgnn_tfinance_evidence_20260909.zip":
        "3125fab6c0a3c199d7249dfb50311e35e5cadf9bb3d767523d5a693a0496bff8",

    "bwgnn_tsocial_compatibility_evidence_20260909.zip":
        "a50d1b4b4aa0bb6e21b2514c12a2d9d2a302b271aa3521c6cab39fca244d80ce",

    "bwgnn_fdcompcn_complete_evidence_20260910.zip":
        "82bbf91fec7e4166ef59fb0b065da1cb99432ef80cb5533394f26ae67a2126ac",

    "bwgnn_elliptic_complete_evidence_20260910.zip":
        "fe63086e0100d81e7191e46b79911d46713b90cf9156d66d4ea8b58a3b19aa26"
}


SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]


WORKING = Path(
    "/kaggle/working"
)


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


print(
    "===== STEP 36B — RESTORE BWGNN EVIDENCE =====",
    flush=True
)


# ============================================================
# 1. LOCATE RESTORE BUNDLE
# ============================================================

print(
    "\n[1/5] Searching for restore bundle...",
    flush=True
)


candidates = []


for root in SEARCH_ROOTS:

    if root.exists():

        candidates.extend(
            p
            for p in root.rglob("*.zip")
            if p.is_file()
        )


print(
    "ZIP candidates:",
    len(candidates),
    flush=True
)


bundle = None


# First prefer exact filename.

for path in candidates:

    if (
        path.name
        == "bwgnn_evidence_restore_bundle_20260910.zip"
    ):

        actual = sha256_file(
            path
        )

        if actual == EXPECTED_BUNDLE_SHA:

            bundle = path
            break


# Hash fallback in case Kaggle changes the filename.

if bundle is None:

    for path in candidates:

        try:

            actual = sha256_file(
                path
            )

        except Exception:

            continue

        if actual == EXPECTED_BUNDLE_SHA:

            bundle = path
            break


assert bundle is not None, (
    "Verified BWGNN restore bundle not found. "
    "Add bwgnn_evidence_restore_bundle_20260910.zip "
    "to the current Kaggle notebook inputs."
)


print(
    "Bundle:",
    bundle,
    flush=True
)

print(
    "Bundle SHA256:",
    sha256_file(bundle),
    flush=True
)

print(
    "[1/5] PASS",
    flush=True
)


# ============================================================
# 2. VERIFY BUNDLE STRUCTURE
# ============================================================

print(
    "\n[2/5] Verifying restore bundle...",
    flush=True
)


with zipfile.ZipFile(
    bundle
) as z:

    assert z.testzip() is None

    names = set(
        z.namelist()
    )


    assert (
        "MANIFEST.json"
        in names
    )


    for filename in EXPECTED_PACKAGES:

        assert filename in names, (
            f"Missing package inside bundle: {filename}"
        )


print(
    "Required packages inside bundle: 5/5",
    flush=True
)

print(
    "[2/5] PASS",
    flush=True
)


# ============================================================
# 3. VERIFY INNER BYTES BEFORE EXTRACTION
# ============================================================

print(
    "\n[3/5] Verifying inner package SHA256 values...",
    flush=True
)


with zipfile.ZipFile(
    bundle
) as z:

    for filename, expected_sha in EXPECTED_PACKAGES.items():

        data = z.read(
            filename
        )

        actual_sha = hashlib.sha256(
            data
        ).hexdigest()


        print(
            filename,
            actual_sha,
            flush=True
        )


        assert (
            actual_sha == expected_sha
        ), (
            f"Inner ZIP SHA mismatch: {filename}"
        )


print(
    "[3/5] PASS",
    flush=True
)


# ============================================================
# 4. RESTORE TO /KAGGLE/WORKING
# ============================================================

print(
    "\n[4/5] Restoring evidence packages...",
    flush=True
)


with zipfile.ZipFile(
    bundle
) as z:

    for filename in EXPECTED_PACKAGES:

        destination = (
            WORKING /
            filename
        )


        with z.open(
            filename
        ) as src, destination.open(
            "wb"
        ) as dst:

            shutil.copyfileobj(
                src,
                dst
            )


        print(
            "Restored:",
            destination,
            flush=True
        )


print(
    "[4/5] PASS",
    flush=True
)


# ============================================================
# 5. VERIFY RESTORED PACKAGES
# ============================================================

print(
    "\n[5/5] Verifying restored evidence...",
    flush=True
)


for filename, expected_sha in EXPECTED_PACKAGES.items():

    path = (
        WORKING /
        filename
    )


    assert path.exists()


    actual_sha = sha256_file(
        path
    )


    assert (
        actual_sha == expected_sha
    ), (
        f"Restored SHA mismatch: {filename}"
    )


    with zipfile.ZipFile(
        path
    ) as z:

        assert z.testzip() is None


    print(
        filename,
        "PASS",
        flush=True
    )


print(
    "[5/5] PASS",
    flush=True
)


# ============================================================
# GATE
# ============================================================

print(
    "\n===== STEP 36B GATE =====",
    flush=True
)

print(
    "PASS — all completed BWGNN evidence restored.",
    flush=True
)

print(
    "Restore bundle SHA256: "
    + EXPECTED_BUNDLE_SHA,
    flush=True
)

print(
    "Evidence packages restored: 5/5",
    flush=True
)

print(
    "Datasets represented: 6/6",
    flush=True
)

print(
    "Package SHA256 verification: PASS",
    flush=True
)

print(
    "ZIP integrity verification: PASS",
    flush=True
)

print(
    "Training rerun performed: NO",
    flush=True
)

print(
    "Test evaluation performed: NO",
    flush=True
)

print(
    "Ready for Step 36A recheck: YES",
    flush=True
)

===== STEP 36B — RESTORE BWGNN EVIDENCE =====

[1/5] Searching for restore bundle...
ZIP candidates: 0


AssertionError: Verified BWGNN restore bundle not found. Add bwgnn_evidence_restore_bundle_20260910.zip to the current Kaggle notebook inputs.

In [7]:
from pathlib import Path

print("===== KAGGLE ZIP CHECK =====")

zips = []

for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]:
    if root.exists():
        zips.extend(root.rglob("*.zip"))

for p in zips:
    print(p)

print("ZIP count:", len(zips))

===== KAGGLE ZIP CHECK =====
ZIP count: 0
